# V31 — Periodic JEPA/KAN 

**Derived from `Periodic_JEPA_AlFe_Ni_v30_smooth_curriculum.ipynb`.** The central architecture is preserved: periodic graph representation, GNN/KAN structural encoder, shared two-field Fourier latent, online/EMA electronic branch, field decoder and cohesive-energy readout. V31 changes the optimization/governance protocol in response to the V30 run diagnostics.

## V31 changes
1. Keeps the compact nine-stage scientific curriculum: `D_latent → E_density → E_density_potential → E_full → F_periodic_ramp → P_cohesive_refinement → G_cohesive_calibration → G_adapter_warmup → G_joint_finetune`.
2. Adds separate `SMOKE/QUICK` and `QUICK_TRAIN` modes. Smoke modes validate integration; `QUICK_TRAIN` is the minimum mode intended to validate learning dynamics.
3. Replaces time-only D spectral release by **competence-driven, field-specific low→middle→high release**. Density and potential unlock independently from validation alignment.
4. Keeps the EMA target fixed at the trusted AE target until the low-band D gate is satisfied, then resumes the normal EMA trajectory.
5. Makes readiness **fail-closed** outside smoke modes. A stage cannot hand off while its stage-specific physical/latent gate is failing.
6. Replaces the macro-phase LR clock with stage-local warm-up/cosine schedules while preserving compatible AdamW moments across stages.
7. Gives newly activated `G_joint_finetune` parameter groups their own release-local LR ramp/decay so late modules are not activated after the global LR has already collapsed.
8. Adds conflict-aware gradient-norm balancing from periodically measured task gradients on shared parameters. The balancing state is logged and checkpointed; it does not alter targets.
9. Routes cohesive-energy gradients to the energy head first in `E_full`; shared-latent energy gradients are released only after the configured competence fraction.
10. Strengthens the potential-amplitude objective and adds explicit potential RMS-ratio gates because V30 learned potential shape while severely underestimating amplitude.
11. Enables the independent Al/Fe retention distillation stream by default during Ni adaptation; a configured retention weight can no longer silently multiply a zero loss.
12. Uses worst-element-aware Al/Fe validation scoring instead of allowing one element to mask failure of the other.
13. Makes `F_periodic_ramp`, `P_cohesive_refinement`, and Ni transfer conditional on the required upstream source-domain gates.
14. Persists spectral competence, gradient-balance state, stage competence diagnostics and transition audits in checkpoints/run artifacts.
15. Uses a new V31 checkpoint/training contract. **Do not resume V30 or earlier checkpoints into V31.**

The architecture itself is intentionally not enlarged in this revision. V31 first tests whether better gating, optimizer clocks, gradient routing/balancing, retention, and amplitude control resolve the V30 failure modes before changing the structural predictor.


In [1]:
# =============================================================================
# STEP 01 — Optional dependencies
# Purpose: Installs only packages that are usually unavailable in Colab.
# Inputs: INSTALL_DEPENDENCIES flag and access to the Python environment.
# Outputs: Environment with PyG, NetCDF4, PyArrow, SciPy, pandas, and Matplotlib.
# Reproducibility: run after all preceding cells; do not change the seed during the run.
# =============================================================================
INSTALL_DEPENDENCIES = False

if INSTALL_DEPENDENCIES:
    import subprocess, sys
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "torch-geometric>=2.5", "netCDF4>=1.6", "pyarrow>=15",
        "scipy>=1.11", "pandas>=2.0", "matplotlib>=3.8"
    ])


In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ.setdefault('PERIODIC_JEPA_RUN_MODE', 'DEVELOPMENT')
os.environ.setdefault('PERIODIC_DFT_INDEX_PATH', '../public_core/index_full.parquet')
os.environ.setdefault('PERIODIC_DFT_DATA_ROOT', '../public_core/data')
os.environ.setdefault('PERIODIC_JEPA_RUN_TAG', 'run04_gate_driven')

KJ_MOL_TO_EV_PER_ATOM = 1.0 / 96.48533212331002
COHESIVE_ANCHOR_KJ_MOL = {
    'Al': 329.70,
    'Fe': 415.47,
    'Ni': 430.10,
}
COHESIVE_ANCHOR_EV_PER_ATOM = {
    element: value * KJ_MOL_TO_EV_PER_ATOM
    for element, value in COHESIVE_ANCHOR_KJ_MOL.items()
}
COHESIVE_CALIBRATION_PROTOCOL = {
    'target_name': 'experimentally_anchored_cohesive_energy',
    'thermochemical_quantity': 'standard atomization enthalpy at 298.15 K',
    'source': 'NIST Chemistry WebBook SRD 69 / NIST-JANAF (Chase, 1998)',
    'source_urls': {
        'Al': 'https://webbook.nist.gov/cgi/cbook.cgi?ID=C7429905&Mask=1',
        'Fe': 'https://webbook.nist.gov/cgi/cbook.cgi?ID=C7439896&Mask=1',
        'Ni': 'https://webbook.nist.gov/cgi/cbook.cgi?ID=C7440020&Mask=1',
    },
    'temperature_k': 298.15,
    'lower_tail_fraction': 0.02,
    'minimum_tail_records': 3,
    'minimum_family_records': 3,
    'reference_pools': {'Al': 'pretrain_train', 'Fe': 'pretrain_train', 'Ni': 'ni_adapt_train'},
    'manual_calc_id_selection': False,
}


In [3]:
# =============================================================================
# STEP 02 — Imports and global contract
# Purpose: Imports libraries, reads the selected profile, and defines versions, aliases, and plot compatibility.
# Inputs: Environment variables defined in the user configuration.
# Outputs: Architecture, schema, and source constants, and the compatible_boxplot helper.
# Reproducibility: run after all preceding cells; do not change the seed during the run.
# =============================================================================
from __future__ import annotations

import copy
import contextlib
from IPython.display import display
import gc
import json
import math
import os
import random
import re
import sys
import tempfile
import time
import warnings
import xml.etree.ElementTree as ET
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from functools import lru_cache
from pathlib import Path


from typing import Any, Iterable, Mapping, Sequence, Optional

import matplotlib.pyplot as plt
import netCDF4
import numpy as np
import pandas as pd
import scipy
from scipy.signal import resample

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW

import torch_geometric
from torch_geometric.data import Batch, Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import to_dense_batch


def compatible_boxplot(
    axis: Any, values: Sequence[Any], tick_labels: Sequence[str], **kwargs: Any
) -> dict[str, Any]:
    """Matplotlib 3.8--3.11 compatibility for the labels→tick_labels rename."""
    try:
        return axis.boxplot(values, tick_labels=tick_labels, **kwargs)
    except TypeError as error:
        # Matplotlib <3.9 does not yet recognize tick_labels. Other TypeErrors are not suppressed.
        if "tick_labels" not in str(error):
            raise
        return axis.boxplot(values, labels=tick_labels, **kwargs)

RAW_RUN_MODE = os.environ.get("PERIODIC_JEPA_RUN_MODE", os.environ.get("PERIODIC_JEPA_RUN_MODE","QUICK_TRAIN"))
RUN_MODE_OPTIONS = ("SMOKE", "QUICK", "QUICK_TRAIN", "DEVELOPMENT", "STANDARD", "FULL")
LEGACY_LOWERCASE_RUN_MODE_ALIASES = {
    "tiny": "SMOKE",
    "smoke": "SMOKE",
    "quick": "QUICK",
    "quick_train": "QUICK_TRAIN",
    "quick-train": "QUICK_TRAIN",
    "full": "FULL",
}


def normalize_run_mode(value: str) -> str:
    text = str(value).strip()
    return LEGACY_LOWERCASE_RUN_MODE_ALIASES.get(text, text.upper())


RUN_MODE = normalize_run_mode(RAW_RUN_MODE)
BACKBONE_OPTIONS = ("transformer", "pyg_gnn_mlp", "pyg_gnn_kan")
LEGACY_BACKBONE_OPTIONS = ("direct_baseline", "gnn_mlp", "gnn_kan")
BACKBONE_ALIASES = {
    "direct_baseline": "transformer",
    "gnn_mlp": "pyg_gnn_mlp",
    "gnn_kan": "pyg_gnn_kan",
    "pyg_gnn": "pyg_gnn_mlp",
}
KAN_BASIS_OPTIONS = ("chebyshev", "legendre", "hermite")
DECODER_OPTIONS = ("periodic_fourier_residual",)
PRIMARY_OUTPUT_OPTIONS = ("fields_plus_cohesive",)

ARCHITECTURE_VERSION = "periodic-field-jepa-v31-shared-anchor-retention"
TRAINING_PROTOCOL_VERSION = "periodic-field-jepa-training-v31-r1-gate-driven-nine-stage"
CURRICULUM_VERSION = "full-pool-batch-curriculum-v31-r1-competence-gated"
CHECKPOINT_SCHEMA_VERSION = "periodic-field-jepa-checkpoint-v31-r1-gate-driven-atomic"
DATA_SCHEMA_VERSION = "siesta-periodic-fields-v1"
SOLVER_VERSION = "precomputed-siesta-5.4.2-pbe-tzp-nonspin"
GRAPH_BUILDER_VERSION = "fractional-cell-neighborhood-v1"
PREPROCESSING_VERSION = "periodic-fourier-canonical-v2-spectral-rms"
SOURCE_REVISION = "2026-10-05-periodic-field-jepa-v31-r1-gate-driven"
if RUN_MODE not in RUN_MODE_OPTIONS:
    raise ValueError(
        f"PERIODIC_JEPA_RUN_MODE={RAW_RUN_MODE!r} normalized to {RUN_MODE!r}; "
        f"canonical options={RUN_MODE_OPTIONS}"
    )


print(json.dumps({"run_mode": RUN_MODE, "source_revision": SOURCE_REVISION}, indent=2))


{
  "run_mode": "QUICK_TRAIN",
  "source_revision": "2026-10-05-periodic-field-jepa-v31-r1-gate-driven"
}


/home/rafael/Documentos/Pos_doc-BIO-IA/pykan-env/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [4]:
import os
from dataclasses import dataclass
from pathlib import Path

# =============================================================================
# STEP 04 — Profiles and scientific configuration
# Purpose: Creates CFG, validates options, and fixes the training, model, and evaluation contract.
# Inputs: Profile, backbone, device, paths, and hyperparameters.
# Outputs: CFG and latent gate tables used throughout the notebook.
# Reproducibility: run after all preceding cells; do not change the seed during the run.
# =============================================================================
@dataclass(frozen=True)
class RunPreset:
    audit_records: int
    normalization_records: int
    ae_epochs: int
    jepa_epochs: int
    ni_epochs: int
    batch_size: int
    num_workers: int


PRESETS = {
    # SMOKE/QUICK are integration checks only; they intentionally do not certify learning quality.
    "SMOKE": RunPreset(12, 24, 2, 2, 3, 2, 0),
    "QUICK": RunPreset(12, 24, 2, 2, 3, 2, 0),
    # QUICK_TRAIN is the minimum preset intended to evaluate optimization dynamics.
    "QUICK_TRAIN": RunPreset(48, 128, 8, 20, 12, 4, 0),
    "DEVELOPMENT": RunPreset(96, 256, 25, 50, 40, 4, 0),
    "STANDARD": RunPreset(300, 800, 40, 80, 60, 6, 0),
    "FULL": RunPreset(600, 1500, 80, 160, 100, 8, 0),
}


@dataclass
class Config:
    data_root: Path = Path(os.getenv("PERIODIC_DFT_DATA_ROOT", ".")).expanduser()
    index_path: Path | None = (
        Path(os.environ["PERIODIC_DFT_INDEX_PATH"]).expanduser()
        if os.environ.get("PERIODIC_DFT_INDEX_PATH") else None
    )
    output_root: Path = Path(os.getenv("PERIODIC_JEPA_OUTPUT", "./outputs"))
    run_mode: str = RUN_MODE
    device: str = os.getenv("PERIODIC_JEPA_DEVICE", "auto")
    seed: int = int(os.getenv('PERIODIC_JEPA_SEED', '42'))
    # V31: core architecture preserved; curriculum and optimization become competence driven.
    data_curriculum_mode: str = os.getenv('PERIODIC_JEPA_DATA_CURRICULUM_MODE', 'physical').strip().lower()
    objective_ramp_fraction: float = 0.70
    d_crossfade_start: float = 0.65
    d_crossfade_end: float = 0.90
    retention_ramp_fraction: float = 0.50
    readiness_enabled: bool = True
    readiness_patience: int = 2
    readiness_min_fraction: float = 0.40
    monitor_every_updates: int = 1
    monitor_jepa_weight: float = 0.30
    monitor_latent_cosine_weight: float = 0.25
    monitor_rho_weight: float = 1.00
    monitor_potential_weight: float = 1.00
    monitor_cohesive_weight: float = 0.20
    joint_predictor_start: float = 0.15
    joint_predictor_full: float = 0.35
    joint_decoder_start: float = 0.35
    joint_decoder_full: float = 0.60
    joint_structural_start: float = 0.65
    joint_structural_full: float = 0.85
    development_gradient_accumulation: int = 4
    optimizer_update_budget_scale: float = float(os.getenv('PERIODIC_JEPA_UPDATE_BUDGET_SCALE', '1.0'))
    gradient_conflict_audit: bool = True
    gradient_conflict_every_updates: int = 250
    gradient_conflict_max_parameters: int = 200_000
    # V31 competence-gated optimization controls.
    gradient_balance_enabled: bool = True
    gradient_balance_every_updates: int = 10
    gradient_balance_ema: float = 0.90
    gradient_balance_min_factor: float = 0.25
    gradient_balance_max_factor: float = 4.00
    gradient_conflict_shrink: float = 0.65
    gradient_conflict_cosine_threshold: float = -0.20
    competence_band_ramp_fraction: float = 0.15
    d_low_band_cosine_min: float = 0.70
    d_middle_band_cosine_min: float = 0.60
    d_high_band_cosine_min: float = 0.50
    d_full_density_cosine_min: float = 0.80
    d_full_potential_cosine_min: float = 0.70
    e_density_rho_l2_max: float = 0.55
    e_density_potential_rho_l2_max: float = 0.45
    e_density_potential_l2_max: float = 0.60
    potential_rms_ratio_min: float = 0.70
    potential_rms_ratio_max: float = 1.30
    potential_shape_cosine_min: float = 0.75
    e_full_field_l2_max: float = 0.45
    periodic_consistency_validation_max: float = 0.15
    energy_shared_release_fraction: float = 0.65
    energy_shared_full_fraction: float = 0.90
    source_gate_hysteresis: int = 2
    quick_train_records_per_element: int = 256
    quick_train_validation_records_per_element: int = 64
    quick_train_d_epochs: int = 12
    source_stage_lr_floor: float = 0.10
    joint_group_lr_floor: float = 0.20
    freeze_ema_until_d_low_ready: bool = True
    freeze_ema_during_ni: bool = True
    ae_min_full_regime_updates: int = 1
    ae_min_element_coverage: float = 0.95
    hard_min_effective_updates: int = 1
    energy_sample_weight_cap: float = 3.0
    retention_stream_enabled: bool = os.getenv('PERIODIC_JEPA_RETENTION_STREAM', '1') == '1'
    retention_stream_every_updates: int = 1
    refinement_batch_size: int = 64
    ni_eligible_fraction: float = 0.80
    physical_difficulty_weights: tuple[float, float, float, float] = (0.35, 0.25, 0.20, 0.20)
    energy_bias_weight: float = 0.10
    energy_slope_weight: float = 0.0
    energy_slope_min_records: int = 8
    energy_slope_variance_floor: float = 1e-4
    energy_upper_tail_weight: float = 1.25
    ni_field_log_gain_limit: float = 0.50
    rho_log_amplitude_weight: float = 0.05
    potential_log_amplitude_weight: float = 0.15
    ni_repair_epochs: int = 20
    ni_repair_learning_rate: float = 1e-5
    ni_repair_budget_relative: float = 0.25
    ni_repair_budget_absolute: float = 0.01
    ni_exploration_retention_weight: float = 1.0
    energy_affine_regularization: float = 1e-3
    evaluation_spearman_points: int = 2048
    final_rho_l2_goal: float = 0.10
    final_potential_l2_goal: float = 0.10
    final_potential_cosine_goal: float = 0.95
    final_energy_bias_goal_ev: float = 0.02

    pretrain_elements: tuple[str, ...] = ("Al", "Fe")
    transfer_element: str = "Ni"
    ood_families: tuple[str, ...] = ("shear", "combo")
    require_family_metadata: bool = True
    allow_readme_family_range_inference: bool = True
    require_complete_output: bool = True
    pretrain_validation_fraction: float = 0.10
    pretrain_test_fraction: float = 0.10
    ni_finetune_fraction: float = 0.10
    ni_finetune_fraction_cap: float = 0.10
    ni_validation_fraction: float = 0.10

    canonical_grid: tuple[int, int, int] = (48, 32, 32)
    potential_scale_floor_ry: float = 0.05
    density_floor: float = 1e-8
    density_log_clip: float = 8.0
    cache_preprocessed_records: bool = False
    preprocessing_version: str = PREPROCESSING_VERSION

    backbone: str = BACKBONE_ALIASES.get(
        os.environ.get("PERIODIC_JEPA_BACKBONE", "pyg_gnn_kan"),
        os.environ.get("PERIODIC_JEPA_BACKBONE", "pyg_gnn_kan"),
    )
    decoder_type: str = "periodic_fourier_residual"
    primary_output: str = "fields_plus_cohesive"
    graph_fractional_extent: float = 1.0
    graph_max_edges: int = 250000
    graph_max_candidate_pairs: int = 1000000
    shared_residual_scale: float = 0.05
    run_tag: str = os.getenv("PERIODIC_JEPA_RUN_TAG", "run04_gate_driven")
    rbf_count: int = 24
    position_frequencies: int = 4
    hidden: int = 128
    latent_dim: int = 1024
    field_spectral_complex_modes: int = 512
    spectral_statistics_floor: float = 1e-4
    spectral_statistics_records: int = 800
    latent_queue_tokens: int = 512  # legacy buffer; v9 losses do not use stale detached samples
    jepa_alignment_dim: int = 128  # legacy low-band diagnostic; v12 optimizes all non-DC bands
    d_latent_epochs: int = 30
    d_latent_learning_rate: float = 1e-4
    ni_cohesive_calibration_epochs: int = 10
    element_loss_worst_weight: float = 0.50
    ae_regularization_weight: float = 1e-3
    spectral_relative_floor: float = 0.05
    enforce_stage_gates: bool = True
    ae_rho_relative_l2_max: float = 0.60
    ae_potential_relative_l2_max: float = 0.80
    d_cosine_min: float = 0.50
    d_rank_absolute_goal: float = 5.0
    d_rank_target_fraction: float = 0.50
    d_dominant_fraction_goal: float = 0.80
    graph_layers: int = 4
    use_global_context: bool = True
    predictor_tokens: int = 2
    transformer_layers: int = 4
    attention_heads: int = 4
    dropout: float = float(os.getenv("PERIODIC_JEPA_DROPOUT", "0.10"))

    kan_basis: str = "chebyshev"
    kan_degree: int = 4
    kan_rank: int = 8
    kan_placements: tuple[str, ...] = ("message", "update", "predictor", "decoder", "cohesive")

    autoencoder_learning_rate: float = 1e-4
    learning_rate: float = 1e-4
    ni_learning_rate: float = 5e-5
    weight_decay: float = 1e-5
    gradient_clip_norm: float = 2.0
    gradient_clip_rms: float = 0.10
    gradient_accumulation: int = 2
    warmup_fraction: float = 0.10
    encoder_joint_lr_multiplier: float = 0.05
    reconstruction_joint_lr_multiplier: float = 0.05
    residual_decoder_lr_multiplier: float = 0.20
    ni_replay_fraction: float = 0.80
    fe_replay_share: float = 0.60
    retention_penalty: float = 12.00
    retention_rho_relative_tolerance: float = 0.05
    retention_cosine_drop_tolerance: float = 0.01
    fe_retention_importance: float = 1.0
    pretrain_element_balance_penalty: float = 0.50
    translation_augmentation_probability: float = 1.00

    use_directional_field_edges: bool = True

    potential_amplitude_log_limit: float = 1.50
    ema_tau_start: float = 0.996
    ema_tau_end: float = 0.999
    amp: bool = True

    loss_jepa: float = 0.30
    loss_latent_cosine: float = 0.25
    loss_rho: float = 1.00
    loss_potential: float = 1.00
    loss_reconstruction: float = 0.20
    loss_latent_variance: float = 0.10
    loss_latent_covariance: float = 0.10
    predicted_latent_regularization_fraction: float = 0.80
    loss_latent_scale: float = 0.05
    loss_kan: float = 1e-6
    potential_relative_loss_fraction: float = 0.20
    potential_gradient_loss_fraction: float = 0.15
    potential_cosine_loss_fraction: float = 0.10
    potential_amplitude_loss_fraction: float = 0.30

    # Changes to these controls require a new contract/run_tag.
    online_field_lr_multiplier: float = 0.05
    electronic_online_lr_multiplier: float = 0.05
    loss_electronic_reconstruction: float = 1.0
    loss_electronic_anchor: float = 0.10
    ni_adapter_lr_multiplier: float = 0.50
    adapter_hidden: int = 64
    ni_joint_min_epochs: int = 20
    retention_distillation_weight: float = 1.00
    retention_potential_relative_tolerance: float = 0.05
    retention_cohesive_relative_tolerance: float = 0.10
    retention_cohesive_floor_ev_per_atom: float = 0.001
    spectral_resolution_audit: bool = True
    spectral_audit_records_per_element: int = 12
    rho_gradient_loss_fraction: float = 0.12

    loss_translation_consistency: float = 0.10
    translation_consistency_probability: float = 0.50
    loss_retention_latent: float = 0.25

    # v12: all retained Fourier coefficients participate in the JEPA objective.
    jepa_band_boundaries: tuple[tuple[int, int], ...] = ((2, 130), (130, 514), (514, 1024))
    jepa_band_weights: tuple[float, ...] = (0.45, 0.35, 0.20)

    # Cohesive energy.
    # Positive cohesive energy:
    # E_coh = E_isolated_atom - E_total / N

    loss_cohesive: float = 0.20
    loss_autoencoder_cohesive: float = 0.15
    cohesive_mse_fraction: float = 0.0
    cohesive_huber_beta_ev_per_atom: float = 0.025
    energy_bias_buffer_capacity: int = 128
    energy_bias_min_samples: int = 64
    checkpoint_field_relative_tolerance: float = 0.05

    # Weakly bound structures lie in the lower tail of cohesive energy.
    cohesive_weak_quantile: float = 0.10
    cohesive_extreme_weak_quantile: float = 0.01

    cohesive_weak_weight: float = 1.5
    cohesive_extreme_weight: float = 2.0

    # For positive cohesive energy, the dangerous error is overestimating binding.
    cohesive_overprediction_weight: float = 1.0
    cohesive_extreme_overprediction_weight: float = 1.0

    cohesive_weak_selection_weight: float = 0.35

    ni_cohesive_stratification_bins: int = 20
    ni_weak_cohesion_oversample_factor: int = 2
    ni_extreme_weak_cohesion_oversample_factor: int = 2

    ni_adapter_warmup_epochs: int = 10
    predictor_joint_lr_multiplier: float = 0.10
    cohesive_head_lr_multiplier: float = 0.50
    freeze_autoencoder_in_ni: bool = True
    constraint_barrier: float = 1000.0

    require_shared_latent_intervention: bool = True
    shared_latent_intervention_min_delta: float = 1e-4
    shared_latent_intervention_min_relative_change: float = 0.05

    # v14: rare-energy coverage and asymmetric tail objective.


    # v14: selective replay and high-frequency field fidelity.
    retention_rho_weight: float = 0.75
    retention_potential_weight: float = 2.50
    retention_cohesive_weight: float = 0.50
    retention_fe_multiplier: float = 1.50
    rho_spectral_loss_fraction: float = 0.05
    potential_spectral_loss_fraction: float = 0.15
    spectral_high_frequency_boost: float = 4.0

    # Train-only thresholds for complementary classification diagnostics.
    density_classification_quantile: float = 0.90
    classification_threshold_records_per_element: int = 64

    # v13/v14 phase-aware controls: change before configuration/data cells.
    progressive_spectral_curriculum: bool = True
    middle_band_ramp: tuple[float, float] = (0.15, 0.40)
    high_band_ramp: tuple[float, float] = (0.40, 0.70)
    freeze_autoencoder_in_d: bool = True
    spectral_mode_embedding_dim: int = 16
    spectral_mode_chunk_size: int = 64
    ae_warm_start_checkpoint: str = os.getenv("PERIODIC_JEPA_AE_CHECKPOINT", "")
    short_overfit_diagnostic: bool = True
    overfit_records_per_element: int = 8
    overfit_batch_size: int = 4
    overfit_steps: int = 200
    overfit_learning_rate: float = 3e-4
    overfit_log_every: int = 25

    # V24: optimization and selection only; no model tensor changes.
    balance_pretrain_batches: bool = True
    ae_energy_ramp_start: float = 0.10
    ae_energy_ramp_end: float = 0.30
    ni_cohesive_lr_multiplier: float = 0.50  # extra factor on Ni head only
    energy_refinement_epochs: int = 10
    energy_refinement_learning_rate: float = 1e-5  # actual head LR at scheduler peak
    early_stopping: bool = True
    early_stopping_patience: int = 8
    early_stopping_min_fraction: float = 0.35
    early_stopping_relative_delta: float = 1e-4
    refinement_patience: int = 3
    refinement_max_element_relative_regression: float = 0.02
    refinement_max_tail_relative_regression: float = 0.05
    refinement_error_floor_normalized: float = 1e-3
    resume: bool = True
    save_every_epochs: int = 1
    bootstrap_repetitions: int = 1000
    inference_benchmark_warmup: int = 5
    inference_benchmark_repetitions: int = 25
    experiment_name: str = "periodic_jepa_al_fe_to_ni_v31_gate_driven_curriculum"

    @property
    def preset(self) -> RunPreset:
        if self.run_mode not in PRESETS:
            raise ValueError(f"invalid run_mode: {self.run_mode}")
        return PRESETS[self.run_mode]

    @property
    def accumulation_steps(self) -> int:
        # DEVELOPMENT uses the same effective batch (16) as FULL without increasing VRAM.
        return self.development_gradient_accumulation if self.run_mode in ("QUICK_TRAIN","DEVELOPMENT") else self.gradient_accumulation

    @property
    def run_dir(self) -> Path:
        return self.output_root / f"{self.experiment_name}_{self.data_curriculum_mode}" / f"seed_{self.seed}_{self.run_tag}"


CFG = Config()
assert math.isfinite(CFG.cohesive_huber_beta_ev_per_atom) and CFG.cohesive_huber_beta_ev_per_atom > 0, "Huber beta must be positive in eV/atom"
assert 64 <= CFG.energy_bias_min_samples <= CFG.energy_bias_buffer_capacity, "Bias FIFO requires at least 64 valid samples"
assert 1 <= CFG.ni_weak_cohesion_oversample_factor <= CFG.ni_extreme_weak_cohesion_oversample_factor <= 2, "Ni oversampling limited to 2x"
assert 0 <= CFG.checkpoint_field_relative_tolerance <= 0.05, "Field tolerance cannot exceed 5%"
if not 0 <= CFG.ae_energy_ramp_start < CFG.ae_energy_ramp_end < 1:
    raise ValueError('Invalid AE energy ramp')
if CFG.balance_pretrain_batches and CFG.preset.batch_size % 2:
    raise ValueError('Al/Fe balancing requires an even batch_size')
if CFG.early_stopping_patience < 1 or not 0 < CFG.early_stopping_min_fraction <= 1:
    raise ValueError('Invalid early stopping')
if CFG.energy_refinement_epochs < 1:
    raise ValueError('Refinement requires at least one epoch')
if not CFG.freeze_autoencoder_in_d or not CFG.freeze_autoencoder_in_ni:
    raise ValueError('V31 permanently freezes the AE: both options must remain True.')
if not re.fullmatch(r"[A-Za-z0-9_-]+", CFG.run_tag):
    raise ValueError("run_tag must contain only letters, digits, _ or -")
if not 0.0 < CFG.graph_fractional_extent <= 2.0:
    raise ValueError("graph_fractional_extent must be in (0,2]")
if CFG.data_curriculum_mode not in {'control', 'physical'}:
    raise ValueError("PERIODIC_JEPA_DATA_CURRICULUM_MODE must be 'control' or 'physical'")
if not 0.0 < CFG.ema_tau_start <= CFG.ema_tau_end < 1.0:
    raise ValueError('Invalid EMA interval')
if not 0.0 < CFG.objective_ramp_fraction <= 1.0:
    raise ValueError('objective_ramp_fraction must be in (0, 1]')
if not 0.0 <= CFG.d_crossfade_start < CFG.d_crossfade_end <= 1.0:
    raise ValueError('Invalid D cross-fade window')
if not 0.0 < CFG.retention_ramp_fraction <= 1.0:
    raise ValueError('retention_ramp_fraction must be in (0,1]')
if CFG.readiness_patience < 1 or not 0.0 < CFG.readiness_min_fraction <= 1.0:
    raise ValueError('Invalid readiness configuration')
if CFG.accumulation_steps < 1:
    raise ValueError('Effective gradient accumulation must be >= 1')
for start, end, name in ((CFG.joint_predictor_start, CFG.joint_predictor_full, 'predictor'),
                         (CFG.joint_decoder_start, CFG.joint_decoder_full, 'decoder'),
                         (CFG.joint_structural_start, CFG.joint_structural_full, 'structural')):
    if not 0.0 <= start < end <= 1.0:
        raise ValueError(f'Invalid progressive-unfreezing window: {name}')
if CFG.optimizer_update_budget_scale <= 0:
    raise ValueError('optimizer_update_budget_scale must be positive')
if not 0 < CFG.gradient_balance_min_factor <= 1 <= CFG.gradient_balance_max_factor:
    raise ValueError('Invalid gradient balance range')
if not 0 <= CFG.gradient_balance_ema < 1:
    raise ValueError('gradient_balance_ema must be in [0,1)')
if not 0 < CFG.potential_rms_ratio_min < 1 < CFG.potential_rms_ratio_max:
    raise ValueError('Potential amplitude gate must include 1.0')
if not 0 <= CFG.energy_shared_release_fraction < CFG.energy_shared_full_fraction <= 1:
    raise ValueError('Invalid energy gradient release window')

if not math.isclose(sum(CFG.physical_difficulty_weights), 1.0, abs_tol=1e-9):
    raise ValueError('Physical difficulty weights must sum to 1')
if not 0.0 <= CFG.cohesive_mse_fraction <= 1.0:
    raise ValueError("cohesive_mse_fraction must be between 0 and 1")
if not (0.0 < CFG.cohesive_extreme_weak_quantile < CFG.cohesive_weak_quantile < 1.0):
    raise ValueError("Quantiles must satisfy: 0 < extreme_weak < weak < 1")
if not (1.0 <= CFG.cohesive_weak_weight <= CFG.cohesive_extreme_weight):
    raise ValueError("Invalid weak/extreme cohesion weights")
if CFG.ni_cohesive_stratification_bins < 2:
    raise ValueError("Cohesion stratification requires at least two bins")
if not 0 <= CFG.rho_spectral_loss_fraction < 0.5:
    raise ValueError("Invalid rho spectral fraction")
FIELD_NAMES = ("density", "potential")
if len(CFG.jepa_band_boundaries) != len(CFG.jepa_band_weights):
    raise ValueError("Each JEPA band must have a weight")
if not math.isclose(sum(CFG.jepa_band_weights), 1.0, rel_tol=0.0, abs_tol=1e-12):
    raise ValueError("JEPA band weights must sum to 1")
if CFG.jepa_band_boundaries[0][0] != 2 or CFG.jepa_band_boundaries[-1][1] != CFG.latent_dim:
    raise ValueError("JEPA bands must cover the entire non-DC latent space")
for left, right in zip(CFG.jepa_band_boundaries, CFG.jepa_band_boundaries[1:]):
    if left[1] != right[0]:
        raise ValueError("JEPA bands must be contiguous and nonoverlapping")
LATENT_GATES = {
    "latent_variance_min": 1e-4,
    "latent_variance_max": 100.0,
    "effective_rank_min": 3.0,
    "active_covariance_rank_min": 4,
    "active_condition_number_max": 1e6,
    "near_zero_variance_fraction_max": 0.95,
    "dominant_covariance_fraction_max": 0.95,
}

CFG.run_mode = normalize_run_mode(CFG.run_mode)
if CFG.jepa_alignment_dim % 2 or not (2 <= CFG.jepa_alignment_dim <= CFG.latent_dim - 2):
    raise ValueError("jepa_alignment_dim must be even and exclude the DC pair")
if CFG.backbone != 'pyg_gnn_kan':
    raise ValueError('V31 production configuration requires pyg_gnn_kan')
if CFG.backbone not in BACKBONE_OPTIONS:
    raise ValueError(f"backbone={CFG.backbone!r}; opcoes={BACKBONE_OPTIONS}")
if CFG.kan_basis not in KAN_BASIS_OPTIONS:
    raise ValueError(f"kan_basis={CFG.kan_basis!r}; opcoes={KAN_BASIS_OPTIONS}")
if CFG.decoder_type not in DECODER_OPTIONS:
    raise ValueError(f"decoder_type={CFG.decoder_type!r}; opcoes={DECODER_OPTIONS}")
if CFG.primary_output not in PRIMARY_OUTPUT_OPTIONS:
    raise ValueError(f"primary_output={CFG.primary_output!r}; opcoes={PRIMARY_OUTPUT_OPTIONS}")
potential_loss_fraction_sum = (
    CFG.potential_relative_loss_fraction
    + CFG.potential_gradient_loss_fraction
    + CFG.potential_cosine_loss_fraction
    + CFG.potential_amplitude_loss_fraction
    + CFG.potential_spectral_loss_fraction
)
if not (0.0 <= CFG.translation_augmentation_probability <= 1.0):
    raise ValueError("translation_augmentation_probability must be in [0, 1]")
if not (0.0 < CFG.fe_replay_share < 1.0):
    raise ValueError("fe_replay_share must be in (0, 1)")
if not (0.5 <= CFG.predicted_latent_regularization_fraction <= 1.0):
    raise ValueError("predicted_latent_regularization_fraction must be in [0.5, 1]")
if CFG.latent_queue_tokens < 2 * CFG.preset.batch_size:
    raise ValueError("latent_queue_tokens must accommodate at least two token minibatches")
if CFG.ni_finetune_fraction > CFG.ni_finetune_fraction_cap:
    raise ValueError("Fine-tuning may use at most 10% of Ni ID data")
if not (0.0 <= potential_loss_fraction_sum <= 1.0):
    raise ValueError("Potential loss fractions must sum to at most 1")
if CFG.predictor_tokens != len(FIELD_NAMES):
    raise ValueError("predictor_tokens must correspond to density and potential")
if 2 * CFG.field_spectral_complex_modes != CFG.latent_dim:
    raise ValueError(
        "latent_dim must be twice field_spectral_complex_modes "
        "(real and imaginary parts of each Fourier coefficient)"
    )
if not (0 < CFG.pretrain_validation_fraction < 1 and 0 < CFG.pretrain_test_fraction < 1):
    raise ValueError("Al-Fe validation/test fractions must be in (0, 1)")
if CFG.pretrain_validation_fraction + CFG.pretrain_test_fraction >= 1:
    raise ValueError("Al-Fe validation + test fraction must be less than 1")
if not (0 < CFG.ni_finetune_fraction < 1 and 0 < CFG.ni_validation_fraction < 1):
    raise ValueError("Ni adaptation/validation fractions must be in (0, 1)")
if CFG.ni_finetune_fraction + CFG.ni_validation_fraction >= 1:
    raise ValueError("Ni adaptation + validation fraction must be less than 1")


def resolve_device(requested: str) -> torch.device:
    requested = requested.lower()
    if requested == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    device = torch.device(requested)
    if device.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("PERIODIC_JEPA_DEVICE requests CUDA, but CUDA is unavailable")
    return device


DEVICE = resolve_device(CFG.device)
if not (0 < CFG.electronic_online_lr_multiplier <= 1):
    raise ValueError('electronic_online_lr_multiplier must be in (0,1].')
if not (CFG.loss_electronic_reconstruction > 0 and CFG.loss_electronic_anchor >= 0):
    raise ValueError('The electronic objective requires positive reconstruction and a nonnegative anchor.')
CFG.run_dir.mkdir(parents=True, exist_ok=True)
print({
    "device": str(DEVICE),
    "run_mode_raw": RAW_RUN_MODE,
    "run_mode": CFG.run_mode,
    "run_dir": str(CFG.run_dir),
})


{'device': 'cuda', 'run_mode_raw': 'QUICK_TRAIN', 'run_mode': 'QUICK_TRAIN', 'run_dir': 'outputs/periodic_jepa_al_fe_to_ni_v31_gate_driven_curriculum_physical/seed_42_run04_gate_driven'}


### V29 r3 protocol — checkpoint policy

Loss controls gradients; score compares candidates; physical gates determine eligibility.
In the autoencoder, fields are evaluated only against the configured absolute physical limits
(`ae_rho_relative_l2_max` and `ae_potential_relative_l2_max`), without comparison with the
initial reconstruction. In later energy phases, the relative field reference is frozen at the start
of the phase; subsequent checks are pure and do not tighten the gate using a moving minimum. AE requires
warm-up, ramp, full-regime updates, Al/Fe coverage, and approved absolute field limits.
Ni uses only the frozen eligible selection and five partial unfreezing stages.
Retention may use a separate Al/Fe stream; repair is conditional on validation degradation.


In [5]:
def seed_everything(seed: int) -> None:
    """Sets the random sources used in this notebook."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def atomic_json_dump(value: Any, path: Path) -> None:
    """Saves JSON without leaving a partial file if writing is interrupted."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, default=str))
    os.replace(temporary, path)


seed_everything(CFG.seed)
BASIC_TEST_CONFIG = {
    "experiment": CFG.experiment_name,
    "seed": CFG.seed,
    "run_mode": RUN_MODE,
    "backbone": CFG.backbone,
    "ni_finetune_fraction": CFG.ni_finetune_fraction,
    "ni_validation_fraction": CFG.ni_validation_fraction,
    "ood_families": list(CFG.ood_families),
}
atomic_json_dump(BASIC_TEST_CONFIG, CFG.run_dir / "test_config.json")
print(json.dumps(BASIC_TEST_CONFIG, indent=2))


{
  "experiment": "periodic_jepa_al_fe_to_ni_v31_gate_driven_curriculum",
  "seed": 42,
  "run_mode": "QUICK_TRAIN",
  "backbone": "pyg_gnn_kan",
  "ni_finetune_fraction": 0.1,
  "ni_validation_fraction": 0.1,
  "ood_families": [
    "shear",
    "combo"
  ]
}


In [6]:
def json_canonical(value):
    return json.loads(json.dumps(value, default=str, sort_keys=True))

def configuration_contract():
    config = json_canonical(asdict(CFG))
    config.pop('resume')
    return {'architecture': ARCHITECTURE_VERSION,
            'training_protocol': TRAINING_PROTOCOL_VERSION,
            'curriculum': CURRICULUM_VERSION,
            'checkpoint_schema': CHECKPOINT_SCHEMA_VERSION,
            'graph': GRAPH_BUILDER_VERSION,
            'config': config, 'preset': asdict(CFG.preset),
            'cohesive_anchor_kj_mol': COHESIVE_ANCHOR_KJ_MOL,
            'cohesive_anchor_ev_per_atom': COHESIVE_ANCHOR_EV_PER_ATOM,
            'cohesive_calibration_protocol': COHESIVE_CALIBRATION_PROTOCOL,
            'torch': str(torch.__version__), 'pyg': str(torch_geometric.__version__)}

def validate_cohesive_anchor_protocol():
    required = set(CFG.pretrain_elements) | {CFG.transfer_element}
    if set(COHESIVE_ANCHOR_KJ_MOL) != required:
        raise ValueError('Thermodynamic anchors must cover exactly Al, Fe, and Ni.')
    if not all(np.isfinite(float(value)) and float(value) > 0
               for value in COHESIVE_ANCHOR_KJ_MOL.values()):
        raise ValueError('Thermodynamic anchors must be positive and finite.')
    protocol = COHESIVE_CALIBRATION_PROTOCOL
    if protocol['manual_calc_id_selection'] or protocol['lower_tail_fraction'] <= 0:
        raise ValueError('Invalid automatic calibration protocol.')
    if set(protocol['reference_pools']) != required or set(protocol['source_urls']) != required:
        raise ValueError('Incomplete calibration protocol.')

validate_cohesive_anchor_protocol()
contract_path = CFG.run_dir / 'run_configuration_contract.json'
contract = json_canonical(configuration_contract())
if contract_path.exists():
    if json.loads(contract_path.read_text()) != contract:
        raise RuntimeError('Configuration changed. Choose another PERIODIC_JEPA_RUN_TAG; checkpoints will not be overwritten.')
    if not CFG.resume and any((CFG.run_dir/'checkpoints').glob('*/*.pt')):
        raise RuntimeError('Existing run has checkpoints and resume=False; choose another tag.')
else:
    atomic_json_dump(contract, contract_path)

## Data index and integrity
Retains the V18 Etot/family parser, but completeness filtering is now strict.

In [7]:
# =============================================================================
# Local index, energy audit, families, and manifest
# Requires existing imports: Path, pd, re, os, math, json
# =============================================================================

REQUIRED_INDEX_COLUMNS = {
    "calc_id", "calc_dir", "element", "scf_converged", "final_energy_ev",
    "rho_nc_path", "vh_nc_path", "out_path", "struct_out_path", "ion_xml_path",
}

CFG_FAMILY_RANGES = (
    (0, 2999, "disp"),
    (3000, 5999, "iso"),
    (6000, 8499, "aniso"),
    (8500, 9499, "shear"),
    (9500, 9999, "combo"),
)

EXPECTED_FAMILY_COUNTS_PER_ELEMENT = {
    "disp": 3000,
    "iso": 3000,
    "aniso": 2500,
    "shear": 1000,
    "combo": 500,
}

ETOT_PATTERN = re.compile(
    r"^siesta:\s+Etot\s*=\s*([-+0-9.Ee]+)",
    re.MULTILINE,
)


def local_path(relative_path: str | Path) -> Path:
    """Resolves paths relative to the local dataset directory."""
    path = Path(relative_path).expanduser()
    if path.is_absolute():
        return path.resolve()
    direct = (DATA_ROOT / path).resolve()
    if direct.exists() or DATA_ROOT.name == "data":
        return direct
    via_data = (DATA_ROOT / "data" / path).resolve()
    return via_data if via_data.exists() else direct


def read_tail(path: Path, bytes_to_read: int = 65_536) -> str:
    """Reads only the end of a large file."""
    with path.open("rb") as handle:
        handle.seek(0, os.SEEK_END)
        handle.seek(max(0, handle.tell() - bytes_to_read))
        return handle.read().decode(errors="replace")


def read_last_etot(path: Path) -> float:
    """Extracts the last Etot from the SIESTA file."""
    values = ETOT_PATTERN.findall(read_tail(path))

    if not values:
        raise ValueError(f"Etot not found in: {path}")

    return float(values[-1])


def read_family(calc_dir: Path, calc_id: str) -> tuple[str | None, str]:
    """
    Priority: meta.json; optional fallback: cfg interval.
    Returns (family, origin).
    """
    meta_path = calc_dir / "meta.json"

    if meta_path.is_file():
        metadata = json.loads(meta_path.read_text())

        for key in ("family", "perturbation_type", "type", "category"):
            if key in metadata:
                return str(metadata[key]), "meta_json"

    if CFG.allow_readme_family_range_inference:
        match = re.search(r"cfg_(\d+)", str(calc_id))

        if match:
            number = int(match.group(1))

            for start, end, family in CFG_FAMILY_RANGES:
                if start <= number <= end:
                    return family, "cfg_range"

    return None, "missing"


# -----------------------------------------------------------------------------
# Locate and read the index
# -----------------------------------------------------------------------------

DATA_ROOT = CFG.data_root.expanduser().resolve()

INDEX_PATH = (
    CFG.index_path.expanduser().resolve()
    if CFG.index_path is not None
    else DATA_ROOT / "index_full.parquet"
)

if not INDEX_PATH.is_file():
    raise FileNotFoundError(
        f"Index not found: {INDEX_PATH}\n"
        "Set CFG.index_path or place index_full.parquet in CFG.data_root."
    )

index_frame = (
    pd.read_parquet(INDEX_PATH)
    if INDEX_PATH.suffix == ".parquet"
    else pd.read_json(INDEX_PATH, lines=True)
)

missing_columns = REQUIRED_INDEX_COLUMNS - set(index_frame.columns)

if missing_columns:
    raise KeyError(f"Colunas ausentes: {sorted(missing_columns)}")

if index_frame["calc_id"].duplicated().any():
    raise ValueError("Duplicate calc_id values in the index.")


# -----------------------------------------------------------------------------
# Reread Etot, check complete outputs, and audit discrepancies
# (tolerates missing/corrupt .out files)
# -----------------------------------------------------------------------------

audit_rows = []
energies = []
complete_outputs = []

for position, row in index_frame.iterrows():
    out_path = local_path(row["out_path"])
    indexed_energy = float(row["final_energy_ev"])

    if not out_path.is_file():
        energies.append(indexed_energy)      # fallback to index energy
        complete_outputs.append(False)
        audit_rows.append({
            "calc_id": row["calc_id"],
            "index_energy_ev": indexed_energy,
            "parsed_etot_ev": None,
            "output_complete": False,
            "corrected": False,
            "issue": "missing_out_file",
            "out_path": str(out_path),
        })
        continue

    try:
        etot = read_last_etot(out_path)
        complete = "Job completed" in read_tail(out_path, bytes_to_read=8_192)
        corrected = not math.isclose(etot, indexed_energy, abs_tol=1e-7)

        energies.append(etot)
        complete_outputs.append(complete)

        if corrected or not complete:
            audit_rows.append({
                "calc_id": row["calc_id"],
                "index_energy_ev": indexed_energy,
                "parsed_etot_ev": etot,
                "output_complete": complete,
                "corrected": corrected,
                "issue": None if corrected else "incomplete_output",
                "out_path": str(out_path),
            })

    except Exception as error:
        energies.append(indexed_energy)      # fallback to index energy
        complete_outputs.append(False)
        audit_rows.append({
            "calc_id": row["calc_id"],
            "index_energy_ev": indexed_energy,
            "parsed_etot_ev": None,
            "output_complete": False,
            "corrected": False,
            "issue": f"out_parse_error: {type(error).__name__}",
            "out_path": str(out_path),
        })

    if (position + 1) % 5_000 == 0:
        print(f"Audited {position + 1}/{len(index_frame)} .out files")

index_frame["final_energy_ev"] = energies
index_frame["output_complete"] = complete_outputs
energy_repair_audit = pd.DataFrame(audit_rows)


# -----------------------------------------------------------------------------
# Define and validate structural families
# -----------------------------------------------------------------------------

families_and_sources = [
    read_family(local_path(calc_dir), calc_id)
    for calc_dir, calc_id in zip(
        index_frame["calc_dir"],
        index_frame["calc_id"],
    )
]

index_frame["family"] = [family for family, _ in families_and_sources]
index_frame["family_source"] = [source for _, source in families_and_sources]

if CFG.require_family_metadata and index_frame["family"].isna().any():
    raise FileNotFoundError(
        f"{int(index_frame['family'].isna().sum())} configurations without a family. "
        "Provide meta.json or enable inference from cfg."
    )

if index_frame["family_source"].eq("cfg_range").any():
    for element, group in index_frame.groupby("element"):
        counts = {
            str(family): int(count)
            for family, count in group["family"].value_counts().items()
        }

        if counts != EXPECTED_FAMILY_COUNTS_PER_ELEMENT:
            raise ValueError(
                f"Incorrect families for {element}: {counts}; "
                f"esperado: {EXPECTED_FAMILY_COUNTS_PER_ELEMENT}"
            )


# -----------------------------------------------------------------------------
# Save the scientific audit and a simple dataset summary
# -----------------------------------------------------------------------------

energy_repair_audit.to_csv(
    CFG.run_dir / "energy_repair_audit.csv",
    index=False,
)

DATASET_MANIFEST = {
    "data_schema_version": DATA_SCHEMA_VERSION,
    "solver_version": SOLVER_VERSION,
    "configured_dataset_root": str(CFG.data_root.resolve()),
    "resolved_dataset_root": str(DATA_ROOT),
    "configured_index_path": str(CFG.index_path.resolve())
    if CFG.index_path else None,
    "index_path": str(INDEX_PATH),
    "index_file_size_bytes": INDEX_PATH.stat().st_size,
    "record_count": len(index_frame),
    "element_counts": index_frame["element"].value_counts().to_dict(),
    "family_counts": index_frame["family"].value_counts(
        dropna=False
    ).to_dict(),
    "family_source_counts": index_frame["family_source"].value_counts(
        dropna=False
    ).to_dict(),
    "family_range_fallback_enabled": CFG.allow_readme_family_range_inference,
    "family_ranges": [
        {"start": start, "end": end, "family": family}
        for start, end, family in CFG_FAMILY_RANGES
    ],
    "expected_family_counts_per_element": EXPECTED_FAMILY_COUNTS_PER_ELEMENT,
    "scf_converged_count": int(index_frame["scf_converged"].sum()),
    "complete_output_count": int(index_frame["output_complete"].sum()),
    "incomplete_output_count": int((~index_frame["output_complete"]).sum()),
    "energy_repair_count": len(energy_repair_audit),
    "energy_range_ev_by_element": {
        element: {
            "minimum": float(group["final_energy_ev"].min()),
            "maximum": float(group["final_energy_ev"].max()),
        }
        for element, group in index_frame.groupby("element")
    },
    "fermi_range_ev_by_element": {
        element: {
            "minimum": float(group["fermi_ev"].min()),
            "maximum": float(group["fermi_ev"].max()),
        }
        for element, group in index_frame.groupby("element")
    } if "fermi_ev" in index_frame else None,
    "required_index_columns": sorted(REQUIRED_INDEX_COLUMNS),
    "path_columns": sorted(
        column for column in index_frame.columns
        if column.endswith("_path")
    ),
}

atomic_json_dump(
    DATASET_MANIFEST,
    CFG.run_dir / "dataset_manifest.json",
)

print({
    "index": str(INDEX_PATH),
    "records": len(index_frame),
    "elements": index_frame["element"].value_counts().to_dict(),
    "families": index_frame["family"].value_counts(
        dropna=False
    ).to_dict(),
    "energy_repairs": len(energy_repair_audit),
    "incomplete_outputs": int((~index_frame["output_complete"]).sum()),
    "dataset_records": DATASET_MANIFEST["record_count"],
})

display(energy_repair_audit.head(20))
if index_frame.calc_id.duplicated().any():
    raise ValueError("Duplicate calc_id in the index")
def strict_boolean(value):
    if isinstance(value, (bool, np.bool_)): return bool(value)
    if str(value).strip().lower() in ('true','1'): return True
    if str(value).strip().lower() in ('false','0'): return False
    raise ValueError(f"Invalid boolean in the index: {value!r}")
index_frame['scf_converged'] = index_frame.scf_converged.map(strict_boolean)
if CFG.require_complete_output:
    eligible_mask = index_frame.scf_converged & index_frame.output_complete
else:
    eligible_mask = index_frame.scf_converged
if not eligible_mask.any():
    raise ValueError('No converged/complete records; fix paths/outputs, do not bypass the filter.')
if not np.isfinite(index_frame.loc[eligible_mask,'final_energy_ev']).all():
    raise ValueError('Nonfinite energy in eligible records')


Audited 5000/30000 .out files
Audited 10000/30000 .out files
Audited 15000/30000 .out files
Audited 20000/30000 .out files
Audited 25000/30000 .out files
Audited 30000/30000 .out files
{'index': '/home/rafael/Documentos/Pos_doc-BIO-IA/Task-1/single-element_GNN-JEPA/siesta-single-elements-30k-public-core/public_core/index_full.parquet', 'records': 30000, 'elements': {'Al': 10000, 'Fe': 10000, 'Ni': 10000}, 'families': {'disp': 9000, 'iso': 9000, 'aniso': 7500, 'shear': 3000, 'combo': 1500}, 'energy_repairs': 3, 'incomplete_outputs': 3, 'dataset_records': 30000}


,calc_id,index_energy_ev,parsed_etot_ev,output_complete,corrected,issue,out_path
0,Fe__cfg_2335,-6820.622674,-6820.622674,False,False,incomplete_output,/home/rafael/Documentos/Pos_doc-BIO-IA/Task-1/...
1,Fe__cfg_6424,-6820.624424,-6820.624424,False,False,incomplete_output,/home/rafael/Documentos/Pos_doc-BIO-IA/Task-1/...
2,Ni__cfg_6092,-1436.698243,-9131.058318,False,True,NaN,/home/rafael/Documentos/Pos_doc-BIO-IA/Task-1/...


In [8]:
# =============================================================================
# STEP 07 — SIESTA reading and periodic fields
# Purpose: Reads structure, XML, and Rho/V NetCDF, and resamples fields onto the canonical grid.
# Inputs: Index rows and structure/NetCDF files.
# Outputs: Canonical records with structure, rho, potential, charge, and energy.
# Reproducibility: run after all preceding cells; do not change the seed during the run.
# =============================================================================
BOHR_TO_ANGSTROM = 0.529177210903
ANGSTROM_TO_BOHR = 1.0 / BOHR_TO_ANGSTROM

UNIT_CONTRACT = {
    'version': 'scientific-units-v29-r2', 'total_energy': 'eV/cell',
    'cohesive_energy': 'eV/atom', 'reported_energy': 'meV/atom',
    'potential': 'Ry', 'density': 'electron/Bohr^3', 'cell': 'Angstrom',
    'netcdf_cell': 'Bohr', 'Ry_to_eV': 13.605693122994,
    'Bohr_to_Angstrom': BOHR_TO_ANGSTROM,
}

NETCDF_UNIT_AUDIT = {
    'missing_metadata': {}, 'declared_metadata': {}, 'struct_cell_crosschecks': 0,
    'struct_cell_crosschecks_passed': 0, 'warning_emitted': False,
}

def _assumed_unit_for(quantity):
    return UNIT_CONTRACT['netcdf_cell' if quantity == 'cell' else 'density' if quantity == 'rho' else 'potential']

def validate_netcdf_unit(unit, quantity, path):
    value = str(unit).strip().lower().replace(' ', '')
    accepted = {'rho': {'1/bohr**3', 'e/bohr**3', 'e/bohr^3', '1/bohr^3', 'electron/bohr^3'},
                'electrostaticpotential': {'ry', 'rydberg'},
                'cell': {'bohr', 'a.u.', 'au'}}
    if not value:
        key=f'{quantity}:{_assumed_unit_for(quantity)}'
        bucket=NETCDF_UNIT_AUDIT['missing_metadata'].setdefault(key, set())
        bucket.add(str(path))
        if not NETCDF_UNIT_AUDIT['warning_emitted']:
            warnings.warn('NetCDF unit metadata are missing in part of the dataset; the SIESTA contract will be applied and audited against STRUCT_OUT. A single summary will be saved to netcdf_unit_audit.json.')
            NETCDF_UNIT_AUDIT['warning_emitted']=True
    elif value not in accepted[quantity]:
        raise ValueError(f'Incompatible NetCDF unit: {path}, {quantity}={unit!r}')
    else:
        NETCDF_UNIT_AUDIT['declared_metadata'].setdefault(quantity, set()).add(value)

def finalize_netcdf_unit_audit():
    payload={
        'unit_contract': copy.deepcopy(UNIT_CONTRACT),
        'missing_metadata_counts': {k:len(v) for k,v in NETCDF_UNIT_AUDIT['missing_metadata'].items()},
        'missing_metadata_examples': {k:sorted(v)[:10] for k,v in NETCDF_UNIT_AUDIT['missing_metadata'].items()},
        'declared_metadata': {k:sorted(v) for k,v in NETCDF_UNIT_AUDIT['declared_metadata'].items()},
        'struct_cell_crosschecks': int(NETCDF_UNIT_AUDIT['struct_cell_crosschecks']),
        'struct_cell_crosschecks_passed': int(NETCDF_UNIT_AUDIT['struct_cell_crosschecks_passed']),
        'struct_out_crosscheck_unit_conversion': 'Angstrom -> Bohr',
        'all_crosschecks_passed': NETCDF_UNIT_AUDIT['struct_cell_crosschecks']==NETCDF_UNIT_AUDIT['struct_cell_crosschecks_passed'],
    }
    if 'CFG' in globals():
        atomic_json_dump(payload, CFG.run_dir/'netcdf_unit_audit.json')
    return payload

ATOMIC_PROPERTIES = {
    "Al": {"Z": 13, "mass": 26.9815385, "valence": 3.0, "group": 13, "period": 3},
    "Fe": {"Z": 26, "mass": 55.845, "valence": 16.0, "group": 8, "period": 4},
    "Ni": {"Z": 28, "mass": 58.6934, "valence": 18.0, "group": 10, "period": 4},
}


def read_struct_out(path: Path) -> dict[str, np.ndarray]:
    lines = [line.split() for line in path.read_text().splitlines() if line.strip()]
    if len(lines) < 5:
        raise ValueError(f"Invalid STRUCT_OUT: {path}")
    cell_angstrom = np.asarray(lines[:3], dtype=np.float64)
    atom_count = int(lines[3][0])
    atom_lines = lines[4:4 + atom_count]
    species_Z = np.asarray([int(line[1]) for line in atom_lines], dtype=np.int64)
    frac = np.asarray([[float(v) for v in line[2:5]] for line in atom_lines], dtype=np.float64) % 1.0
    return {"cell_angstrom": cell_angstrom, "fractional": frac, "atomic_numbers": species_Z}


@lru_cache(maxsize=32)
def read_ion_valence(path_string: str) -> float:
    path = Path(path_string)
    root = ET.parse(path).getroot()
    node = root.find("valence")
    if node is None or node.text is None:
        raise ValueError(f"Missing valence in {path}")
    return float(node.text)


def read_grid(path: Path, expected_description: str) -> dict[str, Any]:
    with netCDF4.Dataset(path) as dataset:
        variable = dataset.variables["gridfunc"]
        field = np.asarray(variable[:], dtype=np.float32)
        cell_bohr = np.asarray(dataset.variables["cell"][:], dtype=np.float64)
        dimensions = tuple(variable.dimensions)
        description = getattr(variable, "Description", "")
        unit = getattr(variable, "units", getattr(variable, "unit", ""))
        validate_netcdf_unit(unit, expected_description.lower(), path)
        cell_variable = dataset.variables["cell"]
        validate_netcdf_unit(getattr(cell_variable, "units", getattr(cell_variable, "unit", "")), "cell", path)
    if dimensions != ("spin", "n3", "n2", "n1"):
        raise ValueError(f"Unexpected grid axis order in {path}: {dimensions}")
    if field.shape[0] != 1:
        raise ValueError(
            f"This pilot notebook expects one unpolarized channel; {path} has {field.shape[0]} channels"
        )
    if expected_description.lower() not in str(description).lower():
        raise ValueError(f"Incorrect field in {path}: {description}")
    assert np.isfinite(field).all() and np.isfinite(cell_bohr).all(), f"Nonfinite field/cell: {path}"
    return {
        "field": field[0],
        "cell_bohr": cell_bohr,
        "shape": tuple(field.shape[1:]),
        "unit": str(unit),
        "description": str(description),
    }


def periodic_fourier_resample(array: np.ndarray, target_shape: tuple[int, int, int]) -> np.ndarray:
    """Resamples a periodic field along each axis using Fourier interpolation."""
    result = np.asarray(array, dtype=np.float64)
    for axis, size in enumerate(target_shape):
        if result.shape[axis] != size:
            result = resample(result, size, axis=axis)
    return np.real(result).astype(np.float32)


def canonical_fields(
    rho_native: np.ndarray,
    potential_native: np.ndarray,
    cell_bohr: np.ndarray,
    electron_count: float,
) -> dict[str, np.ndarray | float]:
    rho = periodic_fourier_resample(rho_native, CFG.canonical_grid)
    potential = periodic_fourier_resample(potential_native, CFG.canonical_grid)
    volume_bohr3 = float(abs(np.linalg.det(cell_bohr)))

    # Small negative values can arise solely from spectral resampling.
    rho = np.maximum(rho, 0.0)
    rho *= electron_count / max(float(rho.mean()) * volume_bohr3, 1e-12)
    rho_scaled = rho * volume_bohr3 / electron_count  # mean exactly 1
    potential_relative = potential - float(potential.mean())
    assert np.isfinite(rho).all() and np.isfinite(potential_relative).all(), "Nonfinite canonical fields"
    assert np.isclose(float(rho.mean()) * volume_bohr3, electron_count, rtol=2e-5), "Density e/Bohr^3: inconsistent charge integral"
    assert abs(float(potential_relative.mean())) <= 2e-6 * max(1.0, float(np.abs(potential).max())), "Ry potential: inconsistent gauge removal"
    return {
        "rho": rho,
        "rho_scaled": rho_scaled,
        "potential_relative": potential_relative,
        "volume_bohr3": volume_bohr3,
        "charge": float(rho.mean()) * volume_bohr3,
    }


def load_raw_record(row: Mapping[str, Any]) -> dict[str, Any]:
    struct = read_struct_out(resolve_dataset_path(DATA_ROOT, row["struct_out_path"]))
    rho_data = read_grid(resolve_dataset_path(DATA_ROOT, row["rho_nc_path"]), "Rho")
    potential_data = read_grid(resolve_dataset_path(DATA_ROOT, row["vh_nc_path"]), "ElectrostaticPotential")
    if rho_data["shape"] != potential_data["shape"]:
        raise ValueError(f"Incompatible rho/V grids in {row['calc_id']}")
    if not np.allclose(rho_data["cell_bohr"], potential_data["cell_bohr"], atol=2e-6):
        raise ValueError(f"Incompatible rho/V cells in {row['calc_id']}")
    cell_from_struct_bohr = struct["cell_angstrom"] * ANGSTROM_TO_BOHR
    NETCDF_UNIT_AUDIT['struct_cell_crosschecks'] += 1
    if not np.allclose(cell_from_struct_bohr, rho_data["cell_bohr"], atol=2e-5):
        raise ValueError(f"Incompatible structural and NetCDF cells in {row['calc_id']}")
    NETCDF_UNIT_AUDIT['struct_cell_crosschecks_passed'] += 1

    element = str(row["element"])
    ion_path = resolve_dataset_path(DATA_ROOT, row["ion_xml_path"])
    valence = read_ion_valence(str(ion_path))
    expected_valence = ATOMIC_PROPERTIES[element]["valence"]
    if not math.isclose(valence, expected_valence, abs_tol=1e-6):
        raise ValueError(f"Unexpected valence for {element}: {valence}")
    electron_count = valence * len(struct["atomic_numbers"])
    fields = canonical_fields(
        rho_data["field"], potential_data["field"], rho_data["cell_bohr"], electron_count
    )
    native_charge = float(rho_data["field"].mean() * fields["volume_bohr3"])
    return {
        **dict(row),
        **struct,
        **fields,
        "cell_bohr": rho_data["cell_bohr"],
        "native_grid_shape": rho_data["shape"],
        "native_charge": native_charge,
        "electron_count": electron_count,
    }



def load_structural_record(row: Mapping[str, Any]) -> dict[str, Any]:
    """Structural whitelist: never opens fields, energy, SCF output, or target cache."""
    structure = read_struct_out(resolve_dataset_path(DATA_ROOT, row['struct_out_path']))
    element = str(row['element'])
    if element not in ATOMIC_PROPERTIES:
        raise ValueError(f'Unsupported element: {element}')
    if not np.all(structure['atomic_numbers'] == ATOMIC_PROPERTIES[element]['Z']):
        raise ValueError('This version supports pure systems; composition does not match element')
    cell = structure['cell_angstrom']
    if not np.isfinite(cell).all() or abs(np.linalg.det(cell)) < 1e-8:
        raise ValueError('Degenerate or nonfinite cell')
    valence = read_ion_valence(str(resolve_dataset_path(DATA_ROOT, row['ion_xml_path'])))
    if not math.isclose(valence, ATOMIC_PROPERTIES[element]['valence'], abs_tol=1e-6):
        raise ValueError('Pseudopotential/valence incompatible with chemical descriptors')
    return {**structure, 'element': element, 'calc_id': str(row['calc_id']),
            'family': str(row.get('family','unknown')), 'family_source': str(row.get('family_source','unknown')),
            'electron_count': valence * len(structure['fractional']),
            'volume_bohr3': float(abs(np.linalg.det(cell * ANGSTROM_TO_BOHR)))}

def cached_raw_record(row: Mapping[str, Any]) -> dict[str, Any]:
    # No persistent grid cache: avoids tens of GB and stale energy metadata.
    if CFG.cache_preprocessed_records:
        raise ValueError('V19 uses direct reading; cache_preprocessed_records must be False')
    return load_raw_record(row)


In [9]:
# =============================================================================
# STEP 08 — Physical data audit
# Purpose: Checks charge, positivity, finiteness, potential gauge, SCF, and completeness.
# Inputs: Deterministic sample of index_frame.
# Outputs: data_audit_manifest.json and data_audit_rows.csv.
# Reproducibility: run after all preceding cells; do not change the seed during the run.
# =============================================================================
def resolve_dataset_path(data_root: Path, relative_path: str | Path) -> Path:
    root = Path(data_root).expanduser().resolve()
    path = Path(relative_path).expanduser()
    if path.is_absolute():
        return path.resolve()
    direct = (root / path).resolve()
    if direct.exists() or root.name == "data":
        return direct
    via_data = (root / "data" / path).resolve()
    return via_data if via_data.exists() else direct


def output_is_complete(path: Path) -> bool:
    if not path.is_file():
        return False
    try:
        return "Job completed" in read_tail(path, bytes_to_read=8_192)
    except Exception:
        return False


def parse_last_etot(path: Path) -> float:
    if not path.is_file():
        return float("nan")
    try:
        return float(read_last_etot(path))
    except Exception:
        return float("nan")


def deterministic_sample(frame: pd.DataFrame, count: int, seed: int) -> pd.DataFrame:
    if len(frame) <= count:
        return frame.sort_values("calc_id")
    return frame.sample(count, random_state=seed).sort_values("calc_id")


def try_cached_raw_record(index_row_dict: dict) -> dict | None:
    """Attempt to load a raw record, returning None if any required file is missing."""
    try:
        return cached_raw_record(index_row_dict)
    except (FileNotFoundError, OSError):
        return None


def _metadata_only_row(index_row, out_path: Path, issue: str, missing_files: list[str] | None = None) -> dict:
    return {
        "calc_id": str(index_row.calc_id),
        "element": str(index_row.element),
        "family": index_row.get("family", None),
        "native_shape": None,
        "volume_bohr3": float("nan"),
        "electron_count": float("nan"),
        "native_charge": float("nan"),
        "canonical_charge": float("nan"),
        "native_charge_error": float("nan"),
        "canonical_charge_error": float("nan"),
        "rho_min": float("nan"),
        "rho_finite": False,
        "potential_mean_ry": float("nan"),
        "potential_finite": False,
        "scf_converged": bool(index_row.get("scf_converged", False)),
        "output_complete": output_is_complete(out_path),
        "energy_index_minus_parsed_ev": float("nan"),
        "audit_level": "metadata_only",
        "audit_issue": issue,
        "missing_files": ",".join(missing_files or []),
    }


def audit_records(frame: pd.DataFrame, count: int) -> pd.DataFrame:
    chosen = []
    per_element = max(1, count // max(frame.element.nunique(), 1))
    for offset, (_, group) in enumerate(frame.groupby("element")):
        chosen.append(deterministic_sample(group, per_element, CFG.seed + offset))
    sample = pd.concat(chosen).head(count)

    rows = []
    skipped_full = 0

    for _, index_row in sample.iterrows():
        rho_path = resolve_dataset_path(DATA_ROOT, index_row.rho_nc_path)
        vh_path = resolve_dataset_path(DATA_ROOT, index_row.vh_nc_path)
        struct_path = resolve_dataset_path(DATA_ROOT, index_row.struct_out_path)
        out_path = resolve_dataset_path(DATA_ROOT, index_row.out_path)

        missing_required = [p.name for p in (rho_path, vh_path, struct_path) if not p.is_file()]
        if missing_required:
            skipped_full += 1
            rows.append(_metadata_only_row(
                index_row=index_row,
                out_path=out_path,
                issue="missing_required_files",
                missing_files=missing_required,
            ))
            continue

        record = try_cached_raw_record(index_row.to_dict())
        if record is None:
            skipped_full += 1
            rows.append(_metadata_only_row(
                index_row=index_row,
                out_path=out_path,
                issue="raw_record_unavailable",
                missing_files=[],
            ))
            continue

        parsed_etot = parse_last_etot(out_path)
        energy_delta = (
            float(record["final_energy_ev"] - parsed_etot)
            if np.isfinite(parsed_etot) else float("nan")
        )

        rows.append({
            "calc_id": record["calc_id"],
            "element": record["element"],
            "family": record.get("family"),
            "native_shape": "x".join(map(str, record["native_grid_shape"])),
            "volume_bohr3": record["volume_bohr3"],
            "electron_count": record["electron_count"],
            "native_charge": record["native_charge"],
            "canonical_charge": record["charge"],
            "native_charge_error": abs(record["native_charge"] - record["electron_count"]),
            "canonical_charge_error": abs(record["charge"] - record["electron_count"]),
            "rho_min": float(np.min(record["rho"])),
            "rho_finite": bool(np.isfinite(record["rho"]).all()),
            "potential_mean_ry": float(np.mean(record["potential_relative"])),
            "potential_finite": bool(np.isfinite(record["potential_relative"]).all()),
            "scf_converged": bool(record["scf_converged"]),
            "output_complete": output_is_complete(out_path),
            "energy_index_minus_parsed_ev": energy_delta,
            "audit_level": "full",
            "audit_issue": "",
            "missing_files": "",
        })

    if skipped_full > 0:
        print(f"\n[AUDIT WARNING] {skipped_full}/{len(sample)} records without full audit; metadata_only fallback used.")

    if not rows:
        raise RuntimeError("Unexpected failure: empty audit sample.")

    return pd.DataFrame(rows)


data_audit = audit_records(index_frame.loc[eligible_mask], CFG.preset.audit_records)
data_audit.to_csv(CFG.run_dir / "data_audit.csv", index=False)

print(data_audit.groupby("element").agg({
    "native_charge_error": "max",
    "canonical_charge_error": "max",
    "rho_min": "min",
    "potential_mean_ry": lambda x: float(np.nanmax(np.abs(x))) if np.isfinite(x).any() else float("nan"),
    "output_complete": "mean",
}).to_string())

# Run strict assertions only where a full audit was performed.
full_mask = data_audit["audit_level"].eq("full")
if full_mask.any():
    full_audit = data_audit.loc[full_mask]
    if full_audit.native_charge_error.max() > 2e-3:
        raise AssertionError("Native density integral failed")
    if full_audit.canonical_charge_error.max() > 2e-5:
        raise AssertionError("Canonical density normalization failed")
    if not full_audit.rho_finite.all() or not full_audit.potential_finite.all():
        raise AssertionError("Nonfinite field found")
else:
    raise RuntimeError("No records with a complete physical audit")

DATA_AUDIT_MANIFEST = {
    "records_audited": len(data_audit),
    "records_audited_full": int(full_mask.sum()),
    "records_audited_metadata_only": int((~full_mask).sum()),
    "calc_ids": data_audit.calc_id.astype(str).tolist(),
    "maximum_native_charge_error": float(np.nanmax(data_audit.native_charge_error.values)) if np.isfinite(data_audit.native_charge_error.values).any() else float("nan"),
    "maximum_canonical_charge_error": float(np.nanmax(data_audit.canonical_charge_error.values)) if np.isfinite(data_audit.canonical_charge_error.values).any() else float("nan"),
    "minimum_density": float(np.nanmin(data_audit.rho_min.values)) if np.isfinite(data_audit.rho_min.values).any() else float("nan"),
    "maximum_absolute_potential_mean_ry": float(np.nanmax(np.abs(data_audit.potential_mean_ry.values))) if np.isfinite(data_audit.potential_mean_ry.values).any() else float("nan"),
    "all_fields_finite": bool(data_audit.rho_finite.all() and data_audit.potential_finite.all()),
    "all_outputs_complete_in_audit": bool(data_audit.output_complete.all()),
}
atomic_json_dump(DATA_AUDIT_MANIFEST, CFG.run_dir / "data_audit_manifest.json")

display(data_audit.head())
if not full_mask.all():
    raise RuntimeError('Incomplete audit: fields/files are missing. Resolve this before training.')

# Structural metadata once; cohesive energy/stratification do not open grids.
atom_counts = []
for _, row in index_frame.loc[eligible_mask].iterrows():
    record = load_structural_record(row.to_dict())
    atom_counts.append(len(record['fractional']))
index_frame.loc[eligible_mask, 'atom_count'] = atom_counts
index_frame['energy_per_atom_ev'] = np.nan
index_frame.loc[eligible_mask, 'energy_per_atom_ev'] = (
    index_frame.loc[eligible_mask, 'final_energy_ev']
    / index_frame.loc[eligible_mask, 'atom_count']
)
if not np.isfinite(index_frame.loc[eligible_mask, 'energy_per_atom_ev']).all():
    raise ValueError('Invalid energy per atom in eligible records.')


/tmp/ipykernel_180304/357844035.py:37: UserWarning: NetCDF unit metadata are missing in part of the dataset; the SIESTA contract will be applied and audited against STRUCT_OUT. A single summary will be saved to netcdf_unit_audit.json.
  warnings.warn('NetCDF unit metadata are missing in part of the dataset; the SIESTA contract will be applied and audited against STRUCT_OUT. A single summary will be saved to netcdf_unit_audit.json.')


         native_charge_error  canonical_charge_error   rho_min  potential_mean_ry  output_complete
element                                                                                           
Al              9.536743e-07            6.284308e-07  0.001882       1.390775e-07              1.0
Fe              3.814697e-06            3.598337e-06  0.028933       1.887480e-07              1.0
Ni              3.814697e-06            4.308056e-06  0.024448       9.934107e-08              1.0


,calc_id,element,family,native_shape,volume_bohr3,electron_count,native_charge,canonical_charge,native_charge_error,canonical_charge_error,rho_min,rho_finite,potential_mean_ry,potential_finite,scf_converged,output_complete,energy_index_minus_parsed_ev,audit_level,audit_issue,missing_files
0,Al__cfg_0439,Al,disp,50x30x30,148.920341,6.0,6.000000,6.0,4.768372e-07,3.088848e-07,0.002077,True,5.960464e-08,True,True,True,0.0,full,,
1,Al__cfg_0576,Al,disp,50x30x30,146.039937,6.0,6.000000,6.0,4.768372e-07,1.739681e-07,0.001882,True,3.476937e-08,True,True,True,0.0,full,,
2,Al__cfg_1731,Al,disp,50x30x30,149.392278,6.0,6.000000,6.0,4.768372e-07,3.614201e-07,0.002076,True,5.463759e-08,True,True,True,0.0,full,,
3,Al__cfg_2750,Al,disp,50x30x30,149.414879,6.0,5.999999,6.0,9.536743e-07,9.011827e-08,0.002076,True,-7.947286e-08,True,True,True,0.0,full,,
4,Al__cfg_3999,Al,iso,50x30x30,146.039937,6.0,5.999999,6.0,9.536743e-07,3.700730e-07,0.002039,True,-7.450581e-09,True,True,True,0.0,full,,


In [10]:
import hashlib
def stable_contract_hash(value):
    return hashlib.sha256(json.dumps(json_canonical(value), sort_keys=True).encode()).hexdigest()


STRUCTURAL_DESCRIPTOR_CACHE = {}

def structural_descriptor_frame(frame):
    """Raw descriptors; no validation-fitted statistics or incompatible displacement reference."""
    rows = []
    for _, row in frame.sort_values(['element', 'calc_id']).iterrows():
        key = str(row.element) + '::' + str(row.calc_id)
        if key not in STRUCTURAL_DESCRIPTOR_CACHE:
            record = load_structural_record(row.to_dict())
            cell = np.asarray(record['cell_angstrom'], dtype=float)
            frac = np.asarray(record['fractional'], dtype=float)
            lengths = np.linalg.norm(cell, axis=1)
            volume = abs(np.linalg.det(cell)) / len(frac)
            if not np.isfinite(cell).all() or not np.isfinite(frac).all() or volume <= 0:
                raise ValueError(f'Invalid structure {key}')
            cosines = np.array([np.dot(cell[i], cell[j]) / (lengths[i]*lengths[j])
                                for i,j in ((0,1),(0,2),(1,2))])
            # Enumerate periodic neighbours, including images of the same atom.
            shifts = np.array(list(__import__('itertools').product(range(-2,3), repeat=3)))
            distances = []
            for atom in frac:
                vectors = (frac[:, None, :] - atom + shifts[None, :, :]) @ cell
                d = np.linalg.norm(vectors, axis=-1).ravel()
                distances.append(d[d > 1e-7])
            nearest = min(float(d.min()) for d in distances)
            coordination = float(np.mean([np.sum(d <= 1.25*nearest) for d in distances]))
            explicit = next((str(row[c]) for c in ('trajectory_id','base_structure_id','structure_group','group_id')
                             if c in row and pd.notna(row[c])), None)
            fingerprint = hashlib.sha256(np.round(np.concatenate([cell.ravel(), np.sort(frac, axis=0).ravel()]), 6).tobytes()).hexdigest()[:20]
            group = str(row.element) + '::' + (explicit if explicit else 'geometry:' + fingerprint)
            descriptor = dict(structural_group=group, group_source='metadata' if explicit else 'geometry_duplicate_only',
                volume_per_atom=volume, anisotropy=float(np.std(np.log(lengths))),
                shear=float(np.linalg.norm(cosines)), cell_ratio_1=float(lengths[1]/lengths[0]),
                cell_ratio_2=float(lengths[2]/lengths[0]), cell_angle_01=float(np.arccos(np.clip(cosines[0],-1,1))),
                cell_angle_02=float(np.arccos(np.clip(cosines[1],-1,1))),
                cell_angle_12=float(np.arccos(np.clip(cosines[2],-1,1))),
                nearest_distance=nearest, coordination=coordination,
                displacement_rms=np.nan, displacement_max=np.nan)
            STRUCTURAL_DESCRIPTOR_CACHE[key] = (descriptor, frac, cell)
        rows.append({**row.to_dict(), **STRUCTURAL_DESCRIPTOR_CACHE[key][0]})
    work = pd.DataFrame(rows).reset_index(drop=True)
    work['energy_relative'] = work.energy_per_atom_ev - work.groupby(['element','family']).energy_per_atom_ev.transform('min')
    work['volumetric_deformation'] = np.abs(np.log(work.volume_per_atom / work.groupby(['element','family']).volume_per_atom.transform('median')))
    work['cell_deformation'] = work.volumetric_deformation + work.shear + work.anisotropy
    for _, indices in work.loc[work.group_source.eq('metadata')].groupby('structural_group').groups.items():
        indices = list(indices); reference = work.loc[indices].energy_per_atom_ev.idxmin()
        refrow = work.loc[reference]; ref = STRUCTURAL_DESCRIPTOR_CACHE[str(refrow.element)+'::'+str(refrow.calc_id)][1]
        for i in indices:
            r = work.loc[i]; _, frac, cell = STRUCTURAL_DESCRIPTOR_CACHE[str(r.element)+'::'+str(r.calc_id)]
            if frac.shape != ref.shape:
                continue
            delta = (frac-ref+.5)%1-.5
            shift = np.angle(np.mean(np.exp(2j*np.pi*delta),axis=0))/(2*np.pi)
            delta = ((delta-shift+.5)%1-.5) @ cell
            norm = np.linalg.norm(delta,axis=1)
            work.loc[i,['displacement_rms','displacement_max']] = [np.sqrt(np.mean(norm**2)), norm.max()]
    columns = ['energy_relative','volume_per_atom','volumetric_deformation','cell_deformation',
               'shear','anisotropy','nearest_distance','coordination','displacement_rms']
    work['physical_difficulty'] = 0.0
    for _, indices in work.groupby('element').groups.items():
        available = [c for c in columns if work.loc[indices,c].notna().all()]
        work.loc[indices,'physical_difficulty'] = work.loc[indices,available].rank(pct=True).mean(axis=1)
    return work


def grouped_ni_split(frame, seed):
    """Whole known trajectories/base structures and exact geometry duplicates stay together."""
    eligible, validation, holdout = [], [], []
    owners=frame.groupby('structural_group').family.agg(lambda values: sorted(values.astype(str).unique())[0])
    for offset, owner in enumerate(sorted(owners.unique())):
        groups = sorted(owners.index[owners.eq(owner)].tolist())
        rng = np.random.default_rng(seed+offset); rng.shuffle(groups)
        if len(groups) < 3:
            raise ValueError('Ni: fewer than three independent groups per family; a safe split is impossible')
        nval = max(1, int(round(CFG.ni_validation_fraction*len(groups))))
        ntest = max(1, int(round((1-CFG.ni_eligible_fraction-CFG.ni_validation_fraction)*len(groups))))
        if nval+ntest >= len(groups): raise ValueError('Ni: split quotas exhaust eligible groups')
        validation.extend(groups[:nval]); holdout.extend(groups[nval:nval+ntest]); eligible.extend(groups[nval+ntest:])
    return tuple(frame.loc[frame.structural_group.isin(g)].copy().reset_index(drop=True)
                 for g in (eligible,validation,holdout))


def select_representative_ni(eligible, seed, fraction=None):
    """Marginal quota coverage then group-stratified farthest-point coverage, eligible only."""
    fraction = CFG.ni_finetune_fraction if fraction is None else fraction
    work = eligible.sort_values('calc_id').drop_duplicates('calc_id').reset_index(drop=True)
    columns = [c for c in ('energy_relative','volume_per_atom','volumetric_deformation','cell_deformation',
        'shear','anisotropy','cell_ratio_1','cell_ratio_2','cell_angle_01','cell_angle_02','cell_angle_12',
        'displacement_rms','displacement_max','nearest_distance','coordination','physical_difficulty','scf_iterations')
        if c in work and (c not in ('displacement_rms','displacement_max','scf_iterations') or work[c].notna().all())]
    valid = np.isfinite(work[columns].to_numpy(dtype=float)).all(axis=1)
    excluded = work.loc[~valid,['calc_id']].assign(reason='nonfinite_selection_descriptor').to_dict('records')
    work = work.loc[valid].reset_index(drop=True)
    count = int(math.floor(fraction*len(work)))
    if count < 1: raise ValueError('Ni: 10% budget is empty after excluding invalid records')
    raw = work[columns].to_numpy(dtype=float); center=raw.mean(axis=0); scale=raw.std(axis=0)
    scale=np.where(scale>1e-12,scale,1.)
    x=np.clip((raw-center)/scale,-3.,3.)  # bounded outlier influence
    rng=np.random.default_rng(seed); tie=rng.permutation(len(work)); tie_rank=np.argsort(tie)
    dimensions = ['family','physical_difficulty','energy_relative','cell_deformation','anisotropy']
    strata=[]
    for column in dimensions:
        labels = work[column].astype(str) if column=='family' else pd.Series(_selection_rank_bins(work[column], 5),index=work.index).astype(str)
        strata += [(column,str(label),np.flatnonzero(labels.to_numpy()==label)) for label in sorted(labels.unique())]
    quotas = {(c,l): max(1,int(round(count*len(ids)/len(work)))) for c,l,ids in strata}
    selected=[]; nearest=np.full(len(work),np.inf); group_counts={}
    while len(selected)<count:
        candidates=np.array([i for i in range(len(work)) if i not in set(selected)],dtype=int)
        # Most unmet marginal quotas first; deterministic redistribution when full.
        deficits=np.zeros(len(work))
        for c,l,ids in strata:
            deficit=max(0,quotas[(c,l)]-len(set(selected)&set(ids)))
            if deficit: deficits[ids]+=deficit/max(quotas[(c,l)],1)
        highest=deficits[candidates].max(); pool=candidates[np.isclose(deficits[candidates],highest)]
        # Prefer under-covered structural groups, then k-center within that pool.
        gc=np.array([group_counts.get(work.structural_group.iloc[i],0) for i in pool]); pool=pool[gc==gc.min()]
        if selected:
            distance=nearest[pool]; pool=pool[np.isclose(distance,distance.max())]
        else:
            distance=np.linalg.norm(x[pool],axis=1); pool=pool[np.isclose(distance,distance.min())]
        chosen=int(pool[np.argmin(tie_rank[pool])]); selected.append(chosen)
        g=work.structural_group.iloc[chosen]; group_counts[g]=group_counts.get(g,0)+1
        nearest=np.minimum(nearest,np.sum((x-x[chosen])**2,axis=1))
    picked=work.iloc[selected].sort_values('calc_id').reset_index(drop=True)
    random_ids=work.iloc[rng.choice(len(work),count,replace=False)].calc_id.astype(str).tolist()
    manifest=dict(version='ni-representative-v29-r2',seed=seed,fraction=fraction,
        eligible_ids=work.calc_id.astype(str).tolist(),selected_ids=picked.calc_id.astype(str).tolist(),
        descriptors=columns,center=center.tolist(),scale=scale.tolist(),standardization_pool='eligible_only',
        excluded=excluded,missing_descriptors=[c for c in ('displacement_rms','displacement_max') if c not in columns],
        quotas=[dict(dimension=c,stratum=l,requested=quotas[(c,l)],realized=len(set(selected)&set(ids))) for c,l,ids in strata],
        group_counts=group_counts,random_baseline_ids=random_ids,random_baseline_training=False,
        records=picked[['calc_id','family','structural_group','physical_difficulty']+columns].loc[:,lambda t:~t.columns.duplicated()].to_dict('records'))
    manifest['hash']=stable_contract_hash(manifest)
    return picked,manifest


def _selection_rank_bins(values, n):
    ranks=pd.Series(values).rank(method='average',pct=True).to_numpy()
    return np.minimum((ranks*n).astype(int),n-1)

# =============================================================================
# STEP 09 — Leak-free splits and Ni stratified by energy/family
# =============================================================================
def seeded_split_order(frame: pd.DataFrame, random_state: int) -> pd.DataFrame:
    """Sorts by ID and shuffles with a fixed seed, without using a hash."""
    return (frame.sort_values("calc_id").reset_index(drop=True)
            .sample(frac=1.0, random_state=random_state).reset_index(drop=True))


def stratified_energy_family_three_way(
    frame: pd.DataFrame, train_count: int, validation_count: int, random_state: int
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Stratifies Ni ID by family and energy ranking, with exact counts.

    Using the target only to predefine the split is stratification, not
    training. Normalization, losses, and checkpoint selection still have no access
    to validation/holdout targets.
    """
    if train_count < 1 or validation_count < 1 or train_count + validation_count >= len(frame):
        raise ValueError("Invalid counts for the Ni split")
    work = frame.sort_values("calc_id").reset_index(drop=True).copy()
    bins = np.zeros(len(work), dtype=np.int64)
    for _, indices in work.groupby("family", sort=True).groups.items():
        indices = np.asarray(list(indices), dtype=np.int64)
        ranks = work.loc[indices, "energy_per_atom_ev"].rank(method="first").to_numpy() - 1
        bins[indices] = np.minimum(
            (ranks * CFG.ni_cohesive_stratification_bins / max(len(indices), 1)).astype(int),
            CFG.ni_cohesive_stratification_bins - 1,
        )
    work["_energy_stratum"] = bins
    train_parts, validation_parts, holdout_parts = [], [], []
    train_fraction, validation_fraction = train_count / len(work), validation_count / len(work)
    for offset, (_, group) in enumerate(work.groupby(["family", "_energy_stratum"], sort=True)):
        group = seeded_split_order(group, random_state + offset)
        n_train = int(math.floor(train_fraction * len(group)))
        n_validation = int(math.floor(validation_fraction * len(group)))
        train_parts.append(group.iloc[:n_train])
        validation_parts.append(group.iloc[n_train:n_train + n_validation])
        holdout_parts.append(group.iloc[n_train + n_validation:])
    train = pd.concat(train_parts, ignore_index=True)
    validation = pd.concat(validation_parts, ignore_index=True)
    holdout = pd.concat(holdout_parts, ignore_index=True)

    # Floors per stratum can leave a small deficit; fill deterministically from
    # the current holdout without altering the predeclared total fractions.
    train_deficit = train_count - len(train)
    if train_deficit:
        ordered = seeded_split_order(holdout, random_state + 10001)
        moved, holdout = ordered.iloc[:train_deficit], ordered.iloc[train_deficit:]
        train = pd.concat([train, moved], ignore_index=True)
    validation_deficit = validation_count - len(validation)
    if validation_deficit:
        ordered = seeded_split_order(holdout, random_state + 20001)
        moved, holdout = ordered.iloc[:validation_deficit], ordered.iloc[validation_deficit:]
        validation = pd.concat([validation, moved], ignore_index=True)

    columns = [column for column in work.columns if column != "_energy_stratum"]
    outputs = tuple(part[columns].sort_values("calc_id").reset_index(drop=True)
                    for part in (train, validation, holdout))
    if tuple(map(len, outputs)) != (train_count, validation_count, len(frame)-train_count-validation_count):
        raise AssertionError("Stratification did not preserve exact counts")
    return outputs


def make_splits(frame: pd.DataFrame) -> dict[str, pd.DataFrame]:
    converged = frame[frame.scf_converged.astype(bool)].copy()
    if CFG.require_complete_output:
        complete = converged[converged.output_complete.astype(bool)]
        if complete.empty:
            raise ValueError("No complete output; the mandatory filter cannot be ignored")
        converged = complete.copy()

    pre = converged[converged.element.isin(CFG.pretrain_elements)].copy()
    if CFG.require_family_metadata:
        ood = pre[pre.family.isin(CFG.ood_families)].copy()
        id_pool = pre[~pre.family.isin(CFG.ood_families)].copy()
    else:
        ood, id_pool = pre.iloc[0:0].copy(), pre

    id_pool = seeded_split_order(id_pool, CFG.seed + 101)
    validation_count = int(math.floor(CFG.pretrain_validation_fraction * len(id_pool)))
    test_count = int(math.floor(CFG.pretrain_test_fraction * len(id_pool)))
    train_count = len(id_pool) - validation_count - test_count
    train = id_pool.iloc[:train_count].copy()
    validation = id_pool.iloc[train_count:train_count + validation_count].copy()
    test = id_pool.iloc[train_count + validation_count:].copy()

    ni = converged[converged.element == CFG.transfer_element].copy()
    ni=structural_descriptor_frame(ni)
    ood_groups=set(ni.loc[ni.family.isin(CFG.ood_families),'structural_group'])
    ni_ood=ni.loc[ni.structural_group.isin(ood_groups)].copy()
    ni_id=ni.loc[~ni.structural_group.isin(ood_groups)].copy()
    ni_eligible, ni_validation, ni_holdout = grouped_ni_split(ni_id, CFG.seed+202)
    frozen = {name: part.calc_id.astype(str).tolist() for name,part in
              [('eligible',ni_eligible),('validation',ni_validation),('id_holdout',ni_holdout),('ood',ni_ood)]}
    frozen['groups'] = {name: sorted(part.structural_group.unique()) for name,part in
              [('eligible',ni_eligible),('validation',ni_validation),('id_holdout',ni_holdout)]}
    frozen['hash'] = stable_contract_hash(frozen)
    manifest_path = CFG.run_dir/'ni_frozen_splits.json'
    if manifest_path.exists() and json.loads(manifest_path.read_text()) != json_canonical(frozen):
        raise RuntimeError('Frozen Ni splits changed')
    atomic_json_dump(frozen,manifest_path)
    # Recompute relative/rank descriptors exclusively within frozen eligible data.
    ni_eligible=structural_descriptor_frame(ni_eligible)
    global NI_SELECTION_MANIFEST
    ni_train, NI_SELECTION_MANIFEST = select_representative_ni(ni_eligible, CFG.seed+203)
    NI_SELECTION_MANIFEST['frozen_split_hash'] = frozen['hash']
    NI_SELECTION_MANIFEST['hash'] = stable_contract_hash({k:v for k,v in NI_SELECTION_MANIFEST.items() if k!='hash'})
    atomic_json_dump(NI_SELECTION_MANIFEST,CFG.run_dir/'ni_selection_manifest.json')
    if ni_id.group_source.eq('geometry_duplicate_only').any():
        warnings.warn('Ni has no trajectory/base structure: groups prevent geometric duplicates, but independence of origin is unproven. Unavailable displacements were omitted.')

    result = {
        "pretrain_train": train,
        "pretrain_validation": validation,
        "al_holdout_test": test[test.element == "Al"].copy(),
        "fe_holdout_test": test[test.element == "Fe"].copy(),
        "pretrain_family_ood": ood,
        "ni_adapt_train": ni_train,
        "ni_adapt_validation": ni_validation,
        "ni_id_holdout_test": ni_holdout,
        "ni_family_ood_test": ni_ood,
    }
    required = ["pretrain_train", "pretrain_validation", "al_holdout_test", "fe_holdout_test",
                "ni_adapt_train", "ni_adapt_validation", "ni_id_holdout_test"]
    if CFG.require_family_metadata:
        required += ["pretrain_family_ood", "ni_family_ood_test"]
    empty = [name for name in required if result[name].empty]
    if empty:
        raise ValueError(f"Required splits are empty: {empty}")
    return result


splits = make_splits(index_frame)

def select_automatic_bulk_reference(element, frame):
    """Select a supported low-energy training record without a manual calc_id."""
    columns = ['calc_id', 'element', 'family', 'energy_per_atom_ev']
    work = frame.loc[frame.element.astype(str).eq(element), columns].copy()
    work = work[np.isfinite(work.energy_per_atom_ev)].sort_values(
        ['family', 'energy_per_atom_ev', 'calc_id'], kind='stable')
    minimum_family = int(COHESIVE_CALIBRATION_PROTOCOL['minimum_family_records'])
    fraction = float(COHESIVE_CALIBRATION_PROTOCOL['lower_tail_fraction'])
    minimum_tail = int(COHESIVE_CALIBRATION_PROTOCOL['minimum_tail_records'])
    candidates = []
    for family, group in work.groupby('family', sort=True, dropna=False):
        if len(group) < minimum_family:
            continue
        k = min(len(group), max(minimum_tail, int(math.ceil(fraction * len(group)))))
        tail = group.sort_values(['energy_per_atom_ev', 'calc_id'], kind='stable').head(k).copy()
        center = float(tail.energy_per_atom_ev.median())
        tail['_distance'] = (tail.energy_per_atom_ev - center).abs()
        selected = tail.sort_values(['_distance', 'calc_id'], kind='stable').iloc[0]
        candidates.append(dict(element=element, family=str(family), family_records=len(group),
            tail_records=k, tail_median_energy_ev_per_atom=center,
            selected_calc_id=str(selected.calc_id),
            selected_energy_ev_per_atom=float(selected.energy_per_atom_ev),
            tail_calc_ids=tail.calc_id.astype(str).tolist()))
    if not candidates:
        raise ValueError(f'{element}: no training family has support for automatic calibration.')
    candidates.sort(key=lambda row: (row['tail_median_energy_ev_per_atom'], row['family'], row['selected_calc_id']))
    return candidates[0], candidates

reference_pool_frames = {
    'Al': splits['pretrain_train'],
    'Fe': splits['pretrain_train'],
    'Ni': splits['ni_adapt_train'],
}
COHESIVE_BULK_REFERENCE_EV_PER_ATOM = {}
COHESIVE_REFERENCE_CALC_ID = {}
calibration_rows = []
for element in sorted(COHESIVE_ANCHOR_EV_PER_ATOM):
    selected, candidates = select_automatic_bulk_reference(element, reference_pool_frames[element])
    COHESIVE_BULK_REFERENCE_EV_PER_ATOM[element] = selected['selected_energy_ev_per_atom']
    COHESIVE_REFERENCE_CALC_ID[element] = selected['selected_calc_id']
    calibration_rows.append({**selected,
        'reference_pool': COHESIVE_CALIBRATION_PROTOCOL['reference_pools'][element],
        'anchor_kj_mol': float(COHESIVE_ANCHOR_KJ_MOL[element]),
        'anchor_ev_per_atom': float(COHESIVE_ANCHOR_EV_PER_ATOM[element]),
        'candidate_families': candidates})

def calibrated_cohesive_energy_ev_per_atom(element, total_energy_ev, atom_count):
    if atom_count < 1:
        raise ValueError('atom_count must be positive')
    return float(COHESIVE_ANCHOR_EV_PER_ATOM[element]
        + COHESIVE_BULK_REFERENCE_EV_PER_ATOM[element]
        - float(total_energy_ev) / int(atom_count))

for split_name, part in splits.items():
    part['cohesive_energy_ev_per_atom'] = [
        calibrated_cohesive_energy_ev_per_atom(str(row.element), row.final_energy_ev, row.atom_count)
        for row in part.itertuples()
    ]
reference_by_id = dict(zip(index_frame.calc_id.astype(str), index_frame.index))
index_frame['cohesive_energy_ev_per_atom'] = np.nan
for part in splits.values():
    positions = [reference_by_id[str(calc_id)] for calc_id in part.calc_id]
    index_frame.loc[positions, 'cohesive_energy_ev_per_atom'] = part.cohesive_energy_ev_per_atom.to_numpy()

COHESIVE_CALIBRATION_MANIFEST = {
    'protocol': COHESIVE_CALIBRATION_PROTOCOL,
    'anchor_kj_mol': COHESIVE_ANCHOR_KJ_MOL,
    'anchor_ev_per_atom': COHESIVE_ANCHOR_EV_PER_ATOM,
    'bulk_reference_ev_per_atom': COHESIVE_BULK_REFERENCE_EV_PER_ATOM,
    'reference_calc_id': COHESIVE_REFERENCE_CALC_ID,
    'selection': calibration_rows,
    'test_or_validation_ids_used': False,
    'ni_zero_shot_scope': 'model weights are zero-shot; target zero is calibrated from ni_adapt_train',
}
atomic_json_dump(COHESIVE_CALIBRATION_MANIFEST, CFG.run_dir/'cohesive_calibration_manifest.json')
display(pd.DataFrame([{k:v for k,v in row.items() if k!='candidate_families'} for row in calibration_rows]))

all_split_sets = {name: set(part.calc_id) for name, part in splits.items()}
names = list(all_split_sets)
for i, left in enumerate(names):
    for right in names[i + 1:]:
        if all_split_sets[left] & all_split_sets[right]:
            raise AssertionError(f"Leakage between {left} and {right}")

split_records = {name: {"count": len(part), "calc_ids": sorted(part.calc_id.tolist())}
                 for name, part in splits.items()}
ni_id_eligible_records = len(NI_SELECTION_MANIFEST["eligible_ids"])
ni_adapt_train_records = len(splits["ni_adapt_train"])
split_manifest = {
    "policy": "Al/Fe seeded order; Ni group-disjoint frozen splits then 10% eligible-only marginal quotas + group k-center",
    "split_random_states": {"pretrain": CFG.seed + 101, "ni_transfer": CFG.seed + 202},
    "ni_cohesive_stratification_bins": CFG.ni_cohesive_stratification_bins,
    "cohesive_calibration": COHESIVE_CALIBRATION_MANIFEST,
    "pretrain_elements": CFG.pretrain_elements,
    "transfer_element": CFG.transfer_element,
    "ood_families": CFG.ood_families,
    "require_complete_output": CFG.require_complete_output,
    "pretrain_validation_fraction": CFG.pretrain_validation_fraction,
    "pretrain_test_fraction": CFG.pretrain_test_fraction,
    "ni_finetune_fraction": CFG.ni_finetune_fraction,
    "ni_finetune_fraction_cap": CFG.ni_finetune_fraction_cap,
    "ni_id_eligible_records": int(ni_id_eligible_records),
    "ni_adapt_train_records": int(ni_adapt_train_records),
    "ni_adapt_train_actual_fraction": float(ni_adapt_train_records / ni_id_eligible_records),
    "ni_validation_fraction": CFG.ni_validation_fraction,
    "holdout_contract": {
        "al_holdout_test": "never optimized",
        "fe_holdout_test": "never optimized",
        "ni_id_holdout_test": "never optimized; energy-stratified ID holdout",
        "ni_family_ood_test": "shear/combo never optimized",
    },
    "records": split_records,
}
atomic_json_dump(split_manifest, CFG.run_dir / "split_manifest.json")


/tmp/ipykernel_180304/2519696628.py:256: UserWarning: Ni has no trajectory/base structure: groups prevent geometric duplicates, but independence of origin is unproven. Unavailable displacements were omitted.
  warnings.warn('Ni has no trajectory/base structure: groups prevent geometric duplicates, but independence of origin is unproven. Unavailable displacements were omitted.')


,element,family,family_records,tail_records,tail_median_energy_ev_per_atom,selected_calc_id,selected_energy_ev_per_atom,tail_calc_ids,reference_pool,anchor_kj_mol,anchor_ev_per_atom
0,Al,disp,2376,48,-62.223807,Al__cfg_0019,-62.223794,"[Al__cfg_2144, Al__cfg_2897, Al__cfg_0459, Al_...",pretrain_train,329.70,3.417100
1,Fe,disp,2427,49,-3410.338094,Fe__cfg_0468,-3410.338094,"[Fe__cfg_1586, Fe__cfg_0683, Fe__cfg_1486, Fe_...",pretrain_train,415.47,4.306043
2,Ni,disp,239,5,-4565.537197,Ni__cfg_0559,-4565.537197,"[Ni__cfg_2038, Ni__cfg_1309, Ni__cfg_0559, Ni_...",ni_adapt_train,430.10,4.457672


In [11]:

def cohesive_energy_ev_per_atom(element, total_energy_ev, atom_count):
    return calibrated_cohesive_energy_ev_per_atom(element, total_energy_ev, atom_count)

@dataclass
class Normalization:
    potential_scale_ry: float
    cohesive_center_ev_per_atom: float
    cohesive_scale_ev_per_atom: float
    cohesive_anchor_ev_per_atom: dict[str, float]
    bulk_reference_energy_ev_per_atom: dict[str, float]
    cohesive_reference_calc_id: dict[str, str]
    potential_fitted_calc_ids: list[str] = field(default_factory=list)
    cohesive_fitted_calc_ids: list[str] = field(default_factory=list)

    def __post_init__(self):
        assert math.isfinite(self.potential_scale_ry) and self.potential_scale_ry > 0, "Potential scale must be positive in Ry"
        assert math.isfinite(self.cohesive_scale_ev_per_atom) and self.cohesive_scale_ev_per_atom > 0, "Global cohesive energy scale must be positive in eV/atom"
        assert math.isfinite(self.cohesive_center_ev_per_atom), "Nonfinite cohesive energy center"

    def physical_cohesive_energy(self, element, total_energy_ev, atom_count):
        assert math.isfinite(float(total_energy_ev)), 'Input total energy must be finite in eV/cell'
        if atom_count < 1:
            raise ValueError('Invalid atom_count')
        return float(self.cohesive_anchor_ev_per_atom[element]
            + self.bulk_reference_energy_ev_per_atom[element]
            - float(total_energy_ev) / int(atom_count))

    def encode_cohesive_energy(self, element, total_energy_ev, atom_count):
        cohesive = self.physical_cohesive_energy(element, total_energy_ev, atom_count)
        return float((cohesive-self.cohesive_center_ev_per_atom)/self.cohesive_scale_ev_per_atom)

    def decode_cohesive_energy(self, normalized_value):
        return float(self.cohesive_center_ev_per_atom
            + self.cohesive_scale_ev_per_atom*float(normalized_value))

def robust_center_scale(values: np.ndarray) -> tuple[float, float]:
    center = float(np.median(values))
    mad = float(np.median(np.abs(values - center)))
    scale = max(1.4826 * mad, 1e-3)
    return center, scale

def fit_normalization(
    frame: pd.DataFrame,
    sample_count: int,
) -> Normalization:
    validate_cohesive_anchor_protocol()

    # Potential scale: sample only from pretrain_train.
    sample = deterministic_sample(
        frame,
        min(sample_count, len(frame)),
        CFG.seed + 77,
    )

    potential_squares = []
    potential_fitted_ids = []

    for _, row in sample.iterrows():
        record = cached_raw_record(row.to_dict())
        potential_squares.append(
            float(np.mean(np.square(record["potential_relative"])))
        )
        potential_fitted_ids.append(str(row.calc_id))

    potential_scale = max(
        math.sqrt(float(np.mean(potential_squares))),
        CFG.potential_scale_floor_ry,
    )

    # A single global cohesive energy normalization, fitted only on Al/Fe training data.
    cohesive_values = frame.cohesive_energy_ev_per_atom.to_numpy(dtype=np.float64)
    cohesive_fitted_ids = frame.calc_id.astype(str).tolist()

    cohesive_values = np.asarray(cohesive_values, dtype=np.float64)
    cohesive_center, cohesive_scale = robust_center_scale(cohesive_values)

    return Normalization(
        potential_scale_ry=potential_scale,
        cohesive_center_ev_per_atom=cohesive_center,
        cohesive_scale_ev_per_atom=cohesive_scale,
        cohesive_anchor_ev_per_atom={k:float(v) for k,v in COHESIVE_ANCHOR_EV_PER_ATOM.items()},
        bulk_reference_energy_ev_per_atom={k:float(v) for k,v in COHESIVE_BULK_REFERENCE_EV_PER_ATOM.items()},
        cohesive_reference_calc_id={k:str(v) for k,v in COHESIVE_REFERENCE_CALC_ID.items()},
        potential_fitted_calc_ids=potential_fitted_ids,
        cohesive_fitted_calc_ids=cohesive_fitted_ids,
    )
pretrain_normalization = fit_normalization(splits['pretrain_train'], CFG.preset.normalization_records)
atomic_json_dump(asdict(pretrain_normalization), CFG.run_dir/'pretrain_normalization.json')
display(asdict(pretrain_normalization))

# Explicit numerical and physical contract; no scale or anchor is replaced.
def audit_v29_energy_units(normalization):
    for element in normalization.cohesive_anchor_ev_per_atom:
        atoms = 2
        total = atoms * normalization.bulk_reference_energy_ev_per_atom[element]
        encoded = normalization.encode_cohesive_energy(element, total, atoms)
        physical = normalization.physical_cohesive_energy(element, total, atoms)
        assert math.isclose(normalization.decode_cohesive_energy(encoded), physical, abs_tol=1e-9), "Cohesive energy eV/atom: invalid round trip"
        for error_norm in (-0.1, 0.0, 0.1):
            error_ev = normalization.decode_cohesive_energy(encoded + error_norm) - physical
            assert math.isclose(error_ev, error_norm * normalization.cohesive_scale_ev_per_atom, abs_tol=1e-9), "error_eV/atom != normalized_error × global_scale"
    print(json.dumps(dict(audit='V29_units', density='e/Bohr^3', potential='Ry; mean removed',
        total_energy_input='eV/cell', cohesive_report='eV/atom',
        cohesive_scale_ev_per_atom=normalization.cohesive_scale_ev_per_atom,
        cohesive_huber_beta_ev_per_atom=CFG.cohesive_huber_beta_ev_per_atom,
        cohesive_huber_beta_normalized=CFG.cohesive_huber_beta_ev_per_atom / normalization.cohesive_scale_ev_per_atom)))
audit_v29_energy_units(pretrain_normalization)


{'potential_scale_ry': 1.87452839990275,
 'cohesive_center_ev_per_atom': 4.051812364087027,
 'cohesive_scale_ev_per_atom': 0.37908043425007915,
 'cohesive_anchor_ev_per_atom': {'Al': 3.4170997056696386,
  'Fe': 4.306043114087245,
  'Ni': 4.457672379158361},
 'bulk_reference_energy_ev_per_atom': {'Al': -62.223794,
  'Fe': -3410.3380945,
  'Ni': -4565.5371975},
 'cohesive_reference_calc_id': {'Al': 'Al__cfg_0019',
  'Fe': 'Fe__cfg_0468',
  'Ni': 'Ni__cfg_0559'},
 'potential_fitted_calc_ids': ['Al__cfg_0123',
  'Al__cfg_0376',
  'Al__cfg_0486',
  'Al__cfg_0516',
  'Al__cfg_1004',
  'Al__cfg_1195',
  'Al__cfg_1358',
  'Al__cfg_1482',
  'Al__cfg_1615',
  'Al__cfg_1725',
  'Al__cfg_1801',
  'Al__cfg_2024',
  'Al__cfg_2031',
  'Al__cfg_2133',
  'Al__cfg_2231',
  'Al__cfg_2271',
  'Al__cfg_2325',
  'Al__cfg_2874',
  'Al__cfg_3018',
  'Al__cfg_3052',
  'Al__cfg_3065',
  'Al__cfg_3129',
  'Al__cfg_3264',
  'Al__cfg_3317',
  'Al__cfg_3339',
  'Al__cfg_3401',
  'Al__cfg_3524',
  'Al__cfg_3581',
  

{"audit": "V29_units", "density": "e/Bohr^3", "potential": "Ry; mean removed", "total_energy_input": "eV/cell", "cohesive_report": "eV/atom", "cohesive_scale_ev_per_atom": 0.37908043425007915, "cohesive_huber_beta_ev_per_atom": 0.025, "cohesive_huber_beta_normalized": 0.06594906447613573}


## Cell-dependent periodic graph
RBFs use each record’s geometry; no call accepts a fixed cutoff.

In [12]:
def seed_loader_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# =============================================================================
# STEP 11 — PyG periodic graph
# Purpose: Builds nodes, periodic images, RBFs, cell, and batchable targets.
# Inputs: Canonical record and normalization.
# Outputs: Data/Batch objects and DataLoaders ready for the backbones.
# Reproducibility: run after all preceding cells; do not change the seed during the run.
# =============================================================================



def gaussian_rbf(distances: torch.Tensor, count: int, radial_scale: float) -> torch.Tensor:
    centers = torch.linspace(0.0, radial_scale, count, device=distances.device, dtype=distances.dtype)
    width = max(radial_scale / max(count - 1, 1), 1e-6)
    return torch.exp(-0.5 * ((distances[..., None] - centers) / width) ** 2)


def enumerate_periodic_edges(fractional, cell_angstrom):
    fractional = np.asarray(fractional, dtype=np.float64) % 1.0
    cell = np.asarray(cell_angstrom, dtype=np.float64)
    if cell.shape != (3,3) or not np.isfinite(cell).all() or abs(np.linalg.det(cell)) < 1e-8:
        raise ValueError('Invalid cell')
    extent = float(CFG.graph_fractional_extent)
    limit = int(math.ceil(extent))
    shifts = np.asarray([(a,b,c) for a in range(-limit,limit+1)
                         for b in range(-limit,limit+1) for c in range(-limit,limit+1)], dtype=np.int64)
    count = len(fractional)
    if not count or not np.isfinite(fractional).all():
        raise ValueError('Empty or nonfinite coordinates')
    if count*count*len(shifts) > CFG.graph_max_candidate_pairs:
        raise ValueError('Graph exceeds the candidate budget; review the rule before raising limits')
    sources, destinations, images, vectors = [], [], [], []
    for i in range(count):
        for j in range(count):
            q = fractional[j] + shifts - fractional[i]
            v = q @ cell
            distance = np.linalg.norm(v, axis=1)
            keep = (np.max(np.abs(q), axis=1) <= extent + 1e-7) & (distance > 1e-8)
            sources.extend([i]*int(keep.sum())); destinations.extend([j]*int(keep.sum()))
            images.extend(shifts[keep]); vectors.extend(v[keep])
    if not sources:
        raise ValueError('No edges: use a fractional extent that includes neighbors/images')
    if len(sources) > CFG.graph_max_edges:
        raise ValueError(f'Graph with {len(sources)} edges exceeds limit; no edges were truncated')
    return (np.asarray([sources,destinations],dtype=np.int64),
            np.asarray(images,dtype=np.int64), np.asarray(vectors,dtype=np.float32))



def periodic_position_encoding(fractional: torch.Tensor, frequencies: int) -> torch.Tensor:
    modes = torch.arange(1, frequencies + 1, device=fractional.device, dtype=fractional.dtype)
    phase = 2 * math.pi * fractional[..., None] * modes
    return torch.cat([torch.sin(phase), torch.cos(phase)], dim=-1).flatten(-2)


def cell_invariants(cell_angstrom: np.ndarray, atom_count: int) -> np.ndarray:
    lengths = np.linalg.norm(cell_angstrom, axis=1)
    cosines = []
    for i, j in ((1, 2), (0, 2), (0, 1)):
        cosines.append(np.dot(cell_angstrom[i], cell_angstrom[j]) / (lengths[i] * lengths[j]))
    volume = abs(np.linalg.det(cell_angstrom))
    return np.asarray([
        *(lengths / 5.0), *cosines, math.log(max(volume / atom_count, 1e-8)), atom_count / 8.0
    ], dtype=np.float32)


def graph_from_record(record: Mapping[str, Any], normalization: Normalization, include_targets: bool) -> Data:
    element = str(record["element"])
    properties = ATOMIC_PROPERTIES[element]
    if 'atomic_numbers' in record and not np.all(np.asarray(record['atomic_numbers']) == properties['Z']):
        raise ValueError('Only pure systems with a consistent element are supported')
    fractional = np.asarray(record["fractional"], dtype=np.float32) % 1.0
    cell_angstrom = np.asarray(record["cell_angstrom"], dtype=np.float32)
    edge_index_np, shifts_np, vectors_np = enumerate_periodic_edges(
        fractional, cell_angstrom
    )
    distances = torch.as_tensor(np.linalg.norm(vectors_np, axis=1), dtype=torch.float32)
    # Scale follows the cell; log1p(d/Å) preserves absolute distance.
    distance_scale = float(CFG.graph_fractional_extent * np.linalg.norm(cell_angstrom,axis=1).sum())
    rbf = gaussian_rbf(distances, CFG.rbf_count, distance_scale)
    fractional_edges = (fractional[edge_index_np[1]] + shifts_np - fractional[edge_index_np[0]])
    cell_envelope = torch.as_tensor(np.exp(-0.5*(fractional_edges**2).sum(-1)),dtype=torch.float32)
    edge_attr = torch.cat([distances[:,None]/distance_scale,
                           torch.log1p(distances[:,None]), rbf*cell_envelope[:,None]], dim=-1)

    edge_vectors = torch.as_tensor(vectors_np, dtype=torch.float32)

    edge_direction = edge_vectors / distances[:, None].clamp_min(1e-8)

    atom_count = len(fractional)
    base = torch.tensor([
        properties["Z"] / 30.0,
        properties["mass"] / 60.0,
        properties["valence"] / 20.0,
        properties["group"] / 18.0,
        properties["period"] / 7.0,
    ], dtype=torch.float32).expand(atom_count, -1).clone()
    environment = torch.zeros(atom_count, CFG.rbf_count, dtype=torch.float32)
    environment.index_add_(0, torch.as_tensor(edge_index_np[0]), rbf * cell_envelope[:, None])
    x_invariant = torch.cat([base, environment], dim=-1)
    frac_tensor = torch.as_tensor(fractional, dtype=torch.float32)

    data = Data(
        x_invariant=x_invariant,
        pos_fractional=frac_tensor,
        pos_periodic=periodic_position_encoding(frac_tensor, CFG.position_frequencies),
        atomic_number=torch.full((atom_count,), int(properties["Z"]), dtype=torch.long),
        edge_index=torch.as_tensor(edge_index_np, dtype=torch.long),
        edge_attr=edge_attr,
        edge_direction=edge_direction,
        edge_shift=torch.as_tensor(shifts_np, dtype=torch.long),
        cell=torch.as_tensor(cell_angstrom, dtype=torch.float32).unsqueeze(0),
        global_features=torch.as_tensor(cell_invariants(cell_angstrom, atom_count)).unsqueeze(0),
        electron_count=torch.tensor([float(record["electron_count"])], dtype=torch.float32),
        volume_bohr3=torch.tensor([float(record["volume_bohr3"])], dtype=torch.float32),
        atom_count=torch.tensor([atom_count], dtype=torch.long),
        calc_id=str(record["calc_id"]),
        element=element,
        family=str(record.get("family", "unknown")),
        family_source=str(record.get("family_source", "unknown")),
        num_nodes=atom_count,
    )
    if include_targets:
        rho_scaled = torch.as_tensor(record["rho_scaled"], dtype=torch.float32)
        potential_scaled = torch.as_tensor(
            record["potential_relative"] / normalization.potential_scale_ry, dtype=torch.float32
        )
        field_input = torch.stack([
            torch.log(rho_scaled.clamp_min(CFG.density_floor)), potential_scaled
        ], dim=0).unsqueeze(0)

        cohesive_ev_per_atom = normalization.physical_cohesive_energy(
            element, float(record["final_energy_ev"]), atom_count
        )

        cohesive_normalized = normalization.encode_cohesive_energy(
            element,
            float(record["final_energy_ev"]),
            atom_count,
        )

        assert math.isclose(normalization.decode_cohesive_energy(cohesive_normalized), cohesive_ev_per_atom, abs_tol=1e-8), "Cohesive energy: inconsistent normalization"
        data.field_input = field_input
        data.rho_target = rho_scaled[None, None]
        data.potential_target = potential_scaled[None, None]

        data.cohesive_energy_target = torch.tensor(
            [cohesive_normalized],
            dtype=torch.float32,
        )

        data.cohesive_energy_ev_per_atom = torch.tensor(
            [cohesive_ev_per_atom],
            dtype=torch.float64,
        )

        data.native_grid_shape = torch.tensor([record["native_grid_shape"]], dtype=torch.long)
    return data


class PeriodicDFTDataset(torch.utils.data.Dataset):
    def __init__(self, frame: pd.DataFrame, normalization: Normalization, include_targets: bool = True):
        self.rows = frame.reset_index(drop=True)
        self.normalization = normalization
        self.include_targets = include_targets

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, index: int) -> Data:
        row = self.rows.iloc[index].to_dict()
        record = cached_raw_record(row) if self.include_targets else load_structural_record(row)
        return graph_from_record(record, self.normalization, self.include_targets)


class BalancedElementBatchSampler(torch.utils.data.Sampler):
    """Balanced batches, complete majority coverage; minority cycles uniformly.

    Shares the DataLoader generator, whose state is saved in epoch checkpoints.
    Balances elements, never energy bins or validation/holdout records.
    """
    def __init__(self, elements, batch_size, generator):
        labels = np.asarray(elements, dtype=str)
        if set(labels) != set(CFG.pretrain_elements):
            raise ValueError('Balancing requires only Al and Fe, both present')
        if batch_size < 2 or batch_size % 2:
            raise ValueError('batch_size must be even and >=2')
        self.pools = [torch.as_tensor(np.flatnonzero(labels == e), dtype=torch.long)
                      for e in CFG.pretrain_elements]
        self.half = batch_size // 2
        self.generator = generator
        self.batch_count = math.ceil(max(map(len, self.pools)) / self.half)

    def __len__(self):
        return self.batch_count

    def __iter__(self):
        count = self.batch_count * self.half
        streams = []
        for pool in self.pools:
            pieces, total = [], 0
            while total < count:
                pieces.append(pool[torch.randperm(len(pool), generator=self.generator)])
                total += len(pool)
            streams.append(torch.cat(pieces)[:count])
        for start in range(0, count, self.half):
            batch = torch.cat([s[start:start+self.half] for s in streams])
            yield batch[torch.randperm(len(batch), generator=self.generator)].tolist()


def make_loader(
    frame: pd.DataFrame,
    normalization: Normalization,
    shuffle: bool,
    include_targets: bool = True,
    balance_elements: bool = False,
) -> DataLoader:
    dataset = PeriodicDFTDataset(frame, normalization, include_targets)
    generator = torch.Generator().manual_seed(CFG.seed)
    if balance_elements:
        if not shuffle or not include_targets:
            raise ValueError('Balancing is allowed only in supervised training')
        sampler = BalancedElementBatchSampler(dataset.rows.element, CFG.preset.batch_size, generator)
        return DataLoader(dataset, batch_sampler=sampler,
            num_workers=CFG.preset.num_workers, pin_memory=DEVICE.type == 'cuda',
            generator=generator, persistent_workers=False, worker_init_fn=seed_loader_worker)
    return DataLoader(
        dataset,
        batch_size=min(CFG.preset.batch_size, max(1, len(dataset))),
        shuffle=shuffle,
        num_workers=CFG.preset.num_workers,
        pin_memory=DEVICE.type == "cuda",
        generator=generator,
        persistent_workers=False,
        worker_init_fn=seed_loader_worker,
    )


## Fully available curriculum

CONTROL and PHYSICAL share IDs, seed, initialization, batches, and budget.
PHYSICAL progressively orders batches by difficulty without excluding IDs.
The plan and loader use curriculum_epoch_spec and simulate coverage before training.


In [13]:
import hashlib
import math
from dataclasses import dataclass

@dataclass(frozen=True)
class DataCurriculumConfig:
    enabled: bool = True
    minimum_full_data_epochs: int = 1
    energy_bins: int = 10
    mode: str = CFG.data_curriculum_mode
    version: str = CURRICULUM_VERSION

DATA_CURRICULUM_CFG=DataCurriculumConfig()
STAGE_DATA_FRACTIONS={s:(1.,1.,1.) for s in ('autoencoder','D_latent','E_density',
    'E_density_potential','E_full','F_periodic_ramp','P_cohesive_refinement',
    'G_cohesive_calibration','G_adapter_warmup','G_joint_finetune','G_retention_repair')}
DATA_CURRICULUM_REGISTRY={}
DATA_CURRICULUM_MANIFEST_CACHE={}
DATA_CURRICULUM_EXPOSURES={}
CURRICULUM_PROGRESS={'completed_epochs':0,'planned_epochs':1,'last_progress':0.0}
SAMPLE_STAGE_EXPOSURE={}


def _curriculum_key_frame(frame):
    result=frame.copy()
    result['calc_id']=result.calc_id.astype(str); result['element']=result.element.astype(str)
    result['curriculum_key']=result.element+'::'+result.calc_id
    return result


def build_data_curriculum_manifest(frame, normalization, scope):
    unique=_curriculum_key_frame(frame).drop_duplicates('curriculum_key').reset_index(drop=True)
    work=structural_descriptor_frame(unique)
    columns=['curriculum_key','calc_id','element','family','structural_group','group_source',
             'physical_difficulty','energy_relative','volume_per_atom','volumetric_deformation',
             'cell_deformation','shear','anisotropy','displacement_rms','displacement_max',
             'coordination','nearest_distance']
    manifest=work[columns].sort_values('curriculum_key').reset_index(drop=True)
    manifest.attrs.update(manifest_hash=hashlib.sha256((CURRICULUM_VERSION+manifest.to_csv(index=False)).encode()).hexdigest(),scope=scope)
    return manifest


def register_data_curriculum_stage(stage,frame,normalization,*,scope,balance_elements=False,focus='all'):
    if stage not in STAGE_DATA_FRACTIONS: raise ValueError(stage)
    frame=frame.drop_duplicates(['element','calc_id']).reset_index(drop=True)
    if scope not in DATA_CURRICULUM_MANIFEST_CACHE:
        manifest=build_data_curriculum_manifest(frame,normalization,scope)
        DATA_CURRICULUM_MANIFEST_CACHE[scope]=manifest
        manifest.to_csv(CFG.run_dir/f'data_curriculum_manifest_{scope}.csv',index=False)
    manifest=DATA_CURRICULUM_MANIFEST_CACHE[scope]
    if not set(_curriculum_key_frame(frame).curriculum_key)<=set(manifest.curriculum_key):
        raise AssertionError('IDs outside the curriculum manifest')
    DATA_CURRICULUM_REGISTRY[stage]=dict(frame=frame,normalization=normalization,scope=scope,
        manifest=manifest,manifest_hash=manifest.attrs['manifest_hash'],balance_elements=balance_elements,focus=focus)


class ProgressiveCompositionBatchSampler(torch.utils.data.Sampler):
    """Weighted ordering without replacement; every ID is visited every epoch.

    Later batches carry more hard cases early in the curriculum. Balance cycles
    the minority only; total draws are identical across CONTROL and PHYSICAL.
    """
    def __init__(self, elements, difficulty, batch_size, generator, priority, balance):
        self.elements=np.asarray(elements); self.difficulty=np.asarray(difficulty,dtype=float)
        self.batch_size=int(batch_size); self.generator=generator; self.priority=float(priority)
        self.balance=bool(balance)
        if balance and (set(self.elements)!=set(CFG.pretrain_elements) or batch_size<2 or batch_size%2):
            raise ValueError('Al/Fe batch must be even and contain both elements')
        self.pools=[np.flatnonzero(self.elements==e) for e in CFG.pretrain_elements] if balance else [np.arange(len(elements))]
        self.quota=self.batch_size//2 if balance else self.batch_size
        self.batch_count=math.ceil(max(map(len,self.pools))/self.quota)

    def __len__(self): return self.batch_count

    def _ordered(self,pool):
        u=torch.rand(len(pool),generator=self.generator).numpy().clip(1e-12,1.)
        weight=np.exp(-self.priority*self.difficulty[pool])
        return pool[np.argsort(-np.log(u)/weight,kind='stable')]

    def __iter__(self):
        count=self.batch_count*self.quota
        streams=[]
        for pool in self.pools:
            ordered=self._ordered(pool).tolist()
            while len(ordered)<count: ordered.extend(self._ordered(pool).tolist())
            streams.append(ordered[:count] if self.balance else ordered[:len(pool)])
        for start in range(0,count,self.quota):
            batch=[i for stream in streams for i in stream[start:start+self.quota]]
            if batch: yield [batch[i] for i in torch.randperm(len(batch),generator=self.generator).tolist()]


def _curriculum_epoch_seed(stage,epoch):
    digest=hashlib.sha256(f'{CFG.seed}|{stage}|{epoch}|batch-curriculum'.encode()).digest()
    return int.from_bytes(digest[:8],'little')%(2**63-1)


def curriculum_epoch_spec(stage,epoch,epochs,fallback_loader):
    spec=DATA_CURRICULUM_REGISTRY.get(stage)
    size=len(spec['frame']) if spec else len(fallback_loader.dataset)
    batch=CFG.preset.batch_size
    if stage=='P_cohesive_refinement':
        batch=CFG.refinement_batch_size if DEVICE.type=='cuda' else CFG.preset.batch_size
    balanced=bool(spec and spec['balance_elements'])
    batches=math.ceil(max(spec['frame'].element.value_counts())/(batch//2)) if balanced else math.ceil(size/batch)
    stage_offset=CURRICULUM_PROGRESS['completed_epochs']-int(epoch)
    progress=min(1.,max(0.,(stage_offset+int(epoch))/max(CURRICULUM_PROGRESS['planned_epochs'],1)))
    # Complete distribution after latent learning; Ni adapters may progressively order
    # the same IDs, while calibration/joint/repair always use the full distribution.
    ordered=stage in ('autoencoder','D_latent','G_adapter_warmup')
    priority=4.*(1-progress) if ordered and DATA_CURRICULUM_CFG.mode=='physical' else 0.
    return dict(active_fraction=1.,available_fraction=1.,curriculum_tier='full_pool',active_records=size,
        active_unique_records=size,new_unique_records=size if epoch==0 else 0,
        full_data_epochs_completed=epoch+1,checkpoint_eligible=True,
        difficulty_progress=progress,sampling_priority=priority,mean_sampling_weight=1.,
        manifest_hash=spec['manifest_hash'] if spec else 'disabled',scope=spec['scope'] if spec else 'fallback',
        batches=batches,optimizer_updates=math.ceil(batches/max(CFG.accumulation_steps,1)),
        batch_size=batch,refinement_batch_fallback='CPU_memory_conservative' if stage=='P_cohesive_refinement' and DEVICE.type!='cuda' else '',
        loader_seed=_curriculum_epoch_seed(stage,epoch))


def data_curriculum_loader(stage,epoch,epochs,fallback_loader):
    info=curriculum_epoch_spec(stage,epoch,epochs,fallback_loader)
    spec=DATA_CURRICULUM_REGISTRY.get(stage)
    if spec is None: return fallback_loader,info
    frame=spec['frame']; manifest=spec['manifest'].set_index('curriculum_key')
    difficulty=manifest.loc[_curriculum_key_frame(frame).curriculum_key,'physical_difficulty'].to_numpy()
    generator=torch.Generator().manual_seed(info['loader_seed'])
    sampler=ProgressiveCompositionBatchSampler(frame.element,difficulty,info['batch_size'],generator,
        info['sampling_priority'],spec['balance_elements'])
    loader=DataLoader(PeriodicDFTDataset(frame,spec['normalization']),batch_sampler=sampler,
        num_workers=CFG.preset.num_workers,pin_memory=DEVICE.type=='cuda',generator=generator,
        persistent_workers=False,worker_init_fn=seed_loader_worker)
    if len(loader)!=info['batches']: raise AssertionError('Plan/loader mismatch')
    return loader,info


def data_curriculum_plan(stage,epochs,fallback_loader):
    base=CURRICULUM_PROGRESS['completed_epochs']; plan=[]
    try:
        for e in range(epochs):
            CURRICULUM_PROGRESS['completed_epochs']=base+e
            plan.append(curriculum_epoch_spec(stage,e,epochs,fallback_loader))
    finally: CURRICULUM_PROGRESS['completed_epochs']=base
    return plan


def validate_data_curriculum_plan(stage,plan):
    if not plan or any(r['available_fraction']!=1. or r['active_fraction']!=1. for r in plan):
        raise AssertionError('Incomplete curriculum availability')
    if len({r['active_unique_records'] for r in plan})!=1: raise AssertionError('Curriculum IDs changed')
    spec=DATA_CURRICULUM_REGISTRY.get(stage)
    if spec:
        dummy=type('Fallback',(),{'dataset':spec['frame']})()
        for epoch,row in enumerate(plan):
            old=CURRICULUM_PROGRESS['completed_epochs']
            CURRICULUM_PROGRESS['completed_epochs']=int(round(row['difficulty_progress']*CURRICULUM_PROGRESS['planned_epochs']))
            try: loader,actual=data_curriculum_loader(stage,epoch,len(plan),dummy)
            finally: CURRICULUM_PROGRESS['completed_epochs']=old
            drawn=[i for b in loader.batch_sampler for i in b]
            if set(drawn)!=set(range(len(spec['frame']))): raise AssertionError('Samples without exposure in the plan')
            if len(loader)!=row['batches']: raise AssertionError('Mismatched batch budget')
    return True


def _sample_keys_from_batch(batch):
    return [f'{e}::{i}' for e,i in zip(map(str,batch.element),map(str,batch.calc_id))]


def record_sample_batch_exposure(stage,epoch,batch):
    keys=_sample_keys_from_batch(batch)
    for key in keys:
        state=SAMPLE_STAGE_EXPOSURE.setdefault((stage,key),dict(epochs_exposed=0,batches_seen=0,
            draws=0,optimizer_updates=0,last_epoch=None,first_exposure_epoch=int(epoch),weight_sum=0.))
        if state['first_exposure_epoch'] is None: state['first_exposure_epoch']=int(epoch)
        state['draws']+=1; state['weight_sum']+=1.
    for key in set(keys):
        state=SAMPLE_STAGE_EXPOSURE[(stage,key)]; state['batches_seen']+=1
        if state['last_epoch']!=int(epoch): state['epochs_exposed']+=1; state['last_epoch']=int(epoch)
    return keys


def record_sample_optimizer_update(stage,keys):
    for key in set(keys):
        state=SAMPLE_STAGE_EXPOSURE.setdefault((stage,key),dict(epochs_exposed=0,batches_seen=0,
            draws=0,optimizer_updates=0,last_epoch=None,first_exposure_epoch=None,weight_sum=0.))
        state['optimizer_updates']+=1


def flush_sample_stage_exposure():
    rows=[]
    metadata_by_stage={stage:spec['manifest'].set_index('curriculum_key')[['structural_group','physical_difficulty']].to_dict('index')
                       for stage,spec in DATA_CURRICULUM_REGISTRY.items()}
    for (stage,key),values in sorted(SAMPLE_STAGE_EXPOSURE.items()):
        element,calc_id=key.split('::',1)
        metadata=metadata_by_stage.get(stage,{}).get(key,{})
        rows.append(dict(stage=stage,curriculum_key=key,element=element,calc_id=calc_id,**metadata,
            **{k:v for k,v in values.items() if k not in ('last_epoch','weight_sum')},
            mean_weight=values['weight_sum']/max(values['draws'],1),
            oversampling_frequency=values['draws']/max(values['epochs_exposed'],1)))
    _atomic_write_dataframe_csv(pd.DataFrame(rows),CFG.run_dir/'sample_stage_exposure.csv')


def record_data_curriculum_epoch(stage,epoch,info,active_loader):
    spec=DATA_CURRICULUM_REGISTRY.get(stage)
    if spec:
        for key in spec['manifest'].curriculum_key:
            SAMPLE_STAGE_EXPOSURE.setdefault((stage,key),dict(epochs_exposed=0,batches_seen=0,
                draws=0,optimizer_updates=0,last_epoch=None,first_exposure_epoch=None,weight_sum=0.))
    row=dict(stage=stage,epoch=epoch,**info)
    path=CFG.run_dir/'data_curriculum_history.csv'; old=pd.read_csv(path) if path.exists() else pd.DataFrame()
    pd.concat([old,pd.DataFrame([row])],ignore_index=True).drop_duplicates(['stage','epoch'],keep='last').to_csv(path,index=False)
    return row


def data_curriculum_contract():
    return dict(version=CURRICULUM_VERSION,mode=DATA_CURRICULUM_CFG.mode,availability='all_allowed_ids_every_epoch',
        composition='weighted_permutation_without_replacement_then_uniform_full_distribution',
        manifest_hashes={k:v.attrs['manifest_hash'] for k,v in DATA_CURRICULUM_MANIFEST_CACHE.items()},
        validation_and_holdout_curriculum=False,architecture_changed=False)


def sample_coverage(stage,element):
    spec=DATA_CURRICULUM_REGISTRY.get(stage)
    if spec is None: return 0.
    keys=set(spec['manifest'].loc[spec['manifest'].element.eq(element),'curriculum_key'])
    covered={k for (s,k),v in SAMPLE_STAGE_EXPOSURE.items() if s==stage and v['optimizer_updates']>0}
    return len(keys&covered)/max(len(keys),1)


def hard_exposure_gate(stage):
    spec=DATA_CURRICULUM_REGISTRY.get(stage)
    if spec is None: return True,[]
    cutoff=float(spec['manifest'].physical_difficulty.quantile(.80))
    hard=set(spec['manifest'].loc[spec['manifest'].physical_difficulty.ge(cutoff),'curriculum_key'])
    missing=[k for k in sorted(hard) if SAMPLE_STAGE_EXPOSURE.get((stage,k),{}).get('optimizer_updates',0)<CFG.hard_min_effective_updates]
    return not missing,missing


In [14]:
# Policy and exposure are consolidated in the previous cell; no active redefinition.


In [15]:
# =============================================================================
# STEP 12 — KAN layers and feed-forward blocks
# Purpose: Implements Chebyshev/Legendre/Hermite bases and factorized KAN.
# Inputs: Feature tensors and KAN options in CFG.
# Outputs: Layers reused in message, update, predictor, decoder, and energy.
# Reproducibility: run after all preceding cells; do not change the seed during the run.
# =============================================================================
def polynomial_basis_values(z: torch.Tensor, degree: int, basis: str) -> torch.Tensor:
    if degree < 1:
        raise ValueError("degree must be positive")
    values = [torch.ones_like(z), z]
    for order in range(2, degree + 1):
        if basis == "chebyshev":
            next_value = 2 * z * values[-1] - values[-2]
        elif basis == "legendre":
            next_value = ((2 * order - 1) * z * values[-1] - (order - 1) * values[-2]) / order
        elif basis == "hermite":
            next_value = (
                z * values[-1] / math.sqrt(order)
                - math.sqrt((order - 1) / order) * values[-2]
            )
        else:
            raise ValueError(f"Unimplemented basis: {basis}")
        values.append(next_value)
    return torch.stack(values[1:], dim=-1)


class BasisTransform(nn.Module):
    """Same domain transform used by the KANs in the original notebook."""
    def __init__(self, width: int, basis: str, momentum: float = 0.05):
        super().__init__()
        self.basis = basis
        self.momentum = momentum
        self.register_buffer("running_mean", torch.zeros(width))
        self.register_buffer("running_var", torch.ones(width))
        self.register_buffer("updates", torch.tensor(0, dtype=torch.long))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.basis in ("chebyshev", "legendre"):
            return torch.tanh(x)
        if self.basis != "hermite":
            raise ValueError(self.basis)
        if self.training and x.numel() > 0:
            dimensions = tuple(range(x.ndim - 1))
            mean = x.detach().float().mean(dimensions)
            variance = x.detach().float().var(dimensions, unbiased=False).clamp_min(1e-6)
            self.running_mean.mul_(1 - self.momentum).add_(mean, alpha=self.momentum)
            self.running_var.mul_(1 - self.momentum).add_(variance, alpha=self.momentum)
            self.updates.add_(1)
        return ((x - self.running_mean) / torch.sqrt(self.running_var + 1e-6)).clamp(-3, 3)


class FactorizedBasisKANLinear(nn.Module):
    """Approximates c[o,i,d] with low-rank factors to reduce parameters."""
    def __init__(self, in_features: int, out_features: int, degree: int, rank: int, basis: str):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.degree = degree
        self.rank = rank
        self.basis = basis
        self.transform = BasisTransform(in_features, basis)
        self.base = nn.Linear(in_features, out_features)
        self.output_factor = nn.Parameter(torch.empty(out_features, rank))
        self.basis_factor = nn.Parameter(torch.empty(rank, in_features, degree))
        nn.init.normal_(self.output_factor, std=0.01)
        nn.init.normal_(self.basis_factor, std=0.01)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        transformed = self.transform(x)
        basis = polynomial_basis_values(transformed, self.degree, self.basis)
        compressed = torch.einsum("...id,rid->...r", basis, self.basis_factor)
        return self.base(F.silu(x)) + compressed @ self.output_factor.T

    def dense_coefficients(self) -> torch.Tensor:
        return torch.einsum("or,rid->oid", self.output_factor, self.basis_factor)


def make_linear(in_features: int, out_features: int, use_kan: bool, placement: str) -> nn.Module:
    if use_kan and placement in CFG.kan_placements:
        return FactorizedBasisKANLinear(
            in_features, out_features, CFG.kan_degree, CFG.kan_rank, CFG.kan_basis
        )
    return nn.Linear(in_features, out_features)


class FeedForwardBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int, use_kan: bool, placement: str):
        super().__init__()
        self.first = make_linear(in_features, out_features, use_kan, placement)
        self.norm = nn.LayerNorm(out_features)
        self.second = make_linear(out_features, out_features, use_kan, placement)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.second(F.silu(self.norm(self.first(x))))


In [16]:
# =============================================================================
# STEP 13 — Structural backbones
# Purpose: Defines GNN-MLP, GNN-KAN, and Transformer with field and energy contexts.
# Inputs: PyG batch with invariant features, periodic position, edges, and cell.
# Outputs: Global field context, invariant energy context, and node embeddings.
# Reproducibility: run after all preceding cells; do not change the seed during the run.
# =============================================================================
def deterministic_segment_mean(
    values: torch.Tensor, index: torch.Tensor, dim_size: Optional[int] = None
) -> torch.Tensor:
    """Segmented mean with fixed order, compatible with deterministic CUDA."""
    if dim_size is None:
        dim_size = int(index.max()) + 1 if index.numel() else 0
    order = torch.argsort(index, stable=True)
    sorted_index = index[order]
    unique, counts = torch.unique_consecutive(sorted_index, return_counts=True)
    lengths = torch.zeros(dim_size, dtype=torch.long, device=index.device)
    lengths[unique] = counts
    summed = torch.segment_reduce(values[order], "sum", lengths=lengths)
    denominator = lengths.clamp_min(1).to(values.dtype)
    return summed / denominator.reshape((-1,) + (1,) * (values.ndim - 1))


class PeriodicMessagePassing(MessagePassing):
    """PyG version of the original OperatorGraphBlock, including global context."""
    def __init__(
        self, hidden: int, edge_features: int, use_kan: bool,
        use_global_context: bool = True,
    ):
        super().__init__(aggr="mean", node_dim=0)
        self.use_global_context = use_global_context
        self.message_network = FeedForwardBlock(
            3 * hidden + edge_features, hidden, use_kan, "message"
        )
        self.update_network = FeedForwardBlock(3 * hidden, hidden, use_kan, "update")
        self.norm = nn.LayerNorm(hidden)
        self.gate = nn.Parameter(torch.tensor(-1.5))

    def forward(
        self, x: torch.Tensor, edge_index: torch.Tensor,
        edge_attr: torch.Tensor, node_batch: torch.Tensor,
    ) -> torch.Tensor:
        normalized = self.norm(x)
        global_context = deterministic_segment_mean(normalized, node_batch)
        if not self.use_global_context:
            global_context = torch.zeros_like(global_context)
        global_nodes = global_context[node_batch]
        aggregate = self.propagate(
            edge_index, x=normalized, edge_attr=edge_attr, global_node=global_nodes
        )
        update = self.update_network(
            torch.cat([normalized, aggregate, global_nodes], dim=-1)
        )
        return x + torch.sigmoid(self.gate) * update

    def aggregate(
        self, inputs: torch.Tensor, index: torch.Tensor,
        ptr: Optional[torch.Tensor] = None, dim_size: Optional[int] = None,
    ) -> torch.Tensor:
        del ptr
        return deterministic_segment_mean(inputs, index, dim_size)

    def message(
        self, x_i: torch.Tensor, x_j: torch.Tensor,
        edge_attr: torch.Tensor, global_node_i: torch.Tensor,
    ) -> torch.Tensor:
        return self.message_network(
            torch.cat([x_i, x_j - x_i, global_node_i, edge_attr], dim=-1)
        )

class NiEmbeddingCorrection(nn.Module):
    """Zero-initialized, trainable correction for Ni only; not a new random embedding."""
    def __init__(self, hidden):
        super().__init__()
        self.delta = nn.Parameter(torch.zeros(hidden))

    def forward(self, atomic_number):
        ni_z = ATOMIC_PROPERTIES[CFG.transfer_element]["Z"]
        return atomic_number.eq(int(ni_z)).unsqueeze(-1).to(self.delta.dtype) * self.delta


class GraphContextEncoder(nn.Module):
    def __init__(self, node_features: int, global_features: int, edge_features: int, hidden: int, use_kan: bool):
        super().__init__()
        self.ni_embedding = NiEmbeddingCorrection(hidden)
        self.node_stem = nn.Linear(node_features, hidden)
        self.blocks = nn.ModuleList([
            PeriodicMessagePassing(
                hidden, edge_features, use_kan, CFG.use_global_context
            ) for _ in range(CFG.graph_layers)
        ])

        self.field_readout = FeedForwardBlock(
            hidden + global_features,
            hidden,
            use_kan,
            "update",
        )
        self.final_norm = nn.LayerNorm(hidden)

    def forward(
        self,
        data: Batch,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        h = (
            self.node_stem(data.x_invariant)
            + self.ni_embedding(data.atomic_number)
        )

        for block in self.blocks:
            h = block(h, data.edge_index, data.edge_attr, data.batch)

        field_nodes = self.final_norm(h)
        field_pool = deterministic_segment_mean(field_nodes, data.batch)

        global_features = data.global_features.reshape(
            field_pool.shape[0],
            -1,
        )

        field_context = self.field_readout(
            torch.cat([field_pool, global_features], dim=-1)
        )

        return field_context, field_nodes

class TransformerContextEncoder(nn.Module):
    def __init__(self, node_features: int, global_features: int, hidden: int):
        super().__init__()
        self.ni_embedding = NiEmbeddingCorrection(hidden)
        self.node_stem = nn.Linear(node_features, hidden)
        self.cell_token = nn.Linear(global_features, hidden)
        layer = nn.TransformerEncoderLayer(
            hidden, CFG.attention_heads, 4 * hidden, CFG.dropout,
            batch_first=True, norm_first=True, activation="gelu"
        )
        # norm_first=True disables the nested-tensor path; stating this prevents
        # the warning emitted by PyTorch 2.12 without changing computation.
        try:
            self.transformer = nn.TransformerEncoder(
                layer, CFG.transformer_layers, enable_nested_tensor=False
            )
        except TypeError:  # compatibility with older PyTorch versions
            self.transformer = nn.TransformerEncoder(layer, CFG.transformer_layers)
        self.norm = nn.LayerNorm(hidden)

    def _encode(
        self, atom_tokens: torch.Tensor, atom_mask: torch.Tensor, cell_token: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        tokens = torch.cat([cell_token[:, None], atom_tokens], dim=1)
        padding = torch.cat([
            torch.zeros((atom_mask.shape[0], 1), dtype=torch.bool, device=atom_mask.device),
            ~atom_mask,
        ], dim=1)
        encoded = self.transformer(tokens, src_key_padding_mask=padding)
        encoded = self.norm(encoded)
        return encoded[:, 1:][atom_mask], encoded[:, 0]

    def forward(
        self,
        data: Batch,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        invariant_nodes = (
            self.node_stem(data.x_invariant)
            + self.ni_embedding(data.atomic_number)
        )

        dense_invariant, mask = to_dense_batch(
            invariant_nodes,
            data.batch,
        )

        batch_size = dense_invariant.shape[0]
        global_features = data.global_features.reshape(batch_size, -1)
        cell = self.cell_token(global_features)

        field_nodes, field_context = self._encode(
            dense_invariant,
            mask,
            cell,
        )

        return field_context, field_nodes

def make_context_encoder(sample: Data) -> nn.Module:
    node_features = sample.x_invariant.shape[-1]
    global_features = sample.global_features.shape[-1]
    edge_features = sample.edge_attr.shape[-1]
    if CFG.backbone == "transformer":
        return TransformerContextEncoder(node_features, global_features, CFG.hidden)
    if CFG.backbone == "pyg_gnn_mlp":
        return GraphContextEncoder(node_features, global_features, edge_features, CFG.hidden, use_kan=False)
    if CFG.backbone == "pyg_gnn_kan":
        return GraphContextEncoder(node_features, global_features, edge_features, CFG.hidden, use_kan=True)
    raise ValueError(f"Invalid backbone: {CFG.backbone}")


## Learned shared space
`z_shared[B,2,2M]` represents both fields. A real 2×2 mixture per mode,
conditioned on invariants, allows cohesive energy to train the encoder without breaking
phase law. The head uses `z_shared`, rather than raw coefficients disconnected from the encoder.
DC modes are controlled and are not used as an unconstrained scalar path for cohesive energy.

In [17]:
def invariant_shared_latent_view(z, spectral_rms):
    if z.ndim != 3 or z.shape[1] != 2 or z.shape[-1] % 2:
        raise ValueError('Expected latent [B,2,2M]')
    with torch.autocast(device_type=z.device.type, enabled=False):
        # A shared real scale for Re/Im preserves phase.
        rms = spectral_rms.float().reshape(2,-1,2)
        common = rms.square().mean(-1).sqrt().clamp_min(1e-6)
        pairs = z.float().reshape(len(z),2,-1,2) * rms[None] / common[None,:,:,None]
        # DC: excluded from the head, avoiding a channel unaligned by non-DC JEPA.
        mask = torch.ones(pairs.shape[2], device=z.device); mask[0] = 0
        pairs = pairs * mask[None,None,:,None]
        r, v = pairs[:,0], pairs[:,1]
        ar = r.square().sum(-1).add(1e-8).sqrt()
        av = v.square().sum(-1).add(1e-8).sqrt()
        den = 1.0 + ar*av  # damped cross-spectrum for near-zero modes
        real = (r[...,0]*v[...,0] + r[...,1]*v[...,1])/den
        imag = (r[...,1]*v[...,0] - r[...,0]*v[...,1])/den
        return torch.cat([torch.log1p(ar),torch.log1p(av),real,imag],-1)

class PhaseEquivariantResidual(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.mode_count = latent_dim//2
        self.register_buffer('spectral_rms', spectral_rms_tensor())
        self.net = nn.Sequential(nn.Linear(4,32), nn.SiLU(), nn.Linear(32,4))
        nn.init.zeros_(self.net[-1].weight); nn.init.zeros_(self.net[-1].bias)

    def forward(self, z):
        with torch.autocast(device_type=z.device.type, enabled=False):
            rms = self.spectral_rms.float().reshape(2,-1,2)
            common = rms.square().mean(-1).sqrt().clamp_min(1e-6)
            physical = z.float().reshape(len(z),2,-1,2)*rms[None]
            pairs = physical/common[None,:,:,None]
            descriptor = invariant_shared_latent_view(z,self.spectral_rms)
            features = descriptor.reshape(len(z),4,self.mode_count).transpose(1,2)
            weights = torch.tanh(self.net(features)).reshape(len(z),self.mode_count,2,2)
            delta = torch.einsum('bmfg,bgmc->bfmc',weights,pairs)
            mask = torch.ones(self.mode_count,device=z.device); mask[0] = 0
            delta = delta*mask[None,None,:,None]*common[None,:,:,None]/rms.clamp_min(1e-8)[None]
            return delta.flatten(-2)


In [18]:
# =============================================================================
# STEP 14 — 3D Fourier autoencoder
# Purpose: Projects rho/V onto rFFT modes, encodes latents, and reconstructs with iFFT and physical constraints.
# Inputs: field_input normalized on the canonical grid.
# Outputs: z_online, positive/normalized reconstructed rho, and zero-mean V.
# Reproducibility: run after all preceding cells; do not change the seed during the run.
# =============================================================================
def low_frequency_rfft_indices(grid: tuple[int, int, int], count: int) -> torch.Tensor:
    """Independent low-frequency modes INCLUDING the kx=0 plane.

    Tensor axes are (z,y,x). Nyquist planes are deliberately omitted. On kx=0
    retain one of (kz,ky) and (-kz,-ky); synthesis writes its conjugate as well.
    """
    depth, height, width = grid
    candidates = []
    for iz in range(depth):
        kz = iz if iz <= (depth - 1) // 2 else iz - depth
        if depth % 2 == 0 and iz == depth // 2:
            continue
        for iy in range(height):
            ky = iy if iy <= (height - 1) // 2 else iy - height
            if height % 2 == 0 and iy == height // 2:
                continue
            for kx in range((width - 1) // 2 + 1):
                if kx == 0 and not (kz > 0 or (kz == 0 and ky >= 0)):
                    continue
                index = (iz * height + iy) * (width // 2 + 1) + kx
                key = (kz*kz + ky*ky + kx*kx, abs(kz), abs(ky), kx, kz, ky)
                candidates.append((key, index))
    candidates.sort()
    if not 1 <= count <= len(candidates):
        raise ValueError(f"Number of modes {count} outside [1,{len(candidates)}]")
    return torch.tensor([index for _, index in candidates[:count]], dtype=torch.long)


def periodic_fields_to_spectral(
    field_input: torch.Tensor, spectral_indices: torch.Tensor
) -> torch.Tensor:
    """Converts [B,F,D,H,W] to real/imaginary pairs [B,F,2M]."""
    if field_input.ndim != 5:
        raise ValueError(f"field_input must have rank 5; received {field_input.shape}")
    device_type = field_input.device.type if field_input.device.type in ("cpu", "cuda") else "cpu"
    with torch.autocast(device_type=device_type, enabled=False):
        spectrum = torch.fft.rfftn(
            field_input.float(), dim=(-3, -2, -1), norm="ortho"
        ).flatten(start_dim=-3)
        selected = spectrum.index_select(-1, spectral_indices)
        coefficients = torch.stack((selected.real, selected.imag), dim=-1).flatten(-2)
    return coefficients.to(field_input.dtype)


def spectral_to_periodic_raw(
    coefficients: torch.Tensor,
    spectral_indices: torch.Tensor,
    grid: tuple[int, int, int],
) -> torch.Tensor:
    """Reconstructs real periodic fields using only retained modes."""
    if coefficients.ndim != 3 or coefficients.shape[-1] % 2:
        raise ValueError(f"Coefficients must have shape [B,F,2M]; received {coefficients.shape}")
    batch_size, field_count, latent_dim = coefficients.shape
    mode_count = latent_dim // 2
    if mode_count != spectral_indices.numel():
        raise ValueError("Incompatible number of coefficients and spectral indices")
    device_type = coefficients.device.type if coefficients.device.type in ("cpu", "cuda") else "cpu"
    with torch.autocast(device_type=device_type, enabled=False):
        pairs = coefficients.float().reshape(batch_size, field_count, mode_count, 2)
        complex_coefficients = torch.complex(pairs[..., 0], pairs[..., 1])
        # The imaginary DC component does not represent a real degree of freedom.
        dc_mask = spectral_indices.eq(0)[None, None]
        complex_coefficients = torch.where(
            dc_mask, torch.complex(complex_coefficients.real, torch.zeros_like(complex_coefficients.real)),
            complex_coefficients,
        )
        depth, height, width = grid
        flattened_size = depth * height * (width // 2 + 1)
        flattened = torch.zeros(
            batch_size, field_count, flattened_size,
            dtype=complex_coefficients.dtype, device=coefficients.device,
        )
        flattened = flattened.index_copy(-1, spectral_indices, complex_coefficients)
        # irfftn does not infer the missing in-plane Hermitian partner for us.
        half_width = width // 2 + 1
        iz = spectral_indices // (height * half_width)
        iy = (spectral_indices // half_width) % height
        ix = spectral_indices % half_width
        plane_positions = torch.nonzero((ix == 0) & (spectral_indices != 0)).flatten()
        mirror = (((-iz) % depth) * height + ((-iy) % height)) * half_width
        flattened = flattened.index_copy(
            -1, mirror.index_select(0, plane_positions),
            complex_coefficients.index_select(-1, plane_positions).conj(),
        )
        spectrum = flattened.reshape(batch_size, field_count, depth, height, width // 2 + 1)
        raw = torch.fft.irfftn(spectrum, s=grid, dim=(-3, -2, -1), norm="ortho")
    return raw


def complex_pair_rms(square_mean: torch.Tensor, floor: float, relative_floor: float) -> torch.Tensor:
    """One RMS per complex pair, invariant under a phase rotation."""
    pair_power = square_mean.reshape(*square_mean.shape[:-1], -1, 2).mean(-1)
    rms = torch.sqrt(pair_power.clamp_min(0))
    reference = rms[..., 1:].median(dim=-1, keepdim=True).values
    minimum = (relative_floor * reference).clamp_min(floor)
    rms = torch.maximum(rms, minimum)
    return rms.repeat_interleave(2, dim=-1)


def fit_spectral_rms_statistics(frame, normalization, sample_count):
    """Train-only statistics with equal element contribution, no test/Ni leakage."""
    sample = deterministic_sample(frame, min(int(sample_count), len(frame)), CFG.seed + 41)
    indices = low_frequency_rfft_indices(CFG.canonical_grid, CFG.field_spectral_complex_modes)
    element_power, fitted_ids = [], []
    for _, group in sample.groupby("element"):
        total = torch.zeros(len(FIELD_NAMES), CFG.latent_dim, dtype=torch.float64)
        for _, row in group.iterrows():
            graph = graph_from_record(cached_raw_record(row.to_dict()), normalization, include_targets=True)
            values = periodic_fields_to_spectral(graph.field_input, indices)[0].double()
            total += values.square()
            fitted_ids.append(str(row.calc_id))
        element_power.append(total / len(group))
    if not element_power:
        raise ValueError("Spectral fitting requires training records")
    rms = complex_pair_rms(torch.stack(element_power).mean(0),
                           CFG.spectral_statistics_floor, CFG.spectral_relative_floor)
    return {"records": len(fitted_ids), "fitted_calc_ids": fitted_ids,
            "rms": rms.float().tolist(), "minimum_rms": float(rms.min()),
            "maximum_rms": float(rms.max()), "pair_shared_scale": True,
            "element_weighting": "equal means", "selection": "all-axis Hermitian representatives"}


def spectral_rms_tensor() -> torch.Tensor:

    values = torch.as_tensor(SPECTRAL_RMS_STATISTICS["rms"], dtype=torch.float32)

    expected = (len(FIELD_NAMES), CFG.latent_dim)

    if tuple(values.shape) != expected:

        raise RuntimeError(f"Spectral RMS {tuple(values.shape)} incompatible with {expected}")

    return values

def constrain_periodic_fields(raw: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Applies density positivity/charge and the additive potential gauge."""
    if raw.shape[1] != len(FIELD_NAMES):
        raise ValueError(f"Expected {len(FIELD_NAMES)} fields; received {raw.shape[1]}")
    log_rho = CFG.density_log_clip * torch.tanh(raw[:, :1] / CFG.density_log_clip)
    rho_scaled = torch.exp(log_rho).clamp_min(CFG.density_floor)
    rho_scaled = rho_scaled / rho_scaled.mean(
        dim=(-3, -2, -1), keepdim=True
    ).clamp_min(1e-12)
    potential_scaled = raw[:, 1:2]
    potential_scaled = potential_scaled - potential_scaled.mean(
        dim=(-3, -2, -1), keepdim=True
    )
    return rho_scaled, potential_scaled

class PeriodicFieldEncoder(nn.Module):
    """Global spectral bottleneck for density and potential."""

    def __init__(self, latent_dim: int):
        super().__init__()

        self.register_buffer(
            "spectral_indices",
            low_frequency_rfft_indices(
                CFG.canonical_grid,
                CFG.field_spectral_complex_modes,
            ),
            persistent=True,
        )

        self.residual = PhaseEquivariantResidual(latent_dim)

        self.register_buffer(
            "spectral_rms",
            spectral_rms_tensor(),
            persistent=True,
        )

    def forward_spectral(self, raw_latent: torch.Tensor) -> torch.Tensor:
        """Same learned map for field Fourier coordinates and graph-predicted coordinates."""
        return raw_latent + CFG.shared_residual_scale * self.residual(raw_latent)

    def raw_spectral_latent(
        self,
        field_input: torch.Tensor,
    ) -> torch.Tensor:
        """
        Returns the normalized spectral coefficients before the
        learned residual transform.

        Shape:
            field_input: [B, 2, D, H, W]
            raw_latent:  [B, 2, latent_dim]
        """
        coefficients = periodic_fields_to_spectral(
            field_input,
            self.spectral_indices,
        )

        spectral_rms = self.spectral_rms.to(
            device=coefficients.device,
            dtype=coefficients.dtype,
        )

        return coefficients / spectral_rms.clamp_min(1e-8)

    def forward(
        self,
        field_input: torch.Tensor,
    ) -> torch.Tensor:
        """
        Produces the latent used by the autoencoder and as the JEPA EMA target.
        """
        raw_latent = self.raw_spectral_latent(field_input)

        return self.forward_spectral(raw_latent)

class FieldwiseLinear(nn.Module):
    """Independent parameters for density and potential, not independent element models."""
    def __init__(self, in_features, out_features, identity=False):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(in_features, out_features) for _ in FIELD_NAMES])
        if identity:
            if in_features != out_features:
                raise ValueError("Identity map must be square")
            with torch.no_grad():
                for layer in self.layers:
                    layer.weight.copy_(torch.eye(in_features))
                    layer.bias.zero_()

    def forward(self, values):
        return torch.stack([layer(values[:,f]) for f,layer in enumerate(self.layers)], dim=1)


class PeriodicFieldReconstructionDecoder(nn.Module):
    """Decoder specific to the autoencoder, analogous to SolutionReconstructionDecoder."""
    def __init__(self, latent_dim: int):
        super().__init__()
        self.register_buffer(
            "spectral_indices",
            low_frequency_rfft_indices(CFG.canonical_grid, CFG.field_spectral_complex_modes),
            persistent=True,
        )
        self.head = FieldwiseLinear(latent_dim, latent_dim, identity=True)
        self.register_buffer(
            "spectral_rms", spectral_rms_tensor(), persistent=True
        )

    def forward(self, z: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        coefficients = self.head(z).float() * self.spectral_rms.float()
        raw = spectral_to_periodic_raw(
            coefficients, self.spectral_indices, CFG.canonical_grid
        )
        return constrain_periodic_fields(raw)


class PeriodicFieldAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = PeriodicFieldEncoder(CFG.latent_dim)
        self.decoder = PeriodicFieldReconstructionDecoder(CFG.latent_dim)

    def forward(self, field_input: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        z = self.encoder(field_input)
        rho, potential = self.decoder(z)
        return z, rho, potential


In [19]:
# =============================================================================
# STEP 15 — JEPA, EMA, predictor, decoder, and energy
# Purpose: Builds online/target routes, rho/V queries, predictive decoder, and energy-per-atom head.
# Inputs: Structural context and target fields during training only.
# Outputs: z_pred/z_target, rho_pred, V_pred, and normalized cohesive energy.
# Reproducibility: run after all preceding cells; do not change the seed during the run.
# =============================================================================



@torch.no_grad()
def initialize_ema_target(online: nn.Module) -> nn.Module:
    target = copy.deepcopy(online).eval()
    for parameter in target.parameters():
        parameter.requires_grad_(False)
    return target


@torch.no_grad()
def ema_update(online: nn.Module, target: nn.Module, tau: float) -> None:
    if not 0.0 <= tau <= 1.0:
        raise ValueError("EMA tau must be in [0,1]")
    online_parameters = dict(online.named_parameters())
    target_parameters = dict(target.named_parameters())
    if online_parameters.keys() != target_parameters.keys():
        raise RuntimeError("Incompatible online/target structures")
    for name, target_parameter in target_parameters.items():
        target_parameter.mul_(tau).add_(online_parameters[name].detach(), alpha=1 - tau)
    online_buffers = dict(online.named_buffers())
    for name, target_buffer in target.named_buffers():
        source = online_buffers[name]
        if torch.is_floating_point(target_buffer) and not name.endswith("spectral_rms"):
            target_buffer.mul_(tau).add_(source.detach(), alpha=1 - tau)
        else:
            target_buffer.copy_(source)


def ema_tau(step: int, total_steps: int) -> float:
    progress = min(max(int(step) / max(int(total_steps) - 1, 1), 0.0), 1.0)
    return float(CFG.ema_tau_end - (CFG.ema_tau_end - CFG.ema_tau_start)
                 * 0.5 * (1.0 + math.cos(math.pi * progress)))


class NiResidualAdapter(nn.Module):
    """Residual Ni adapter with a KAN bottleneck and zero-initialized output."""
    def __init__(self, hidden, use_kan):
        super().__init__()
        self.norm = nn.LayerNorm(hidden)
        self.down = nn.Linear(hidden, CFG.adapter_hidden)
        self.kan = FeedForwardBlock(CFG.adapter_hidden, CFG.adapter_hidden, use_kan, "predictor")
        self.up = nn.Linear(CFG.adapter_hidden, hidden)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, values):
        return self.up(F.silu(self.kan(F.silu(self.down(self.norm(values))))))


def spectral_wavevectors_xyz(indices, grid):
    """Integer fractional wavevectors, respecting the stored (z,y,x) FFT axes."""
    depth, height, width = grid
    half = width // 2 + 1
    iz = indices // (height * half)
    iy = (indices // half) % height
    ix = indices % half
    kz = torch.where(iz <= (depth - 1) // 2, iz, iz - depth)
    ky = torch.where(iy <= (height - 1) // 2, iy, iy - height)
    return torch.stack((ix, ky, kz), dim=-1).float()

def rotate_spectral_pairs(coefficients, wavevectors, fractional_shift):
    """f(r-s) has Fourier coefficient exp(-2 pi i k.s) C(k)."""
    pairs = coefficients.float().reshape(*coefficients.shape[:-1], -1, 2)
    theta = -2 * math.pi * (wavevectors.float() @ fractional_shift.float())
    c, s = theta.cos(), theta.sin()
    return torch.stack((pairs[..., 0]*c - pairs[..., 1]*s,
                        pairs[..., 0]*s + pairs[..., 1]*c), dim=-1).flatten(-2)

class FieldConditionedPredictor(nn.Module):
    """KAN amplitudes plus analytic atomic Fourier phases, without target inputs.

    Only the raw spectral output obeys an exact translation law. The subsequent
    trained AE latent lift and residual decoder are audited separately.
    """
    def __init__(self, hidden, edge_features, use_kan):
        super().__init__()
        self.field_encoder = PeriodicFieldEncoder(CFG.latent_dim)
        self.field_queries = nn.Embedding(len(FIELD_NAMES), hidden)
        self.input = FeedForwardBlock(3*hidden, hidden, use_kan, 'predictor')
        self.graph = PeriodicMessagePassing(
            hidden, edge_features + (3 if CFG.use_directional_field_edges else 0),
            use_kan, CFG.use_global_context)
        heads = 4 if hidden % 4 == 0 else 2
        self.field_attention = nn.MultiheadAttention(hidden, heads, batch_first=True)
        self.token_update = FeedForwardBlock(2*hidden, hidden, use_kan, 'predictor')
        self.norm = nn.LayerNorm(hidden)
        indices = low_frequency_rfft_indices(CFG.canonical_grid, CFG.field_spectral_complex_modes)
        self.register_buffer('wavevectors_xyz', spectral_wavevectors_xyz(indices, CFG.canonical_grid))
        self.register_buffer('spectral_indices', indices)
        self.register_buffer('spectral_rms', spectral_rms_tensor())
        self.mode_embedding = nn.Embedding(len(indices), CFG.spectral_mode_embedding_dim)
        self.mode_network = FeedForwardBlock(
            hidden + CFG.spectral_mode_embedding_dim + 4, hidden, use_kan, 'predictor')
        self.complex_head = nn.Linear(hidden, 2)  # linear readout of KAN; NOT an MLP backbone
        nn.init.normal_(self.complex_head.weight, std=0.01)
        nn.init.zeros_(self.complex_head.bias)

    def forward(self, node_context, global_context, data):
        states, tokens = [], []
        edge_attr = (torch.cat((data.edge_attr, data.edge_direction), dim=-1)
                     if CFG.use_directional_field_edges else data.edge_attr)
        for f in range(len(FIELD_NAMES)):
            query = self.field_queries.weight[f].expand_as(node_context)
            state = self.input(torch.cat((node_context, query, global_context[data.batch]), -1))
            state = self.graph(state, data.edge_index, edge_attr, data.batch)
            states.append(state)
            tokens.append(deterministic_segment_mean(state, data.batch))
        stacked, token_tensor = torch.stack(states, 1), torch.stack(tokens, 1)
        mixed, _ = self.field_attention(token_tensor, token_tensor, token_tensor, need_weights=False)
        delta = self.token_update(torch.cat((token_tensor, mixed), -1))
        updated_nodes = self.norm(stacked + delta[data.batch])
        updated_tokens = deterministic_segment_mean(updated_nodes, data.batch)
        count = len(self.wavevectors_xyz)
        outputs = []
        # Phase/cell algebra stays float32 even under mixed-precision training.
        with torch.autocast(device_type=data.pos_fractional.device.type, enabled=False):
            reciprocal = torch.linalg.inv(data.cell.float()).transpose(-1, -2)
            xyz = self.wavevectors_xyz.float()
            cartesian = torch.einsum('md,bdk->bmk', xyz, reciprocal)
            norm = cartesian.square().sum(-1).sqrt()
            direction = xyz / xyz.square().sum(-1, keepdim=True).sqrt().clamp_min(1.0)
            theta = -2 * math.pi * (data.pos_fractional.float() @ xyz.T)
        for start in range(0, count, CFG.spectral_mode_chunk_size):
            end = min(start + CFG.spectral_mode_chunk_size, count)
            m = end - start
            mode = self.mode_embedding.weight[start:end]
            mode = mode[None, None].expand(len(updated_nodes), len(FIELD_NAMES), m, -1)
            context = updated_nodes[:, :, None].expand(-1, -1, m, -1)
            features = torch.cat((
                direction[None, start:end].expand(len(updated_nodes), -1, -1),
                torch.log1p(norm[data.batch, start:end])[..., None]), -1)
            features = features[:, None].expand(-1, len(FIELD_NAMES), -1, -1)
            amplitudes = self.complex_head(self.mode_network(
                torch.cat((context, mode, features), -1))).float()
            c, s = theta[:, None, start:end].cos(), theta[:, None, start:end].sin()
            rotated = torch.stack((amplitudes[..., 0]*c - amplitudes[..., 1]*s,
                                   amplitudes[..., 0]*s + amplitudes[..., 1]*c), -1)
            outputs.append(deterministic_segment_mean(rotated, data.batch))
        pairs = torch.cat(outputs, dim=-2)
        mask = self.spectral_indices.ne(0).to(pairs.dtype)
        pairs = torch.stack((pairs[..., 0], pairs[..., 1]*mask), -1)
        # Shared pair scaling preserves phase, including imported AE checkpoints.
        rms = self.spectral_rms.float().reshape(len(FIELD_NAMES), count, 2)
        common = rms.square().mean(-1).sqrt()[None, ..., None]
        normalized = (pairs * common / rms.clamp_min(1e-8)).flatten(-2)
        return normalized, updated_tokens, updated_nodes


class SharedLatentCohesiveEnergyHead(nn.Module):
    """
    Predicts only normalized cohesive energy from the shared
    spectral latent. It receives no structural context.
    """
    def __init__(
        self,
        mode_count: int,
        hidden: int,
        use_kan: bool,
    ):
        super().__init__()

        input_dim = 4 * mode_count

        self.input = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, hidden),
            nn.SiLU(),
        )

        self.body = FeedForwardBlock(
            hidden,
            hidden,
            use_kan,
            "cohesive",
        )

        self.output = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, 1),
        )

        nn.init.normal_(self.output[-1].weight, std=0.001)
        nn.init.zeros_(self.output[-1].bias)

    def forward(self, invariant_descriptor: torch.Tensor) -> torch.Tensor:
        # KAN head and output stay FP32, not just the final scalar loss.
        with torch.autocast(device_type=invariant_descriptor.device.type, enabled=False):
            hidden = self.input(invariant_descriptor.float())
            hidden = self.body(hidden)
            return self.output(hidden).squeeze(-1)


class NiEnergyCalibration(nn.Module):
    """Two trainable scalars; initial identity, no independent energy latent."""
    def __init__(self):
        super().__init__()
        self.log_scale = nn.Parameter(torch.zeros(()))
        self.offset = nn.Parameter(torch.zeros(()))

    def forward(self, residual, ni_mask):
        return residual * torch.exp(ni_mask * self.log_scale.clamp(-1., 1.)) + ni_mask*self.offset


class NiSharedSpectralGain(nn.Module):
    """Real, phase-preserving FiLM gain on the SAME latent used by every head.

    No complex additive bias: it would break translation equivariance.
    Only Ni is changed. No density scaling after charge normalization.
    """
    def __init__(self):
        super().__init__()
        self.log_gain = nn.Parameter(torch.zeros(2))

    def forward(self, z, ni_mask):
        gain = torch.exp(CFG.ni_field_log_gain_limit*torch.tanh(self.log_gain))
        non_dc = torch.ones(z.shape[-1], device=z.device); non_dc[:2] = 0
        scale = 1 + ni_mask[:,None,None]*(gain[None,:,None]-1)*non_dc[None,None,:]
        return z*scale


class PeriodicFourierResidualDecoder(nn.Module):
    """Separate field decoders, initialized from the corresponding AE maps."""
    def __init__(self, latent_dim, hidden, use_kan):
        super().__init__()
        self.register_buffer("spectral_indices",low_frequency_rfft_indices(
            CFG.canonical_grid,CFG.field_spectral_complex_modes),persistent=True)
        self.register_buffer("spectral_rms",spectral_rms_tensor(),persistent=True)
        self.base = FieldwiseLinear(latent_dim,latent_dim,identity=True)
        self.pre = nn.ModuleList([FeedForwardBlock(latent_dim,hidden,use_kan,"decoder") for _ in FIELD_NAMES])
        self.head = FieldwiseLinear(hidden,latent_dim)
        self.residual_gate = nn.Parameter(torch.full((len(FIELD_NAMES),),-0.5))
        self.potential_amplitude = nn.Linear(hidden,1)
        for layer in self.head.layers:
            nn.init.normal_(layer.weight,std=0.002); nn.init.zeros_(layer.bias)
        nn.init.zeros_(self.potential_amplitude.weight); nn.init.zeros_(self.potential_amplitude.bias)

    def forward(self,z_pred):
        if z_pred.ndim != 3 or z_pred.shape[1] != len(FIELD_NAMES):
            raise ValueError("Expected [B,2,L] field latents")
        features = torch.stack([module(z_pred[:,f]) for f,module in enumerate(self.pre)],dim=1)
        normalized = self.base(z_pred) + self.residual_gate.sigmoid()[None,:,None]*self.head(features)
        raw = spectral_to_periodic_raw(normalized.float()*self.spectral_rms.float(),
                                      self.spectral_indices,CFG.canonical_grid)
        log_scale = CFG.potential_amplitude_log_limit*torch.tanh(self.potential_amplitude(features[:,1]))
        raw = torch.cat([raw[:,:1], raw[:,1:2]*log_scale.exp().reshape(-1,1,1,1,1)],dim=1)
        return constrain_periodic_fields(raw)


TARGET_KEYS = {"energy_target", "energy_ev", "energy_mask", "native_grid_shape", "field_input", "rho_target", "potential_target", "cohesive_energy_target", "cohesive_energy_ev_per_atom"}

class PeriodicFieldJEPA(nn.Module):
    def __init__(self, sample: Data):
        super().__init__()
        use_kan = CFG.backbone == "pyg_gnn_kan"
        self.context_encoder = make_context_encoder(sample)
        self.ni_energy_calibration = NiEnergyCalibration()
        self.ni_shared_gain = NiSharedSpectralGain()
        self.register_buffer('energy_anchor_normalized', torch.tensor([
            (pretrain_normalization.cohesive_anchor_ev_per_atom[e]
             -pretrain_normalization.cohesive_center_ev_per_atom)
            /pretrain_normalization.cohesive_scale_ev_per_atom for e in ('Al','Fe','Ni')
        ], dtype=torch.float32))
        self.ni_field_adapter = NiResidualAdapter(
        CFG.hidden,
        use_kan,
        )
        self.cohesive_head = SharedLatentCohesiveEnergyHead(
        mode_count=CFG.field_spectral_complex_modes,
        hidden=CFG.hidden,
        use_kan=use_kan,
        )
        self.field_online_encoder = PeriodicFieldEncoder(CFG.latent_dim)
        # Temporary architecture copy only: NOT a trained JEPA teacher yet.
        self.field_target_encoder = initialize_ema_target(self.field_online_encoder)
        # deepcopy consumes no RNG: preserve V22 initialization of all common modules.
        self.electronic_online_encoder = copy.deepcopy(self.field_online_encoder)
        self.register_buffer("field_target_initialized_from_ae", torch.tensor(False))
        self.register_buffer("field_target_ema_updates", torch.zeros((), dtype=torch.long))
        self.field_reconstruction_decoder = PeriodicFieldReconstructionDecoder(
            CFG.latent_dim
        )
        self.field_prediction_decoder = PeriodicFourierResidualDecoder(
            CFG.latent_dim, CFG.hidden, use_kan
        )
        self.predictor = FieldConditionedPredictor(
            CFG.hidden, sample.edge_attr.shape[-1], use_kan
        )

        self.register_buffer(

            "predicted_latent_queue",

            torch.zeros(CFG.latent_queue_tokens, CFG.latent_dim),

            persistent=False,

        )

        self.register_buffer(

            "predicted_latent_queue_count", torch.zeros((), dtype=torch.long),

            persistent=False,

        )

        self.register_buffer(

            "predicted_latent_queue_cursor", torch.zeros((), dtype=torch.long),

            persistent=False,

        )



    def train(self, mode: bool = True):
        super().train(mode)
        self.field_target_encoder.eval()
        if bool(self.field_target_initialized_from_ae):
            self.field_online_encoder.requires_grad_(False).eval()
            self.field_reconstruction_decoder.requires_grad_(False).eval()
        return self

    @torch.no_grad()

    def update_predicted_latent_queue(self, z_pred: torch.Tensor) -> None:

        values = z_pred.detach().reshape(-1, z_pred.shape[-1]).float()

        capacity = self.predicted_latent_queue.shape[0]

        if len(values) >= capacity:

            self.predicted_latent_queue.copy_(values[-capacity:])

            self.predicted_latent_queue_count.fill_(capacity)

            self.predicted_latent_queue_cursor.zero_()

            return

        cursor = int(self.predicted_latent_queue_cursor)

        first = min(len(values), capacity - cursor)

        self.predicted_latent_queue[cursor:cursor + first].copy_(values[:first])

        remaining = len(values) - first

        if remaining:

            self.predicted_latent_queue[:remaining].copy_(values[first:])

        self.predicted_latent_queue_cursor.fill_((cursor + len(values)) % capacity)

        self.predicted_latent_queue_count.fill_(

            min(capacity, int(self.predicted_latent_queue_count) + len(values))

        )



    def predicted_regularization_population(self, z_pred: torch.Tensor) -> torch.Tensor:

        current = z_pred.reshape(-1, z_pred.shape[-1])

        count = int(self.predicted_latent_queue_count)

        if count == 0:

            return current

        history = self.predicted_latent_queue[:count].to(current.dtype).detach()

        return torch.cat([current, history], dim=0)



    @torch.no_grad()

    def reset_predicted_latent_queue(self) -> None:
        self.predicted_latent_queue.zero_()
        self.predicted_latent_queue_count.zero_()
        self.predicted_latent_queue_cursor.zero_()

    def encode_structure(self, data: Batch) -> tuple[torch.Tensor, torch.Tensor]:
        field_context, nodes = self.context_encoder(data)

        ni_z = int(ATOMIC_PROPERTIES[CFG.transfer_element]["Z"])
        mask = deterministic_segment_mean(
            data.atomic_number.eq(ni_z).float()[:, None],
            data.batch,
        )

        field_context = (
            field_context
            + mask * self.ni_field_adapter(field_context)
        )

        return field_context, nodes

    def predict_fields(
        self,
        node_context: torch.Tensor,
        global_context: torch.Tensor,
        data: Batch,
    ):
        z_raw, tokens, nodes = self.predictor(
            node_context,
            global_context,
            data,
        )

        # Latent used by the density and potential decoders.
        z_shared = self.predictor.field_encoder.forward_spectral(z_raw)
        ni_mask = self.element_masks(data.element, z_shared.device)[1]
        z_shared = self.ni_shared_gain(z_shared, ni_mask)

        return z_shared, z_raw, tokens, nodes


    def predict_cohesive_energy(
        self,
        z_shared: torch.Tensor,
        elements=None,
    ) -> torch.Tensor:
        descriptor = invariant_shared_latent_view(
            z_shared,
            self.predictor.spectral_rms,
        )
        return self.cohesive_from_descriptor(descriptor, elements)

    @staticmethod
    def element_masks(elements, device):
        names = [str(e) for e in elements]
        mapping = {'Al':0, 'Fe':1, 'Ni':2}
        if any(e not in mapping for e in names):
            raise ValueError('V29 calibrated for Al/Fe/Ni only')
        index = torch.tensor([mapping[e] for e in names], device=device, dtype=torch.long)
        return index, index.eq(2).float()

    def cohesive_from_descriptor(self, descriptor, elements=None):
        residual = self.cohesive_head(descriptor)
        # No-elements mode is only the residual readout, for latent-only interventions.
        if elements is None:
            return residual
        index, ni_mask = self.element_masks(elements, descriptor.device)
        if len(index) != len(residual):
            raise ValueError('Element metadata and latent batch disagree')
        return self.energy_anchor_normalized[index] + self.ni_energy_calibration(residual, ni_mask)

    def forward_train(self, data: Batch) -> dict[str, torch.Tensor]:
        if not hasattr(data, "field_input"):
            raise KeyError("forward_train requires field targets")

        field_context, node_embeddings = self.encode_structure(data)

        z_online = self.electronic_online_encoder(data.field_input)

        with torch.no_grad():
            z_target = self.field_target_encoder(data.field_input)

        (
            z_shared_pred,
            z_raw_pred,
            prediction_tokens,
            prediction_nodes,
        ) = self.predict_fields(
            node_embeddings,
            field_context,
            data,
        )

        rho_pred, potential_pred = self.field_prediction_decoder(
            z_shared_pred
        )

        rho_reconstruction, potential_reconstruction = (
            self.field_reconstruction_decoder(z_online)
        )

        cohesive_prediction = self.predict_cohesive_energy(
            z_shared_pred, data.element
        )

        return {
            "z_online": z_online,
            "z_target": z_target,
            "z_pred": z_shared_pred,
            "z_raw_pred": z_raw_pred,
            "rho_pred": rho_pred,
            "potential_pred": potential_pred,
            "rho_reconstruction": rho_reconstruction,
            "potential_reconstruction": potential_reconstruction,
            "cohesive_energy_pred": cohesive_prediction,
            "field_context": field_context,
            "node_embeddings": node_embeddings,
            "prediction_tokens": prediction_tokens,
            "prediction_nodes": prediction_nodes,
        }

    def forward_operator(
        self,
        data: Batch,
    ) -> dict[str, torch.Tensor]:
        leaked = sorted(
            key for key in TARGET_KEYS if hasattr(data, key)
        )

        if leaked:
            raise RuntimeError(
                f"Inference received forbidden targets: {leaked}"
            )

        field_context, node_embeddings = self.encode_structure(data)

        (
            z_shared_pred,
            z_raw_pred,
            prediction_tokens,
            _,
        ) = self.predict_fields(
            node_embeddings,    
            field_context,
            data,
        )

        rho_scaled, potential_scaled = (
            self.field_prediction_decoder(z_shared_pred)
        )

        cohesive_normalized = self.predict_cohesive_energy(
            z_shared_pred, data.element
        )

        return {
            "z_pred": z_shared_pred,
            "z_raw_pred": z_raw_pred,
            "rho_scaled": rho_scaled,
            "potential_scaled": potential_scaled,
            "cohesive_energy_normalized": cohesive_normalized,
            "prediction_tokens": prediction_tokens,
        }


@torch.no_grad()
def initialize_target_from_trained_autoencoder(model, source):
    """Exact copy of the selected trained AE; never reset an active JEPA EMA."""
    if bool(model.field_target_initialized_from_ae):
        raise RuntimeError("Target already initialized: do not copy the AE again; restore the JEPA checkpoint.")
    model.predictor.field_encoder.load_state_dict(model.field_online_encoder.state_dict(), strict=True)
    model.electronic_online_encoder.load_state_dict(model.field_online_encoder.state_dict(), strict=True)
    model.electronic_online_encoder.requires_grad_(False).eval()
    model.field_online_encoder.requires_grad_(False).eval()
    model.field_reconstruction_decoder.requires_grad_(False).eval()
    model.field_target_encoder.load_state_dict(model.field_online_encoder.state_dict(), strict=True)
    model.field_target_encoder.requires_grad_(False)
    model.field_target_encoder.eval()
    online = model.field_online_encoder.state_dict()
    target = model.field_target_encoder.state_dict()
    predictor_state = model.predictor.field_encoder.state_dict()
    electronic_state = model.electronic_online_encoder.state_dict()
    for key in online:
        states = [online[key], target[key], predictor_state[key], electronic_state[key]]
        if len({v.data_ptr() for v in states}) != 4 or not all(torch.equal(states[0],v) for v in states[1:]):
            raise RuntimeError('Electronic initialization requires exact, independent copies.')
    if any(online[k].data_ptr() == target[k].data_ptr() or
           online[k].data_ptr() == predictor_state[k].data_ptr() for k in online):
        raise RuntimeError("AE, predictor, and target must have independent storage.")
    if not all(torch.equal(online[k], target[k]) and torch.equal(online[k], predictor_state[k]) for k in online):
        raise RuntimeError("The AE -> target copy was not exact.")
    model.field_target_initialized_from_ae.fill_(True)
    model.field_target_ema_updates.zero_()
    audit = dict(source=str(source), exact_state_copy=True, target_requires_grad=False,
                 ema_updates=0, ema_source="electronic_online_encoder",
                 initialization="selected trained autoencoder, before first JEPA update")
    atomic_json_dump(audit, CFG.run_dir / "target_encoder_initialization.json")
    return audit


def require_initialized_jepa_target(model):
    if not bool(model.field_target_initialized_from_ae):
        raise RuntimeError("Train/load the autoencoder and initialize the target before JEPA.")
    if any(p.requires_grad for module in (model.field_online_encoder, model.field_reconstruction_decoder)
           for p in module.parameters()):
        raise RuntimeError("The autoencoder must remain frozen throughout JEPA.")
    if any(p.requires_grad for p in model.field_target_encoder.parameters()):
        raise RuntimeError("The target encoder must be updated exclusively by EMA.")


@torch.no_grad()
def update_jepa_target(model, stage, tau):
    if stage == "autoencoder":
        raise RuntimeError("EMA is not allowed during autoencoder training.")
    if stage.startswith('G_'):
        raise RuntimeError('EMA frozen in all Ni stages')
    require_initialized_jepa_target(model)
    ema_update(model.electronic_online_encoder, model.field_target_encoder, tau)
    model.field_target_encoder.eval()
    model.field_target_ema_updates.add_(1)


In [20]:
# =============================================================================
# STEP 16 — Losses and latent diagnostics
# Purpose: Combines JEPA, fields, energy, reconstruction, variance, covariance, and KAN regularization.
# Inputs: Model outputs and Batch targets.
# Outputs: Total loss, separate terms, and collapse/rank metrics.
# Reproducibility: run after all preceding cells; do not change the seed during the run.
# =============================================================================
def latent_regularization(z: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    flat = z.reshape(-1, z.shape[-1]).float()

    # Real(DC) and Imag(DC) must not govern rank control. In particular,

    # Imag(DC) is structurally zero and potential DC is removed by the gauge.

    if flat.shape[-1] == CFG.latent_dim and flat.shape[-1] > 2:

        flat = flat[:, 2:]
    if flat.shape[0] < 2:
        zero = z.sum() * 0
        return zero, zero
    centered = flat - flat.mean(0, keepdim=True)
    variance = centered.square().mean(0)
    variance_loss = (torch.sqrt(variance + 1e-4) - 1.0).square().mean()
    standardized = centered / torch.sqrt(variance + 1e-4)
    covariance = standardized.T @ standardized / max(flat.shape[0] - 1, 1)
    off_diagonal = covariance - torch.diag(torch.diagonal(covariance))
    denominator = max(flat.shape[-1] * (flat.shape[-1] - 1), 1)
    covariance_loss = off_diagonal.square().sum() / denominator
    return variance_loss, covariance_loss


def kan_coefficient_penalty(module: nn.Module) -> torch.Tensor:
    penalties = []
    for child in module.modules():
        if isinstance(child, FactorizedBasisKANLinear):
            dense = child.dense_coefficients()
            degree = torch.arange(1, child.degree + 1, device=dense.device, dtype=dense.dtype)
            penalties.append((dense.square() * degree.square()).mean())
    if penalties:
        return torch.stack(penalties).mean()
    return next(module.parameters()).sum() * 0


def gauge_center(field: torch.Tensor) -> torch.Tensor:
    return field - field.mean(dim=(-3, -2, -1), keepdim=True)


def periodic_gradient_smooth_l1(
    prediction: torch.Tensor, target: torch.Tensor
) -> torch.Tensor:
    losses = []
    for dimension in (-3, -2, -1):
        prediction_gradient = torch.roll(prediction, -1, dimension) - prediction
        target_gradient = torch.roll(target, -1, dimension) - target
        losses.append(F.smooth_l1_loss(prediction_gradient, target_gradient))
    return torch.stack(losses).mean()


def balanced_sample_mean(values: torch.Tensor, data: Batch) -> torch.Tensor:
    """Equal element means + mild worst-element weight, applied to gradients."""
    labels = getattr(data, "element", None)
    if not isinstance(labels, (list, tuple)) or len(labels) != len(values):
        return values.mean()
    means = torch.stack([values[torch.tensor([x == label for x in labels],
                         device=values.device)].mean() for label in sorted(set(labels))])
    alpha = CFG.element_loss_worst_weight
    return (1-alpha) * means.mean() + alpha * means.max()


def relative_field_error(pred, target):
    error = (pred.float() - target.float()).square().flatten(1).mean(-1)
    reference = target.float().square().flatten(1).mean(-1).clamp_min(1e-8)
    return torch.sqrt(error / reference + 1e-12)



def periodic_cartesian_gradient(values, cell):
    """Central periodic differences: r = s @ cell, grid axes (z,y,x).

    Returns [B,F,3,D,H,W] in field-units/Angstrom. Unlike index differences,
    this accounts for anisotropy and off-diagonal cell terms.
    """
    with torch.autocast(device_type=values.device.type,enabled=False):
        values = values.float()
        frac = torch.stack([(torch.roll(values,-1,axis)-torch.roll(values,1,axis))
                            * (values.shape[axis]/2) for axis in (-1,-2,-3)],dim=2)
        inverse = torch.linalg.inv(cell.reshape(-1,3,3).float())
        return torch.einsum('bij,bfjdhw->bfidhw',inverse,frac)


def cartesian_gradient_error(prediction,target,cell):
    pg,tg = periodic_cartesian_gradient(prediction,cell),periodic_cartesian_gradient(target,cell)
    reference = tg.square().flatten(1).mean(-1).clamp_min(1e-8).sqrt().detach()
    return F.smooth_l1_loss(pg/reference[:,None,None,None,None,None],
                          tg/reference[:,None,None,None,None,None],reduction='none').flatten(1).mean(-1)


def weighted_spectral_relative_error(prediction, target):
    """Relative rFFT error with extra weight on physically unresolved high modes."""
    with torch.autocast(device_type=prediction.device.type, enabled=False):
        prediction = prediction.float()
        target = target.float()
        ps = torch.fft.rfftn(prediction, dim=(-3, -2, -1), norm="ortho")
        ts = torch.fft.rfftn(target, dim=(-3, -2, -1), norm="ortho")
        depth, height, width = prediction.shape[-3:]
        kz = torch.fft.fftfreq(depth, device=prediction.device).reshape(1, 1, depth, 1, 1)
        ky = torch.fft.fftfreq(height, device=prediction.device).reshape(1, 1, 1, height, 1)
        kx = torch.fft.rfftfreq(width, device=prediction.device).reshape(1, 1, 1, 1, -1)
        radius2 = (kx.square() + ky.square() + kz.square()) / 0.75
        weight = 1.0 + CFG.spectral_high_frequency_boost * radius2.clamp(0, 1)
        error = (weight * (ps - ts).abs().square()).flatten(1).sum(-1)
        reference = (weight * ts.abs().square()).flatten(1).sum(-1).clamp_min(1e-8)
        return torch.sqrt(error / reference + 1e-12)


def field_losses(rho_pred, potential_pred, data):
    # All reductions in float32; small fields must not underflow with AMP.
    rho_pred, rho_target = rho_pred.float(), data.rho_target.float()
    log_loss = F.smooth_l1_loss(
        torch.log(rho_pred.clamp_min(CFG.density_floor)),
        torch.log(rho_target.clamp_min(CFG.density_floor)), reduction="none"
    ).flatten(1).mean(-1)
    rho_grad = cartesian_gradient_error(rho_pred, rho_target, data.cell)
    rho_spectral = weighted_spectral_relative_error(rho_pred, rho_target)
    gw, sw = CFG.rho_gradient_loss_fraction, CFG.rho_spectral_loss_fraction
    rho_point = ((0.5-gw-sw)*log_loss + 0.5*relative_field_error(rho_pred, rho_target)
                 + gw*rho_grad + sw*rho_spectral)
    rho_amp = (rho_pred.square().flatten(1).mean(-1).clamp_min(1e-12).log()
               -rho_target.square().flatten(1).mean(-1).clamp_min(1e-12).log()).abs()*0.5
    rho_loss = balanced_sample_mean(rho_point + CFG.rho_log_amplitude_weight*rho_amp, data)

    prediction = gauge_center(potential_pred.float())
    target = gauge_center(data.potential_target.float())
    scale = target.square().mean((-3,-2,-1), keepdim=True).clamp_min(1e-8).sqrt().detach()
    shape = F.smooth_l1_loss(prediction / scale, target / scale, reduction="none").flatten(1).mean(-1)
    p_rms = prediction.square().mean((-3,-2,-1), keepdim=True).add(1e-12).sqrt()
    amplitude = F.smooth_l1_loss(p_rms / scale, torch.ones_like(scale), reduction="none").flatten(1).mean(-1)
    cosine = 1 - F.cosine_similarity(prediction.flatten(1), target.flatten(1), dim=-1, eps=1e-6)
    gradient_loss = cartesian_gradient_error(prediction, target, data.cell)
    spectral_loss = weighted_spectral_relative_error(prediction, target)
    alpha, beta, gamma, delta, spectral_fraction = (
        CFG.potential_relative_loss_fraction, CFG.potential_gradient_loss_fraction,
        CFG.potential_cosine_loss_fraction, CFG.potential_amplitude_loss_fraction,
        CFG.potential_spectral_loss_fraction,
    )
    residual_fraction = 1-alpha-beta-gamma-delta-spectral_fraction
    potential = (residual_fraction*shape + alpha*relative_field_error(prediction,target)
                 + delta*amplitude + gamma*cosine + beta*gradient_loss
                 + spectral_fraction*spectral_loss)
    log_amplitude = (p_rms.clamp_min(1e-6).log()-scale.clamp_min(1e-6).log()).abs().flatten(1).mean(-1)
    return rho_loss, balanced_sample_mean(potential + CFG.potential_log_amplitude_weight*log_amplitude, data)


def jepa_alignment_view(z):
    """Complete non-DC spectral latent used by v12 alignment and audits."""
    return z[..., 2:].float()


def target_moment_losses(prediction, target):
    """Match moments across records, independently for each field.

    Accept [B,F,D] or [B,D] (one field). The target is always stop-gradient.
    Population covariance retains the original normalization by batch size.
    """
    if prediction.ndim not in (2, 3):
        raise ValueError(f"Latents must have shape [B,D] or [B,F,D]; received {tuple(prediction.shape)}")
    if prediction.shape != target.shape:
        raise ValueError(f"Incompatible shapes: prediction={tuple(prediction.shape)}, target={tuple(target.shape)}")
    if any(size == 0 for size in prediction.shape):
        raise ValueError("Latents cannot have empty axes")
    if prediction.device != target.device:
        raise ValueError("Prediction and target must be on the same device")
    if prediction.ndim == 2:
        prediction, target = prediction.unsqueeze(1), target.unsqueeze(1)
    with torch.autocast(device_type=prediction.device.type, enabled=False):
        target = target.detach().float()
        prediction = prediction.float()
        if len(prediction) < 2:
            zero = prediction.sum() * 0
            return zero, zero
        p = prediction - prediction.mean(0, keepdim=True)
        t = target - target.mean(0, keepdim=True)
        ps = (p.square().mean(0) + 1e-4).sqrt()
        ts = (t.square().mean(0) + 1e-4).sqrt()
        variance = F.smooth_l1_loss(ps, ts)
        pn, tn = p / ts.clamp_min(0.1), t / ts.clamp_min(0.1)
        pc = torch.einsum('bfd,bfe->fde', pn, pn) / len(p)
        tc = torch.einsum('bfd,bfe->fde', tn, tn) / len(t)
        covariance = F.smooth_l1_loss(pc, tc)
        return variance, covariance



COHESIVE_WEAK_THRESHOLDS: dict[str, float] = {}
COHESIVE_EXTREME_WEAK_THRESHOLDS: dict[str, float] = {}
RETENTION_TEACHER = None

def normalized_cohesive_coordinates(frame, normalization):
    values = np.asarray(frame.cohesive_energy_ev_per_atom, dtype=np.float64)
    return (values-normalization.cohesive_center_ev_per_atom)/normalization.cohesive_scale_ev_per_atom



def fit_cohesive_weak_thresholds(
    frame: pd.DataFrame,
    normalization: Normalization,
) -> dict[str, float]:
    thresholds = {}

    for element, group in frame.groupby("element"):
        coordinates = normalized_cohesive_coordinates(
            group,
            normalization,
        )

        thresholds[str(element)] = float(
            np.quantile(
                coordinates,
                CFG.cohesive_weak_quantile,
            )
        )

    return thresholds


def fit_cohesive_extreme_weak_thresholds(
    frame: pd.DataFrame,
    normalization: Normalization,
) -> dict[str, float]:
    thresholds = {}

    for element, group in frame.groupby("element"):
        coordinates = normalized_cohesive_coordinates(
            group,
            normalization,
        )

        thresholds[str(element)] = float(
            np.quantile(
                coordinates,
                CFG.cohesive_extreme_weak_quantile,
            )
        )

    return thresholds


from collections import deque

class ElementEnergyResidualFIFO:
    """Bounded, training-only residual history; stores no tensors or graphs."""
    def __init__(self, capacity, minimum):
        self.capacity, self.minimum = int(capacity), int(minimum)
        assert 64 <= self.minimum <= self.capacity
        self.values = {}

    def update(self, element, residual):
        values = residual.detach().float().cpu().reshape(-1)
        if not bool(torch.isfinite(values).all()):
            raise FloatingPointError(f"Nonfinite residual in the FIFO for {element}")
        self.values.setdefault(str(element), deque(maxlen=self.capacity)).extend(values.tolist())

    def mean(self, element):
        values = self.values.get(str(element), ())
        return float(np.mean(values)) if len(values) >= self.minimum else None

    def state_dict(self):
        return {e: list(v) for e, v in self.values.items()}

    def load_state_dict(self, state):
        self.values = {}
        for element, values in state.items():
            assert len(values) <= self.capacity and np.isfinite(values).all(), "Invalid FIFO in checkpoint"
            self.values[str(element)] = deque(map(float, values), maxlen=self.capacity)

ENERGY_BIAS_FIFO = ElementEnergyResidualFIFO(CFG.energy_bias_buffer_capacity, CFG.energy_bias_min_samples)

def cohesive_energy_objective(
    prediction: torch.Tensor,
    data: Batch,
    training: bool,
) -> torch.Tensor:
    target = data.cohesive_energy_target.reshape(-1).float()
    pred = prediction.reshape(-1).float()
    labels = list(map(str, data.element))

    weak_thresholds = torch.tensor(
        [
            COHESIVE_WEAK_THRESHOLDS.get(
                element,
                -float("inf"),
            )
            for element in labels
        ],
        device=pred.device,
        dtype=pred.dtype,
    )

    extreme_thresholds = torch.tensor(
        [
            COHESIVE_EXTREME_WEAK_THRESHOLDS.get(
                element,
                -float("inf"),
            )
            for element in labels
        ],
        device=pred.device,
        dtype=pred.dtype,
    )

    # Positive cohesive energy: weak binding corresponds to the lower tail.
    weak = target <= weak_thresholds
    extreme = target <= extreme_thresholds

    point = (
        (1.0 - CFG.cohesive_mse_fraction)
        * F.smooth_l1_loss(
            pred,
            target,
            reduction="none",
            beta=CFG.cohesive_huber_beta_ev_per_atom / pretrain_normalization.cohesive_scale_ev_per_atom,
        )
        + (CFG.cohesive_mse_fraction * (pred - target).square()
           if CFG.cohesive_mse_fraction > 0 else torch.zeros_like(pred))
    )

    if training:
        sample_weights = (
            torch.ones_like(target)
            + (CFG.cohesive_weak_weight - 1.0)
            * weak.float()
            + (
                CFG.cohesive_extreme_weight
                - CFG.cohesive_weak_weight
            )
            * extreme.float()
        )

        # With positive cohesive energy, pred > target means overestimating binding.
        overprediction = (torch.relu(pred - target).square()
            if max(CFG.cohesive_overprediction_weight, CFG.cohesive_extreme_overprediction_weight) > 1.0
            else torch.zeros_like(pred))

        asymmetric_weight = torch.where(
            extreme,
            torch.full_like(
                target,
                max(CFG.cohesive_extreme_overprediction_weight - 1.0, 0.0),
            ),
            torch.where(
                weak,
                torch.full_like(
                    target,
                    max(CFG.cohesive_overprediction_weight - 1.0, 0.0),
                ),
                torch.zeros_like(target),
            ),
        )

        point = sample_weights.clamp(max=CFG.energy_sample_weight_cap) * (
            point + asymmetric_weight * overprediction
        )

    if training:
        upper = torch.tensor([COHESIVE_UPPER_THRESHOLDS.get(e,float('inf')) for e in labels], device=pred.device)
        upper_weight = 1 + (CFG.energy_upper_tail_weight-1)*(target>=upper).float()
        point = point * (sample_weights * upper_weight).clamp(max=CFG.energy_sample_weight_cap) / sample_weights.clamp(max=CFG.energy_sample_weight_cap)
    extra = pred.sum()*0
    penalties = []
    if training:
        for element in sorted(set(labels)):
            mask = torch.tensor([e == element for e in labels], device=pred.device)
            p, t = pred[mask], target[mask]
            residual = p - t
            ENERGY_BIAS_FIFO.update(element, residual)
            bias = ENERGY_BIAS_FIFO.mean(element)
            penalty = pred.sum()*0
            if bias is not None:
                # Historical value, current minibatch derivative; no historical graph.
                proxy = residual.mean() - residual.mean().detach() + residual.new_tensor(bias)
                penalty = CFG.energy_bias_weight * proxy.square()
            if CFG.energy_slope_weight > 0 and len(t) >= CFG.energy_slope_min_records:
                tc = t - t.mean(); variance = tc.square().mean()
                if float(variance.detach()) >= CFG.energy_slope_variance_floor:
                    slope = ((p-p.mean())*tc).mean()/variance
                    penalty = penalty + CFG.energy_slope_weight*F.smooth_l1_loss(slope, torch.ones_like(slope))
            penalties.append(penalty)
    if penalties: extra = torch.stack(penalties).mean()
    return balanced_sample_mean(point, data) + extra



@torch.no_grad()
def latent_diagnostics(z):
    """No artificial eigenvalue floor: a constant representation has rank ZERO.

    Float64 covariance reduces spurious tiny directions. Positive clipping of all
    eigenvalues to 1e-12 would make an exactly constant latent look full-rank.
    """
    z=z.detach().double()
    if z.ndim==3: z=z-z.mean(0,keepdim=True)
    z=z.reshape(-1,z.shape[-1])
    if z.shape[-1]==CFG.latent_dim and z.shape[-1]>2: z=z[:,2:]
    zero=dict(latent_variance=0.0,effective_rank=0.0,active_covariance_rank=0,
        condition_number=float('inf'),active_condition_number=float('inf'),
        dominant_covariance_fraction=1.0,near_zero_variance_fraction=1.0,
        covariance_eigenvalues=[0.0]*z.shape[-1])
    if len(z)<2: return zero
    centered=z-z.mean(0)
    covariance=centered.T@centered/(len(z)-1)
    eigenvalues=torch.linalg.eigvalsh(covariance).clamp_min(0)
    eigenvalues=torch.where(eigenvalues>1e-12*eigenvalues.max(),eigenvalues,torch.zeros_like(eigenvalues))
    total=eigenvalues.sum()
    if float(total)<=1e-20: return zero
    probability=eigenvalues[eigenvalues>0]/total
    active=eigenvalues[eigenvalues>1e-6*eigenvalues.max()]
    return dict(latent_variance=float(z.var(0,unbiased=False).mean()),
        effective_rank=float(torch.exp(-(probability*probability.log()).sum())),
        active_covariance_rank=int(len(active)),
        condition_number=float(eigenvalues.max()/eigenvalues.min()) if float(eigenvalues.min())>0 else float('inf'),
        active_condition_number=float(active.max()/active.min()) if len(active) else float('inf'),
        dominant_covariance_fraction=float(eigenvalues.max()/total),
        near_zero_variance_fraction=float((z.std(0,unbiased=False)<1e-3).double().mean()),
        covariance_eigenvalues=eigenvalues.cpu().tolist())

def latent_gate_decisions(values: Mapping[str, Any]) -> dict[str, bool]:
    selected = {
        key: float(values[key])
        for key in (
            "latent_variance", "effective_rank", "active_covariance_rank",
            "active_condition_number", "dominant_covariance_fraction",
            "near_zero_variance_fraction",
        )
    }
    return {
        "finite": bool(all(np.isfinite(value) for value in selected.values())),
        "variance_in_range": bool(
            LATENT_GATES["latent_variance_min"] <= selected["latent_variance"]
            <= LATENT_GATES["latent_variance_max"]
        ),
        "effective_rank_ok": bool(
            selected["effective_rank"] >= LATENT_GATES["effective_rank_min"]
        ),
        "active_rank_ok": bool(
            selected["active_covariance_rank"] >= LATENT_GATES["active_covariance_rank_min"]
        ),
        "active_condition_ok": bool(
            selected["active_condition_number"] <= LATENT_GATES["active_condition_number_max"]
        ),
        "near_zero_fraction_ok": bool(
            selected["near_zero_variance_fraction"] <= LATENT_GATES["near_zero_variance_fraction_max"]
        ),
        "dominant_fraction_ok": bool(
            selected["dominant_covariance_fraction"] <= LATENT_GATES["dominant_covariance_fraction_max"]
        ),
    }


In [21]:

# =============================================================================
# V31 — per-field spectral competence (D_latent)
# =============================================================================
# Spectral difficulty is no longer released solely by the stage clock.
# Each field (density/potential) unlocks middle/high independently based on
# of cosines measured during validation. The unlock time is persisted and the
# new band enters through a short cosine ramp to avoid abrupt changes.
SPECTRAL_COMPETENCE_STATE = {
    field: {
        'middle_unlocked_at': None,
        'high_unlocked_at': None,
        'low_ready': False,
        'middle_ready': False,
        'high_ready': False,
        'low_streak': 0,
        'middle_streak': 0,
        'high_streak': 0,
    }
    for field in FIELD_NAMES
}
D_LOW_GATE_READY = False
D_COMPETENCE_META = {'crossfade_unlocked_at': None, 'low_gate_ready': False}

def _competence_ramp(progress: float, unlocked_at: float | None) -> float:
    if unlocked_at is None:
        return 0.0
    start=float(unlocked_at)
    end=min(1.0,start+CFG.competence_band_ramp_fraction)
    if end <= start + 1e-12:
        return 1.0
    return cosine_window(progress,start,end)

def progressive_band_weights(stage, progress, training, field_name=None):
    """Return field-specific JEPA band weights.

    Outside D training the final three-band objective is used. During D, low is
    always active while middle/high are competence-gated and smoothly ramped
    after their validation unlock.
    """
    final = tuple(float(x) for x in CFG.jepa_band_weights)
    if len(final) != 3 or any(x <= 0 for x in final) or abs(sum(final)-1.0) > 1e-8:
        raise ValueError('Configure three positive spectral weights that sum to 1.')
    if not training or stage != 'D_latent' or not CFG.progressive_spectral_curriculum:
        return final
    if field_name not in FIELD_NAMES:
        raise ValueError(f'Invalid field_name for spectral curriculum: {field_name}')
    state=SPECTRAL_COMPETENCE_STATE[field_name]
    factors=(
        1.0,
        _competence_ramp(progress,state['middle_unlocked_at']),
        _competence_ramp(progress,state['high_unlocked_at']),
    )
    weighted=[w*f for w,f in zip(final,factors)]
    total=sum(weighted)
    if total <= 0:
        return (1.0,0.0,0.0)
    return tuple(w/total for w in weighted)

def spectral_alignment_terms(prediction, target, stage, progress, training):
    """JEPA alignment with independent field-specific competence schedules."""
    if prediction.shape != target.shape or prediction.ndim != 3:
        raise ValueError('Expected aligned [B,F,L] prediction/target tensors')
    field_results=[]
    metrics={}
    for f,field_name in enumerate(FIELD_NAMES):
        weights=progressive_band_weights(stage,progress,training,field_name)
        losses=[]; moments=[]; scales=[]
        for name,weight,(start,stop) in zip(('low','middle','high'),weights,CFG.jepa_band_boundaries):
            p=prediction[:,f:f+1,start:stop].float()
            t=target[:,f:f+1,start:stop].float()
            normalized=F.smooth_l1_loss(F.layer_norm(p,p.shape[-1:]),F.layer_norm(t,t.shape[-1:]))
            cosine_loss=(1-F.cosine_similarity(p,t,dim=-1,eps=1e-6)).mean()
            losses.append((weight,F.smooth_l1_loss(p,t)+0.25*normalized,cosine_loss))
            scales.append(weight*F.smooth_l1_loss(
                p.square().mean(-1).add(1e-6).sqrt(),
                t.square().mean(-1).add(1e-6).sqrt()))
            moments.append((weight,target_moment_losses(p,t)))
            metrics[f'{field_name}_cosine_{name}']=1-cosine_loss
            metrics[f'{field_name}_band_weight_{name}']=prediction.new_tensor(weight)
        full_cosine=F.cosine_similarity(
            prediction[:,f,2:].float(),target[:,f,2:].float(),dim=-1,eps=1e-6).mean()
        metrics[f'{field_name}_latent_cosine_full_similarity']=full_cosine
        field_results.append(dict(
            jepa=sum(w*l for w,l,_ in losses),
            latent_cosine=sum(w*c for w,_,c in losses),
            latent_variance_penalty=sum(w*v[0] for w,v in moments),
            latent_covariance_penalty=sum(w*v[1] for w,v in moments),
            latent_scale=sum(scales),
        ))
    # Equal field weighting prevents the easier field from dominating the shared JEPA objective.
    result={key:sum(item[key] for item in field_results)/len(field_results)
            for key in field_results[0]}
    metrics['latent_cosine_full_similarity']=torch.stack([
        metrics[f'{name}_latent_cosine_full_similarity'] for name in FIELD_NAMES]).mean()
    metrics['latent_cosine_full_loss']=1-metrics['latent_cosine_full_similarity']
    return result,metrics


In [22]:
def translate_periodic_batch(data, shifts):
    """Exact discrete periodic translation used only for an equivariance comparison."""
    augmented = data.clone()
    fractional_shift = torch.tensor([shifts[2] / CFG.canonical_grid[2], shifts[1] / CFG.canonical_grid[1],
                                     shifts[0] / CFG.canonical_grid[0]], dtype=augmented.pos_fractional.dtype,
                                    device=augmented.pos_fractional.device)
    shifted = augmented.pos_fractional + fractional_shift
    wraps = torch.floor(shifted).to(augmented.edge_shift.dtype)
    src, dst = augmented.edge_index
    augmented.edge_shift = augmented.edge_shift + wraps[dst] - wraps[src]
    augmented.pos_fractional = shifted % 1.0
    augmented.pos_periodic = periodic_position_encoding(augmented.pos_fractional, CFG.position_frequencies)
    for key in ("field_input", "rho_target", "potential_target"):
        if hasattr(augmented, key):
            setattr(augmented, key, torch.roll(getattr(augmented, key), shifts=shifts, dims=(-3, -2, -1)))
    return augmented


In [23]:
def operator_batch(data):
    result = data.clone()
    for key in TARGET_KEYS:
        if key in result: del result[key]
    return result

def predicted_outputs(model, data):
    fc, nodes = model.encode_structure(data)
    z, raw, _, _ = model.predict_fields(nodes, fc, data)
    rho, potential = model.field_prediction_decoder(z)
    return z, rho, potential, model.predict_cohesive_energy(z, data.element)

def shared_replay_distillation(model, data, z, rho, potential, cohesive):
    if RETENTION_TEACHER is None or not model.training:
        return z.sum()*0
    labels = list(map(str,data.element))
    if not any(e in CFG.pretrain_elements for e in labels): return z.sum()*0
    with torch.no_grad():
        tz, tr, tv, te = predicted_outputs(RETENTION_TEACHER, operator_batch(data))
    values=[]
    for element in CFG.pretrain_elements:
        mask=torch.tensor([e==element for e in labels],device=z.device)
        if not mask.any(): continue
        lr=F.smooth_l1_loss(rho[mask].clamp_min(CFG.density_floor).log(),tr[mask].clamp_min(CFG.density_floor).log())
        scale=tv[mask].square().flatten(1).mean(-1).clamp_min(1e-6).sqrt().reshape(-1,1,1,1,1)
        lv=F.smooth_l1_loss(potential[mask]/scale,tv[mask]/scale)
        le=F.smooth_l1_loss(cohesive[mask].float(),te[mask].float())
        lz=F.smooth_l1_loss(z[mask].float(),tz[mask].float())
        value=(CFG.retention_rho_weight*lr + CFG.retention_potential_weight*lv
               + CFG.retention_cohesive_weight*le + CFG.loss_retention_latent*lz)
        values.append(value*(CFG.retention_fe_multiplier if element=='Fe' else 1.0))
    return torch.stack(values).mean()

def shared_translation_consistency(model,data):
    if (not model.training or CFG.loss_translation_consistency <= 0
        or float(torch.rand(())) >= CFG.translation_consistency_probability):
        return next(model.parameters()).sum()*0
    modes={module:module.training for module in model.modules()}
    try:
        model.eval()
        shifts=tuple(max(1,n//5) for n in CFG.canonical_grid)
        _,r,v,e=predicted_outputs(model,operator_batch(data))
        translated=translate_periodic_batch(operator_batch(data),shifts)
        _,rt,vt,et=predicted_outputs(model,translated)
        return (relative_field_error(rt,torch.roll(r,shifts,(-3,-2,-1))).mean()
                +relative_field_error(vt,torch.roll(v,shifts,(-3,-2,-1))).mean()
                +F.smooth_l1_loss(et,e))
    finally:
        for module,training in modes.items(): module.training=training

def electronic_online_active(stage):
    return stage in ('D_latent','E_density','E_density_potential','E_full',
                     'F_periodic_ramp')




def fixed_monitor_objective(model, data, stage, z, terms):
    """Fixed diagnostic objective; detached and excluded from backward/checkpoint selection."""
    monitor = {}
    # D does not optimize fields yet; calibration does not optimize JEPA. Recompute only
    # the missing diagnostic pieces from the already available structural latent.
    need_alignment = stage in ('D_latent','G_cohesive_calibration')
    need_fields = stage == 'D_latent'
    for key in ('jepa','latent_cosine'):
        if not need_alignment and key in terms:
            monitor[key] = terms[key].detach()
    for key in ('rho','potential','cohesive'):
        if not need_fields and key in terms:
            monitor[key] = terms[key].detach()
    with torch.no_grad():
        zd = z.detach()
        if need_alignment or any(key not in monitor for key in ('jepa','latent_cosine')):
            target = model.field_target_encoder(data.field_input)
            alignment, _ = spectral_alignment_terms(zd, target, 'D_latent', 1.0, False)
            monitor['jepa'] = alignment['jepa']
            monitor['latent_cosine'] = alignment['latent_cosine']
        if need_fields or any(key not in monitor for key in ('rho','potential','cohesive')):
            rho, potential = model.field_prediction_decoder(zd)
            rho_loss, potential_loss = field_losses(rho, potential, data)
            energy = model.predict_cohesive_energy(zd, data.element)
            cohesive = cohesive_energy_objective(energy, data, False)
            monitor['rho'] = rho_loss
            monitor['potential'] = potential_loss
            monitor['cohesive'] = cohesive
    total = (CFG.monitor_jepa_weight*monitor['jepa']
             + CFG.monitor_latent_cosine_weight*monitor['latent_cosine']
             + CFG.monitor_rho_weight*monitor['rho']
             + CFG.monitor_potential_weight*monitor['potential']
             + CFG.monitor_cohesive_weight*monitor['cohesive'])
    return total.detach(), {f'monitor_{key}': value.detach() for key,value in monitor.items()}


def compute_loss(model,data,stage,progress=1.0):
    if stage=='autoencoder':
        z=model.field_online_encoder(data.field_input)
        rr,vv=model.field_reconstruction_decoder(z)
        rl,vl=field_losses(rr,vv,data)
        raw=model.field_online_encoder.raw_spectral_latent(data.field_input)
        anchor=F.smooth_l1_loss(z.float(),raw.float())
        # Head and encoder receive cohesive-energy gradients; the label is never an input.
        descriptor=invariant_shared_latent_view(z,model.field_online_encoder.spectral_rms)
        prediction=model.cohesive_from_descriptor(descriptor, data.element)
        cohesive=cohesive_energy_objective(prediction,data,model.training)
        weight=ae_cohesive_weight(progress, model.training)
        total=rl+vl+CFG.ae_regularization_weight*anchor+weight*cohesive
        return total,dict(total=total,monitor_total=total.detach(),rho=rl,potential=vl,cohesive=cohesive,reconstruction=rl+vl,
                         ae_anchor=anchor,cohesive_weight=total.new_tensor(weight))
    if stage == 'P_cohesive_refinement':
        # Frozen representation, deterministic dropout policy, head gradients only.
        with torch.no_grad():
            fc,nodes=model.encode_structure(data)
            z,_,_,_=model.predict_fields(nodes,fc,data)
        prediction=model.predict_cohesive_energy(z.detach(), data.element)
        cohesive=cohesive_energy_objective(prediction,data,model.training)
        kan = kan_coefficient_penalty(model.cohesive_head)
        total = cohesive + CFG.loss_kan * kan
        return total, {'total':total,'monitor_total':cohesive.detach(),'cohesive':cohesive,'kan':kan}
    fc,nodes=model.encode_structure(data)
    z,raw,_,_=model.predict_fields(nodes,fc,data)
    zero=z.sum()*0
    weights=stage_loss_weights(stage,progress)
    terms={key:zero for key in weights}
    metrics={}
    if stage!='G_cohesive_calibration':
        with torch.no_grad(): target=model.field_target_encoder(data.field_input)
        alignment,metrics=spectral_alignment_terms(z,target,stage,progress,model.training)
        terms.update(alignment)
    if stage!='D_latent':
        rp,vp=model.field_prediction_decoder(z)
        # E_full first trains the cohesive readout against a detached shared latent.
        # Shared representation gradients from energy are released only after the
        # field pathway has had time to stabilize.
        if stage=='E_full' and model.training:
            shared_factor=cosine_window(
                progress,CFG.energy_shared_release_fraction,CFG.energy_shared_full_fraction)
            # Forward value is exactly z; only the gradient entering the shared latent is scaled.
            energy_latent=z.detach()+shared_factor*(z-z.detach())
        else:
            shared_factor=1.0
            energy_latent=z
        ep=model.predict_cohesive_energy(energy_latent, data.element)
        terms['rho'],terms['potential']=field_losses(rp,vp,data)
        terms['cohesive']=cohesive_energy_objective(ep,data,model.training)
        metrics['energy_shared_gradient_factor']=z.new_tensor(float(shared_factor))
        if stage.startswith('G_'):
            terms['retention_distillation']=shared_replay_distillation(model,data,z,rp,vp,ep)
    if electronic_online_active(stage):
        electronic = model.electronic_online_encoder(data.field_input)
        # Frozen decoder parameters, but NO no_grad: gradients must reach electronic encoder.
        er, ev = model.field_reconstruction_decoder(electronic)
        erl, evl = field_losses(er, ev, data)
        with torch.no_grad():
            reference_latent = model.field_online_encoder(data.field_input)
        terms['electronic_reconstruction'] = erl + evl
        terms['electronic_anchor'] = F.smooth_l1_loss(electronic.float(), reference_latent.float())
        metrics['electronic_rho'] = erl
        metrics['electronic_potential'] = evl
    if weights['translation_consistency']: terms['translation_consistency']=shared_translation_consistency(model,data)
    if weights['kan']: terms['kan']=kan_coefficient_penalty(model)
    total=sum(weights[k]*v for k,v in terms.items())
    if stage.startswith('G_'):
        calibration_reg = CFG.energy_affine_regularization * (
            model.ni_energy_calibration.log_scale.square()
            + model.ni_energy_calibration.offset.square())
        total = total + calibration_reg
        metrics['calibration_regularization'] = calibration_reg
    # Arithmetic zeros stay internal; uncomputed quantities are absent from logs.
    logged=dict(terms)
    if stage == 'D_latent':
        for key in ('rho','potential','cohesive','retention_distillation'):
            logged.pop(key, None)
    if stage == 'G_cohesive_calibration':
        for key in ('jepa','latent_cosine','latent_variance_penalty','latent_covariance_penalty','latent_scale'):
            logged.pop(key, None)
    if not electronic_online_active(stage):
        logged.pop('electronic_reconstruction', None); logged.pop('electronic_anchor', None)
    if not weights['kan']: logged.pop('kan', None)
    if not weights['translation_consistency']: logged.pop('translation_consistency', None)
    if not stage.startswith('G_'): logged.pop('retention_distillation', None)
    logged.pop('reconstruction', None); logged.pop('ae_anchor', None)
    for key,value in weights.items():
        if value:
            metrics[f'effective_weight_{key}']=z.new_tensor(float(value))
    monitor_total, monitor_terms = fixed_monitor_objective(model, data, stage, z, terms)
    return total,{'total':total,'monitor_total':monitor_total,**monitor_terms,**logged,**metrics}


In [24]:

# =============================================================================
# V31 — within-stage ramps + adaptive weights based on conflict/magnitude
# =============================================================================
def cosine_ramp(progress, ramp_fraction=None):
    ramp_fraction = CFG.objective_ramp_fraction if ramp_fraction is None else float(ramp_fraction)
    p = min(max(float(progress) / max(ramp_fraction, 1e-12), 0.0), 1.0)
    return 0.5 * (1.0 - math.cos(math.pi * p))

def cosine_window(progress, start, end):
    if end <= start:
        raise ValueError('cosine_window requires end > start')
    p=min(max((float(progress)-float(start))/(float(end)-float(start)),0.0),1.0)
    return 0.5*(1.0-math.cos(math.pi*p))

def lerp(a,b,t):
    return float(a)+(float(b)-float(a))*float(t)

TASK_GRADIENT_BALANCE_STATE = {
    'factors': {},
    'norm_ema': {},
    'conflict_ema': {},
    'updates': 0,
}

BALANCEABLE_TASKS=('jepa','rho','potential','cohesive','translation_consistency','retention_distillation')

def _gradient_balance_factor(stage,key):
    if not CFG.gradient_balance_enabled or key not in BALANCEABLE_TASKS:
        return 1.0
    return float(TASK_GRADIENT_BALANCE_STATE['factors'].get(f'{stage}:{key}',1.0))

def objective_ramp_complete(stage, progress):
    ramped = {'D_latent','E_density','E_density_potential','E_full','F_periodic_ramp',
              'G_adapter_warmup','G_joint_finetune'}
    if stage == 'autoencoder':
        return float(progress) >= CFG.ae_energy_ramp_end
    if stage == 'D_latent':
        spectral_ready=all(bool(SPECTRAL_COMPETENCE_STATE[field]['high_ready']) for field in FIELD_NAMES)
        start=D_COMPETENCE_META.get('crossfade_unlocked_at')
        cross_complete=(start is not None and
            _competence_ramp(float(progress),float(start)) >= 1.0-1e-9)
        return spectral_ready and cross_complete
    if stage in ('G_adapter_warmup','G_joint_finetune'):
        needed=max(CFG.retention_ramp_fraction,
                   CFG.joint_structural_full if stage=='G_joint_finetune' else 0.0)
        return float(progress) >= needed
    return stage not in ramped or float(progress) >= CFG.objective_ramp_fraction

def _base_stage_loss_weights(stage, progress=1.0):
    common = dict(
        jepa=CFG.loss_jepa, latent_cosine=CFG.loss_latent_cosine,
        rho=CFG.loss_rho, potential=CFG.loss_potential, cohesive=CFG.loss_cohesive,
        reconstruction=0.0, ae_anchor=0.0,
        electronic_reconstruction=(CFG.loss_electronic_reconstruction if electronic_online_active(stage) else 0.0),
        electronic_anchor=(CFG.loss_electronic_anchor if electronic_online_active(stage) else 0.0),
        latent_variance_penalty=CFG.loss_latent_variance,
        latent_covariance_penalty=CFG.loss_latent_covariance,
        latent_scale=CFG.loss_latent_scale, kan=CFG.loss_kan,
        translation_consistency=CFG.loss_translation_consistency,
        retention_distillation=0.0,
    )
    ramp = cosine_ramp(progress)
    if stage == 'D_latent':
        cross=_competence_ramp(progress,D_COMPETENCE_META.get('crossfade_unlocked_at'))
        return {**common,
                'jepa':lerp(1.0,CFG.loss_jepa,cross),
                'latent_cosine':lerp(0.5,CFG.loss_latent_cosine,cross),
                'rho':0.0,'potential':0.0,'cohesive':0.0,
                'latent_variance_penalty':lerp(0.5,CFG.loss_latent_variance,cross),
                'latent_covariance_penalty':lerp(0.2,CFG.loss_latent_covariance,cross),
                'kan':CFG.loss_kan*cross,'translation_consistency':0.0}
    if stage == 'E_density':
        return {**common, 'rho':CFG.loss_rho*ramp, 'potential':0.0,
                'cohesive':0.0, 'translation_consistency':0.0}
    if stage == 'E_density_potential':
        return {**common, 'rho':CFG.loss_rho, 'potential':CFG.loss_potential*ramp,
                'cohesive':0.0, 'translation_consistency':0.0}
    if stage == 'E_full':
        return {**common, 'rho':CFG.loss_rho, 'potential':CFG.loss_potential,
                'cohesive':CFG.loss_cohesive*ramp, 'translation_consistency':0.0}
    if stage == 'F_periodic_ramp':
        return {**common, 'rho':CFG.loss_rho, 'potential':CFG.loss_potential,
                'cohesive':CFG.loss_cohesive,
                'translation_consistency':CFG.loss_translation_consistency*ramp}
    if stage == 'G_cohesive_calibration':
        return {key: (1.0 if key == 'cohesive' else 0.0) for key in common}
    if stage == 'P_cohesive_refinement':
        return {key: 1.0 if key == 'cohesive' else CFG.loss_kan if key == 'kan' else 0.0 for key in common}
    if stage in ('G_adapter_warmup','G_joint_finetune','G_retention_repair'):
        retention_ramp=cosine_ramp(progress,CFG.retention_ramp_fraction)
        return {**common, 'reconstruction':0.0, 'ae_anchor':0.0,
                'retention_distillation':CFG.retention_distillation_weight*retention_ramp}
    raise ValueError(stage)

def stage_loss_weights(stage, progress=1.0, adaptive=True):
    weights=_base_stage_loss_weights(stage,progress)
    if not adaptive:
        return weights
    for key in BALANCEABLE_TASKS:
        if key in weights and weights[key] > 0:
            weights[key] *= _gradient_balance_factor(stage,key)
    return weights

def ae_cohesive_weight(progress, training):
    if not training:
        return float(CFG.loss_autoencoder_cohesive)
    local = ((float(progress) - CFG.ae_energy_ramp_start) /
             max(CFG.ae_energy_ramp_end - CFG.ae_energy_ramp_start, 1e-12))
    local = min(max(local, 0.0), 1.0)
    return float(CFG.loss_autoencoder_cohesive) * 0.5 * (1.0 - math.cos(math.pi * local))


## Checkpoints and resumption

last.pt stores the literal epoch; best.pt stores a validated candidate. There is no copy last→best.
Resumption validates contracts, units, hashes, shapes, Adam, scheduler, and scaler before modifying
state, with rollback of the model, optimizer, RNG, and counters if any restoration fails.


In [25]:
def atomic_torch_save(payload: Mapping[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(dict(payload), temporary)
    os.replace(temporary, path)
def capture_rng_state(loader_generator=None):
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),
            'cuda':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
            'loader':loader_generator.get_state() if loader_generator is not None else None}

def restore_rng_state(state,loader_generator=None):
    random.setstate(state['python']); np.random.set_state(state['numpy']); torch.set_rng_state(state['torch'])
    if state['cuda']:
        if not torch.cuda.is_available() or len(state['cuda'])!=torch.cuda.device_count():
            raise RuntimeError('CUDA RNG restoration requires the same device count')
        torch.cuda.set_rng_state_all(state['cuda'])
    if loader_generator is not None and state['loader'] is not None:
        loader_generator.set_state(state['loader'])

@contextlib.contextmanager
def isolated_rng():
    state=capture_rng_state()
    try: yield
    finally: restore_rng_state(state)

def training_contract():
    return json_canonical({**configuration_contract(),'splits':split_manifest['records'],
                           'normalization':asdict(pretrain_normalization), 'units':UNIT_CONTRACT,
                           'ni_selection':NI_SELECTION_MANIFEST, 'split_hash':stable_contract_hash(split_manifest),
                           'spectral_statistics':SPECTRAL_RMS_STATISTICS,
                           'index':{'path':str(INDEX_PATH),'bytes':INDEX_PATH.stat().st_size,
                                    'mtime_ns':INDEX_PATH.stat().st_mtime_ns}})





In [26]:

# =============================================================================
# V31 — transactional checkpoint, Adam handoff, and variable EMA
# =============================================================================
OPTIMIZER_HANDOFF_PATH = CFG.run_dir / 'checkpoints' / 'optimizer_handoff.pt'
GLOBAL_OPTIMIZER_STATE={'valid_updates':0,'alfe_valid_updates':0,'ni_valid_updates':0,
    'source_curriculum_updates':0,'ni_curriculum_updates':0,
    'source_curriculum_planned':0,'ni_curriculum_planned':0}
EMA_PHASE_STATE = {'name':'AlFe','planned_updates':0,'completed_updates':0,'last_tau':0.996,'ema_frozen':False}
SOURCE_CURRICULUM_STAGES=('D_latent','E_density','E_density_potential','E_full','F_periodic_ramp')
NI_CURRICULUM_STAGES=('G_cohesive_calibration','G_adapter_warmup','G_joint_finetune')

def configure_lr_macro_phase(name, planned_updates):
    key='source_curriculum_planned' if name=='AlFe' else 'ni_curriculum_planned'
    planned=max(int(planned_updates),1)
    current=int(GLOBAL_OPTIMIZER_STATE.get(key,0))
    if current not in (0,planned):
        raise RuntimeError(f'LR budget for macro phase {name} changed: {current} -> {planned}')
    GLOBAL_OPTIMIZER_STATE[key]=planned
    return planned

def macro_lr_clock(stage):
    if stage in SOURCE_CURRICULUM_STAGES:
        return ('source_curriculum_updates','source_curriculum_planned')
    if stage in NI_CURRICULUM_STAGES:
        return ('ni_curriculum_updates','ni_curriculum_planned')
    return None


def configure_ema_phase(name,planned_updates,reset=True):
    if name=='Ni':
        EMA_PHASE_STATE['ema_frozen']=True
        return dict(EMA_PHASE_STATE)
    if EMA_PHASE_STATE['planned_updates'] and int(planned_updates)!=EMA_PHASE_STATE['planned_updates']:
        raise RuntimeError('Global AlFe EMA budget changed')
    EMA_PHASE_STATE['planned_updates']=max(int(planned_updates),1)
    EMA_PHASE_STATE['ema_frozen']=False
    return dict(EMA_PHASE_STATE)


def ema_tau_at_progress(progress):
    p=min(max(float(progress),0.),1.)
    return float(.999-(.999-.996)*.5*(1.+math.cos(math.pi*p)))


def effective_ema_tau(stage,stage_update,stage_total_updates):
    if stage.startswith('G_'): return float(EMA_PHASE_STATE['last_tau'])
    if EMA_PHASE_STATE['planned_updates']<1: raise RuntimeError('AlFe EMA has no global budget')
    return ema_tau_at_progress(EMA_PHASE_STATE['completed_updates']/EMA_PHASE_STATE['planned_updates'])


def advance_ema_phase(stage):
    if stage.startswith('G_'): return
    EMA_PHASE_STATE['completed_updates']+=1
    EMA_PHASE_STATE['last_tau']=ema_tau_at_progress(EMA_PHASE_STATE['completed_updates']/max(EMA_PHASE_STATE['planned_updates'],1))


def validate_ema_restore(state,allow_historical=False):
    required={'planned_updates','completed_updates','last_tau','ema_frozen','name'}
    if required-set(state) or state['completed_updates']<0 or state['planned_updates']<1:
        raise RuntimeError('Incomplete global EMA state')
    expected=ema_tau_at_progress(state['completed_updates']/state['planned_updates'])
    if abs(float(state['last_tau'])-expected)>1e-12: raise RuntimeError('EMA tau/counter mismatch')
    if EMA_PHASE_STATE['planned_updates'] not in (0,state['planned_updates']): raise RuntimeError('Incompatible EMA budget')
    if not allow_historical and float(state['last_tau'])<float(EMA_PHASE_STATE['last_tau'])-1e-12:
        raise RuntimeError('Restoration would move global tau backward')


def named_optimizer_state(model, optimizer):
    name_by_id = {id(parameter): name for name, parameter in model.named_parameters()}
    state = {}
    for parameter, values in optimizer.state.items():
        name = name_by_id.get(id(parameter))
        if name is None:
            continue
        state[name] = {
            key: value.detach().cpu().clone() if torch.is_tensor(value) else copy.deepcopy(value)
            for key, value in values.items()
        }
    return state


def restore_named_optimizer_state(model, optimizer, saved_state):
    restored, initialized = [], []
    parameters = dict(model.named_parameters())
    active = {id(parameter) for group in optimizer.param_groups for parameter in group['params']}
    for name, parameter in parameters.items():
        if id(parameter) not in active:
            continue
        incoming = saved_state.get(name)
        if incoming is None:
            initialized.append(name)
            continue
        compatible = True
        for key, value in incoming.items():
            if torch.is_tensor(value) and value.ndim > 0 and value.shape != parameter.shape:
                compatible = False
                break
        if not compatible:
            initialized.append(name)
            continue
        optimizer.state[parameter] = {
            key: value.to(parameter.device) if torch.is_tensor(value) else copy.deepcopy(value)
            for key, value in incoming.items()
        }
        restored.append(name)
    removed=sorted(set(saved_state)-set(restored)-set(initialized))
    return {'restored_parameters': restored, 'new_parameters': initialized,
            'removed_parameters':removed,'removed_count':len(removed),
            'restored_count': len(restored), 'new_count': len(initialized)}


def save_optimizer_handoff(model, optimizer, stage):
    payload = {
        'architecture_version': ARCHITECTURE_VERSION,
        'training_protocol_version': TRAINING_PROTOCOL_VERSION,
        'checkpoint_schema_version': CHECKPOINT_SCHEMA_VERSION,
        'training_contract': training_contract(),
        'completed_stage': stage,
        'optimizer_named_state': named_optimizer_state(model, optimizer),
        'ema_phase_state': dict(EMA_PHASE_STATE),
    }
    atomic_torch_save(payload, OPTIMIZER_HANDOFF_PATH)


def load_optimizer_handoff(model, optimizer):
    global EMA_PHASE_STATE
    if not OPTIMIZER_HANDOFF_PATH.exists():
        return {'restored_count': 0, 'new_count': sum(1 for g in optimizer.param_groups for _ in g['params']),
                'restored_parameters': [], 'new_parameters': []}
    payload = torch.load(OPTIMIZER_HANDOFF_PATH, map_location='cpu', weights_only=False)
    if payload.get('architecture_version') != ARCHITECTURE_VERSION:
        raise RuntimeError('Adam handoff incompatible with the current architecture')
    if payload.get('training_protocol_version') != TRAINING_PROTOCOL_VERSION:
        raise RuntimeError('Adam handoff incompatible with the current protocol')
    if payload.get('training_contract') != training_contract():
        raise RuntimeError('Adam handoff incompatible with the run contract')
    validate_ema_restore(payload['ema_phase_state'])
    report = restore_named_optimizer_state(model, optimizer, payload.get('optimizer_named_state', {}))
    # Handoff migrates Adam only; phase progress belongs to the live global clock.
    report['source_stage'] = payload.get('completed_stage')
    return report


def checkpoint_payload(model, optimizer, scheduler, scaler, stage, epoch, global_step,
                       best_validation, normalization, metrics, loader_generator):
    return {
        'architecture_version': ARCHITECTURE_VERSION,
        'training_protocol_version': TRAINING_PROTOCOL_VERSION,
        'curriculum_version': CURRICULUM_VERSION,
        'checkpoint_schema_version': CHECKPOINT_SCHEMA_VERSION,
        'training_contract': training_contract(), 'seed': CFG.seed,
        'checkpoint_role': 'literal_last', 'stage': stage, 'epoch': epoch,
        'global_step': global_step, 'best_validation': best_validation,
        'normalization': asdict(normalization), 'metrics': dict(metrics),
        'model_state': {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
        'optimizer_state': optimizer.state_dict(),
        'optimizer_named_state': named_optimizer_state(model, optimizer),
        'scheduler_state': scheduler.state_dict(), 'scaler_state': scaler.state_dict(),
        'ema_phase_state': dict(EMA_PHASE_STATE),
        'global_optimizer_progress':dict(GLOBAL_OPTIMIZER_STATE),
        'ema_frozen': stage.startswith('G_'),
        'units': copy.deepcopy(UNIT_CONTRACT), 'normalization_hash': stable_contract_hash(asdict(normalization)),
        'split_manifest_hash': stable_contract_hash(split_manifest), 'ni_manifest_hash': NI_SELECTION_MANIFEST['hash'],
        'data_curriculum_contract': data_curriculum_contract(),
        'curriculum_progress': copy.deepcopy(CURRICULUM_PROGRESS),
        'sample_stage_exposure': copy.deepcopy(SAMPLE_STAGE_EXPOSURE),
        'trainable_modules': [g['module_name'] for g in optimizer.param_groups],
        'rng_state': capture_rng_state(loader_generator),
    }


def validate_checkpoint(payload, normalization):
    required = {'model_state', 'optimizer_state', 'optimizer_named_state', 'scheduler_state',
                'scaler_state', 'rng_state', 'stage', 'epoch', 'global_step', 'checkpoint_role','ema_phase_state',
                'curriculum_progress', 'global_optimizer_progress','sample_stage_exposure','units','normalization_hash','split_manifest_hash','ni_manifest_hash'}
    if required - set(payload):
        raise RuntimeError(f'Checkpoint incompleto: {required - set(payload)}')
    if payload.get('checkpoint_role') not in ('literal_last','validation_best','validated_stage_final_fallback'):
        raise RuntimeError('Selection/inference artifact cannot resume training')
    expected = {
        'architecture_version': ARCHITECTURE_VERSION,
        'training_protocol_version': TRAINING_PROTOCOL_VERSION,
        'curriculum_version': CURRICULUM_VERSION,
        'checkpoint_schema_version': CHECKPOINT_SCHEMA_VERSION,
    }
    mismatches = {key: (payload.get(key), value) for key, value in expected.items()
                  if payload.get(key) != value}
    if mismatches:
        raise RuntimeError(f'Incompatible versions in checkpoint: {mismatches}')
    if payload.get('training_contract') != training_contract():
        raise RuntimeError('Run contract changed; use another run_tag')
    expected_contracts={'units':UNIT_CONTRACT,'normalization_hash':stable_contract_hash(asdict(normalization)),
        'split_manifest_hash':stable_contract_hash(split_manifest),'ni_manifest_hash':NI_SELECTION_MANIFEST['hash']}
    for key,value in expected_contracts.items():
        if payload.get(key)!=value: raise RuntimeError(f'Incompatible contract/hash: {key}')
    if payload.get('normalization') != asdict(normalization):
        raise RuntimeError('Incompatible normalization')


def validate_model_state_before_restore(model, payload):
    current = model.state_dict()
    incoming = payload['model_state']
    missing = sorted(set(current) - set(incoming))
    unexpected = sorted(set(incoming) - set(current))
    shapes = {key: (tuple(incoming[key].shape), tuple(current[key].shape))
              for key in set(current) & set(incoming) if incoming[key].shape != current[key].shape}
    if any(not torch.isfinite(v).all() for v in incoming.values() if v.is_floating_point()):
        raise RuntimeError('Checkpoint contains a nonfinite model')
    if missing or unexpected or shapes:
        raise RuntimeError(f'Incompatible model state: missing={missing[:5]}, '
                           f'unexpected={unexpected[:5]}, shapes={dict(list(shapes.items())[:5])}')


def validate_optimizer_restore(optimizer, state):
    current=optimizer.state_dict()
    if len(state['param_groups'])!=len(current['param_groups']): raise RuntimeError('Incompatible Adam groups')
    for incoming,existing,live in zip(state['param_groups'],current['param_groups'],optimizer.param_groups):
        if len(incoming['params'])!=len(existing['params']) or incoming.get('module_name')!=existing.get('module_name'):
            raise RuntimeError('Incompatible Adam parameters/groups')
        for identity,parameter in zip(incoming['params'],live['params']):
            for key,value in state['state'].get(identity,{}).items():
                if torch.is_tensor(value):
                    if not torch.isfinite(value).all(): raise RuntimeError('Nonfinite Adam moment')
                    if key in ('exp_avg','exp_avg_sq','max_exp_avg_sq') and value.shape!=parameter.shape:
                        raise RuntimeError('Incompatible Adam shape')


def load_stage_checkpoint(path,model,optimizer,scheduler,scaler,normalization,loader_generator):
    payload=torch.load(path,map_location='cpu',weights_only=False)
    validate_checkpoint(payload,normalization)
    validate_model_state_before_restore(model,payload)
    validate_optimizer_restore(optimizer,payload['optimizer_state'])
    validate_ema_restore(payload['ema_phase_state'])
    if payload.get('data_curriculum_contract')!=data_curriculum_contract(): raise RuntimeError('Incompatible curriculum')
    schedule=payload['scheduler_state']
    if set(schedule)!=set(scheduler.state_dict()) or schedule['total_updates']!=scheduler.total_updates or not 0<=schedule['update_index']<=schedule['total_updates']:
        raise RuntimeError('Incompatible scheduler')
    if scaler.is_enabled() and (set(payload['scaler_state'])!=set(scaler.state_dict()) or payload['scaler_state'].get('scale',0)<=0):
        raise RuntimeError('Incompatible scaler')
    # Dry-run on isolated objects, followed by a transaction including RNG and clocks.
    probe_optimizer=copy.deepcopy(optimizer); probe_optimizer.load_state_dict(payload['optimizer_state'])
    probe_scheduler=copy.copy(scheduler); probe_scheduler.optimizer=probe_optimizer
    probe_scheduler.load_state_dict(schedule)
    probe_scaler=copy.deepcopy(scaler); probe_scaler.load_state_dict(payload['scaler_state'])
    previous=dict(model=copy.deepcopy(model.state_dict()),optimizer=copy.deepcopy(optimizer.state_dict()),
        scheduler=copy.deepcopy(scheduler.state_dict()),scaler=copy.deepcopy(scaler.state_dict()),
        rng=capture_rng_state(loader_generator),ema=copy.deepcopy(EMA_PHASE_STATE),
        curriculum=copy.deepcopy(CURRICULUM_PROGRESS),exposure=copy.deepcopy(SAMPLE_STAGE_EXPOSURE),
        global_progress=copy.deepcopy(GLOBAL_OPTIMIZER_STATE))
    try:
        model.load_state_dict(payload['model_state']); optimizer.load_state_dict(payload['optimizer_state'])
        scheduler.load_state_dict(schedule); scaler.load_state_dict(payload['scaler_state'])
        restore_rng_state(payload['rng_state'],loader_generator)
        EMA_PHASE_STATE.clear(); EMA_PHASE_STATE.update(payload['ema_phase_state'])
        GLOBAL_OPTIMIZER_STATE.clear(); GLOBAL_OPTIMIZER_STATE.update(payload['global_optimizer_progress'])
        CURRICULUM_PROGRESS.clear(); CURRICULUM_PROGRESS.update(payload['curriculum_progress'])
        SAMPLE_STAGE_EXPOSURE.clear(); SAMPLE_STAGE_EXPOSURE.update(payload['sample_stage_exposure'])
    except Exception:
        model.load_state_dict(previous['model']); optimizer.load_state_dict(previous['optimizer'])
        scheduler.load_state_dict(previous['scheduler']); scaler.load_state_dict(previous['scaler'])
        restore_rng_state(previous['rng'],loader_generator)
        EMA_PHASE_STATE.clear(); EMA_PHASE_STATE.update(previous['ema'])
        GLOBAL_OPTIMIZER_STATE.clear(); GLOBAL_OPTIMIZER_STATE.update(previous['global_progress'])
        CURRICULUM_PROGRESS.clear(); CURRICULUM_PROGRESS.update(previous['curriculum'])
        SAMPLE_STAGE_EXPOSURE.clear(); SAMPLE_STAGE_EXPOSURE.update(previous['exposure'])
        raise
    return payload


In [27]:
def v29_store_candidate(payload, row, stage_dir):
    """Validation-only Pareto archive; full optimizer resume remains in last.pt."""
    ni=float(row.get('validation_unbounded_ni_score',float('inf')))
    excess=float(row.get('validation_retention_excess',float('inf')))
    if not (math.isfinite(ni) and math.isfinite(excess)):
        raise FloatingPointError('Invalid candidate metrics')
    path=stage_dir/'pareto_archive.json'
    entries=json.loads(path.read_text()) if path.exists() else []
    if any(e['ni_score']<=ni and e['retention_excess']<=excess for e in entries): return
    entries=[e for e in entries if not (ni<=e['ni_score'] and excess<=e['retention_excess'])]
    candidate_path=stage_dir/f"candidate_epoch_{int(row['epoch']):04d}.pt"
    atomic_torch_save({'model_state':payload['model_state'],'training_contract':training_contract(),
        'architecture_version':ARCHITECTURE_VERSION,
        'training_protocol_version':TRAINING_PROTOCOL_VERSION,
        'curriculum_version':CURRICULUM_VERSION,
        'checkpoint_schema_version':CHECKPOINT_SCHEMA_VERSION,
        'validation_metrics':dict(row),
        'selection_snapshot':copy.deepcopy(payload.get('selection_snapshot', {})),
        'stage':row['stage'],'epoch':row['epoch'],'global_step':row['global_step']},candidate_path)
    entries.append({'stage':str(row['stage']),'epoch':int(row['epoch']),
        'ni_score':ni,'retention_excess':excess,'checkpoint':str(candidate_path),
        'validation_unbounded_ni_score':ni,'validation_retention_excess':excess,
        'validation_constraints_passed':excess<=1e-12,
        **{k:row[k] for k in ('validation_field_constraints_passed','validation_field_violations','validation_energy_mae_ev_per_atom') if k in row}})
    atomic_json_dump(entries,path)


def v29_retention_components(current, baseline, element):
    definitions=[
        ('rho_relative_l2_mean',CFG.retention_rho_relative_tolerance,'relative',1e-8),
        ('potential_relative_l2_mean',CFG.retention_potential_relative_tolerance,'relative',1e-8),
        ('cohesive_mae_ev_per_atom',CFG.retention_cohesive_relative_tolerance,'relative',CFG.retention_cohesive_floor_ev_per_atom),
        ('rho_cosine_similarity_mean',CFG.retention_cosine_drop_tolerance,'drop',1e-8),
        ('cohesive_absolute_bias_ev_per_atom',CFG.retention_cohesive_relative_tolerance,'relative',CFG.retention_cohesive_floor_ev_per_atom),
        ('cohesive_error_p95_ev_per_atom',.05,'relative',CFG.retention_cohesive_floor_ev_per_atom),
        ('cohesive_max_absolute_error_ev_per_atom',.05,'relative',CFG.retention_cohesive_floor_ev_per_atom)]
    rows=[]
    for metric,tolerance,kind,floor in definitions:
        value,reference=float(current[metric]),float(baseline[metric])
        if not (math.isfinite(value) and math.isfinite(reference)):
            raise FloatingPointError(f'Nonfinite retention: {element}/{metric}')
        degradation=(reference-value if kind=='drop' else (value-reference)/max(abs(reference),floor))
        rows.append(dict(element=element,metric=metric,value=value,baseline=reference,
            degradation=degradation,tolerance=tolerance,excess=max(0.,degradation-tolerance)))
    return rows


def v29_load_model_candidate(candidate):
    payload=torch.load(candidate['checkpoint'],map_location='cpu',weights_only=False)
    expected={'architecture_version':ARCHITECTURE_VERSION,
              'training_protocol_version':TRAINING_PROTOCOL_VERSION,
              'curriculum_version':CURRICULUM_VERSION,
              'checkpoint_schema_version':CHECKPOINT_SCHEMA_VERSION}
    mismatch={key:(payload.get(key),value) for key,value in expected.items()
              if payload.get(key)!=value}
    if mismatch or payload.get('training_contract')!=training_contract():
        raise RuntimeError(f'Candidate incompatible with this V29 run: {mismatch}')
    validate_model_state_before_restore(model,payload)
    model.load_state_dict(payload['model_state'])
    if 'optimizer_named_state' in payload:
        atomic_torch_save({**payload,'completed_stage':payload.get('stage',candidate['stage']),
            'ema_phase_state':dict(EMA_PHASE_STATE)},OPTIMIZER_HANDOFF_PATH)
    return payload

def _v29_checked_field_value(element, field, value):
    value = float(value)
    if not math.isfinite(value) or value < 0:
        raise FloatingPointError(f"Invalid validation field: {element}/{field}")
    return value


def v29_make_field_references(summaries):
    """Creates a frozen reference from a snapshot, without side effects."""
    references = {}
    for element, metrics in summaries.items():
        references[str(element)] = {
            field: _v29_checked_field_value(element, field, metrics[field])
            for field in ('rho', 'potential')
        }
    return references


def v29_update_field_references(summaries, references):
    """Explicitly updates the best envelope across candidates.

    This function is used only when BUILDING a global reference across candidates.
    The eligibility check (`v29_field_constraints`) is deliberately pure.
    """
    for element, metrics in summaries.items():
        reference = references.setdefault(str(element), {})
        for field in ('rho', 'potential'):
            value = _v29_checked_field_value(element, field, metrics[field])
            reference[field] = min(float(reference.get(field, value)), value)
    return references


def v29_field_constraints(summaries, references):
    """Checks fields against frozen references; never modifies `references`."""
    violations = []
    for element, metrics in summaries.items():
        key = str(element)
        if key not in references:
            raise KeyError(f"Missing field reference for {key}")
        for field in ('rho', 'potential'):
            if field not in references[key]:
                raise KeyError(f"Missing field reference for {key}/{field}")
            value = _v29_checked_field_value(key, field, metrics[field])
            reference = _v29_checked_field_value(key, field, references[key][field])
            if value > (1 + CFG.checkpoint_field_relative_tolerance) * reference + 1e-12:
                violations.append(f'{key}:{field}')
    return not violations, violations


def v29_ae_absolute_field_constraints(summaries):
    """Scientific AE gate: absolute limits independent of input reconstruction."""
    violations = []
    for element, metrics in summaries.items():
        rho = _v29_checked_field_value(element, 'rho', metrics['rho'])
        potential = _v29_checked_field_value(element, 'potential', metrics['potential'])
        if rho > CFG.ae_rho_relative_l2_max:
            violations.append(f'{element}:rho')
        if potential > CFG.ae_potential_relative_l2_max:
            violations.append(f'{element}:potential')
    return not violations, violations


def v29_energy_selection_stage(stage):
    return stage == 'autoencoder' or stage in ('E_full', 'F_periodic_ramp', 'P_cohesive_refinement') or stage.startswith('G_')


def v29_selection_snapshot(model, loader, stage):
    summaries, _ = physical_validation_snapshot(model, loader, autoencoder=stage == 'autoencoder')
    if stage.startswith('G_'):
        for element, retention_loader in retention_loaders.items():
            retention, _ = physical_validation_snapshot(model, retention_loader)
            summaries.update(retention)
    return summaries

def v29_candidate_field_audit(path):
    payload = torch.load(path, map_location='cpu', weights_only=False)
    metrics = payload.get('metrics', payload.get('validation_metrics', {}))
    result={key:metrics[key] for key in ('validation_field_constraints_passed',
        'validation_field_violations','validation_energy_mae_ev_per_atom') if key in metrics}
    gates=metrics.get('physical_energy_gates')
    if gates is None:
        gates={k:v for k,v in json.loads(metrics.get('checkpoint_gates','{}')).items() if k.startswith(('Al_','Fe_','Ni_'))}
    result['validation_energy_gates_passed']=bool(gates) and all(gates.values())
    result['candidate_trained']=int(payload.get('epoch',-1))>=0
    return result


def ae_checkpoint_gates(global_step,total_steps,warmup_updates,full_updates,absolute_fields_passed,finite_validation):
    coverage={e:sample_coverage('autoencoder',e) for e in CFG.pretrain_elements}
    gates=dict(warmup_complete=global_step>=warmup_updates,
        ae_ramp_complete=objective_ramp_complete('autoencoder',global_step/max(total_steps,1)),
        minimum_full_regime_updates=full_updates>=CFG.ae_min_full_regime_updates,
        coverage_Al=coverage['Al']>=CFG.ae_min_element_coverage,
        coverage_Fe=coverage['Fe']>=CFG.ae_min_element_coverage,
        finite_validation=bool(finite_validation),
        absolute_field_constraints=bool(absolute_fields_passed))
    return gates,coverage


# Absolute physical quality gates are independent of loss weights and selection score.
ENERGY_GATE_LIMITS={e:dict(mae=.10,bias=.02,p95=.30,maximum=.60,slope_tolerance=.25)
                   for e in ('Al','Fe','Ni')}

def per_element_energy_gates(snapshot,normalization):
    gates={}; diagnostics={}
    scale=normalization.cohesive_scale_ev_per_atom
    for element,metrics in snapshot.items():
        limits=ENERGY_GATE_LIMITS[element]
        for name,key in [('mae','energy'),('bias','energy_bias'),('p95','energy_p95'),('maximum','energy_max')]:
            value=abs(float(metrics[key]))*scale
            gates[f'{element}_{name}']=math.isfinite(value) and value<=limits[name]
            diagnostics[f'{element}_{name}_ev_per_atom']=value
        supported=bool(metrics['energy_slope_supported'])
        gates[f'{element}_slope']=not supported or abs(metrics['energy_slope']-1)<=limits['slope_tolerance']
        diagnostics[f'{element}_slope_supported']=supported
        if element=='Al': gates['Al_tail']=gates['Al_p95'] and gates['Al_maximum']
    return gates,diagnostics


In [28]:
def structural_last_block(model):
    encoder = model.context_encoder
    return encoder.blocks[-1] if hasattr(encoder, 'blocks') else encoder.transformer.layers[-1]

# =============================================================================
# STEP 18 — Training loop
# Purpose: Applies LR groups, warmup+cosine, accumulation, per-module clipping, EMA, and rollback.
# Inputs: DataLoaders, number of epochs, and stage learning rate.
# Outputs: CSV history and checkpoints for A/D/E/F/G.
# Reproducibility: run after all preceding cells; do not change the seed during the run.
# =============================================================================
def move_batch(data: Batch) -> Batch:
    return data.to(DEVICE, non_blocking=True)


def stage_module_groups(model,stage):
    online=[('solution_online_encoder',model.field_online_encoder,CFG.online_field_lr_multiplier),
            ('reconstruction_decoder',model.field_reconstruction_decoder,CFG.online_field_lr_multiplier)]
    ni=[('ni_embedding',model.context_encoder.ni_embedding,CFG.ni_adapter_lr_multiplier),
        ('ni_field_adapter',model.ni_field_adapter,CFG.ni_adapter_lr_multiplier),
        ('ni_shared_gain',model.ni_shared_gain,1.0),
        ('ni_energy_calibration',model.ni_energy_calibration,1.0)]
    head=[('cohesive_head',model.cohesive_head,CFG.cohesive_head_lr_multiplier)]
    if stage=='autoencoder':
        if bool(model.field_target_initialized_from_ae):
            raise RuntimeError('AE already frozen: use a new model to resume the AE stage.')
        specs=[(n,m,1.) for n,m,_ in online]+head
    elif stage=='D_latent':
        specs=[('potential_context',model.context_encoder,1.),('predictor',model.predictor,1.)]
        # AE permanently excluded from JEPA optimizers.
    elif stage=='P_cohesive_refinement':
        specs=[('cohesive_head',model.cohesive_head,1.0)]
    elif stage=='G_cohesive_calibration':
        specs=[('ni_energy_calibration',model.ni_energy_calibration,1.0)]
    elif stage=='G_adapter_warmup': specs=ni
    elif stage=='G_joint_finetune':
        specs=ni+head+[('predictor',model.predictor,CFG.predictor_joint_lr_multiplier),
                      ('prediction_decoder',model.field_prediction_decoder,CFG.residual_decoder_lr_multiplier),
                      ('structural_last_block',structural_last_block(model),CFG.encoder_joint_lr_multiplier)]
        # Ni tuning also leaves the original AE frozen.
    elif stage=='G_retention_repair':
        # Ni-specific gains/embedding/calibration and both electronic encoders frozen.
        # Shared modules are repaired with Al/Fe replay plus Ni supervised examples.
        specs=[('cohesive_head',model.cohesive_head,.25),
               ('predictor',model.predictor,.05),
               ('prediction_decoder',model.field_prediction_decoder,.10)]
    elif stage in ('E_density','E_density_potential','E_full','F_periodic_ramp'):
        specs=[('potential_context',model.context_encoder,.5),('predictor',model.predictor,.5),
               ('prediction_decoder',model.field_prediction_decoder,1.)]
        if stage in ('E_full','F_periodic_ramp'): specs+=head
    else: raise ValueError(stage)
    if electronic_online_active(stage):
        specs += [('electronic_online_encoder', model.electronic_online_encoder,
                   CFG.electronic_online_lr_multiplier)]
    if stage.startswith('G_'):
        specs=[(n,m,w*CFG.ni_cohesive_lr_multiplier if n=='cohesive_head' else w) for n,m,w in specs]
    for p in model.parameters(): p.requires_grad_(False); p.grad=None
    groups=[]; seen=set()
    for name,module,multiplier in specs:
        parameters=list(module.parameters())
        # Ni correction must stay exactly zero before Ni adaptation (including weight decay).
        if not stage.startswith('G_'):
            ni_ids={id(p) for p in model.context_encoder.ni_embedding.parameters()}
            parameters=[p for p in parameters if id(p) not in ni_ids]
        if any(id(p) in seen for p in parameters): raise RuntimeError(f'Duplicate parameter: {name}')
        seen.update(id(p) for p in parameters)
        for p in parameters: p.requires_grad_(True)
        groups.append(dict(name=name,params=parameters,lr_multiplier=multiplier))
    return groups



def make_stage_optimizer(model: PeriodicFieldJEPA, stage: str, base_lr: float) -> AdamW:
    groups = [
        {"params": group["params"], "lr": base_lr * group["lr_multiplier"],
         "lr_multiplier": group["lr_multiplier"], "module_name": group["name"]}
        for group in stage_module_groups(model, stage)
    ]
    return AdamW(groups, lr=base_lr, weight_decay=CFG.weight_decay)


class WarmupCosineScheduler:
    """Stage-local warm-up + cosine decay.

    V31 intentionally does not share one decaying clock across D/E/F or G.
    Compatible AdamW moments are still transferred, but each newly posed
    optimization problem receives a fresh useful LR budget.
    """
    def __init__(self, optimizer: torch.optim.Optimizer, base_lr: float, total_updates: int, stage: str):
        self.optimizer=optimizer; self.base_lr=float(base_lr); self.stage=str(stage)
        self.total_updates=max(int(total_updates),1)
        self.warmup_updates=max(1,int(round(CFG.warmup_fraction*self.total_updates)))
        self.update_index=0
        self._apply()

    def factor(self) -> float:
        index=min(max(int(self.update_index),0),self.total_updates)
        if index < self.warmup_updates:
            return max((index+1)/self.warmup_updates,1e-3)
        progress=(index-self.warmup_updates)/max(self.total_updates-self.warmup_updates,1)
        floor=CFG.source_stage_lr_floor
        return floor+(1-floor)*0.5*(1.0+math.cos(math.pi*min(max(progress,0.0),1.0)))

    def _apply(self) -> None:
        factor=self.factor()
        for group in self.optimizer.param_groups:
            group['lr']=self.base_lr*float(group.get('lr_multiplier',1.0))*factor

    def step(self) -> None:
        self.update_index+=1
        self._apply()

    def state_dict(self) -> dict[str, Any]:
        return {'base_lr':self.base_lr,'total_updates':self.total_updates,
                'warmup_updates':self.warmup_updates,'update_index':self.update_index,'stage':self.stage}

    def load_state_dict(self,state:Mapping[str,Any]) -> None:
        self.base_lr=float(state['base_lr']); self.total_updates=int(state['total_updates'])
        self.warmup_updates=int(state['warmup_updates']); self.update_index=int(state['update_index'])
        if str(state.get('stage',self.stage))!=self.stage:
            raise RuntimeError('Scheduler restored for an incompatible stage')
        self._apply()


def _progressive_factor(progress,start,end):
    return cosine_window(progress,start,end)

def _joint_release_factor(progress,start,full):
    """Own LR clock for a late-released G_joint parameter group."""
    p=float(progress)
    if p < start:
        return 0.0
    if p < full:
        return cosine_window(p,start,full)
    if full >= 1.0:
        return 1.0
    tail=(p-full)/max(1-full,1e-12)
    return CFG.joint_group_lr_floor+(1-CFG.joint_group_lr_floor)*0.5*(1+math.cos(math.pi*min(max(tail,0.0),1.0)))

def apply_progressive_joint_lrs(optimizer,scheduler,stage,progress):
    """Stage-local LR; G_joint late modules receive release-local clocks."""
    scheduler._apply()
    if stage!='G_joint_finetune':
        for group in optimizer.param_groups:
            group['progressive_unfreeze_factor']=1.0
        return
    windows={
        'predictor':(CFG.joint_predictor_start,CFG.joint_predictor_full),
        'prediction_decoder':(CFG.joint_decoder_start,CFG.joint_decoder_full),
        'structural_last_block':(CFG.joint_structural_start,CFG.joint_structural_full),
    }
    for group in optimizer.param_groups:
        name=group.get('module_name')
        if name in windows:
            factor=_joint_release_factor(progress,*windows[name])
            group['progressive_unfreeze_factor']=float(factor)
            group['lr']=scheduler.base_lr*float(group.get('lr_multiplier',1.0))*factor
        else:
            group['progressive_unfreeze_factor']=1.0



LAST_CLIPPING_DIAGNOSTICS={}

def clip_gradients_by_module(optimizer):
    norms={}; LAST_CLIPPING_DIAGNOSTICS.clear()
    for group in optimizer.param_groups:
        params=[p for p in group['params'] if p.grad is not None]
        if not params: continue
        name=group.get('module_name','module')
        norm=torch.linalg.vector_norm(torch.stack([p.grad.detach().float().norm() for p in params]))
        limit=min(CFG.gradient_clip_norm,CFG.gradient_clip_rms*math.sqrt(max(sum(p.numel() for p in params),1)))
        if torch.isfinite(norm): torch.nn.utils.clip_grad_norm_(params,limit)
        post=torch.linalg.vector_norm(torch.stack([p.grad.detach().float().norm() for p in params]))
        norms[name]=float(norm)
        LAST_CLIPPING_DIAGNOSTICS[name]=dict(pre_norm=float(norm),post_norm=float(post),
            limit=limit,clipped=bool(norm>limit),nonfinite=not bool(torch.isfinite(norm) and torch.isfinite(post)))
    return norms


def autocast_context():
    enabled = CFG.amp and DEVICE.type == "cuda"
    return torch.autocast(device_type=DEVICE.type, enabled=enabled, dtype=torch.float16)


def make_grad_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=CFG.amp and DEVICE.type == "cuda", init_scale=1024.)
    except TypeError:
        return torch.cuda.amp.GradScaler(enabled=CFG.amp and DEVICE.type == "cuda", init_scale=1024.)


def periodic_translation_augment_batch(data):
    probability=float(CFG.translation_augmentation_probability)
    if probability<=0 or float(torch.rand((),device=data.pos_fractional.device))>=probability:
        return data
    shifts=tuple(int(torch.randint(0,n,(),device=data.pos_fractional.device)) for n in CFG.canonical_grid)
    return translate_periodic_batch(data,shifts)



def aggregate_terms(rows):
    if not rows: raise ValueError('Empty loader')
    weights=np.asarray([r['_records'] for r in rows],dtype=float)
    return {key:float(np.average([r[key] for r in rows],weights=weights)) for key in rows[0] if key!='_records'}




# =============================================================================
# Detailed history per optimizer update (instrumentation; does not change the architecture)
# =============================================================================
def _atomic_write_dataframe_csv(frame: pd.DataFrame, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".{os.getpid()}.tmp")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def optimizer_update_history_path(stage: str) -> Path:
    return CFG.run_dir / f"optimizer_history_{stage}.csv"


def trim_optimizer_update_history(stage: str, max_valid_update: int) -> None:
    """Aligns the detailed log with the restored checkpoint without touching model weights."""
    path = optimizer_update_history_path(stage)
    if not path.exists():
        return
    frame = pd.read_csv(path)
    if "stage_optimizer_update" not in frame.columns:
        print(f"Warning: {path.name} has no stage_optimizer_update; old log preserved.")
        return
    frame = frame.loc[frame["stage_optimizer_update"] <= int(max_valid_update)].copy()
    if len(frame):
        _atomic_write_dataframe_csv(frame, path)
    else:
        path.unlink()


def append_optimizer_update_history(stage: str, rows: list[dict[str, Any]]) -> None:
    """Persists one row per successful optimizer.step; does not participate in training."""
    if not rows:
        return
    path = optimizer_update_history_path(stage)
    new = pd.DataFrame(rows)
    if path.exists():
        old = pd.read_csv(path)
        new = pd.concat([old, new], ignore_index=True)
    new = (
        new.drop_duplicates(["stage", "stage_optimizer_update"], keep="last")
           .sort_values("stage_optimizer_update")
           .reset_index(drop=True)
    )
    _atomic_write_dataframe_csv(new, path)


def _mean_accumulated_terms(rows: list[dict[str, Any]]) -> dict[str, float]:
    """Mean weighted by graphs in the gradient accumulation window."""
    if not rows:
        return {}
    weights = np.asarray([float(row["_records"]) for row in rows], dtype=float)
    keys = sorted({key for row in rows for key in row if key != "_records"})
    result = {}
    for key in keys:
        values = [float(row[key]) for row in rows if key in row]
        key_weights = [weights[i] for i, row in enumerate(rows) if key in row]
        if values:
            result[key] = float(np.average(values, weights=np.asarray(key_weights, dtype=float)))
    return result


def run_epoch(
    model: PeriodicFieldJEPA,
    loader: DataLoader,
    stage: str,
    optimizer: torch.optim.Optimizer | None,
    scaler: Any,
    global_step: int,
    total_steps: int,
    scheduler: WarmupCosineScheduler | None = None,
    epoch_index: int | None = None,
) -> tuple[dict[str, float], int, float]:
    training = optimizer is not None
    if stage != "autoencoder":
        require_initialized_jepa_target(model)
    elif training and bool(model.field_target_initialized_from_ae):
        raise RuntimeError("AE already transferred to the target: resume the AE checkpoint or use a new model.")
    model.train(training)
    model.field_target_encoder.eval()
    for module in model.children():
        if not any(p.requires_grad for p in module.parameters()):
            module.eval()
    if stage.startswith("G_"):
        # eval() disables dropout in the frozen backbone, but still allows
        # gradients through the trainable Ni embedding correction.
        model.context_encoder.eval()
    if stage == 'P_cohesive_refinement':
        model.context_encoder.eval(); model.predictor.eval()
    rows, tau = [], float(CFG.ema_tau_start)
    lr_samples = {g['module_name']: [] for g in optimizer.param_groups} if training else {}
    gradient_samples = {g['module_name']: [] for g in optimizer.param_groups} if training else {}
    skipped_updates = 0
    accumulation = max(CFG.accumulation_steps, 1) if training else 1
    optimizer_update_rows: list[dict[str, Any]] = []
    accumulation_log_rows: list[dict[str, Any]] = []
    accumulation_sample_keys: list[str] = []
    gradient_conflict_rows: list[dict[str, Any]] = []
    retention_loader=globals().get('RETENTION_STREAM_LOADER') if stage.startswith('G_') else None
    retention_iterator=iter(retention_loader) if training and retention_loader is not None else None
    if training:
        optimizer.zero_grad(set_to_none=True)
    for batch_index, batch in enumerate(loader):
        if training and global_step >= total_steps: break
        batch = move_batch(batch)
        if training:
            accumulation_sample_keys.extend(record_sample_batch_exposure(
                stage, -1 if epoch_index is None else epoch_index, batch))
            batch = periodic_translation_augment_batch(batch)
        with torch.set_grad_enabled(training):
            with autocast_context():
                stage_progress = min(max(
                    (global_step + (1 if training else 0)) / max(total_steps, 1), 0.0), 1.0)
                if training and scheduler is not None:
                    apply_progressive_joint_lrs(optimizer,scheduler,stage,stage_progress)

                loss, terms = compute_loss(model, batch, stage, stage_progress)
                if not bool(torch.isfinite(loss)):
                    raise FloatingPointError(f"Nonfinite loss in {stage}; training stopped before optimizer.step")
            detached_terms = {key: value.detach() for key, value in terms.items()}
            if training:
                accumulation_log_rows.append({"_records": batch.num_graphs, **detached_terms})
                # Last accumulation group can contain fewer batches.
                group_start = (batch_index // accumulation) * accumulation
                group_size = min(accumulation, len(loader) - group_start)
                boundary = (batch_index + 1) % accumulation == 0 or batch_index + 1 == len(loader)
                if boundary and should_audit_gradient_conflicts(global_step + 1):
                    gradient_conflict_rows.extend(audit_gradient_conflicts(
                        model, stage, terms, stage_progress, global_step + 1))
                if stage=='P_cohesive_refinement':
                    illegal=[n for n,p in model.named_parameters() if p.requires_grad and not n.startswith('cohesive_head.')]
                    if illegal: raise RuntimeError(f'P: forbidden parameters {illegal}')
                if boundary and retention_iterator is not None and (global_step+1)%CFG.retention_stream_every_updates==0:
                    try: replay_batch=next(retention_iterator)
                    except StopIteration:
                        retention_iterator=iter(retention_loader); replay_batch=next(retention_iterator)
                    replay_batch=move_batch(replay_batch)
                    if not all(str(e) in CFG.pretrain_elements for e in replay_batch.element):
                        raise AssertionError('Retention stream must contain only training Al/Fe')
                    accumulation_sample_keys.extend(record_sample_batch_exposure(stage,epoch_index,replay_batch))
                    with autocast_context():
                        rz,rr,rv,re=predicted_outputs(model,operator_batch(replay_batch))
                        retention_weight=stage_loss_weights(stage,stage_progress).get('retention_distillation',0.0)
                        retention_loss=retention_weight*shared_replay_distillation(model,replay_batch,rz,rr,rv,re)
                    if not torch.isfinite(retention_loss): raise FloatingPointError('Nonfinite retention loss')
                    scaler.scale(retention_loss).backward()
                    accumulation_log_rows.append(dict(_records=replay_batch.num_graphs,retention_separate_stream=retention_loss.detach()))
                scaler.scale(loss / group_size).backward()
                if boundary:
                    scaler.unscale_(optimizer)
                    # Truly frozen progressive groups must not accumulate Adam moments before release.
                    if stage=='G_joint_finetune':
                        for group in optimizer.param_groups:
                            if float(group.get('progressive_unfreeze_factor',1.0)) <= 1e-12:
                                for parameter in group['params']:
                                    parameter.grad=None
                    if stage=='P_cohesive_refinement':
                        illegal=[n for n,p in model.named_parameters() if p.grad is not None and not n.startswith('cohesive_head.')]
                        if illegal: raise RuntimeError(f'P: forbidden gradients {illegal}')
                    norms = clip_gradients_by_module(optimizer)
                    for group in optimizer.param_groups:
                        name=group['module_name']
                        if name in norms: gradient_samples[name].append(norms[name])
                    # LR actually applied to this optimizer.step, before the scheduler advances.
                    applied_lrs = {group['module_name']: float(group['lr']) for group in optimizer.param_groups}
                    previous_scale = scaler.get_scale()
                    finite_gradients=all(math.isfinite(v) for v in norms.values())
                    if finite_gradients: scaler.step(optimizer)
                    if scaler.is_enabled() or finite_gradients: scaler.update()
                    update_succeeded = scaler.get_scale() >= previous_scale and all(math.isfinite(v) for v in norms.values())
                    optimizer.zero_grad(set_to_none=True)
                    if update_succeeded:
                        for group in optimizer.param_groups:
                            lr_samples[group['module_name']].append(float(group['lr']))
                        global_step += 1
                        GLOBAL_OPTIMIZER_STATE['valid_updates']+=1
                        GLOBAL_OPTIMIZER_STATE['ni_valid_updates' if stage.startswith('G_') else 'alfe_valid_updates']+=1
                        if not stage.startswith('G_'): advance_ema_phase(stage)

                        # Instrumentation only: aggregates terms already computed in the accumulation window.
                        mean_terms = _mean_accumulated_terms(accumulation_log_rows)
                        optimizer_update_rows.append({
                            'stage': stage,
                            'epoch': int(epoch_index) if epoch_index is not None else -1,
                            'stage_optimizer_update': int(global_step),
                            'stage_progress': float(stage_progress),
                            'training_objective': float(mean_terms.get('total', float('nan'))),
                            **{f'loss_{key}': value for key, value in mean_terms.items()},
                            **{f'lr_{name}': value for name, value in applied_lrs.items()},
                            **{f'unfreeze_{group["module_name"]}': float(group.get('progressive_unfreeze_factor',1.0)) for group in optimizer.param_groups},
                            **{f'grad_norm_{name}': float(value) for name, value in norms.items()},
                            **{f'balance_{key.split(":",1)[1]}':float(value)
                               for key,value in TASK_GRADIENT_BALANCE_STATE.get('factors',{}).items()
                               if key.startswith(stage+':')},
                            **{f'clip_{name}_{key}':value for name,diag in LAST_CLIPPING_DIAGNOSTICS.items() for key,value in diag.items()},
                            'ema_frozen':stage.startswith('G_'), 'global_alfe_updates':EMA_PHASE_STATE['completed_updates'],
                            'global_optimizer_updates':GLOBAL_OPTIMIZER_STATE['valid_updates'], 'ema_tau':EMA_PHASE_STATE['last_tau'],
                        })

                        record_sample_optimizer_update(stage, accumulation_sample_keys)
                        if stage in SOURCE_CURRICULUM_STAGES:
                            GLOBAL_OPTIMIZER_STATE['source_curriculum_updates'] += 1
                        elif stage in NI_CURRICULUM_STAGES:
                            GLOBAL_OPTIMIZER_STATE['ni_curriculum_updates'] += 1
                        if scheduler is not None:
                            scheduler.step()
                        tau = effective_ema_tau(stage, global_step - 1, total_steps)
                        # Exactly one EMA per successful JEPA optimizer update.
                        # No EMA in AE pretraining, validation, or AMP-skipped steps.
                        # A frozen source still participates in the EMA recurrence.
                        if not stage.startswith('G_'):
                            tau=effective_ema_tau(stage,global_step,total_steps)
                            ema_authorized = not (
                                stage=='D_latent' and CFG.freeze_ema_until_d_low_ready
                                and not bool(globals().get('D_LOW_GATE_READY',False))
                            )
                            if stage not in ('autoencoder','P_cohesive_refinement') and ema_authorized:
                                update_jepa_target(model,stage,tau)
                            elif not ema_authorized:
                                tau=1.0
                    else:
                        skipped_updates += 1
                        skipped_row=dict(stage=stage,epoch=epoch_index,batch=batch_index,stage_valid_updates=global_step,
                            overflow=scaler.get_scale()<previous_scale,nonfinite_gradients=not finite_gradients,
                            scale_before=previous_scale,scale_after=scaler.get_scale(),
                            clipping=LAST_CLIPPING_DIAGNOSTICS.copy())
                        with (CFG.run_dir/'invalid_optimizer_updates.jsonl').open('a') as handle:
                            handle.write(json.dumps(skipped_row)+'\n')
                    # The window ends even if AMP skips the update, because gradients have already been zeroed.
                    accumulation_log_rows.clear()
                    accumulation_sample_keys.clear()
        rows.append({"_records": batch.num_graphs, **{key: float(value) for key, value in detached_terms.items()}})
    result=aggregate_terms(rows)
    result['records_seen']=sum(r['_records'] for r in rows)
    if training:
        result['amp_skipped_updates']=skipped_updates
        result['nonfinite_or_overflow_updates']=skipped_updates
        result['ema_frozen']=stage.startswith('G_')
        for element, values in ENERGY_BIAS_FIFO.values.items():
            result[f'energy_bias_fifo_count_{element}'] = len(values)
            result[f'energy_bias_fifo_ready_{element}'] = len(values) >= CFG.energy_bias_min_samples
            if len(values) >= CFG.energy_bias_min_samples:
                result[f'energy_bias_fifo_mean_normalized_{element}'] = float(np.mean(values))
        for name,values in lr_samples.items():
            result[f'lr_{name}_mean']=float(np.mean(values)) if values else float('nan')
            result[f'lr_{name}_min']=float(min(values)) if values else float('nan')
            result[f'lr_{name}_max']=float(max(values)) if values else float('nan')
        for name,values in gradient_samples.items():
            result[f'grad_norm_{name}_max']=float(max(values)) if values else float('nan')
        # One write per epoch; the files are for diagnostics only.
        append_optimizer_update_history(stage, optimizer_update_rows)
        append_gradient_conflict_history(gradient_conflict_rows)
        flush_sample_stage_exposure()
    return result, global_step, tau


def early_stop_policy(stage, epochs):
    fraction=CFG.early_stopping_min_fraction
    if stage == 'autoencoder': fraction=max(fraction, CFG.ae_energy_ramp_end+0.05)
    if stage == 'D_latent': fraction=max(fraction, CFG.high_band_ramp[1]+0.05)
    minimum=min(epochs,max(3,math.ceil(epochs*fraction)))
    patience=CFG.early_stopping_patience
    if stage == 'G_joint_finetune': minimum=min(epochs,max(minimum,CFG.ni_joint_min_epochs))
    if stage == 'P_cohesive_refinement': minimum=min(3,epochs); patience=CFG.refinement_patience
    return minimum,patience


def update_early_stop(state, score, epoch, stage, epochs):
    state=dict(state)
    minimum,patience=early_stop_policy(stage,epochs)
    # Ignore patience before the curriculum has reached its stable objective.
    if epoch+1 < minimum:
        state.update(reference=float(score),bad_epochs=0,stopped=False)
        return state
    reference=float(state.get('reference',float('inf')))
    delta=CFG.early_stopping_relative_delta*max(abs(reference),1e-12)
    if not math.isfinite(reference) or score < reference-delta:
        state.update(reference=float(score),bad_epochs=0)
    else:
        state['bad_epochs']=int(state.get('bad_epochs',0))+1
    state['stopped']=bool(CFG.early_stopping and state['bad_epochs']>=patience)
    state['minimum_epochs']=minimum; state['patience']=patience
    return state


def train_stage(
    model: PeriodicFieldJEPA,
    stage: str,
    train_loader: DataLoader,
    validation_loader: DataLoader,
    epochs: int,
    learning_rate: float,
    normalization: Normalization,
    optimizer: torch.optim.Optimizer | None = None,
    validation_score_callback: Any | None = None,
) -> pd.DataFrame:
    global ENERGY_BIAS_FIFO
    if stage.startswith('G_') and validation_score_callback is None:
        validation_score_callback = globals().get('ni_retention_validation_score')
        if validation_score_callback is None:
            raise ValueError('Ni phases require the initialized validation/retention callback')
    ENERGY_BIAS_FIFO = ElementEnergyResidualFIFO(CFG.energy_bias_buffer_capacity, CFG.energy_bias_min_samples)
    field_references = {}
    full_regime_updates=0
    stage_dir = CFG.run_dir / "checkpoints" / stage
    stage_dir.mkdir(parents=True, exist_ok=True)
    last_path, best_path = stage_dir / "last.pt", stage_dir / "best.pt"
    if optimizer is not None:
        raise ValueError('Pass optimizer=None; train_stage creates the groups and migrates compatible Adam moments')
    optimizer = make_stage_optimizer(model, stage, learning_rate)
    has_stage_checkpoint = bool(CFG.resume and last_path.exists())
    adam_handoff = ({'restored_count': 0, 'new_count': 0, 'source_stage': None}
                    if has_stage_checkpoint else load_optimizer_handoff(model, optimizer))
    write_trainability_report(model, optimizer, stage, adam_handoff)
    curriculum_plan = data_curriculum_plan(stage, epochs, train_loader)
    validate_data_curriculum_plan(stage, curriculum_plan)
    planned_steps = sum(row['optimizer_updates'] for row in curriculum_plan)
    total_steps = max(1, int(round(planned_steps * CFG.optimizer_update_budget_scale)))
    atomic_json_dump({'stage': stage, 'planned_from_epochs': planned_steps,
        'budget_scale': CFG.optimizer_update_budget_scale, 'optimizer_update_budget': total_steps},
        stage_dir / 'optimizer_update_budget.json')
    scheduler = WarmupCosineScheduler(optimizer, learning_rate, total_steps, stage)
    if stage=='P_cohesive_refinement':
        scheduler.warmup_updates=max(scheduler.warmup_updates, math.ceil(.20*scheduler.total_updates))
        scheduler._apply()
    scaler = make_grad_scaler()
    start_epoch, global_step, best_validation = 0, 0, float("inf")
    early_state={'reference':float('inf'),'bad_epochs':0,'stopped':False}
    readiness_state={'streak':0,'passed':False,'required':1 if CFG.run_mode in ('SMOKE','QUICK') else CFG.readiness_patience}

    if CFG.resume and last_path.exists():
        payload = load_stage_checkpoint(
            last_path, model, optimizer, scheduler, scaler, normalization, train_loader.generator
        )
        start_epoch = int(payload["epoch"]) + 1
        global_step = int(payload["global_step"])
        best_validation = float(payload["best_validation"])
        early_state=payload.get('early_stopping_state',early_state)
        readiness_state=payload.get('readiness_state',readiness_state)
        if 'spectral_competence_state' in payload:
            SPECTRAL_COMPETENCE_STATE.clear(); SPECTRAL_COMPETENCE_STATE.update(copy.deepcopy(payload['spectral_competence_state']))
        if 'task_gradient_balance_state' in payload:
            TASK_GRADIENT_BALANCE_STATE.clear(); TASK_GRADIENT_BALANCE_STATE.update(copy.deepcopy(payload['task_gradient_balance_state']))
        if 'd_competence_meta' in payload:
            D_COMPETENCE_META.clear(); D_COMPETENCE_META.update(copy.deepcopy(payload['d_competence_meta']))
            globals()['D_LOW_GATE_READY']=bool(D_COMPETENCE_META.get('low_gate_ready',False))
        ENERGY_BIAS_FIFO.load_state_dict(payload.get('energy_bias_fifo', {}))
        field_references = copy.deepcopy(payload.get('field_selection_references', {}))
        full_regime_updates=int(payload.get('full_regime_updates',0))
        saved_curriculum_contract = payload.get('data_curriculum_contract')
        current_curriculum_contract = data_curriculum_contract()
        if saved_curriculum_contract != current_curriculum_contract:
            raise RuntimeError(
                f'Checkpoint for {stage} incompatible with the current V29 curriculum. '
                'Do not resume after changing the manifest, fractions, or seed.')
        if early_state.get('stopped',False): start_epoch=epochs
        print(f"Resuming {stage} at epoch {start_epoch}")

    # Synchronizes only the detailed log with the checkpoint actually restored.
    trim_optimizer_update_history(stage, global_step)

    if stage != "autoencoder" and start_epoch == 0:
        model.reset_predicted_latent_queue()
    # A resumed JEPA checkpoint owns its accumulated teacher; never recopy AE here.
    if stage != "autoencoder":
        require_initialized_jepa_target(model)
    for module in model.children():
        if not any(p.requires_grad for p in module.parameters()):
            module.eval()

    # Relative field constraints are meaningful only after the AE. Their reference is
    # captured once at phase entry and remains frozen throughout the stage.
    if v29_energy_selection_stage(stage) and stage != 'autoencoder' and not field_references:
        entry = v29_selection_snapshot(model, validation_loader, stage)
        field_references = v29_make_field_references(entry)
        if not best_path.exists():
            entry_score = (entry[CFG.transfer_element]['energy'] * normalization.cohesive_scale_ev_per_atom
                           if stage.startswith('G_') else fixed_physical_score(entry))
            entry_metrics = dict(stage=stage, epoch=-1, validation_field_constraints_passed=True,
                validation_energy_mae_ev_per_atom=entry_score, validation_field_violations='',
                checkpoint_reason='phase_entry_fallback',trained_candidate=False,
                field_reference_policy='frozen_phase_entry',
                physical_energy_gates=per_element_energy_gates(entry,normalization)[0])
            entry_payload = checkpoint_payload(model, optimizer, scheduler, scaler, stage, -1,
                global_step, entry_score, normalization, entry_metrics, train_loader.generator)
            entry_payload['selection_snapshot'] = copy.deepcopy(entry)
            entry_payload['field_selection_references'] = copy.deepcopy(field_references)
            entry_payload['energy_bias_fifo'] = ENERGY_BIAS_FIFO.state_dict()
            atomic_torch_save(entry_payload, best_path)
            best_validation = entry_score
    history = []
    for epoch in range(start_epoch, epochs):
        epoch_started = time.perf_counter()
        steps_at_start=global_step
        active_train_loader, curriculum_info = data_curriculum_loader(
            stage, epoch, epochs, train_loader)
        record_data_curriculum_epoch(stage, epoch, curriculum_info, active_train_loader)
        # Transactional epoch rollback, one conservative retry at half LR/scaler.
        rollback = dict(model={k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
            optimizer=copy.deepcopy(optimizer.state_dict()), scheduler=copy.deepcopy(scheduler.state_dict()),
            scaler=copy.deepcopy(scaler.state_dict()), rng=capture_rng_state(active_train_loader.generator),
            fifo=ENERGY_BIAS_FIFO.state_dict(), ema=copy.deepcopy(EMA_PHASE_STATE),
            exposure=copy.deepcopy(SAMPLE_STAGE_EXPOSURE),global_progress=copy.deepcopy(GLOBAL_OPTIMIZER_STATE))
        for attempt in range(2):
            try:
                train_metrics, global_step, tau = run_epoch(
                    model, active_train_loader, stage, optimizer, scaler, global_step, total_steps, scheduler,
                    epoch_index=epoch,
                )
                if not all(bool(torch.isfinite(v).all()) for v in model.state_dict().values() if v.is_floating_point()):
                    raise FloatingPointError('Nonfinite parameter/buffer after optimizer.step')
                break
            except FloatingPointError as error:
                model.load_state_dict(rollback['model'])
                optimizer.load_state_dict(rollback['optimizer']); optimizer.zero_grad(set_to_none=True)
                scheduler.load_state_dict(rollback['scheduler'])
                if scaler.is_enabled():
                    state = copy.deepcopy(rollback['scaler'])
                    state['scale'] = max(float(state['scale']) * 0.5, 1.0)
                    scaler.load_state_dict(state)
                restore_rng_state(rollback['rng'], active_train_loader.generator)
                ENERGY_BIAS_FIFO.load_state_dict(rollback['fifo'])
                EMA_PHASE_STATE.clear(); EMA_PHASE_STATE.update(rollback['ema'])
                GLOBAL_OPTIMIZER_STATE.clear(); GLOBAL_OPTIMIZER_STATE.update(rollback['global_progress'])
                SAMPLE_STAGE_EXPOSURE.clear(); SAMPLE_STAGE_EXPOSURE.update(rollback['exposure'])
                global_step = steps_at_start
                trim_optimizer_update_history(stage, global_step)
                scheduler.base_lr *= 0.5; scheduler._apply()
                print(json.dumps(dict(event='V29_nonfinite_rollback', stage=stage, epoch=epoch,
                    attempt=attempt + 1, base_lr=scheduler.base_lr, reason=str(error))))
                if attempt == 1:
                    raise FloatingPointError('Nonfinite values persist after rollback; phase stopped with weights restored') from error
        del rollback
        # Count successful optimizer updates whose applied objective was fully ramped.
        threshold=max(scheduler.warmup_updates,math.ceil(CFG.ae_energy_ramp_end*total_steps)) if stage=='autoencoder' else 0
        full_regime_updates+=max(0,global_step-max(steps_at_start,threshold-1))
        CURRICULUM_PROGRESS['completed_epochs']+=1
        CURRICULUM_PROGRESS['last_progress']=max(CURRICULUM_PROGRESS['last_progress'],curriculum_info['difficulty_progress'])
        with isolated_rng():
            validation_metrics, _, _ = run_epoch(
                model, validation_loader, stage, None, scaler, global_step, total_steps
            )
        epoch_seconds = time.perf_counter() - epoch_started
        row = {
            "stage": stage, "epoch": epoch, "global_step": global_step,
            "field_target_ema_updates": int(model.field_target_ema_updates),
            "field_target_initialized_from_ae": bool(model.field_target_initialized_from_ae),
            "learning_rate": optimizer.param_groups[0]["lr"], "ema_tau": tau,
            "field_ema_source_trainable": any(p.requires_grad for p in model.electronic_online_encoder.parameters()),
            "field_ema_source": "electronic_online_encoder",
            "epoch_seconds": epoch_seconds,
            "training_records_per_second": train_metrics['records_seen'] / max(epoch_seconds, 1e-12),
            "optimizer_updates_this_epoch": global_step-steps_at_start,
            "stage_progress": global_step/max(total_steps,1),
            "optimizer_reinitialized_at_stage_start": False,
            "adam_states_restored": int(adam_handoff.get("restored_count", 0)),
            "adam_states_initialized": int(adam_handoff.get("new_count", 0)),
            "optimizer_update_budget": int(total_steps),
            "objective_ramp_complete": bool(objective_ramp_complete(stage, global_step/max(total_steps, 1))),
            "data_curriculum_version": DATA_CURRICULUM_CFG.version,
            "data_curriculum_tier": curriculum_info['curriculum_tier'],
            "data_curriculum_fraction": curriculum_info['active_fraction'],
            "data_curriculum_active_records": curriculum_info['active_records'],
            "data_curriculum_active_unique_records": curriculum_info['active_unique_records'],
            "data_curriculum_new_unique_records": curriculum_info['new_unique_records'],
            "data_curriculum_full_epochs": curriculum_info['full_data_epochs_completed'],
            "data_curriculum_checkpoint_eligible": curriculum_info['checkpoint_eligible'],
            "data_curriculum_manifest_hash": curriculum_info['manifest_hash'],
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"validation_{k}": v for k, v in validation_metrics.items()},
        }
        history.append(row)
        if validation_score_callback is None:
            validation_score = stage_validation_score(stage, validation_metrics)
            score_extras = {}
        else:
            with isolated_rng():
                validation_score, score_extras = validation_score_callback(
                    model, validation_metrics, scaler, global_step, total_steps
                )
        if not math.isfinite(validation_score):
            raise FloatingPointError('Nonfinite validation score')
        row["validation_score"] = float(validation_score)
        row.update(score_extras)
        fields_passed = True
        violations = []
        if v29_energy_selection_stage(stage):
            snapshot = v29_selection_snapshot(model, validation_loader, stage)
            if stage == 'autoencoder':
                fields_passed, violations = v29_ae_absolute_field_constraints(snapshot)
                field_reference_record = {
                    'policy': 'absolute_ae_limits',
                    'rho_relative_l2_max': CFG.ae_rho_relative_l2_max,
                    'potential_relative_l2_max': CFG.ae_potential_relative_l2_max,
                }
                checkpoint_criterion = 'validation_energy_mae_ev_per_atom_with_absolute_ae_field_gates'
            else:
                fields_passed, violations = v29_field_constraints(snapshot, field_references)
                field_reference_record = field_references
                checkpoint_criterion = 'validation_energy_mae_ev_per_atom_with_frozen_field_constraints'
            energy_mae = (snapshot[CFG.transfer_element]['energy'] * normalization.cohesive_scale_ev_per_atom
                          if stage.startswith('G_') else fixed_physical_score(snapshot))
            validation_score = energy_mae
            row.update(validation_score=energy_mae, validation_energy_mae_ev_per_atom=energy_mae,
                validation_field_constraints_passed=fields_passed,
                validation_field_violations=';'.join(violations),
                validation_field_references=json.dumps(field_reference_record, sort_keys=True),
                checkpoint_criterion=checkpoint_criterion)
        else:
            row['checkpoint_criterion'] = 'field_or_latent_curriculum_score'
        phase_constraints_passed = bool(row.get('validation_constraints_passed', True))
        competence_gates, competence_diagnostics = stage_competence_gates(
            model, validation_loader, stage, normalization,
            progress=global_step/max(total_steps,1), update_state=True)
        row.update(competence_diagnostics)
        if v29_energy_selection_stage(stage) and stage != 'autoencoder' and best_path.exists():
            old_best = torch.load(best_path, map_location='cpu', weights_only=False)
            old_snapshot = old_best.get('selection_snapshot')
            if old_snapshot is not None and not v29_field_constraints(old_snapshot, field_references)[0]:
                best_validation = float('inf')
            del old_best
        if stage.startswith('G_'):
            row['ni_energy_scale']=float(model.ni_energy_calibration.log_scale.detach().clamp(-1,1).exp())
            row['ni_energy_offset_ev']=float(model.ni_energy_calibration.offset.detach())*normalization.cohesive_scale_ev_per_atom
            for channel,name in enumerate(FIELD_NAMES):
                row[f'ni_latent_gain_{name}']=float(torch.exp(CFG.ni_field_log_gain_limit*torch.tanh(model.ni_shared_gain.log_gain[channel].detach())))
        hard_passed,hard_missing=hard_exposure_gate(stage)
        if stage=='autoencoder':
            # AE eligibility uses the absolute physical limits above. No relative comparison
            # with the near-identity spectral reconstruction is allowed here.
            ae_gates,coverage=ae_checkpoint_gates(global_step,total_steps,scheduler.warmup_updates,
                full_regime_updates,fields_passed,math.isfinite(validation_score))
            scientific_gates=dict(phase_constraints=phase_constraints_passed,hard_exposure=hard_passed,
                **ae_gates)
            row.update({f'coverage_{e}':v for e,v in coverage.items()})
        else:
            scientific_gates=dict(objective_ramp_complete=row['objective_ramp_complete'],
                fields=fields_passed,phase_constraints=phase_constraints_passed,
                finite_validation=math.isfinite(validation_score),hard_exposure=hard_passed,
                **competence_gates)
            if v29_energy_selection_stage(stage):
                energy_gates,energy_diagnostics=per_element_energy_gates(snapshot,normalization)
                scientific_gates.update(energy_gates); row.update(energy_diagnostics)
        checkpoint_eligible=all(scientific_gates.values())
        readiness_floor=(epoch+1)>=max(1,math.ceil(CFG.readiness_min_fraction*epochs))
        if checkpoint_eligible and row['objective_ramp_complete'] and readiness_floor:
            readiness_state['streak'] += 1
        else:
            readiness_state['streak'] = 0
        readiness_state['passed'] = bool((not CFG.readiness_enabled) or readiness_state['streak']>=readiness_state['required'])
        row['readiness_streak']=int(readiness_state['streak'])
        row['readiness_required']=int(readiness_state['required'])
        row['stage_readiness_passed']=bool(readiness_state['passed'])
        row.update(checkpoint_eligible=checkpoint_eligible,
            checkpoint_gates=json.dumps(scientific_gates,sort_keys=True),
            full_regime_updates=full_regime_updates,hard_underexposed_ids=';'.join(hard_missing),
            ema_frozen=stage.startswith('G_'),global_alfe_updates=EMA_PHASE_STATE['completed_updates'])
        row['checkpoint_accepted']=bool(checkpoint_eligible and validation_score<best_validation)
        rejected=[name for name,passed in scientific_gates.items() if not passed]
        row['checkpoint_reason']=('accepted_lower_energy_or_curriculum_score' if row['checkpoint_accepted'] else
            'rejected:'+','.join(rejected) if rejected else 'rejected_no_score_improvement')
        if checkpoint_eligible and row['objective_ramp_complete'] and readiness_floor:
            early_state=update_early_stop(early_state,validation_score,epoch,stage,epochs)
        else:
            early_state.update(reference=float('inf'),bad_epochs=0,stopped=False)
        row['early_stopping_bad_epochs']=early_state['bad_epochs']
        row['early_stopped']=early_state['stopped']
        payload = checkpoint_payload(
            model, optimizer, scheduler, scaler, stage, epoch, global_step,
            (min(best_validation, validation_score)
             if checkpoint_eligible else best_validation),
            normalization, row, train_loader.generator,
        )
        payload['early_stopping_state']=dict(early_state)
        payload['readiness_state']=dict(readiness_state)
        payload['spectral_competence_state']=copy.deepcopy(SPECTRAL_COMPETENCE_STATE)
        payload['task_gradient_balance_state']=copy.deepcopy(TASK_GRADIENT_BALANCE_STATE)
        payload['d_competence_meta']=copy.deepcopy(D_COMPETENCE_META)
        payload['full_regime_updates']=full_regime_updates
        payload['energy_bias_fifo']=ENERGY_BIAS_FIFO.state_dict()
        payload['field_selection_references']=copy.deepcopy(field_references)
        if v29_energy_selection_stage(stage): payload['selection_snapshot']=copy.deepcopy(snapshot)
        payload['data_curriculum']=dict(curriculum_info)
        payload['data_curriculum_contract']=data_curriculum_contract()
        if stage.startswith('G_') and checkpoint_eligible:
            # Only full-data candidates may enter the validation Pareto frontier.
            v29_store_candidate(payload, row, stage_dir)
        if epoch % CFG.save_every_epochs == 0 or epoch == epochs - 1 or early_state['stopped']:
            atomic_torch_save(payload, last_path)
        if checkpoint_eligible and validation_score < best_validation:
            best_validation = validation_score
            payload["checkpoint_role"] = "validation_best"
            payload["best_validation"] = best_validation
            atomic_torch_save(payload, best_path)
        history_path = CFG.run_dir / f"history_{stage}.csv"
        previous = pd.read_csv(history_path) if history_path.exists() else pd.DataFrame()
        pd.concat([previous,pd.DataFrame([row])],ignore_index=True).drop_duplicates(
            ['stage','epoch'],keep='last').to_csv(history_path,index=False)
        print(json.dumps(row, indent=None))
        if early_state['stopped']:
            print(f'Early stopping: {stage}, epoch {epoch+1}; restoring best checkpoint.')
            break

    history_frame = pd.DataFrame(history)
    history_path = CFG.run_dir / f"history_{stage}.csv"
    if history_path.exists():
        previous = pd.read_csv(history_path)
        if len(history_frame):
            history_frame = pd.concat([previous, history_frame], ignore_index=True).drop_duplicates(
                ["stage", "epoch"], keep="last"
            )
        else:
            history_frame = previous
    history_frame.to_csv(history_path, index=False)
    ae_recovery_failure = ''
    if stage=='autoencoder' and not best_path.exists() and len(history_frame):
        # Resume-safe recovery: when last.pt already consumed the full epoch budget, revalidate
        # the restored model under the CURRENT scientific gates instead of requiring a new epoch.
        final_row=history_frame.iloc[-1].to_dict()
        with isolated_rng():
            fresh=v29_selection_snapshot(model,validation_loader,stage)
        fields_ok, field_violations = v29_ae_absolute_field_constraints(fresh)
        fresh_score = fixed_physical_score(fresh)
        hard_passed, hard_missing = hard_exposure_gate(stage)
        raw_phase = final_row.get('validation_constraints_passed', True)
        phase_ok = (str(raw_phase).strip().lower() not in {'false','0','nan','none',''})
        ae_gates, coverage = ae_checkpoint_gates(global_step,total_steps,scheduler.warmup_updates,
            full_regime_updates,fields_ok,math.isfinite(fresh_score))
        recovery_gates = dict(phase_constraints=phase_ok,hard_exposure=hard_passed,**ae_gates)
        if all(recovery_gates.values()):
            final_row.update(validation_score=fresh_score,validation_energy_mae_ev_per_atom=fresh_score,
                validation_field_constraints_passed=True,validation_field_violations='',
                validation_field_references=json.dumps({
                    'policy':'absolute_ae_limits',
                    'rho_relative_l2_max':CFG.ae_rho_relative_l2_max,
                    'potential_relative_l2_max':CFG.ae_potential_relative_l2_max},sort_keys=True),
                checkpoint_criterion='validation_energy_mae_ev_per_atom_with_absolute_ae_field_gates',
                checkpoint_gates=json.dumps(recovery_gates,sort_keys=True),
                checkpoint_reason='validated_stage_final_recovery',checkpoint_accepted=True,
                hard_underexposed_ids=';'.join(hard_missing),
                **{f'coverage_{e}':v for e,v in coverage.items()})
            fallback_epoch=int(final_row.get('epoch',max(epochs-1,0)))
            fallback=checkpoint_payload(model,optimizer,scheduler,scaler,stage,fallback_epoch,global_step,
                fresh_score,normalization,final_row,train_loader.generator)
            fallback.update(selection_snapshot=fresh,field_selection_references={},
                full_regime_updates=full_regime_updates,checkpoint_role='validated_stage_final_recovery')
            atomic_torch_save(fallback,best_path)
            history_frame.loc[history_frame.index[-1],'checkpoint_reason']='validated_stage_final_recovery'
            history_frame.loc[history_frame.index[-1],'checkpoint_accepted']=True
            history_frame.loc[history_frame.index[-1],'checkpoint_gates']=json.dumps(recovery_gates,sort_keys=True)
            history_frame.to_csv(history_path,index=False)
        else:
            rejected=[name for name,passed in recovery_gates.items() if not passed]
            ae_recovery_failure='recovery_rejected:'+','.join(rejected)
            if field_violations:
                ae_recovery_failure += ';field_violations=' + ','.join(field_violations)
    if stage=='autoencoder':
        diagnostic=dict(epochs_completed=len(history_frame),valid_updates=global_step,full_regime_updates=full_regime_updates,
            gates=json.loads(history_frame.iloc[-1]['checkpoint_gates']) if len(history_frame) else {},
            last_pt=str(last_path),last_exists=last_path.exists(),best_pt=str(best_path),best_exists=best_path.exists(),
            block_reason='' if best_path.exists() else (ae_recovery_failure or
                (str(history_frame.iloc[-1]['checkpoint_reason']) if len(history_frame) else 'no_epochs')))
        atomic_json_dump(diagnostic,stage_dir/'ae_checkpoint_diagnostic.json')
        print('AE checkpoint diagnostic:',json.dumps(diagnostic))
    transition_audit={
        'from_stage':adam_handoff.get('source_stage'), 'to_stage':stage,
        'readiness':dict(readiness_state),
        'objective_weights_start':stage_loss_weights(stage,0.0,adaptive=False) if stage!='autoencoder' else {'cohesive':0.0},
        'objective_weights_end':stage_loss_weights(stage,1.0,adaptive=False) if stage!='autoencoder' else {'cohesive':CFG.loss_autoencoder_cohesive},
        'optimizer_states_restored':int(adam_handoff.get('restored_count',0)),
        'optimizer_states_initialized':int(adam_handoff.get('new_count',0)),
        'macro_lr_progress':copy.deepcopy(GLOBAL_OPTIMIZER_STATE),
        'ema_phase_state':copy.deepcopy(EMA_PHASE_STATE),
        'spectral_competence_state':copy.deepcopy(SPECTRAL_COMPETENCE_STATE),
        'task_gradient_balance_state':copy.deepcopy(TASK_GRADIENT_BALANCE_STATE),
        'd_competence_meta':copy.deepcopy(D_COMPETENCE_META),
        'best_checkpoint_exists':best_path.exists(),
    }
    with (CFG.run_dir/'stage_transition_audit.jsonl').open('a') as handle:
        handle.write(json.dumps(transition_audit,default=str)+'\n')
    if (CFG.readiness_enabled and CFG.run_mode not in ('SMOKE','QUICK') and stage not in ('autoencoder','P_cohesive_refinement','G_retention_repair')
            and not readiness_state.get('passed',False)):
        raise RuntimeError(f'Stage {stage} exhausted its budget without achieving readiness for {readiness_state["required"]} consecutive validations. Use a new run/budget; do not advance silently.')
    if best_path.exists():
        best_payload = torch.load(best_path, map_location="cpu", weights_only=False)
        if stage == 'autoencoder':
            feasible_best, violations = v29_ae_absolute_field_constraints(best_payload['selection_snapshot'])
            if not feasible_best:
                raise RuntimeError(f'Best AE checkpoint violates absolute field limits: {violations}')
        elif v29_energy_selection_stage(stage):
            feasible_best, violations = v29_field_constraints(best_payload['selection_snapshot'], field_references)
            if not feasible_best:
                raise RuntimeError(f'No best checkpoint satisfies frozen field references: {violations}')
        validate_checkpoint(best_payload, normalization)
        validate_model_state_before_restore(model, best_payload)
        model.load_state_dict(best_payload["model_state"])
        optimizer.load_state_dict(best_payload["optimizer_state"])
        save_optimizer_handoff(model, optimizer, stage)
        history_frame.attrs["best_validation"] = float(best_payload["best_validation"])
        history_frame.attrs["best_checkpoint"] = str(best_path)
    if stage.startswith('G_'):
        archive_path = stage_dir / 'pareto_archive.json'
        if archive_path.exists():
            entries = json.loads(archive_path.read_text())
            for candidate in entries:
                candidate_payload = torch.load(candidate['checkpoint'], map_location='cpu', weights_only=False)
                passed, violations = v29_field_constraints(candidate_payload['selection_snapshot'], field_references)
                candidate['validation_field_constraints_passed'] = passed
                candidate['validation_field_violations'] = ';'.join(violations)
            atomic_json_dump(entries, archive_path)
    if stage == "autoencoder":
        if not best_path.is_file():
            raise RuntimeError("No trained AE checkpoint was selected; JEPA will not start.")
        # train_stage has just restored validation-best weights, not last-epoch weights.
        initialize_target_from_trained_autoencoder(model, source=best_path)
    with isolated_rng():
        v29_stage_snapshot(model,validation_loader,normalization,stage)
    return history_frame


def scientific_validation_score(metrics):
    """Fixed physical score within v9; frozen AE reconstruction is not included."""
    return float(metrics.get("rho",0.0) + metrics.get("potential",0.0)
        + 0.20*metrics.get("cohesive",0.0) + 0.10*metrics.get("jepa",0.0)
        + 0.10*metrics.get("latent_cosine",0.0))

def stage_validation_score(stage, metrics):
    if stage == "autoencoder":
        return float(metrics["rho"] + metrics["potential"] + CFG.loss_autoencoder_cohesive*metrics["cohesive"])
    if stage == "D_latent":
        return float(metrics["jepa"] + 0.5*metrics.get("latent_cosine_full_loss", metrics["latent_cosine"])
            + 0.5*metrics["latent_variance_penalty"] + 0.2*metrics["latent_covariance_penalty"]
            + CFG.loss_latent_scale*metrics["latent_scale"])
    if stage == "E_density":
        return float(metrics["rho"] + 0.1*metrics["jepa"] + 0.1*metrics["latent_cosine"])
    if stage == "E_density_potential":
        return float(metrics["rho"] + metrics["potential"] + 0.1*metrics["jepa"] + 0.1*metrics["latent_cosine"])
    if stage in ("G_cohesive_calibration", "P_cohesive_refinement"):
        return float(metrics["cohesive"])
    if stage in ("G_adapter_warmup", "G_joint_finetune", "G_retention_repair"):
        return float(metrics["rho"] + metrics["potential"] + CFG.loss_cohesive*metrics["cohesive"] + 0.1*metrics["jepa"])
    return scientific_validation_score(metrics)


In [29]:

# =============================================================================
# V29 — trainability reports and gradient conflict audit
# =============================================================================
def write_trainability_report(model, optimizer, stage, adam_handoff):
    active = {id(parameter) for group in optimizer.param_groups for parameter in group['params']}
    rows = []
    for name, module in model.named_children():
        parameters = list(module.parameters())
        total = sum(parameter.numel() for parameter in parameters)
        trainable = sum(parameter.numel() for parameter in parameters if id(parameter) in active)
        rows.append({'stage': stage, 'module': name, 'parameters': total,
                     'trainable_parameters': trainable,
                     'trainable_percent': 100.0 * trainable / max(total, 1),
                     'frozen_parameters': total - trainable,
                     'adam_source_stage': adam_handoff.get('source_stage'),
                     'adam_states_restored': adam_handoff.get('restored_count', 0),
                     'adam_states_initialized': adam_handoff.get('new_count', 0),
                     'adam_states_removed':adam_handoff.get('removed_count',0),
                     'learning_rates':json.dumps({g['module_name']:g['lr'] for g in optimizer.param_groups}),
                     'ema_frozen':stage.startswith('G_')})
    path = CFG.run_dir / 'trainability_by_stage.csv'
    frame = pd.DataFrame(rows)
    if path.exists():
        frame = pd.concat([pd.read_csv(path), frame], ignore_index=True)
    _atomic_write_dataframe_csv(frame.drop_duplicates(['stage', 'module'], keep='last'), path)


def should_audit_gradient_conflicts(next_update):
    next_update=int(next_update)
    conflict_due=bool(CFG.gradient_conflict_audit and
        (next_update == 1 or next_update % CFG.gradient_conflict_every_updates == 0))
    balance_due=bool(CFG.gradient_balance_enabled and
        (next_update == 1 or next_update % CFG.gradient_balance_every_updates == 0))
    return conflict_due or balance_due


def _gradient_audit_parameters(model):
    selected, count = [], 0
    preferred = ('context_encoder', 'predictor', 'field_prediction_decoder', 'cohesive_head',
                 'field_online_encoder', 'field_reconstruction_decoder', 'ni_')
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad or not name.startswith(preferred):
            continue
        if selected and count + parameter.numel() > CFG.gradient_conflict_max_parameters:
            continue
        selected.append(parameter); count += parameter.numel()
    return selected


def audit_gradient_conflicts(model, stage, terms, progress, optimizer_update):
    """Measure task gradients and update the next-step conflict-aware balance state.

    The optimizer step itself remains ordinary AdamW. V31 uses the sampled shared-parameter
    gradients only to rescale future task coefficients; it never replaces gradients with an
    unlogged projection. This keeps the intervention auditable and cheaper than full PCGrad.
    """
    parameters = _gradient_audit_parameters(model)
    if not parameters:
        return []
    weights = (_base_stage_loss_weights(stage, progress) if stage != 'autoencoder'
               else dict(rho=1., potential=1., cohesive=ae_cohesive_weight(progress, True),
                         ae_anchor=CFG.ae_regularization_weight))
    candidates = []
    for name in ('jepa', 'rho', 'potential', 'cohesive', 'translation_consistency',
                 'retention_distillation', 'kan', 'ae_anchor'):
        value = terms.get(name)
        weight = float(weights.get(name, 1.0 if stage == 'autoencoder' and name == 'cohesive' else 0.0))
        if value is not None and value.requires_grad and weight > 0:
            candidates.append((name, weight * value))
    if len(candidates) < 2:
        return []
    vectors = {}
    for name, value in candidates:
        gradients = torch.autograd.grad(value, parameters, retain_graph=True, allow_unused=True)
        pieces = [(gradient.detach().float() if gradient is not None
                   else torch.zeros_like(parameter, dtype=torch.float32)).reshape(-1)
                  for parameter, gradient in zip(parameters, gradients)]
        if pieces:
            vectors[name] = torch.cat(pieces)
    rows = []
    names = sorted(vectors)
    cosines={name:[] for name in names}
    for i, left in enumerate(names):
        for right in names[i + 1:]:
            a, b = vectors[left], vectors[right]
            cosine = float(F.cosine_similarity(a, b, dim=0, eps=1e-12))
            cosines[left].append(cosine); cosines[right].append(cosine)
            rows.append({'stage': stage, 'optimizer_update': int(optimizer_update),
                         'objective_a': left, 'objective_b': right, 'gradient_cosine': cosine,
                         'conflict': cosine < 0.0, 'norm_a': float(a.norm()), 'norm_b': float(b.norm()),
                         'parameters_audited': int(a.numel())})
    # GradNorm-like magnitude equalization plus a conservative shrink for strong conflicts.
    if CFG.gradient_balance_enabled and vectors:
        norms={name:max(float(vec.norm()),1e-12) for name,vec in vectors.items()}
        valid=np.asarray([value for value in norms.values() if math.isfinite(value) and value>0],dtype=float)
        if len(valid):
            target=float(np.exp(np.mean(np.log(valid))))
            for name,norm in norms.items():
                if not math.isfinite(norm): continue
                desired=target/norm
                if any(value < CFG.gradient_conflict_cosine_threshold for value in cosines.get(name,[])):
                    desired*=CFG.gradient_conflict_shrink
                desired=float(np.clip(desired,CFG.gradient_balance_min_factor,CFG.gradient_balance_max_factor))
                key=f'{stage}:{name}'
                old=float(TASK_GRADIENT_BALANCE_STATE['factors'].get(key,1.0))
                smoothed=CFG.gradient_balance_ema*old+(1-CFG.gradient_balance_ema)*desired
                TASK_GRADIENT_BALANCE_STATE['factors'][key]=float(np.clip(
                    smoothed,CFG.gradient_balance_min_factor,CFG.gradient_balance_max_factor))
                old_norm=float(TASK_GRADIENT_BALANCE_STATE['norm_ema'].get(key,norm))
                TASK_GRADIENT_BALANCE_STATE['norm_ema'][key]=(
                    CFG.gradient_balance_ema*old_norm+(1-CFG.gradient_balance_ema)*norm)
                conflict_value=min(cosines.get(name,[1.0]))
                old_conflict=float(TASK_GRADIENT_BALANCE_STATE['conflict_ema'].get(key,conflict_value))
                TASK_GRADIENT_BALANCE_STATE['conflict_ema'][key]=(
                    CFG.gradient_balance_ema*old_conflict+(1-CFG.gradient_balance_ema)*conflict_value)
            TASK_GRADIENT_BALANCE_STATE['updates']+=1
        for row in rows:
            for side in ('a','b'):
                name=row[f'objective_{side}']
                row[f'balance_factor_{side}']=float(
                    TASK_GRADIENT_BALANCE_STATE['factors'].get(f'{stage}:{name}',1.0))
    return rows


def append_gradient_conflict_history(rows):
    if not rows:
        return
    path = CFG.run_dir / 'gradient_conflicts.csv'
    frame = pd.DataFrame(rows)
    if path.exists():
        frame = pd.concat([pd.read_csv(path), frame], ignore_index=True)
    frame=frame.drop_duplicates(['stage','optimizer_update','objective_a','objective_b'],keep='last')
    frame['conflict_frequency']=frame.groupby(['stage','objective_a','objective_b']).conflict.transform('mean')
    _atomic_write_dataframe_csv(frame,path)


## Initialization and tests before training

In [30]:
COHESIVE_WEAK_THRESHOLDS=fit_cohesive_weak_thresholds(splits['pretrain_train'],pretrain_normalization)
COHESIVE_EXTREME_WEAK_THRESHOLDS=fit_cohesive_extreme_weak_thresholds(splits['pretrain_train'],pretrain_normalization)
COHESIVE_UPPER_THRESHOLDS={str(e):float(np.quantile(normalized_cohesive_coordinates(g,pretrain_normalization),.90))
    for e,g in splits['pretrain_train'].groupby('element')}
SPECTRAL_RMS_STATISTICS=fit_spectral_rms_statistics(
    splits['pretrain_train'],pretrain_normalization,min(CFG.spectral_statistics_records,CFG.preset.normalization_records))
atomic_json_dump(SPECTRAL_RMS_STATISTICS,CFG.run_dir/'spectral_rms_statistics.json')
sample_graph=PeriodicDFTDataset(splits['pretrain_train'].head(1),pretrain_normalization,True)[0]
sample_batch=Batch.from_data_list([sample_graph]).to(DEVICE)
model=PeriodicFieldJEPA(sample_graph).to(DEVICE)
architecture_manifest={'version':ARCHITECTURE_VERSION,'latent_shape':[2,CFG.latent_dim],
    'shared_energy_input':'learned shared spectral tensor, non-DC invariant view',
    'energy_structural_bypass':False,'energy_encoder_input':False,
    'energy_readout':'known elemental anchor + shared-latent KAN residual; Ni-only affine residual calibration',
    'energy_head_extra_inputs':'element identity only for fixed anchor and two Ni calibration scalars; no graph context',
    'ni_field_modulation':'positive real non-DC gain on shared tensor before ALL readouts',
    'electronic_online_frozen_during_Ni':True,
    'autoencoder_frozen_after_pretraining':True,
    'ema_source':'electronic_online_encoder',
    'ema_scope':'global valid AlFe optimizer clock; frozen throughout Ni',
    'optimizer_lr_scope':'stage-local warmup+cosine; late G_joint groups use release-local LR clocks',
    'curriculum_availability':'all IDs each epoch; progressive batch composition',
    'ni_selection_hash':NI_SELECTION_MANIFEST['hash'],
    'ni_unfreeze_stages':['cohesive_calibration','adapter_warmup','joint(progressive predictor/decoder/structural)','conditional_repair'],
    'electronic_online_lr_multiplier':CFG.electronic_online_lr_multiplier,
    'electronic_objective':{'reconstruction':CFG.loss_electronic_reconstruction,
                            'frozen_AE_anchor':CFG.loss_electronic_anchor},
    'predictor_architecture_preserved_from_v22':True,
    'property':'experimentally anchored cohesive energy eV/atom at 298.15 K','graph':GRAPH_BUILDER_VERSION,
    'graph_rule':'max(abs(f_j+s-f_i)) <= fractional_extent; distances from full cell matrix',
    'graph_fractional_extent':CFG.graph_fractional_extent,
    'shared_encoder_residual':'phase-equivariant real channel mixing per reciprocal mode',
    'known_limits':['cell-basis dependent neighborhood','no exact supercell extensivity',
                    'decoder translation covariance audited, not exact','thermochemical anchor differs from a 0 K DFT isolated-atom reference'],
    'cohesive_calibration':COHESIVE_CALIBRATION_MANIFEST}
atomic_json_dump(architecture_manifest,CFG.run_dir/'architecture_manifest.json')


In [31]:
def run_v29_architecture_contract_tests(model, sample_batch):
    results={}
    state=copy.deepcopy(model.state_dict())
    flags={n:p.requires_grad for n,p in model.named_parameters()}
    modes={m:m.training for m in model.modules()}
    with isolated_rng():
        try:
            model.eval()
            # Canonical graph: image reciprocity, translation, cell dilation/shear.
            f=np.asarray([[.11,.23,.37],[.59,.64,.73]])
            cell=np.asarray([[3.2,0.,0.],[.6,2.8,0.],[.3,.4,4.1]])
            e,s,v=enumerate_periodic_edges(f,cell)
            edges={(int(e[0,k]),int(e[1,k]),*map(int,s[k])) for k in range(e.shape[1])}
            assert all((j,i,-a,-b,-c) in edges for i,j,a,b,c in edges)
            assert any(i==j and (a or b or c) for i,j,a,b,c in edges)
            _,_,vt=enumerate_periodic_edges((f+np.asarray([.6,.5,.7]))%1,cell)
            np.testing.assert_allclose(np.sort(np.linalg.norm(v,axis=1)),np.sort(np.linalg.norm(vt,axis=1)),rtol=1e-6)
            ed,sd,vd=enumerate_periodic_edges(f,cell*1.7)
            np.testing.assert_array_equal(e,ed); np.testing.assert_array_equal(s,sd)
            np.testing.assert_allclose(vd,1.7*v,rtol=1e-6,atol=1e-6)
            _,_,vs=enumerate_periodic_edges(f,cell@np.asarray([[1,.2,0],[0,1,0],[0,0,1]]))
            assert not np.allclose(np.linalg.norm(v,axis=1),np.linalg.norm(vs,axis=1))
            results['cell_graph_translation_dilation_shear_and_reciprocity']=True
            # Shape and operator target guard.
            batch=operator_batch(sample_batch)
            out=model.forward_operator(batch)
            assert out['z_pred'].shape==(batch.num_graphs,2,CFG.latent_dim)
            assert out['cohesive_energy_normalized'].shape==(batch.num_graphs,)
            try: model.forward_operator(sample_batch)
            except RuntimeError: pass
            else: raise AssertionError('Target guard failed')
            results['operator_target_guard']=True
            assert not any('element_embedding' in n for n,_ in model.named_parameters())
            assert torch.count_nonzero(model.context_encoder.ni_embedding.delta)==0
            results['no_random_ni_lookup']=True
            # Shared residual phase law, with nonzero trained-like mixing.
            residual=model.field_online_encoder.residual
            with torch.no_grad(): residual.net[-1].weight.normal_(0,.02)
            z=torch.randn(2,2,CFG.latent_dim,device=DEVICE)
            delta=z.new_tensor([.137,-.263,.491])
            wave=model.predictor.wavevectors_xyz
            rotated=rotate_spectral_pairs(z,wave,delta)
            lifted=z+CFG.shared_residual_scale*residual(z)
            lifted_rotated=rotated+CFG.shared_residual_scale*residual(rotated)
            torch.testing.assert_close(lifted_rotated,rotate_spectral_pairs(lifted,wave,delta),atol=3e-5,rtol=3e-5)
            torch.testing.assert_close(model.predict_cohesive_energy(lifted),model.predict_cohesive_energy(lifted_rotated),atol=2e-5,rtol=2e-5)
            results['learned_shared_latent_and_cohesive_translation']=True
            model.load_state_dict(state)
            # Cohesive gradients reach BOTH channels and the AE learned residual.
            for p in model.parameters(): p.requires_grad_(True)
            z=torch.randn(2,2,CFG.latent_dim,device=DEVICE,requires_grad=True)
            model.predict_cohesive_energy(z).sum().backward()
            assert all(float(z.grad[:,i,2:].abs().sum())>0 for i in range(2))
            model.zero_grad(set_to_none=True)
            online=model.field_online_encoder(sample_batch.field_input)
            model.predict_cohesive_energy(online).sum().backward()
            assert any(p.grad is not None and float(p.grad.abs().sum())>0 for p in model.field_online_encoder.parameters())
            results['cohesive_gradient_both_channels_and_encoder']=True
            model.zero_grad(set_to_none=True)
            # Every optimizer stage executes forward/backward without removed modules.
            for stage in ('autoencoder','D_latent','E_density','E_density_potential','E_full',
                          'F_periodic_ramp','G_cohesive_calibration','G_adapter_warmup','G_joint_finetune'):
                groups=stage_module_groups(model,stage); model.train(); model.field_target_encoder.eval()
                data=sample_batch.clone()
                if stage.startswith('G_'):
                    data.atomic_number.fill_(ATOMIC_PROPERTIES['Ni']['Z']); data.element=['Ni']*data.num_graphs
                loss,_=compute_loss(model,data,stage)
                assert torch.isfinite(loss), stage
                loss.backward()
                assert any(p.grad is not None and float(p.grad.abs().sum())>0 for g in groups for p in g['params']),stage
                assert all(p.grad is None for p in model.field_target_encoder.parameters()),stage
                assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None),stage
                model.zero_grad(set_to_none=True)
            results['all_nine_stage_backward_contracts']=True
            # Global normalization round-trip; Ni doesn't require a fitted Ni center.
            n=pretrain_normalization
            for element in ('Al','Fe','Ni'):
                desired=3.25
                total=2*(n.cohesive_anchor_ev_per_atom[element]
                    +n.bulk_reference_energy_ev_per_atom[element]-desired)
                normalized=n.encode_cohesive_energy(element,total,2)
                assert math.isclose(n.decode_cohesive_energy(normalized),desired,abs_tol=1e-9)
                reference_id=COHESIVE_REFERENCE_CALC_ID[element]
                assert reference_id in set(reference_pool_frames[element].calc_id)
                reference_row=reference_pool_frames[element].loc[
                    reference_pool_frames[element].calc_id.astype(str).eq(reference_id)].iloc[0]
                reference_cohesive=n.physical_cohesive_energy(
                    element,reference_row.final_energy_ev,int(reference_row.atom_count))
                assert math.isclose(reference_cohesive,n.cohesive_anchor_ev_per_atom[element],abs_tol=1e-9)
                forbidden=set().union(*(set(frame.calc_id.astype(str)) for name,frame in splits.items()
                    if name not in ('pretrain_train','ni_adapt_train')))
                assert reference_id not in forbidden
                audit=next(row for row in COHESIVE_CALIBRATION_MANIFEST['selection'] if row['element']==element)
                assert reference_id in audit['tail_calc_ids'] and audit['tail_records']>=1
            results['automatic_train_only_cohesive_calibration']=True
            assert set(n.cohesive_fitted_calc_ids)<=set(splits['pretrain_train'].calc_id)
            assert not set(n.cohesive_fitted_calc_ids)&set(splits['ni_adapt_train'].calc_id)
            results['global_train_only_cohesive_roundtrip']=True
            generator=torch.Generator().manual_seed(123)
            rng=capture_rng_state(generator)
            first=(random.random(),np.random.rand(),torch.rand(3),torch.rand(3,generator=generator))
            restore_rng_state(rng,generator)
            second=(random.random(),np.random.rand(),torch.rand(3),torch.rand(3,generator=generator))
            assert first[0]==second[0] and first[1]==second[1]
            torch.testing.assert_close(first[2],second[2]); torch.testing.assert_close(first[3],second[3])
            results['rng_roundtrip']=True
        finally:
            model.load_state_dict(state)
            for n,p in model.named_parameters(): p.requires_grad_(flags[n]); p.grad=None
            for module,mode in modes.items(): module.training=mode
    return results

def test_structural_reader_without_targets():
    # The source row intentionally contains no energy/NetCDF/output keys.
    row=splits['pretrain_train'].iloc[0]
    structural_row={k:row[k] for k in ('calc_id','element','struct_out_path','ion_xml_path')}
    old_grid=globals()['read_grid']; old_cached=globals()['cached_raw_record']
    def forbidden(*args,**kwargs): raise AssertionError('Attempt to read targets during inference')
    try:
        globals()['read_grid']=forbidden; globals()['cached_raw_record']=forbidden
        dataset=PeriodicDFTDataset(pd.DataFrame([structural_row]),pretrain_normalization,False)
        data=dataset[0]
        assert not TARGET_KEYS.intersection(data.keys())
        with torch.no_grad(): model.forward_operator(Batch.from_data_list([data]).to(DEVICE))
    finally:
        globals()['read_grid']=old_grid; globals()['cached_raw_record']=old_cached
    return True

CONTRACT_TESTS=run_v29_architecture_contract_tests(model,sample_batch)
CONTRACT_TESTS['structural_reader_no_target_io']=test_structural_reader_without_targets()
atomic_json_dump(CONTRACT_TESTS,CFG.run_dir/'pretraining_contract_tests.json')
display(CONTRACT_TESTS)


{'cell_graph_translation_dilation_shear_and_reciprocity': True,
 'operator_target_guard': True,
 'no_random_ni_lookup': True,
 'learned_shared_latent_and_cohesive_translation': True,
 'cohesive_gradient_both_channels_and_encoder': True,
 'all_nine_stage_backward_contracts': True,
 'automatic_train_only_cohesive_calibration': True,
 'global_train_only_cohesive_roundtrip': True,
 'rng_roundtrip': True,
 'structural_reader_no_target_io': True}

In [32]:

# =============================================================================
# V31 — protocol tests (do not train the model)
# =============================================================================
V31_PROTOCOL_TESTS = {}
ramp_grid = np.linspace(0.0, CFG.objective_ramp_fraction, 101)
ramp_values = np.asarray([cosine_ramp(value) for value in ramp_grid])
assert math.isclose(ramp_values[0], 0.0, abs_tol=1e-12)
assert math.isclose(ramp_values[-1], 1.0, abs_tol=1e-12)
assert np.all(np.diff(ramp_values) >= -1e-12)
V31_PROTOCOL_TESTS['cosine_objective_ramp_monotonic'] = True

# D must start low-band only before any validation competence unlock.
for field in FIELD_NAMES:
    w=progressive_band_weights('D_latent',0.1,True,field)
    assert math.isclose(w[0],1.0,abs_tol=1e-12) and w[1]==0.0 and w[2]==0.0
V31_PROTOCOL_TESTS['D_starts_low_band_only'] = True

# Simulate a fully competent D state only for boundary-continuity unit tests.
_saved_spectral=copy.deepcopy(SPECTRAL_COMPETENCE_STATE)
_saved_meta=copy.deepcopy(D_COMPETENCE_META)
try:
    for field in FIELD_NAMES:
        SPECTRAL_COMPETENCE_STATE[field].update(
            low_ready=True,middle_ready=True,high_ready=True,
            middle_unlocked_at=0.10,high_unlocked_at=0.30)
    D_COMPETENCE_META['crossfade_unlocked_at']=0.50
    D_COMPETENCE_META['low_gate_ready']=True
    boundary_pairs=[('D_latent','E_density'),('E_density','E_density_potential'),
                    ('E_density_potential','E_full'),('E_full','F_periodic_ramp')]
    continuity_keys=('jepa','latent_cosine','rho','potential','cohesive','latent_variance_penalty',
                     'latent_covariance_penalty','latent_scale','kan','translation_consistency',
                     'electronic_reconstruction','electronic_anchor')
    for previous,next_stage in boundary_pairs:
        left=stage_loss_weights(previous,1.0,adaptive=False)
        right=stage_loss_weights(next_stage,0.0,adaptive=False)
        for key in continuity_keys:
            assert math.isclose(float(left[key]),float(right[key]),rel_tol=0.0,abs_tol=1e-12),(
                previous,next_stage,key,left[key],right[key])
finally:
    SPECTRAL_COMPETENCE_STATE.clear(); SPECTRAL_COMPETENCE_STATE.update(_saved_spectral)
    D_COMPETENCE_META.clear(); D_COMPETENCE_META.update(_saved_meta)
V31_PROTOCOL_TESTS['source_objective_boundary_continuity_after_competence'] = True

assert math.isclose(stage_loss_weights('E_density',0.0,adaptive=False)['rho'],0.0,abs_tol=1e-12)
assert math.isclose(stage_loss_weights('E_density',CFG.objective_ramp_fraction,adaptive=False)['rho'],CFG.loss_rho,abs_tol=1e-12)
assert math.isclose(stage_loss_weights('G_adapter_warmup',0.0,adaptive=False)['retention_distillation'],0.0,abs_tol=1e-12)
V31_PROTOCOL_TESTS['density_and_retention_ramps'] = True

# Stage-local scheduler must start fresh for every stage.
dummy_parameter=torch.nn.Parameter(torch.tensor(1.0))
dummy_optimizer=torch.optim.AdamW([{'params':[dummy_parameter],'lr':1e-4,'lr_multiplier':1.0,'module_name':'dummy'}],lr=1e-4)
dummy_scheduler=WarmupCosineScheduler(dummy_optimizer,1e-4,100,'E_full')
first_factor=dummy_scheduler.factor()
for _ in range(50): dummy_scheduler.step()
mid_factor=dummy_scheduler.factor()
dummy_optimizer2=torch.optim.AdamW([{'params':[dummy_parameter],'lr':1e-4,'lr_multiplier':1.0,'module_name':'dummy'}],lr=1e-4)
dummy_scheduler2=WarmupCosineScheduler(dummy_optimizer2,1e-4,100,'F_periodic_ramp')
assert math.isclose(dummy_scheduler2.factor(),first_factor,rel_tol=0,abs_tol=1e-12)
assert not math.isclose(mid_factor,dummy_scheduler2.factor(),rel_tol=0,abs_tol=1e-6)
V31_PROTOCOL_TESTS['stage_local_scheduler_resets_clock'] = True

tau_values = np.asarray([ema_tau_at_progress(value) for value in np.linspace(0, 1, 101)])
assert math.isclose(tau_values[0], CFG.ema_tau_start, abs_tol=1e-12)
assert math.isclose(tau_values[-1], CFG.ema_tau_end, abs_tol=1e-12)
assert np.all(np.diff(tau_values) >= -1e-12)
V31_PROTOCOL_TESTS['variable_ema_monotonic'] = True

stage_trainable = {}
original_requires_grad = {id(parameter): parameter.requires_grad for parameter in model.parameters()}
try:
    for stage in ('P_cohesive_refinement', 'G_cohesive_calibration', 'G_adapter_warmup',
                  'G_joint_finetune', 'G_retention_repair'):
        groups = stage_module_groups(model, stage)
        stage_trainable[stage] = sorted(group['name'] for group in groups)
finally:
    for parameter in model.parameters():
        parameter.requires_grad_(original_requires_grad[id(parameter)])
        parameter.grad = None
assert stage_trainable['G_cohesive_calibration'] == ['ni_energy_calibration']
assert 'predictor' not in stage_trainable['G_adapter_warmup']
assert 'prediction_decoder' in stage_trainable['G_joint_finetune']
assert not any(name.startswith('electronic_') for name in stage_trainable['G_retention_repair'])
assert 'predictor' in stage_trainable['G_joint_finetune'] and 'structural_last_block' in stage_trainable['G_joint_finetune']
assert math.isclose(_joint_release_factor(0.0,CFG.joint_predictor_start,CFG.joint_predictor_full),0.0,abs_tol=1e-12)
assert _joint_release_factor(CFG.joint_predictor_full,CFG.joint_predictor_start,CFG.joint_predictor_full) > .99
assert _joint_release_factor(1.0,CFG.joint_structural_start,CFG.joint_structural_full) >= CFG.joint_group_lr_floor
V31_PROTOCOL_TESTS['compact_progressive_unfreezing_with_release_local_lr'] = True

V31_PROTOCOL_TESTS['retention_stream_enabled'] = bool(CFG.retention_stream_enabled)
if not CFG.retention_stream_enabled:
    warnings.warn('Retention stream explicitly disabled: treat this run as an ablation, not as the main V31 protocol.')

atomic_json_dump({'tests': V31_PROTOCOL_TESTS, 'trainable_groups': stage_trainable,
                  'architecture_version': ARCHITECTURE_VERSION,
                  'training_protocol_version': TRAINING_PROTOCOL_VERSION,
                  'curriculum_version': CURRICULUM_VERSION},
                 CFG.run_dir / 'v31_protocol_tests.json')
print(json.dumps(V31_PROTOCOL_TESTS, indent=2))


{
  "cosine_objective_ramp_monotonic": true,
  "D_starts_low_band_only": true,
  "source_objective_boundary_continuity_after_competence": true,
  "density_and_retention_ramps": true,
  "stage_local_scheduler_resets_clock": true,
  "variable_ema_monotonic": true,
  "compact_progressive_unfreezing_with_release_local_lr": true,
  "retention_stream_enabled": true
}


In [33]:
def run_v29_contract_tests(model):
    state=copy.deepcopy(model.state_dict())
    flags={n:p.requires_grad for n,p in model.named_parameters()}
    modes={m:m.training for m in model.modules()}
    results={}
    with isolated_rng():
        try:
            model.eval()
            z=torch.randn(3,2,CFG.latent_dim,device=DEVICE)
            z[:]=z[:1].clone()
            elements=['Al','Fe','Ni']
            with torch.no_grad():
                residual=model.predict_cohesive_energy(z)
                calibrated=model.predict_cohesive_energy(z,elements)
            torch.testing.assert_close(calibrated-residual,model.energy_anchor_normalized)
            results['known_anchor_plus_latent_residual']=True
            with torch.no_grad():
                model.ni_energy_calibration.offset.fill_(.2)
                changed=model.predict_cohesive_energy(z,elements)
            torch.testing.assert_close(changed[:2],calibrated[:2],rtol=0,atol=0)
            torch.testing.assert_close(changed[2]-calibrated[2],z.new_tensor(.2),atol=1e-6,rtol=1e-6)
            results['Ni_calibration_preserves_AlFe_exactly']=True
            mask=z.new_tensor([0.,0.,1.])
            with torch.no_grad(): model.ni_shared_gain.log_gain.fill_(.15)
            gained=model.ni_shared_gain(z,mask)
            torch.testing.assert_close(gained[:2],z[:2],atol=0,rtol=0)
            torch.testing.assert_close(gained[:,:,:2],z[:,:,:2],atol=0,rtol=0)
            shift=z.new_tensor([.13,.21,-.37])
            rotated=rotate_spectral_pairs(z,model.predictor.wavevectors_xyz,shift)
            torch.testing.assert_close(model.ni_shared_gain(rotated,mask),
                rotate_spectral_pairs(gained,model.predictor.wavevectors_xyz,shift),atol=1e-5,rtol=1e-5)
            torch.testing.assert_close(model.predict_cohesive_energy(gained,elements),
                model.predict_cohesive_energy(rotate_spectral_pairs(gained,model.predictor.wavevectors_xyz,shift),elements),atol=2e-5,rtol=2e-5)
            results['calibrated_energy_and_gain_translation_contract']=True
            groups=stage_module_groups(model,'G_retention_repair')
            trainable={id(p) for g in groups for p in g['params']}
            for module in (model.ni_energy_calibration,model.ni_shared_gain,model.ni_field_adapter,
                model.context_encoder.ni_embedding,model.electronic_online_encoder,
                model.field_online_encoder,model.field_reconstruction_decoder):
                assert not any(id(p) in trainable for p in module.parameters())
            results['repair_freezes_Ni_specific_and_electronic_modules']=True
        finally:
            model.load_state_dict(state)
            for n,p in model.named_parameters(): p.requires_grad_(flags[n]); p.grad=None
            for module,mode in modes.items(): module.training=mode
    atomic_json_dump(results,CFG.run_dir/'v29_pretraining_contracts.json')
    return results

V29_CONTRACT_TESTS=run_v29_contract_tests(model)
display(V29_CONTRACT_TESTS)


{'known_anchor_plus_latent_residual': True,
 'Ni_calibration_preserves_AlFe_exactly': True,
 'calibrated_energy_and_gain_translation_contract': True,
 'repair_freezes_Ni_specific_and_electronic_modules': True}

In [34]:
# =============================================================================
# STEP 21 — DataLoaders by stage
# Purpose: Applies QUICK/DEVELOPMENT limits without contaminating holdouts.
# Inputs: Splits and pretrain_normalization.
# Outputs: Training/validation/holdout/OOD/Ni loaders.
# Reproducibility: run after all preceding cells; do not change the seed during the run.
# =============================================================================
def frame_for_mode(frame: pd.DataFrame, maximum_tiny: int = 32, maximum_quick: int = 2000) -> pd.DataFrame:
    if CFG.run_mode in ("SMOKE","QUICK"):
        return deterministic_sample(frame, min(maximum_tiny, len(frame)), CFG.seed + 400)
    if CFG.run_mode == "QUICK_TRAIN":
        return deterministic_sample(frame, min(max(128, maximum_tiny*8), len(frame)), CFG.seed + 405)
    if CFG.run_mode == "DEVELOPMENT":
        return deterministic_sample(frame, min(maximum_quick, len(frame)), CFG.seed + 401)
    if CFG.run_mode == "STANDARD":
        return deterministic_sample(frame, min(8000, len(frame)), CFG.seed + 402)
    return frame


if CFG.run_mode in ("SMOKE","QUICK"):
    pretrain_train_frame=pd.concat([frame_for_mode(g,16) for _,g in splits["pretrain_train"].groupby("element")],ignore_index=True)
elif CFG.run_mode=="QUICK_TRAIN":
    pretrain_train_frame=pd.concat([
        deterministic_sample(g,min(CFG.quick_train_records_per_element,len(g)),CFG.seed+450+i)
        for i,(_,g) in enumerate(splits["pretrain_train"].groupby("element"))
    ],ignore_index=True)
else:
    pretrain_train_frame=splits["pretrain_train"].copy()
pretrain_validation_frame = frame_for_mode(splits["pretrain_validation"], 16, 400)
al_retention_validation_frame = frame_for_mode(
    splits["pretrain_validation"][splits["pretrain_validation"].element == "Al"], 8, 200
)
fe_retention_validation_frame = frame_for_mode(
    splits["pretrain_validation"][splits["pretrain_validation"].element == "Fe"], 8, 200
)
al_holdout_frame = frame_for_mode(splits["al_holdout_test"], 24, 600)
fe_holdout_frame = frame_for_mode(splits["fe_holdout_test"], 24, 600)
pretrain_ood_frame = frame_for_mode(splits["pretrain_family_ood"], 24, 600)

pretrain_train_loader = make_loader(pretrain_train_frame, pretrain_normalization, shuffle=True,
    balance_elements=CFG.balance_pretrain_batches)

for _stage in ('autoencoder','D_latent','E_density','E_density_potential','E_full',
               'F_periodic_ramp','P_cohesive_refinement'):
    register_data_curriculum_stage(
        _stage, pretrain_train_frame, pretrain_normalization,
        scope='pretrain_AlFe', balance_elements=CFG.balance_pretrain_batches, focus='all')
PRETRAIN_SAMPLING_AUDIT = {
    'balanced_batches': CFG.balance_pretrain_batches,
    'unique_records_per_element': pretrain_train_frame.groupby('element').calc_id.nunique().to_dict(),
    'batches_per_epoch': len(pretrain_train_loader),
    'draws_per_epoch': (len(pretrain_train_loader)*CFG.preset.batch_size
                        if CFG.balance_pretrain_batches else len(pretrain_train_frame)),
    'tail_oversampling_added': False,
    'rule': 'complete shuffled pool cycles; equal Al/Fe per batch; minority uniformly repeated'}
atomic_json_dump(PRETRAIN_SAMPLING_AUDIT, CFG.run_dir/'pretrain_sampling_audit.json')
pretrain_validation_loader = make_loader(pretrain_validation_frame, pretrain_normalization, shuffle=False)
al_retention_validation_loader = make_loader(al_retention_validation_frame, pretrain_normalization, shuffle=False)
fe_retention_validation_loader = make_loader(fe_retention_validation_frame, pretrain_normalization, shuffle=False)
al_holdout_loader = make_loader(al_holdout_frame, pretrain_normalization, shuffle=False)
fe_holdout_loader = make_loader(fe_holdout_frame, pretrain_normalization, shuffle=False)
pretrain_ood_loader = (
    make_loader(pretrain_ood_frame, pretrain_normalization, shuffle=False)
    if len(pretrain_ood_frame) else None
)
print({
    "pretrain_train": len(pretrain_train_frame),
    "pretrain_validation": len(pretrain_validation_frame),
    "al_holdout_test": len(al_holdout_frame),
    "fe_holdout_test": len(fe_holdout_frame),
    "pretrain_ood": len(pretrain_ood_frame),
})


def make_balanced_al_fe_validation_callback(stage: str):
    def callback(model, combined_metrics, scaler, global_step, total_steps):
        al_metrics, _, _ = run_epoch(
            model, al_retention_validation_loader, stage, None,
            scaler, global_step, total_steps,
        )
        fe_metrics, _, _ = run_epoch(
            model, fe_retention_validation_loader, stage, None,
            scaler, global_step, total_steps,
        )
        al_score = stage_validation_score(stage, al_metrics)
        fe_score = stage_validation_score(stage, fe_metrics)
        mean_score = 0.5 * (al_score + fe_score)
        worst_score = max(al_score, fe_score)
        imbalance = abs(al_score - fe_score)
        w=float(CFG.element_loss_worst_weight)
        balanced_score = (1-w)*mean_score + w*worst_score
        return balanced_score, {
            "validation_al_score": al_score,
            "validation_fe_score": fe_score,
            "validation_al_fe_mean_score": mean_score,
            "validation_al_fe_imbalance": imbalance,
            "validation_al_fe_worst_score": worst_score,
        }
    return callback


@torch.no_grad()
def physical_validation_snapshot(model, loader, autoencoder=False):
    """Validation only: per-configuration metrics, no stochastic loss weights."""
    model.eval(); rows=[]
    for data in loader:
        data=move_batch(data)
        if autoencoder:
            z=model.field_online_encoder(data.field_input)
            rho,potential=model.field_reconstruction_decoder(z)
            descriptor=invariant_shared_latent_view(z,model.field_online_encoder.spectral_rms)
            energy=model.cohesive_from_descriptor(descriptor, data.element)
        else:
            out=model.forward_operator(operator_batch(data))
            rho,potential,energy=out['rho_scaled'],out['potential_scaled'],out['cohesive_energy_normalized']
        rho_f=rho.float(); rho_t=data.rho_target.float()
        pot_f=gauge_center(potential.float()); pot_t=gauge_center(data.potential_target.float())
        re=relative_field_error(rho_f,rho_t)
        ve=relative_field_error(pot_f,pot_t)
        rho_cos=F.cosine_similarity(rho_f.flatten(1),rho_t.flatten(1),dim=-1,eps=1e-6)
        pot_cos=F.cosine_similarity(pot_f.flatten(1),pot_t.flatten(1),dim=-1,eps=1e-6)
        pot_pred_rms=pot_f.square().flatten(1).mean(-1).clamp_min(1e-12).sqrt()
        pot_true_rms=pot_t.square().flatten(1).mean(-1).clamp_min(1e-12).sqrt()
        pot_rms_ratio=pot_pred_rms/pot_true_rms
        target=data.cohesive_energy_target.reshape(-1).float()
        signed=energy.reshape(-1).float()-target
        for j,element in enumerate(map(str,data.element)):
            rows.append({'calc_id':str(data.calc_id[j]),'element':element,'family':str(data.family[j]),
                'rho':float(re[j]),'potential':float(ve[j]),
                'rho_cosine':float(rho_cos[j]),'potential_cosine':float(pot_cos[j]),
                'potential_rms_ratio':float(pot_rms_ratio[j]),
                'energy':float(signed[j].abs()),'energy_signed':float(signed[j]),
                'energy_target':float(target[j]),'energy_prediction':float(energy.reshape(-1)[j]),
                'weak':float(target[j]) <= COHESIVE_WEAK_THRESHOLDS.get(element,-float('inf'))})
    frame=pd.DataFrame(rows)
    if frame.empty or frame.calc_id.duplicated().any():
        raise ValueError('Physical validation requires unique, nonempty configurations')
    summaries={}
    for element,g in frame.groupby('element'):
        tail=g.loc[g.weak]
        summary={k:float(g[k].mean()) for k in ('rho','potential','energy')}
        summary.update(rho_cosine=float(g.rho_cosine.mean()),
                       potential_cosine=float(g.potential_cosine.mean()),
                       potential_rms_ratio=float(g.potential_rms_ratio.mean()),
                       potential_rms_ratio_min=float(g.potential_rms_ratio.min()),
                       potential_rms_ratio_max=float(g.potential_rms_ratio.max()))
        summary.update(energy_p95=float(g.energy.quantile(.95)),energy_bias=float(g.energy_signed.mean()),
                       weak_energy=float(tail.energy.mean()) if len(tail) else float(g.energy.mean()),
                       weak_records=int(len(tail)),records=int(len(g)))
        if not all(math.isfinite(summary[k]) for k in ('rho','potential','energy','energy_p95','weak_energy')):
            raise FloatingPointError('Nonfinite physical validation')
        summary.update(energy_max=float(g.energy.max()))
        t=g.energy_target.to_numpy(); p=g.energy_prediction.to_numpy()
        supported=len(t)>=CFG.energy_slope_min_records and float(np.var(t))>=CFG.energy_slope_variance_floor
        summary.update(energy_slope=float(np.polyfit(t,p,1)[0]) if supported else float('nan'),
            energy_slope_supported=supported)
        summaries[str(element)]=summary
    return summaries,frame


def fixed_physical_score(summaries):
    """Mean physical cohesive-energy MAE in eV/atom; intentionally independent of loss weights."""
    return float(np.mean([s['energy'] for s in summaries.values()]) * pretrain_normalization.cohesive_scale_ev_per_atom)


def physical_selection_callback(model, combined_metrics, scaler, global_step, total_steps):
    summaries,_=physical_validation_snapshot(model,pretrain_validation_loader,autoencoder=True)
    score=fixed_physical_score(summaries)
    extras={f'validation_physical_{e}_{k}':v for e,s in summaries.items() for k,v in s.items()}
    return score,extras


def alfe_jepa_schedule():
    if CFG.run_mode in ('SMOKE','QUICK'):
        d_epochs=1
    elif CFG.run_mode=='QUICK_TRAIN':
        d_epochs=CFG.quick_train_d_epochs
    else:
        d_epochs=max(CFG.d_latent_epochs,CFG.preset.jepa_epochs//2)
    budget=max(4,CFG.preset.jepa_epochs); weights=np.asarray([.20,.30,.25,.25])
    allocated=np.maximum(1,np.floor(weights*budget).astype(int))
    while allocated.sum()<budget: allocated[int(np.argmax(weights*budget-allocated))]+=1
    return [('D_latent',d_epochs,CFG.d_latent_learning_rate)]+[
        (name,int(count),CFG.learning_rate) for name,count in zip(
            ['E_density','E_density_potential','E_full','F_periodic_ramp'],allocated)]



# =============================================================================
# V31 — stage-specific competence gates
# =============================================================================
@torch.no_grad()
def latent_alignment_snapshot(model, loader):
    model.eval()
    store={(element,field):[] for element in CFG.pretrain_elements for field in FIELD_NAMES}
    for data in loader:
        data=move_batch(data)
        fc,nodes=model.encode_structure(data)
        pred,_,_,_=model.predict_fields(nodes,fc,data)
        target=model.field_target_encoder(data.field_input)
        for j,element in enumerate(map(str,data.element)):
            for f,field in enumerate(FIELD_NAMES):
                record={'full':float(F.cosine_similarity(
                    pred[j,f,2:].float(),target[j,f,2:].float(),dim=0,eps=1e-6))}
                for band,(a,b) in zip(('low','middle','high'),CFG.jepa_band_boundaries):
                    record[band]=float(F.cosine_similarity(
                        pred[j,f,a:b].float(),target[j,f,a:b].float(),dim=0,eps=1e-6))
                store[(element,field)].append(record)
    rows=[]
    for (element,field),records in store.items():
        if not records: continue
        rows.append({'element':element,'field':field,**{
            key:float(np.mean([row[key] for row in records]))
            for key in ('full','low','middle','high')}})
    return rows

def _advance_gate_streak(state,key,passed):
    streak_key=key+'_streak'
    state[streak_key]=int(state.get(streak_key,0))+1 if passed else 0
    return state[streak_key] >= CFG.source_gate_hysteresis

@torch.no_grad()
def update_d_spectral_competence(model,loader,progress):
    """Unlock middle/high per field from validation competence, never from time alone."""
    global D_LOW_GATE_READY
    rows=latent_alignment_snapshot(model,loader)
    for field in FIELD_NAMES:
        field_rows=[row for row in rows if row['field']==field]
        if not field_rows: continue
        state=SPECTRAL_COMPETENCE_STATE[field]
        low_ok=min(row['low'] for row in field_rows) >= CFG.d_low_band_cosine_min
        if _advance_gate_streak(state,'low',low_ok):
            state['low_ready']=True
            if state['middle_unlocked_at'] is None:
                state['middle_unlocked_at']=float(progress)
        middle_ok=(state['middle_unlocked_at'] is not None and
                   min(row['middle'] for row in field_rows) >= CFG.d_middle_band_cosine_min)
        if _advance_gate_streak(state,'middle',middle_ok):
            state['middle_ready']=True
            if state['high_unlocked_at'] is None:
                state['high_unlocked_at']=float(progress)
        high_ok=(state['high_unlocked_at'] is not None and
                 min(row['high'] for row in field_rows) >= CFG.d_high_band_cosine_min)
        if _advance_gate_streak(state,'high',high_ok):
            state['high_ready']=True
    D_LOW_GATE_READY=all(SPECTRAL_COMPETENCE_STATE[field]['low_ready'] for field in FIELD_NAMES)
    D_COMPETENCE_META['low_gate_ready']=bool(D_LOW_GATE_READY)
    all_high=all(SPECTRAL_COMPETENCE_STATE[field]['high_ready'] for field in FIELD_NAMES)
    if all_high and D_COMPETENCE_META.get('crossfade_unlocked_at') is None:
        D_COMPETENCE_META['crossfade_unlocked_at']=float(progress)
    return rows

@torch.no_grad()
def periodic_consistency_validation(model,loader,max_batches=2):
    model.eval(); values=[]
    for batch_index,data in enumerate(loader):
        if batch_index>=max_batches: break
        data=move_batch(data)
        shifts=tuple(max(1,n//5) for n in CFG.canonical_grid)
        _,rho,potential,energy=predicted_outputs(model,operator_batch(data))
        translated=translate_periodic_batch(operator_batch(data),shifts)
        _,rho_t,potential_t,energy_t=predicted_outputs(model,translated)
        rho_err=relative_field_error(rho_t,torch.roll(rho,shifts,(-3,-2,-1))).mean()
        pot_err=relative_field_error(potential_t,torch.roll(potential,shifts,(-3,-2,-1))).mean()
        energy_err=F.smooth_l1_loss(energy_t,energy)
        values.append(float(rho_err+pot_err+energy_err))
    return float(np.mean(values)) if values else float('inf')

@torch.no_grad()
def stage_competence_gates(model,loader,stage,normalization,progress,update_state=False):
    """Independent readiness contract. These gates are added to checkpoint eligibility."""
    diagnostics={}
    gates={}
    if stage=='D_latent':
        rows=(update_d_spectral_competence(model,loader,progress)
              if update_state else latent_alignment_snapshot(model,loader))
        if not rows:
            return {'competence_D_nonempty':False},{'competence_rows':'[]'}
        for field in FIELD_NAMES:
            fr=[row for row in rows if row['field']==field]
            full_min=min(row['full'] for row in fr)
            diagnostics[f'competence_{field}_full_cosine_min']=full_min
            for band in ('low','middle','high'):
                diagnostics[f'competence_{field}_{band}_cosine_min']=min(row[band] for row in fr)
            threshold=(CFG.d_full_density_cosine_min if field=='density'
                       else CFG.d_full_potential_cosine_min)
            gates[f'competence_D_{field}_full']=full_min>=threshold
            gates[f'competence_D_{field}_high']=bool(SPECTRAL_COMPETENCE_STATE[field]['high_ready'])
        diagnostics['spectral_competence_state']=json.dumps(SPECTRAL_COMPETENCE_STATE,sort_keys=True)
        diagnostics['ema_low_band_authorized']=bool(D_LOW_GATE_READY)
        return gates,diagnostics

    if stage in ('E_density','E_density_potential','E_full','F_periodic_ramp',
                 'P_cohesive_refinement') or stage.startswith('G_'):
        summaries,_=physical_validation_snapshot(model,loader,autoencoder=False)
        if not summaries:
            return {'competence_physical_nonempty':False},{}
        for element,metrics in summaries.items():
            for key in ('rho','potential','rho_cosine','potential_cosine','potential_rms_ratio'):
                diagnostics[f'competence_{element}_{key}']=float(metrics[key])
        if stage=='E_density':
            gates['competence_E_density_rho']=all(
                metrics['rho']<=CFG.e_density_rho_l2_max for metrics in summaries.values())
        elif stage=='E_density_potential':
            gates['competence_E_rho']=all(
                metrics['rho']<=CFG.e_density_potential_rho_l2_max for metrics in summaries.values())
            gates['competence_E_potential_l2']=all(
                metrics['potential']<=CFG.e_density_potential_l2_max for metrics in summaries.values())
            gates['competence_E_potential_shape']=all(
                metrics['potential_cosine']>=CFG.potential_shape_cosine_min for metrics in summaries.values())
            gates['competence_E_potential_amplitude']=all(
                CFG.potential_rms_ratio_min<=metrics['potential_rms_ratio']<=CFG.potential_rms_ratio_max
                for metrics in summaries.values())
        elif stage in ('E_full','F_periodic_ramp','P_cohesive_refinement'):
            gates['competence_full_fields']=all(
                metrics['rho']<=CFG.e_full_field_l2_max and metrics['potential']<=CFG.e_full_field_l2_max
                for metrics in summaries.values())
            gates['competence_full_potential_shape']=all(
                metrics['potential_cosine']>=CFG.potential_shape_cosine_min for metrics in summaries.values())
            gates['competence_full_potential_amplitude']=all(
                CFG.potential_rms_ratio_min<=metrics['potential_rms_ratio']<=CFG.potential_rms_ratio_max
                for metrics in summaries.values())
            if stage=='F_periodic_ramp':
                periodic_value=periodic_consistency_validation(model,loader)
                diagnostics['competence_periodic_consistency']=periodic_value
                gates['competence_periodic_consistency']=(
                    math.isfinite(periodic_value) and
                    periodic_value<=CFG.periodic_consistency_validation_max)
        # G-stage final feasibility is still governed by Ni + Al/Fe retention callbacks.
        return gates,diagnostics
    return {},diagnostics

def source_domain_ready(model,loader,normalization):
    gates,diag=stage_competence_gates(model,loader,'F_periodic_ramp',normalization,1.0,False)
    snapshot,_=physical_validation_snapshot(model,loader)
    energy_gates,energy_diag=per_element_energy_gates(snapshot,normalization)
    all_gates={**gates,**energy_gates}
    return bool(all(all_gates.values())),all_gates,{**diag,**energy_diag}


{'pretrain_train': 512, 'pretrain_validation': 128, 'al_holdout_test': 192, 'fe_holdout_test': 192, 'pretrain_ood': 192}


## V29 — Train-only diagnostic thresholds and stage-wise evaluation
Detailed metrics are computed once per restored stage, not at every optimizer update.

In [35]:
# Detailed diagnostics run only for named evaluation stages, not every training batch.
from scipy.stats import spearmanr
V29_DETAILED_EVAL=False
V29_DETAIL_ROWS=[]
V29_CLASS_ROWS=[]
V29_SPECTRAL_ROWS=[]
V29_STAGE_VALIDATION_FRAMES=[]
V29_AE_EVALUATION=False

def v29_stage_snapshot(model,loader,normalization,stage):
    global V29_DETAILED_EVAL,V29_AE_EVALUATION
    previous=(V29_DETAILED_EVAL,V29_AE_EVALUATION)
    V29_DETAILED_EVAL=True; V29_AE_EVALUATION=(stage=='autoencoder')
    try:
        frame,_=evaluate_loader(model,loader,normalization,f'validation_{stage}')
        V29_STAGE_VALIDATION_FRAMES.append(frame)
        frame.to_csv(CFG.run_dir/f'physical_validation_{stage}.csv',index=False)
    finally: V29_DETAILED_EVAL,V29_AE_EVALUATION=previous

def v29_regression_metrics(truth, prediction, rank_points=None):
    t=np.asarray(truth,dtype=np.float64).ravel(); p=np.asarray(prediction,dtype=np.float64).ravel()
    if t.shape!=p.shape or not len(t) or not (np.isfinite(t).all() and np.isfinite(p).all()):
        raise ValueError('Empty, nonfinite or mismatched regression arrays')
    e=p-t; ae=np.abs(e); tc=t-t.mean(); pc=p-p.mean()
    vt=np.dot(tc,tc); vp=np.dot(pc,pc); norm=np.dot(t,t)
    slope=float(np.dot(tc,pc)/vt) if vt>1e-20 else np.nan
    idx=np.arange(len(t)) if rank_points is None or len(t)<=rank_points else np.linspace(0,len(t)-1,rank_points,dtype=int)
    rank=(float(spearmanr(t[idx],p[idx]).statistic)
          if np.ptp(t[idx])>0 and np.ptp(p[idx])>0 and len(idx)>1 else np.nan)
    return dict(n=len(t),mae=float(ae.mean()),rmse=float(np.sqrt(np.mean(e*e))),
        median_absolute_error=float(np.median(ae)),bias=float(e.mean()),error_std=float(e.std()),
        relative_l2=float(np.linalg.norm(e)/np.sqrt(norm)) if norm>1e-20 else np.nan,
        r2=float(1-np.dot(e,e)/vt) if vt>1e-20 else np.nan,
        pearson=float(np.dot(tc,pc)/np.sqrt(vt*vp)) if vt*vp>1e-30 else np.nan,
        spearman=rank,spearman_points=len(idx),slope=slope,
        intercept=float(p.mean()-slope*t.mean()) if np.isfinite(slope) else np.nan,
        error_p90=float(np.quantile(ae,.90)),error_p95=float(np.quantile(ae,.95)),
        error_p99=float(np.quantile(ae,.99)),maximum_absolute_error=float(ae.max()),
        target_min=float(t.min()),target_max=float(t.max()),pred_min=float(p.min()),pred_max=float(p.max()),
        rms_ratio=float(np.sqrt(np.dot(p,p)/norm)) if norm>1e-20 else np.nan,
        cosine=float(np.dot(t,p)/(np.linalg.norm(t)*np.linalg.norm(p))) if np.linalg.norm(t)*np.linalg.norm(p)>1e-20 else np.nan,
        constant_target=bool(vt<=1e-20))

def v29_binary_row(truth,prediction,**metadata):
    t=np.asarray(truth,dtype=bool).ravel(); p=np.asarray(prediction,dtype=bool).ravel()
    tn=int(np.sum(~t & ~p)); fp=int(np.sum(~t & p)); fn=int(np.sum(t & ~p)); tp=int(np.sum(t & p))
    return dict(**metadata,tn=tn,fp=fp,fn=fn,tp=tp)

def v29_binary_metrics(tn,fp,fn,tp):
    tn,fp,fn,tp=map(float,(tn,fp,fn,tp))
    safe=lambda a,b:a/b if b else 0.
    recall=safe(tp,tp+fn); specificity=safe(tn,tn+fp)
    f1=safe(2*tp,2*tp+fp+fn)
    return dict(precision=safe(tp,tp+fp),recall=recall,f1=f1,specificity=specificity,
        accuracy=safe(tp+tn,tp+tn+fp+fn),balanced_accuracy=(recall+specificity)/2,
        mcc=safe(tp*tn-fp*fn,math.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))),
        f1_negative=safe(2*tn,2*tn+fp+fn),macro_class_f1=(f1+safe(2*tn,2*tn+fp+fn))/2,
        support_positive=int(tp+fn),support_negative=int(tn+fp),
        single_true_class=bool(tp+fn==0 or tn+fp==0))

@torch.no_grad()
def v29_collect_field_metrics(data,out,normalization,category):
    for i,element in enumerate(map(str,data.element)):
        meta=dict(category=category,calc_id=str(data.calc_id[i]),element=element)
        for name,pred,truth,scale in (
            ('rho',out['rho_scaled'][i],data.rho_target[i],1.),
            ('potential',out['potential_scaled'][i],data.potential_target[i],normalization.potential_scale_ry)):
            p=pred.detach().float().cpu().numpy()*scale
            t=truth.detach().float().cpu().numpy()*scale
            row=dict(**meta,target=name,unit='rho/mean_rho' if name=='rho' else 'Ry',
                **v29_regression_metrics(t,p,CFG.evaluation_spearman_points))
            V29_DETAIL_ROWS.append(row)
            threshold=DENSITY_CLASSIFICATION_THRESHOLDS[element] if name=='rho' else 0.
            V29_CLASS_ROWS.append(v29_binary_row(t>threshold,p>threshold,**meta,
                task='density_high' if name=='rho' else 'potential_positive'))
            # Physical Fourier coefficients. Error includes amplitude and phase.
            ft=np.fft.rfftn(t.squeeze(),norm='ortho').ravel()
            fp=np.fft.rfftn(p.squeeze(),norm='ortho').ravel()
            ids=V29_SPECTRAL_INDICES
            for band,(a,b) in zip(('low','middle','high'),CFG.jepa_band_boundaries):
                target_modes=ft[ids[a//2:b//2]]; pred_modes=fp[ids[a//2:b//2]]
                norm=np.linalg.norm(target_modes)
                valid=np.abs(target_modes)>max(np.max(np.abs(target_modes))*1e-6,1e-12)
                weights=np.abs(target_modes[valid])
                phase=np.abs(np.angle(pred_modes[valid]*np.conj(target_modes[valid])))
                V29_SPECTRAL_ROWS.append(dict(**meta,target=name,band=band,
                    relative_l2=float(np.linalg.norm(pred_modes-target_modes)/norm) if norm>1e-12 else np.nan,
                    amplitude_relative_l2=float(np.linalg.norm(np.abs(pred_modes)-np.abs(target_modes))/norm) if norm>1e-12 else np.nan,
                    amplitude_ratio=float(np.linalg.norm(pred_modes)/norm) if norm>1e-12 else np.nan,
                    weighted_phase_mae_rad=float(np.average(phase,weights=weights)) if weights.sum()>0 else np.nan,
                    phase_valid_modes=int(valid.sum())))

def v29_evaluate_validation_stage(model,stage):
    global V29_DETAILED_EVAL
    previous=V29_DETAILED_EVAL; V29_DETAILED_EVAL=True
    frames=[]
    try:
        for element,loader in {**retention_loaders,'Ni':ni_validation_loader}.items():
            frame,_=evaluate_loader(model,loader,ni_normalization,f'{element}_validation_{stage}')
            frames.append(frame)
    finally: V29_DETAILED_EVAL=previous
    return frames

def v29_fit_thresholds():
    # Exact V24 density threshold estimator; executed before named evaluations.
    pools={'Al':pretrain_train_frame.loc[pretrain_train_frame.element.eq('Al')],
           'Fe':pretrain_train_frame.loc[pretrain_train_frame.element.eq('Fe')],
           'Ni':frame_for_mode(splits['ni_adapt_train'],32,1200)}
    thresholds={}; audit=[]; upper={}
    for offset,(element,frame) in enumerate(pools.items()):
        sample=deterministic_sample(frame,min(CFG.classification_threshold_records_per_element,len(frame)),CFG.seed+710+offset)
        fields=[np.asarray(cached_raw_record(row.to_dict())['rho_scaled']).ravel() for _,row in sample.iterrows()]
        thresholds[element]=float(np.quantile(np.concatenate(fields),CFG.density_classification_quantile))
        coordinates=normalized_cohesive_coordinates(frame,pretrain_normalization)
        upper[element]={str(q):float(pretrain_normalization.decode_cohesive_energy(np.quantile(coordinates,q))) for q in (.90,.95,.99)}
        for cid in frame.calc_id: audit.append(dict(element=element,calc_id=str(cid),purpose='energy_tail_thresholds'))
        for cid in sample.calc_id: audit.append(dict(element=element,calc_id=str(cid),purpose='density_threshold'))
    pd.DataFrame(audit).to_csv(CFG.run_dir/'classification_threshold_training_ids.csv',index=False)
    atomic_json_dump({'density':thresholds,'energy_upper_ev_per_atom':upper,
        'fit_scope':'training configurations only; no holdout quantiles'},CFG.run_dir/'v29_thresholds.json')
    return thresholds,upper

DENSITY_CLASSIFICATION_THRESHOLDS,V29_UPPER_ENERGY_THRESHOLDS=v29_fit_thresholds()
V29_SPECTRAL_INDICES=low_frequency_rfft_indices(CFG.canonical_grid,CFG.field_spectral_complex_modes).cpu().numpy()


In [36]:
def flattened_cosine(left: torch.Tensor, right: torch.Tensor) -> torch.Tensor:
    left_flat = left.flatten(1)
    right_flat = right.flatten(1)
    denominator = left_flat.norm(dim=-1) * right_flat.norm(dim=-1)
    return (left_flat * right_flat).sum(-1) / denominator.clamp_min(1e-12)

def periodic_gradient_relative_l2(prediction,target,cell):
    pg,tg=periodic_cartesian_gradient(prediction,cell),periodic_cartesian_gradient(target,cell)
    return torch.sqrt((pg-tg).square().flatten(1).sum(-1)/tg.square().flatten(1).sum(-1).clamp_min(1e-12))

def finite_r2(target: pd.Series, prediction: pd.Series) -> float:
    valid = target.notna() & prediction.notna()
    target_values = target[valid].to_numpy(dtype=np.float64)
    prediction_values = prediction[valid].to_numpy(dtype=np.float64)
    if len(target_values) < 2:
        return float("nan")
    denominator = np.square(target_values - target_values.mean()).sum()
    return float(1.0 - np.square(target_values - prediction_values).sum() / max(denominator, 1e-12))

def batch_string_values(value: Any, expected: int) -> list[str]:
    values = value if isinstance(value, list) else [value]
    if len(values) != expected:
        raise RuntimeError(f"Categorical metadata with {len(values)} values for batch of {expected}")
    return [str(item) for item in values]
def cohesive_tail_summary(frame):
    tail=frame.loc[frame.cohesive_weak_binding.eq(True)]
    if tail.empty:
        return {'cohesive_weak_records':0,'cohesive_weak_mae_ev_per_atom':float('nan'),
                'cohesive_weak_normalized_mae':float('nan'),'cohesive_weak_bias_ev_per_atom':float('nan')}
    return {'cohesive_weak_records':len(tail),
            'cohesive_weak_mae_ev_per_atom':float(tail.cohesive_abs_error_ev_per_atom.mean()),
            'cohesive_weak_normalized_mae':float(tail.cohesive_abs_error_normalized.mean()),
            'cohesive_weak_bias_ev_per_atom':float(tail.cohesive_signed_error_ev_per_atom.mean())}

def summarize_configuration_metrics(frame):
    if frame.empty: raise ValueError('Evaluation has no configurations')
    error=frame.cohesive_signed_error_ev_per_atom.to_numpy()
    result = {'records':len(frame),
        'rho_relative_l2_mean':float(frame.rho_relative_l2.mean()),
        'rho_cosine_similarity_mean':float(frame.rho_cosine_similarity.mean()),
        'potential_relative_l2_mean':float(frame.potential_relative_l2.mean()),
        'potential_rmse_ry_mean':float(frame.potential_rmse_ry.mean()),
        'potential_cosine_similarity_mean':float(frame.potential_cosine_similarity.mean()),
        'cohesive_mae_ev_per_atom':float(np.abs(error).mean()),
        'cohesive_rmse_ev_per_atom':float(np.sqrt(np.mean(error**2))),
        'cohesive_bias_ev_per_atom':float(error.mean()),
        'cohesive_error_p95_ev_per_atom':float(np.quantile(np.abs(error),.95)),
        'cohesive_normalized_mae':float(frame.cohesive_abs_error_normalized.mean()),
        'cohesive_r2':finite_r2(frame.cohesive_target_ev_per_atom,frame.cohesive_pred_ev_per_atom),
        'cohesive_overprediction_fraction':float(np.mean(error>0)),
        'latent_cosine_mean':float(frame.latent_cosine.mean()),**cohesive_tail_summary(frame)}

    target = frame.cohesive_target_ev_per_atom.to_numpy(dtype=float)
    prediction = frame.cohesive_pred_ev_per_atom.to_numpy(dtype=float)
    error = prediction - target
    target_std = float(np.std(target))
    if len(target) >= 2 and target_std > 1e-12:
        slope, intercept = np.polyfit(target, prediction, 1)
        pearson = float(np.corrcoef(target, prediction)[0, 1])
        spearman = float(scipy.stats.spearmanr(target, prediction).statistic)
    else:
        slope = intercept = pearson = spearman = float('nan')
    result.update({
        'cohesive_median_absolute_error_ev_per_atom': float(np.median(np.abs(error))),
        'cohesive_max_absolute_error_ev_per_atom': float(np.max(np.abs(error))),
        'cohesive_error_std_ev_per_atom': float(np.std(error)),
        'cohesive_slope': float(slope), 'cohesive_intercept_ev_per_atom': float(intercept),
        'cohesive_pearson': pearson, 'cohesive_spearman': spearman,
    })
    result['cohesive_absolute_bias_ev_per_atom'] = abs(float(error.mean()))
    result['cohesive_error_p90_ev_per_atom'] = float(np.quantile(np.abs(error), .90))
    for key, value in list(result.items()):
        if key.endswith('_ev_per_atom'):
            result[key.replace('_ev_per_atom', '_mev_per_atom')] = float(value) * 1000.0
    return result

@torch.no_grad()
def evaluate_loader(model,loader,normalization,category):
    model.eval(); rows=[]
    for data in loader:
        data=move_batch(data)
        # Targets are kept separately; structural branch sees none of them.
        if V29_AE_EVALUATION:
            z=model.field_online_encoder(data.field_input)
            rho,potential=model.field_reconstruction_decoder(z)
            output={'rho_scaled':rho,'potential_scaled':potential,'z_pred':z,
                'cohesive_energy_normalized':model.cohesive_from_descriptor(
                    invariant_shared_latent_view(z,model.field_online_encoder.spectral_rms),data.element)}
        else:
            output=model.forward_operator(operator_batch(data))
        if V29_DETAILED_EVAL:
            v29_collect_field_metrics(data,output,normalization,category)
        rp=output['rho_scaled'].float(); vp=output['potential_scaled'].float()
        target=model.field_target_encoder(data.field_input)
        rerr=relative_field_error(rp,data.rho_target); verr=relative_field_error(vp,data.potential_target)
        rcos=flattened_cosine(rp,data.rho_target); vcos=flattened_cosine(vp,data.potential_target)
        vrmse=(vp-data.potential_target).square().flatten(1).mean(-1).sqrt()*normalization.potential_scale_ry
        lcos=F.cosine_similarity(output['z_pred'][...,2:].float(),target[...,2:].float(),dim=-1).mean(-1)
        ep=(output['cohesive_energy_normalized'].double()*normalization.cohesive_scale_ev_per_atom
            +normalization.cohesive_center_ev_per_atom)
        truth=data.cohesive_energy_ev_per_atom.reshape(-1).double(); error=ep-truth
        charge=(rp.mean((-3,-2,-1)).reshape(-1)-1).abs()*data.electron_count.reshape(-1)
        for i,element in enumerate(map(str,data.element)):
            threshold=COHESIVE_WEAK_THRESHOLDS.get(element)
            extreme=COHESIVE_EXTREME_WEAK_THRESHOLDS.get(element)
            normalized_target=float(data.cohesive_energy_target.reshape(-1)[i])
            rows.append({'category':category,'calc_id':str(data.calc_id[i]),'element':element,
                'family':str(data.family[i]),'atom_count':int(data.atom_count.reshape(-1)[i]),
                'rho_relative_l2':float(rerr[i]),'rho_cosine_similarity':float(rcos[i]),
                'potential_relative_l2':float(verr[i]),'potential_cosine_similarity':float(vcos[i]),
                'potential_rmse_ry':float(vrmse[i]),'latent_cosine':float(lcos[i]),
                'charge_constraint_error_e':float(charge[i]),
                'potential_gauge_error_ry':float(vp[i].mean().abs()*normalization.potential_scale_ry),
                'cohesive_pred_ev_per_atom':float(ep[i]),'cohesive_target_ev_per_atom':float(truth[i]),
                'cohesive_signed_error_ev_per_atom':float(error[i]),'cohesive_abs_error_ev_per_atom':float(error[i].abs()),
                'cohesive_abs_error_normalized':float(error[i].abs()/normalization.cohesive_scale_ev_per_atom),
                'cohesive_weak_binding':normalized_target<=threshold if threshold is not None else None,
                'cohesive_extreme_weak_binding':normalized_target<=extreme if extreme is not None else None})
    frame=pd.DataFrame(rows)
    if frame.duplicated(['calc_id']).any(): raise ValueError('Evaluation requires one row per unique configuration')
    return frame,{'category':category,**summarize_configuration_metrics(frame)}

evaluation_frames=[]; evaluation_summaries=[]
def record_evaluation(model,loader,category):
    global V29_DETAILED_EVAL
    previous=V29_DETAILED_EVAL; V29_DETAILED_EVAL=True
    try:
        frame,summary=evaluate_loader(model,loader,pretrain_normalization,category)
    finally: V29_DETAILED_EVAL=previous
    evaluation_frames.append(frame); evaluation_summaries.append(summary)
    display(pd.DataFrame([summary]))
    return frame,summary



In [37]:
# Physical metrics consolidated in the summarize_configuration_metrics function above.


In [38]:
def load_compatible_ae_warm_start(model, normalization):
    """Explicit transfer of trusted AE weights, never an old predictor/optimizer.

    The path must point to a checkpoint generated by your own notebook. PyTorch
    training checkpoints can contain Python state; do not use an untrusted file.
    """
    source = str(CFG.ae_warm_start_checkpoint).strip()
    if not source:
        return False
    path = Path(source).expanduser()
    if not path.is_file():
        raise FileNotFoundError(f'AE checkpoint not found: {path}')
    payload = torch.load(path, map_location='cpu', weights_only=False)
    if payload.get('stage') != 'autoencoder':
        raise ValueError('Warm-start accepts only an autoencoder-stage checkpoint.')
    if payload.get('seed') != CFG.seed:
        raise ValueError('AE checkpoint seed differs from the current run.')
    expected = json.loads(json.dumps(asdict(normalization), default=str))
    actual = json.loads(json.dumps(payload.get('normalization'), default=str))
    if actual != expected:
        raise ValueError('Incompatible normalization: do not transfer AE across different splits/scales.')
    validate_checkpoint(payload, normalization)
    state = payload['model_state']
    names = ('field_online_encoder', 'field_reconstruction_decoder', 'cohesive_head')
    prepared = {}
    for name in names:
        module = getattr(model, name)
        incoming = {k[len(name)+1:]: v for k, v in state.items() if k.startswith(name+'.')}
        current = module.state_dict()
        if incoming.keys() != current.keys() or any(incoming[k].shape != current[k].shape for k in current):
            raise ValueError(f'Incompatible AE module: {name}')
        for key in ('spectral_indices', 'spectral_rms'):
            if key in current:
                torch.testing.assert_close(incoming[key], current[key].cpu(), rtol=1e-6, atol=1e-7)
        prepared[name] = incoming
    # Verify the old checkpoint's exact train IDs when available through its run.
    old_stats_path = path.parent.parent.parent / 'spectral_rms_statistics.json'
    if not old_stats_path.is_file():
        raise ValueError('Warm-start requires spectral_rms_statistics.json in the source run directory.')
    old_stats = json.loads(old_stats_path.read_text(encoding='utf-8'))
    if sorted(old_stats.get('fitted_calc_ids', [])) != sorted(SPECTRAL_RMS_STATISTICS['fitted_calc_ids']):
        raise ValueError('The IDs used to fit spectral scales do not match.')
    source_split_path = path.parent.parent.parent / 'split_manifest.json'
    split_ids_verified = False
    if source_split_path.is_file():
        source_splits = json.loads(source_split_path.read_text(encoding='utf-8'))
        if source_splits.get('records') != split_manifest['records']:
            raise ValueError('The complete AE checkpoint splits do not match.')
        split_ids_verified = True
    else:
        print('Warning: old run without split_manifest.json; seed/normalization/spectral IDs checked. '
              'Use warm-start only if the dataset and original splits are the same.')
    for name, incoming in prepared.items():
        getattr(model, name).load_state_dict(incoming, strict=True)
    initialize_target_from_trained_autoencoder(model, source=path)
    audit = dict(source=str(path.resolve()), source_architecture=payload.get('architecture_version'),
                 modules=list(names), optimizer_imported=False, predictor_imported=False,
                 normalization_and_spectral_statistics_match=True,
                 complete_split_ids_verified=split_ids_verified,
                 note='AE transfer only; predictor and optimizer start fresh; target copied exactly.')
    atomic_json_dump(audit, CFG.run_dir/'ae_warm_start_audit.json')
    print('AE reused; reconstruction gates will be evaluated again.', str(path))
    return True

def run_short_jepa_overfit(model, frame, normalization):
    """Paired, train-only fit test. Clones and RNG isolation leave the run unchanged."""
    if not CFG.short_overfit_diagnostic:
        return {'skipped': True}
    # Do not silently repeat diagnostics during a resumed curriculum.
    if CFG.resume and (CFG.run_dir/'checkpoints'/'D_latent'/'last.pt').is_file():
        print('Short diagnostic not repeated: this run has a D_latent checkpoint.')
        return {'skipped': True, 'reason': 'resuming D_latent'}
    selected = []
    for element in CFG.pretrain_elements:
        pool = frame[frame.element.astype(str).eq(element)]
        if pool.empty:
            raise ValueError(f'Diagnostic requires training records for {element}.')
        selected.append(deterministic_sample(pool, min(CFG.overfit_records_per_element, len(pool)), CFG.seed+137))
    # Interleave elements for balanced minibatches.
    selected = pd.concat(selected).sort_values('calc_id', kind='stable')
    element_rows = [g.to_dict('records') for _, g in selected.groupby('element')]
    rows = [g[i] for i in range(max(map(len, element_rows))) for g in element_rows if i < len(g)]
    graphs = [graph_from_record(cached_raw_record(row), normalization, True) for row in rows]
    size = max(2, min(CFG.overfit_batch_size, len(graphs)))
    batches = [Batch.from_data_list(graphs[i:i+size]) for i in range(0, len(graphs), size)]
    steps = min(20, CFG.overfit_steps) if CFG.run_mode in ('SMOKE','QUICK') else CFG.overfit_steps
    devices = [DEVICE.index if DEVICE.index is not None else torch.cuda.current_device()] if DEVICE.type == 'cuda' else []
    results, history = [], []
    with torch.random.fork_rng(devices=devices):
        for translated in (False, True):
            torch.manual_seed(CFG.seed+139)
            probe = copy.deepcopy(model).to(DEVICE).eval()
            for p in probe.parameters():
                p.requires_grad_(False)
                p.grad = None
            parameters = list(probe.context_encoder.parameters()) + list(probe.predictor.parameters())
            for p in parameters:
                p.requires_grad_(True)
            optimizer = torch.optim.AdamW(parameters, lr=CFG.overfit_learning_rate, weight_decay=0.0)
            # Same held-fixed AE for both scenarios; cache targets without augmentation.
            prepared = [b.to(DEVICE) for b in batches]
            with torch.no_grad():
                targets = [probe.field_target_encoder(b.field_input) for b in prepared]
            def evaluate():
                collected = []
                with torch.no_grad():
                    for b, target in zip(prepared, targets):
                        fc, nodes = probe.encode_structure(b)
                        pred, _, _, _ = probe.predict_fields(nodes, fc, b)
                        for j, element in enumerate(b.element):
                            for f, name in enumerate(FIELD_NAMES):
                                item = dict(element=str(element), field=name)
                                item['cosine_full'] = float(F.cosine_similarity(pred[j,f,2:], target[j,f,2:], dim=0, eps=1e-6))
                                for label, (a, z) in zip(('low','middle','high'), CFG.jepa_band_boundaries):
                                    item['cosine_'+label] = float(F.cosine_similarity(pred[j,f,a:z], target[j,f,a:z], dim=0, eps=1e-6))
                                collected.append(item)
                return pd.DataFrame(collected).groupby(['element','field']).mean().reset_index().to_dict('records')
            before = evaluate()
            started = time.perf_counter()
            for step in range(steps):
                index = step % len(prepared)
                batch = prepared[index]
                if translated:
                    shifts = tuple(int(torch.randint(0, n, ()).item()) for n in CFG.canonical_grid)
                    batch = translate_periodic_batch(batch, shifts)
                    with torch.no_grad():
                        target = probe.field_target_encoder(batch.field_input)
                else:
                    target = targets[index]
                optimizer.zero_grad(set_to_none=True)
                fc, nodes = probe.encode_structure(batch)
                pred, _, _, _ = probe.predict_fields(nodes, fc, batch)
                # Fixed final bands: fit diagnostic is not the long curriculum.
                alignment, metrics = spectral_alignment_terms(pred, target, 'diagnostic', 1., False)
                loss = alignment['jepa'] + 0.5*alignment['latent_cosine'] + 0.1*alignment['latent_scale']
                if not bool(torch.isfinite(loss)):
                    raise FloatingPointError('JEPA diagnostic produced nonfinite loss.')
                loss.backward()
                torch.nn.utils.clip_grad_norm_(parameters, CFG.gradient_clip_norm)
                optimizer.step()
                if step == 0 or (step+1) % CFG.overfit_log_every == 0 or step+1 == steps:
                    item = dict(translations=translated, step=step+1, loss=float(loss.detach()),
                                cosine_full=float(metrics['latent_cosine_full_similarity'].detach()))
                    history.append(item)
                    print('JEPA short fit:', item)
            after = evaluate()
            results.append(dict(translations=translated, records=len(graphs), steps=steps,
                                seconds=time.perf_counter()-started, before=before, after=after,
                                evaluation='same unshifted training records in both scenarios'))
            del optimizer, parameters
    report = dict(source='pretrain_train ONLY', calc_ids=[str(x['calc_id']) for x in rows],
                  cloned_model=True, dropout_disabled=True, ema_teacher_fixed=True,
                  optimization='fixed final spectral weights; no curriculum', scenarios=results,
                  warning='Fit diagnostic only. Does not establish generalization or optimal hyperparameters.')
    pd.DataFrame(history).to_csv(CFG.run_dir/'diagnostic_overfit_history.csv', index=False)
    atomic_json_dump(report, CFG.run_dir/'diagnostic_overfit_summary.json')
    print(json.dumps(report, indent=2))
    return report


In [39]:
# Plan the complete AlFe optimizer budget before the first AE update.
ALFE_PLANNED_STAGES=[('autoencoder',CFG.preset.ae_epochs,CFG.autoencoder_learning_rate)]+alfe_jepa_schedule()+[
    ('P_cohesive_refinement',1 if CFG.run_mode in ('SMOKE','QUICK') else CFG.energy_refinement_epochs,CFG.energy_refinement_learning_rate)]
ALFE_PLANNED_UPDATES=sum(max(1,int(round(sum(r['optimizer_updates'] for r in
    data_curriculum_plan(stage,epochs,pretrain_train_loader))*CFG.optimizer_update_budget_scale)))
    for stage,epochs,_ in ALFE_PLANNED_STAGES)
configure_ema_phase('AlFe',ALFE_PLANNED_UPDATES,reset=False)
SOURCE_LR_PLANNED_UPDATES=sum(max(1,int(round(sum(r['optimizer_updates'] for r in
    data_curriculum_plan(stage,epochs,pretrain_train_loader))*CFG.optimizer_update_budget_scale)))
    for stage,epochs,_ in alfe_jepa_schedule())
configure_lr_macro_phase('AlFe',SOURCE_LR_PLANNED_UPDATES)
CURRICULUM_PROGRESS['planned_epochs']=sum(e for _,e,_ in ALFE_PLANNED_STAGES)+max(3,CFG.preset.ni_epochs)
atomic_json_dump(dict(stages=ALFE_PLANNED_STAGES,global_alfe_updates=ALFE_PLANNED_UPDATES,
    source_lr_planned_updates=SOURCE_LR_PLANNED_UPDATES),CFG.run_dir/'global_optimizer_plan.json')

@torch.no_grad()
def audit_stage_quality(model, loader, stage, enforce=True, suffix=""):
    """Per-element/per-field gate; no holdout/test set enters checkpoint decisions."""
    model.eval()
    grouped = {}
    for data in loader:
        data = move_batch(data)
        if stage == "autoencoder":
            z = model.field_online_encoder(data.field_input)
            rp, vp = model.field_reconstruction_decoder(z)
            rerr, verr = relative_field_error(rp,data.rho_target), relative_field_error(vp,data.potential_target)
            for j,element in enumerate(data.element):
                grouped.setdefault(element, []).append([float(rerr[j]),float(verr[j])])
        else:
            fc, nodes = model.encode_structure(data)
            pred, _, _, _ = model.predict_fields(nodes,fc,data)
            target = model.field_target_encoder(data.field_input)
            for j,element in enumerate(data.element):
                for f,name in enumerate(FIELD_NAMES):
                    grouped.setdefault((element,name),[]).append((
                        jepa_alignment_view(pred)[j,f].cpu(),jepa_alignment_view(target)[j,f].cpu()))
    rows = []
    for key,values in grouped.items():
        if stage == "autoencoder":
            rho,potential = np.mean(values,axis=0)
            passed = rho <= CFG.ae_rho_relative_l2_max and potential <= CFG.ae_potential_relative_l2_max
            rows.append(dict(element=key,rho_relative_l2=float(rho),potential_relative_l2=float(potential),passed=bool(passed)))
        else:
            pred, target = (torch.stack([x[k] for x in values]) for k in (0,1))
            pdg,tdg = latent_diagnostics(pred),latent_diagnostics(target)
            cosine = float(F.cosine_similarity(pred,target,dim=-1,eps=1e-6).mean())
            rank_min = max(1.2,min(CFG.d_rank_absolute_goal,CFG.d_rank_target_fraction*tdg["effective_rank"]))
            dominant_max = min(0.99,max(CFG.d_dominant_fraction_goal,tdg["dominant_covariance_fraction"]+0.05))
            band_cosines = {
                name: float(F.cosine_similarity(pred[:,a-2:b-2], target[:,a-2:b-2], dim=-1, eps=1e-6).mean())
                for name,(a,b) in zip(('low','middle','high'), CFG.jepa_band_boundaries)
            }
            full_threshold=(CFG.d_full_density_cosine_min if key[1]=='density'
                            else CFG.d_full_potential_cosine_min)
            criteria = dict(
                cosine_ok=cosine >= full_threshold,
                low_band_ok=band_cosines['low'] >= CFG.d_low_band_cosine_min,
                middle_band_ok=band_cosines['middle'] >= CFG.d_middle_band_cosine_min,
                high_band_ok=band_cosines['high'] >= CFG.d_high_band_cosine_min,
                rank_ok=pdg["effective_rank"] >= rank_min,
                dominant_fraction_ok=pdg["dominant_covariance_fraction"] <= dominant_max)
            passed = all(criteria.values())
            rows.append(dict(element=key[0],field=key[1],records=len(values),cosine_similarity=cosine,
                cosine_by_band=band_cosines,criteria=criteria,
                failed_criteria=[k for k,v in criteria.items() if not v],
                predicted_effective_rank=pdg["effective_rank"],target_effective_rank=tdg["effective_rank"],
                predicted_dominant_fraction=pdg["dominant_covariance_fraction"],
                target_dominant_fraction=tdg["dominant_covariance_fraction"],
                required_rank=rank_min,allowed_dominant_fraction=dominant_max,passed=bool(passed)))
    result = dict(stage=stage,run_mode=CFG.run_mode,rows=rows,all_passed=bool(rows) and all(x["passed"] for x in rows))
    stem = "autoencoder_field_gate" if stage == "autoencoder" else "D_latent_alignment_gate"
    path = CFG.run_dir / (stem + suffix + ".json")
    atomic_json_dump(result,path)
    pd.json_normalize(rows).to_csv(path.with_suffix('.csv'),index=False)
    print(json.dumps(result,indent=2))
    if not result["all_passed"] and enforce and CFG.enforce_stage_gates and CFG.run_mode not in ("SMOKE","QUICK"):
        raise RuntimeError(f"Gate for {stage} failed. Diagnostic: {path}. "
            "Do not start the next stage: compare AE errors, spectral coverage, and per-field alignment. "
            "Checkpoints were preserved; increasing epochs requires a new compatible run.")
    return result


# =============================================================================
# STEP 22 — Stage A — autoencoder
# Purpose: Trains the online encoder and field reconstruction decoder.
# Inputs: pretrain_train_loader and pretrain_validation_loader.
# Outputs: history_autoencoder.csv and checkpoints autoencoder.
# Reproducibility: run after all preceding cells; do not change the seed during the run.
# =============================================================================

@torch.no_grad()
def audit_spectral_resolution(frame,normalization):
    rows=[]
    for element,group in frame.groupby('element'):
        selected=deterministic_sample(group,min(len(group),CFG.spectral_audit_records_per_element),CFG.seed+91)
        for modes in (128,256,512):
            indices=low_frequency_rfft_indices(CFG.canonical_grid,modes).to(DEVICE)
            for _,row in selected.iterrows():
                data=Batch.from_data_list([graph_from_record(cached_raw_record(row.to_dict()),normalization,True)]).to(DEVICE)
                coefficients=periodic_fields_to_spectral(data.field_input,indices)
                raw=spectral_to_periodic_raw(coefficients,indices,CFG.canonical_grid)
                rr,vv=constrain_periodic_fields(raw)
                rows.append(dict(element=element,modes=modes,calc_id=str(row.calc_id),
                    rho_relative_l2=float(relative_field_error(rr,data.rho_target)[0]),
                    potential_relative_l2=float(relative_field_error(vv,data.potential_target)[0])))
    frame=pd.DataFrame(rows)
    frame.to_csv(CFG.run_dir/'spectral_resolution_validation.csv',index=False)
    display(frame.groupby(['element','modes'])[['rho_relative_l2','potential_relative_l2']].mean())
    return frame


if CFG.spectral_resolution_audit:
    SPECTRAL_RESOLUTION_AUDIT=audit_spectral_resolution(pretrain_validation_frame,pretrain_normalization)


# Before fitting, AE identity initialization exposes the truncation baseline.
INITIAL_SPECTRAL_RECONSTRUCTION = audit_stage_quality(
    model, pretrain_validation_loader, "autoencoder", enforce=False, suffix="_initial_spectrum")
if not INITIAL_SPECTRAL_RECONSTRUCTION["all_passed"]:
    warnings.warn("The initial spectral projection does not meet field limits. "
        "Compare with the trained AE; if it remains poor, increase field_spectral_complex_modes "
        "and latent_dim=2*modes in a new run before starting D.")
AE_WARM_STARTED = load_compatible_ae_warm_start(model, pretrain_normalization)
RUN_AUTOENCODER_STAGE = not AE_WARM_STARTED
autoencoder_history = pd.DataFrame()

if RUN_AUTOENCODER_STAGE:
    autoencoder_history = train_stage(
        model,
        stage="autoencoder",
        train_loader=pretrain_train_loader,
        validation_loader=pretrain_validation_loader,
        epochs=CFG.preset.ae_epochs,
        learning_rate=CFG.autoencoder_learning_rate,
        normalization=pretrain_normalization,
        validation_score_callback=physical_selection_callback,
    )
    display(autoencoder_history.tail())


@torch.no_grad()
def audit_autoencoder_latent(
    model: PeriodicFieldJEPA, loader: DataLoader
) -> dict[str, Any]:
    model.eval()
    latents, rho_losses, potential_losses = [], [], []
    for batch in loader:
        batch = move_batch(batch)
        z = model.field_online_encoder(batch.field_input)
        rho, potential = model.field_reconstruction_decoder(z)
        rho_loss, potential_loss = field_losses(rho, potential, batch)
        latents.append(z.cpu())
        rho_losses.append(float(rho_loss))
        potential_losses.append(float(potential_loss))
    diagnostics = latent_diagnostics(torch.cat(latents))
    gates = latent_gate_decisions(diagnostics)
    return {
        "records": len(loader.dataset),
        "population_tokens": len(loader.dataset) * len(FIELD_NAMES),
        "rho_reconstruction_loss": float(np.mean(rho_losses)),
        "potential_reconstruction_loss": float(np.mean(potential_losses)),
        **diagnostics, **gates, "all_latent_gates_passed": bool(all(gates.values())),
    }


AUTOENCODER_LATENT_AUDIT = audit_autoencoder_latent(
    model, pretrain_validation_loader
)
atomic_json_dump(
    AUTOENCODER_LATENT_AUDIT, CFG.run_dir / "autoencoder_latent_audit.json"
)
print(json.dumps(AUTOENCODER_LATENT_AUDIT, indent=2))
pd.DataFrame([{k:v for k,v in AUTOENCODER_LATENT_AUDIT.items() if not isinstance(v,(list,dict))}]).to_csv(
    CFG.run_dir/'autoencoder_latent_metrics.csv',index=False)
if 'covariance_eigenvalues' in AUTOENCODER_LATENT_AUDIT:
    pd.DataFrame({'eigenvalue':AUTOENCODER_LATENT_AUDIT['covariance_eigenvalues']}).to_csv(
        CFG.run_dir/'autoencoder_latent_covariance_eigenvalues.csv',index=False)


AUTOENCODER_FIELD_GATE = audit_stage_quality(model,pretrain_validation_loader,"autoencoder")
# Initialization already occurred after best-AE restoration or explicit AE warm-start.
require_initialized_jepa_target(model)
model.field_prediction_decoder.base.load_state_dict(model.field_reconstruction_decoder.head.state_dict())


rho_relative_l2  potential_relative_l2
element modes                                        
Al      128           0.024680               0.074494
        256           0.012372               0.049469
        512           0.007604               0.025737
Fe      128           0.276928               0.183212
        256           0.160418               0.076778
        512           0.084401               0.019477

{
  "stage": "autoencoder",
  "run_mode": "QUICK_TRAIN",
  "rows": [
    {
      "element": "Al",
      "rho_relative_l2": 0.00761089470324805,
      "potential_relative_l2": 0.025970011454774067,
      "passed": true
    },
    {
      "element": "Fe",
      "rho_relative_l2": 0.08355312538333237,
      "potential_relative_l2": 0.019280526699731126,
      "passed": true
    }
  ],
  "all_passed": true
}
{"stage": "autoencoder", "epoch": 0, "global_step": 32, "field_target_ema_updates": 0, "field_target_initialized_from_ae": false, "learning_rate": 9.984896207922383e-05, "ema_tau": 0.9960044828872989, "field_ema_source_trainable": false, "field_ema_source": "electronic_online_encoder", "epoch_seconds": 18.73335914700874, "training_records_per_second": 27.330923193332033, "optimizer_updates_this_epoch": 32, "stage_progress": 0.125, "optimizer_reinitialized_at_stage_start": false, "adam_states_restored": 0, "adam_states_initialized": 26, "optimizer_update_budget": 256, "objective_ramp_co

,stage,epoch,global_step,field_target_ema_updates,field_target_initialized_from_ae,learning_rate,ema_tau,field_ema_source_trainable,field_ema_source,epoch_seconds,...,checkpoint_eligible,checkpoint_gates,full_regime_updates,hard_underexposed_ids,ema_frozen,global_alfe_updates,checkpoint_accepted,checkpoint_reason,early_stopping_bad_epochs,early_stopped
11,autoencoder,3,128,0,False,0.000063,0.996071,False,electronic_online_encoder,18.869239,...,True,"{""absolute_field_constraints"": true, ""ae_ramp_...",52,,False,128,True,accepted_lower_energy_or_curriculum_score,0,False
12,autoencoder,4,160,0,False,0.000043,0.996111,False,electronic_online_encoder,17.808379,...,True,"{""absolute_field_constraints"": true, ""ae_ramp_...",84,,False,160,True,accepted_lower_energy_or_curriculum_score,0,False
13,autoencoder,5,192,0,False,0.000026,0.996159,False,electronic_online_encoder,18.594778,...,True,"{""absolute_field_constraints"": true, ""ae_ramp_...",116,,False,192,True,accepted_lower_energy_or_curriculum_score,0,False
14,autoencoder,6,224,0,False,0.000014,0.996214,False,electronic_online_encoder,18.041547,...,True,"{""absolute_field_constraints"": true, ""ae_ramp_...",148,,False,224,False,rejected_no_score_improvement,1,False
15,autoencoder,7,256,0,False,0.000010,0.996278,False,electronic_online_encoder,18.278282,...,True,"{""absolute_field_constraints"": true, ""ae_ramp_...",180,,False,256,True,accepted_lower_energy_or_curriculum_score,0,False


{
  "records": 128,
  "population_tokens": 256,
  "rho_reconstruction_loss": 0.026743029404315166,
  "potential_reconstruction_loss": 0.010328042932087556,
  "latent_variance": 0.6129267523086671,
  "effective_rank": 11.773472037027984,
  "active_covariance_rank": 117,
  "condition_number": Infinity,
  "active_condition_number": 924329.9227608967,
  "dominant_covariance_fraction": 0.39227764489598826,
  "near_zero_variance_fraction": 0.0,
  "covariance_eigenvalues": [
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0

<All keys matched successfully>

In [40]:
with isolated_rng():
    SHORT_JEPA_DIAGNOSTIC=run_short_jepa_overfit(model,pretrain_train_frame,pretrain_normalization)


JEPA short fit: {'translations': False, 'step': 1, 'loss': 0.9223023653030396, 'cosine_full': 0.3174239695072174}
JEPA short fit: {'translations': False, 'step': 25, 'loss': 0.6252866983413696, 'cosine_full': 0.4826172888278961}
JEPA short fit: {'translations': False, 'step': 50, 'loss': 0.5793597102165222, 'cosine_full': 0.4945405125617981}
JEPA short fit: {'translations': False, 'step': 75, 'loss': 0.5858849883079529, 'cosine_full': 0.4987376928329468}
JEPA short fit: {'translations': False, 'step': 100, 'loss': 0.4706321656703949, 'cosine_full': 0.5522613525390625}
JEPA short fit: {'translations': False, 'step': 125, 'loss': 0.4638408422470093, 'cosine_full': 0.5947962403297424}
JEPA short fit: {'translations': False, 'step': 150, 'loss': 0.3738999366760254, 'cosine_full': 0.6625650525093079}
JEPA short fit: {'translations': False, 'step': 175, 'loss': 0.35965681076049805, 'cosine_full': 0.6605616807937622}
JEPA short fit: {'translations': False, 'step': 200, 'loss': 0.2135583609342

In [41]:
# Curriculum with dedicated D, gates, and independent optimizers.
RUN_JEPA_STAGE = True
if RUN_JEPA_STAGE:
    schedule=alfe_jepa_schedule()
    curriculum_histories = []
    pretrain_candidates=[]
    for curriculum_stage,stage_epochs,lr in schedule:
        history = train_stage(model,stage=curriculum_stage,train_loader=pretrain_train_loader,
            validation_loader=pretrain_validation_loader,epochs=stage_epochs,learning_rate=lr,
            normalization=pretrain_normalization,
            validation_score_callback=make_balanced_al_fe_validation_callback(curriculum_stage))
        curriculum_histories.append(history)
        display(history.tail())
        if curriculum_stage == "D_latent":
            CURRICULUM_LATENT_GATE_AUDIT = audit_stage_quality(model,pretrain_validation_loader,"D_latent")
        if curriculum_stage in ('E_full','F_periodic_ramp'):
            with isolated_rng(): summaries,details=physical_validation_snapshot(model,pretrain_validation_loader)
            details.to_csv(CFG.run_dir/f'validation_physical_{curriculum_stage}.csv',index=False)
            pretrain_candidates.append(dict(stage=curriculum_stage,
                validation_score=fixed_physical_score(summaries),metrics=summaries,
                checkpoint=history.attrs['best_checkpoint']))
    global_field_references = {}
    for candidate in pretrain_candidates:
        v29_update_field_references(candidate['metrics'], global_field_references)
    frozen_global_field_references = copy.deepcopy(global_field_references)
    for candidate in pretrain_candidates:
        candidate['validation_field_constraints_passed'], candidate['field_violations'] = v29_field_constraints(
            candidate['metrics'], frozen_global_field_references)
    feasible_pretrain = [c for c in pretrain_candidates if c['validation_field_constraints_passed']]
    if not feasible_pretrain:
        raise RuntimeError('No E_full/F candidate satisfies the 5% field limit; energy selection refused')
    PRETRAIN_GLOBAL_BEST=min(feasible_pretrain,key=lambda c:c['validation_score'])
    chosen=torch.load(PRETRAIN_GLOBAL_BEST['checkpoint'],map_location='cpu',weights_only=False)
    validate_checkpoint(chosen,pretrain_normalization)
    validate_model_state_before_restore(model,chosen)
    model.load_state_dict(chosen['model_state'])
    atomic_torch_save({**chosen,'completed_stage':chosen['stage'],'ema_phase_state':dict(EMA_PHASE_STATE)},
                      OPTIMIZER_HANDOFF_PATH)
    PRETRAIN_GLOBAL_BEST['selection_rule']='Al/Fe validation energy MAE eV/atom; per-element fields within tolerance of an explicitly constructed frozen best-E_full/F reference'
    atomic_json_dump({'candidates':pretrain_candidates,'selected':PRETRAIN_GLOBAL_BEST},
                    CFG.run_dir/'pretrain_candidate_selection.json')
    atomic_torch_save({**chosen,**PRETRAIN_GLOBAL_BEST,"model_state":model.state_dict(),
        "checkpoint_role":"inference_selection_only",
        "architecture_version":ARCHITECTURE_VERSION,
        "training_protocol_version":TRAINING_PROTOCOL_VERSION,
        "curriculum_version":CURRICULUM_VERSION,
        "checkpoint_schema_version":CHECKPOINT_SCHEMA_VERSION},
        CFG.run_dir/"checkpoints"/"pretrain_global_best.pt")
    jepa_history = pd.concat(curriculum_histories,ignore_index=True)
    SOURCE_DOMAIN_READY,SOURCE_DOMAIN_GATES,SOURCE_DOMAIN_DIAGNOSTICS=source_domain_ready(
        model,pretrain_validation_loader,pretrain_normalization)
    atomic_json_dump({'ready':SOURCE_DOMAIN_READY,'gates':SOURCE_DOMAIN_GATES,
                      'diagnostics':SOURCE_DOMAIN_DIAGNOSTICS},
                     CFG.run_dir/'source_domain_readiness.json')
    if not SOURCE_DOMAIN_READY and CFG.run_mode not in ('SMOKE','QUICK'):
        raise RuntimeError('Al/Fe source did not pass the final gates; F/P/Ni cannot advance. '
                           'See source_domain_readiness.json.')
    if not SOURCE_DOMAIN_READY:
        warnings.warn('SMOKE: source-domain gates failed; continuation allowed only for integration testing.')
    print(PRETRAIN_GLOBAL_BEST)


def refinement_decision(current, baseline):
    floor=CFG.refinement_error_floor_normalized
    violations=[]
    for e,b in baseline.items():
        c=current[e]
        for key,limit in [('energy',CFG.refinement_max_element_relative_regression),
                          ('weak_energy',CFG.refinement_max_tail_relative_regression)]:
            if c[key]-b[key] > limit*max(b[key],floor): violations.append(f'{e}:{key}')
        for key in ('rho','potential'):
            if abs(c[key]-b[key]) > 1e-7+1e-6*abs(b[key]): violations.append(f'{e}:{key}:changed')
    score=fixed_physical_score(current)
    return not violations,score,violations


# P is head-only and therefore conditional: it is useful only after source fields are ready
# and when energy is already within a moderate neighborhood of the final gate.
with isolated_rng(): refinement_baseline,_=physical_validation_snapshot(model,pretrain_validation_loader)
energy_near_gate=all(
    metrics['energy']*pretrain_normalization.cohesive_scale_ev_per_atom <=
    2.0*ENERGY_GATE_LIMITS[element]['mae']
    for element,metrics in refinement_baseline.items()
)
RUN_P_REFINEMENT=bool(SOURCE_DOMAIN_READY and energy_near_gate)
ENERGY_REFINEMENT_AUDIT={'executed':RUN_P_REFINEMENT,'accepted':False,
    'reason':'eligible_head_only_refinement' if RUN_P_REFINEMENT else
             'skipped_source_not_ready_or_energy_too_far_from_gate',
    'baseline':refinement_baseline}
if RUN_P_REFINEMENT:
    refinement_initial_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    refinement_initial_handoff=torch.load(OPTIMIZER_HANDOFF_PATH,map_location='cpu',weights_only=False)
    _,refinement_initial_score,_=refinement_decision(refinement_baseline,refinement_baseline)

    def refinement_validation_callback(model,metrics,scaler,global_step,total_steps):
        current,_=physical_validation_snapshot(model,pretrain_validation_loader)
        feasible,score,violations=refinement_decision(current,refinement_baseline)
        bounded=score/(1+score)
        return bounded if feasible else CFG.constraint_barrier+len(violations)+bounded, {
            'validation_refinement_energy_score':score,'validation_constraints_passed':feasible,
            'validation_refinement_violations':len(violations)}

    refinement_epochs=1 if CFG.run_mode in ('SMOKE','QUICK') else CFG.energy_refinement_epochs
    energy_refinement_history=train_stage(model,'P_cohesive_refinement',pretrain_train_loader,
        pretrain_validation_loader,refinement_epochs,CFG.energy_refinement_learning_rate,
        pretrain_normalization,validation_score_callback=refinement_validation_callback)
    with isolated_rng(): refinement_candidate,refinement_details=physical_validation_snapshot(model,pretrain_validation_loader)
    refinement_feasible,refinement_score,refinement_violations=refinement_decision(refinement_candidate,refinement_baseline)
    refinement_frozen_changes=[k for k,v in model.state_dict().items()
        if not k.startswith('cohesive_head.') and not torch.equal(v.detach().cpu(),refinement_initial_state[k])]
    if refinement_frozen_changes:
        model.load_state_dict(refinement_initial_state)
        raise RuntimeError(f'Refinamento alterou tensores congelados: {refinement_frozen_changes}')
    refinement_accepted=bool(refinement_feasible and refinement_score < refinement_initial_score)
    if not refinement_accepted:
        model.load_state_dict(refinement_initial_state)
        atomic_torch_save({**refinement_initial_handoff,'ema_phase_state':dict(EMA_PHASE_STATE)},OPTIMIZER_HANDOFF_PATH)
    ENERGY_REFINEMENT_AUDIT.update({'accepted':refinement_accepted,'baseline_score':refinement_initial_score,
        'candidate_score':refinement_score,'violations':refinement_violations,
        'candidate':refinement_candidate,'non_head_tensors_bitwise_unchanged':True,
        'peak_head_learning_rate':CFG.energy_refinement_learning_rate,
        'candidate_checkpoint':energy_refinement_history.attrs['best_checkpoint']})
    refinement_details.to_csv(CFG.run_dir/'validation_energy_refinement_candidate.csv',index=False)
    PRETRAIN_GLOBAL_BEST['energy_refinement_accepted']=refinement_accepted
    del refinement_initial_state,refinement_initial_handoff
else:
    PRETRAIN_GLOBAL_BEST['energy_refinement_accepted']=False

atomic_json_dump(ENERGY_REFINEMENT_AUDIT,CFG.run_dir/'energy_refinement_audit.json')
atomic_torch_save({**chosen,**PRETRAIN_GLOBAL_BEST,'model_state':model.state_dict(),
    'checkpoint_role':'inference_selection_only',
    'architecture_version':ARCHITECTURE_VERSION,
    'training_protocol_version':TRAINING_PROTOCOL_VERSION,
    'curriculum_version':CURRICULUM_VERSION,
    'checkpoint_schema_version':CHECKPOINT_SCHEMA_VERSION,
    'training_contract':training_contract()},
    CFG.run_dir/'checkpoints'/'pretrain_global_best.pt')
print('Energy refinement:',ENERGY_REFINEMENT_AUDIT)


{"stage": "D_latent", "epoch": 0, "global_step": 32, "field_target_ema_updates": 0, "field_target_initialized_from_ae": true, "learning_rate": 8.68421052631579e-05, "ema_tau": 1.0, "field_ema_source_trainable": true, "field_ema_source": "electronic_online_encoder", "epoch_seconds": 51.65383973199641, "training_records_per_second": 9.912138239025184, "optimizer_updates_this_epoch": 32, "stage_progress": 0.08333333333333333, "optimizer_reinitialized_at_stage_start": false, "adam_states_restored": 0, "adam_states_initialized": 177, "optimizer_update_budget": 384, "objective_ramp_complete": false, "data_curriculum_version": "full-pool-batch-curriculum-v31-r1-competence-gated", "data_curriculum_tier": "full_pool", "data_curriculum_fraction": 1.0, "data_curriculum_active_records": 512, "data_curriculum_active_unique_records": 512, "data_curriculum_new_unique_records": 512, "data_curriculum_full_epochs": 1, "data_curriculum_checkpoint_eligible": true, "data_curriculum_manifest_hash": "a3b27db

RuntimeError: Stage D_latent exhausted its budget without achieving readiness for 2 consecutive validations. Use a new run/budget; do not advance silently.

## Physical evaluation per configuration
Total energy is used only to construct the cohesive energy label. No metric converts the
prediction back to total energy. The weak-binding mask uses training thresholds.
Zero-shot Ni receives cohesive energy predictions in eV/atom without calibrating normalization on Ni.

In [ ]:
al_pre_frame,al_pre_summary=record_evaluation(model,al_holdout_loader,'Al_ID_before_Ni')
fe_pre_frame,fe_pre_summary=record_evaluation(model,fe_holdout_loader,'Fe_ID_before_Ni')
ood_loaders={}
for element in CFG.pretrain_elements:
    frame=pretrain_ood_frame.loc[pretrain_ood_frame.element.eq(element)]
    if len(frame):
        ood_loaders[element]=make_loader(frame,pretrain_normalization,False)
        record_evaluation(model,ood_loaders[element],f'{element}_OOD_before_Ni')
ni_normalization=pretrain_normalization
ni_id_holdout_frame=frame_for_mode(splits['ni_id_holdout_test'],24,600)
ni_ood_holdout_frame=frame_for_mode(splits['ni_family_ood_test'],24,600)
ni_id_holdout_loader=make_loader(ni_id_holdout_frame,ni_normalization,False)
ni_ood_holdout_loader=make_loader(ni_ood_holdout_frame,ni_normalization,False)
record_evaluation(model,ni_id_holdout_loader,'Ni_ID_zero_shot')
record_evaluation(model,ni_ood_holdout_loader,'Ni_OOD_zero_shot')


## Representative Ni and retention

Group splits are frozen before selecting 10% of the eligible set.
Marginal quotas and k-center coverage use standardized descriptors only within this set.
The main loader contains only Ni; Al/Fe retention is a separate, auditable stream.
The Al–Fe target and EMA clock remain frozen in all G_* stages.


In [ ]:
def build_joint_finetune_frame(ni_frame, alfe_frame, normalization):
    """Ni-only source. Retention data are a separate explicit optimization stream."""
    if not ni_frame.element.eq(CFG.transfer_element).all():
        raise ValueError('Adaptation source must contain only Ni')
    return ni_frame.drop_duplicates(['element','calc_id']).sort_values('calc_id').reset_index(drop=True)


In [ ]:
# V31: Ni transfer is forbidden until source-domain competence is established.
if not bool(globals().get('SOURCE_DOMAIN_READY',False)):
    if CFG.run_mode not in ('SMOKE','QUICK'):
        raise RuntimeError('Ni adaptation blocked: Al/Fe source did not pass the final gates.')
    warnings.warn('SMOKE: Ni ran despite a failed source gate only to validate integration.')

ni_train_frame=splits['ni_adapt_train'].copy()
ni_validation_frame=frame_for_mode(splits['ni_adapt_validation'],16,400)
ni_id_eligible_count=len(NI_SELECTION_MANIFEST['eligible_ids'])
assert set(ni_train_frame.calc_id)<=set(splits['ni_adapt_train'].calc_id)
assert not set(ni_train_frame.family)&set(CFG.ood_families)
assert len(ni_train_frame)<=math.floor(CFG.ni_finetune_fraction_cap*ni_id_eligible_count)
COHESIVE_UPPER_THRESHOLDS.update({str(e):float(np.quantile(normalized_cohesive_coordinates(g,ni_normalization),.90)) for e,g in ni_train_frame.groupby('element')})
COHESIVE_WEAK_THRESHOLDS.update(fit_cohesive_weak_thresholds(ni_train_frame,ni_normalization))
COHESIVE_EXTREME_WEAK_THRESHOLDS.update(fit_cohesive_extreme_weak_thresholds(ni_train_frame,ni_normalization))
atomic_json_dump({'weak':COHESIVE_WEAK_THRESHOLDS,'extreme':COHESIVE_EXTREME_WEAK_THRESHOLDS,
                 'pretrain_ids':pretrain_normalization.cohesive_fitted_calc_ids,
                 'ni_ids':ni_train_frame.calc_id.tolist()},CFG.run_dir/'cohesive_thresholds.json')
joint_finetune_frame=build_joint_finetune_frame(ni_train_frame,pretrain_train_frame,ni_normalization)
joint_finetune_loader=make_loader(joint_finetune_frame,ni_normalization,True)
ni_calibration_loader=make_loader(ni_train_frame,ni_normalization,True)
ni_validation_loader=make_loader(ni_validation_frame,ni_normalization,False)

register_data_curriculum_stage(
    'G_cohesive_calibration', ni_train_frame, ni_normalization,
    scope='Ni_calibration_only', balance_elements=False, focus='all')
for _stage in ('G_adapter_warmup','G_joint_finetune'):
    register_data_curriculum_stage(
        _stage, joint_finetune_frame, ni_normalization,
        scope='Ni_selected_only', balance_elements=False, focus='ni')
register_data_curriculum_stage(
    'G_retention_repair', joint_finetune_frame, ni_normalization,
    scope='Ni_selected_only', balance_elements=False, focus='retention')
NI_FINETUNE_DATA_AUDIT={'eligible_ni_id':ni_id_eligible_count,'unique_training_ni':ni_train_frame.calc_id.nunique(),
    'used_fraction':ni_train_frame.calc_id.nunique()/ni_id_eligible_count,
    'joint_replay_counts':joint_finetune_frame.element.value_counts().to_dict(),
    'normalization_uses_ni':False,'ood_optimized':False,
    'retention_stream_enabled':CFG.retention_stream_enabled, 'hidden_replay':False}
atomic_json_dump(NI_FINETUNE_DATA_AUDIT,CFG.run_dir/'ni_finetune_data_audit.json')

teacher_path=CFG.run_dir/'checkpoints'/'ni_retention_teacher.pt'
if CFG.resume and teacher_path.exists():
    teacher_payload=torch.load(teacher_path,map_location='cpu',weights_only=False)
    teacher_versions={'architecture_version':ARCHITECTURE_VERSION,
        'training_protocol_version':TRAINING_PROTOCOL_VERSION,
        'curriculum_version':CURRICULUM_VERSION,
        'checkpoint_schema_version':CHECKPOINT_SCHEMA_VERSION}
    if (teacher_payload.get('training_contract')!=training_contract() or
            any(teacher_payload.get(key)!=value for key,value in teacher_versions.items())):
        raise RuntimeError('Teacher incompatible with the V29 protocol')
    validate_model_state_before_restore(model,teacher_payload)
else:
    teacher_payload={'training_contract':training_contract(),
        'architecture_version':ARCHITECTURE_VERSION,
        'training_protocol_version':TRAINING_PROTOCOL_VERSION,
        'curriculum_version':CURRICULUM_VERSION,
        'checkpoint_schema_version':CHECKPOINT_SCHEMA_VERSION,
        'model_state':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()}}
    atomic_torch_save(teacher_payload,teacher_path)
RETENTION_TEACHER=copy.deepcopy(model).eval()
RETENTION_TEACHER.load_state_dict(teacher_payload['model_state'])
for p in RETENTION_TEACHER.parameters(): p.requires_grad_(False)
retention_loaders={'Al':al_retention_validation_loader,'Fe':fe_retention_validation_loader}
retention_baselines={e:evaluate_loader(RETENTION_TEACHER,l,ni_normalization,e+'_validation_baseline')[1]
                     for e,l in retention_loaders.items()}

V29_RETENTION_ROWS=[]
V29_SELECTION_PHASE='adaptation'
V29_REPAIR_NI_BUDGET=float('inf')
V29_VALIDATION_CALL=0

def physical_retention_degradation(current,baseline):
    return sum(r['excess'] for r in v29_retention_components(current,baseline,'unknown'))

def ni_retention_validation_score(model,ni_metrics,scaler,global_step,total_steps):
    global V29_VALIDATION_CALL
    V29_VALIDATION_CALL+=1
    ni=evaluate_loader(model,ni_validation_loader,ni_normalization,'Ni_validation')[1]
    components=[]
    for element,loader in retention_loaders.items():
        current=evaluate_loader(model,loader,ni_normalization,element+'_validation')[1]
        components.extend(v29_retention_components(current,retention_baselines[element],element))
    excess=sum(r['excess'] for r in components)
    weak=ni['cohesive_weak_normalized_mae']
    if not math.isfinite(weak): weak=ni['cohesive_normalized_mae']
    score=float(ni['cohesive_mae_ev_per_atom'])
    extras={'validation_unbounded_ni_score':score,'validation_retention_excess':excess,
        'validation_constraints_passed':excess<=1e-12}
    for r in components:
        extras[f"retention_{r['element']}_{r['metric']}_excess"]=r['excess']
        V29_RETENTION_ROWS.append(dict(call=V29_VALIDATION_CALL,stage=globals().get('ACTIVE_NI_STAGE','before_Ni'),
            global_step=global_step,selection_phase=V29_SELECTION_PHASE,**r))
    _atomic_write_dataframe_csv(pd.DataFrame(V29_RETENTION_ROWS),CFG.run_dir/'retention_metrics_by_component.csv')
    for key,value in ni.items():
        if isinstance(value,(float,int)): extras['ni_physical_'+key]=value
    if V29_SELECTION_PHASE=='repair':
        ni_violation=max(0.,score-V29_REPAIR_NI_BUDGET)
        objective=excess + .05*score if ni_violation<=0 else CFG.constraint_barrier+ni_violation+excess
        extras['validation_repair_ni_budget']=V29_REPAIR_NI_BUDGET
    else:
        # Smooth exploration; strict feasibility is checked at final selection.
        objective=score+CFG.ni_exploration_retention_weight*excess
    return float(objective),extras

total_ni_epochs=max(3,CFG.preset.ni_epochs)
if CFG.run_mode in ('SMOKE','QUICK'):
    calibration_epochs,adapter_epochs,joint_epochs=1,1,max(1,total_ni_epochs-2)
else:
    calibration_epochs=min(CFG.ni_cohesive_calibration_epochs,max(1,total_ni_epochs//4))
    adapter_epochs=min(CFG.ni_adapter_warmup_epochs,max(1,total_ni_epochs-calibration_epochs-1))
    joint_epochs=max(1,total_ni_epochs-calibration_epochs-adapter_epochs)
ni_schedule=[('G_cohesive_calibration',calibration_epochs),
             ('G_adapter_warmup',adapter_epochs),
             ('G_joint_finetune',joint_epochs)]
ni_stage_loaders={'G_cohesive_calibration':ni_calibration_loader,
                  'G_adapter_warmup':joint_finetune_loader,
                  'G_joint_finetune':joint_finetune_loader}
NI_LR_PLANNED_UPDATES=sum(max(1,int(round(sum(r['optimizer_updates'] for r in
    data_curriculum_plan(stage,epochs,ni_stage_loaders[stage]))*CFG.optimizer_update_budget_scale)))
    for stage,epochs in ni_schedule)
configure_lr_macro_phase('Ni',NI_LR_PLANNED_UPDATES)
atomic_json_dump({'schedule':ni_schedule,'ni_lr_planned_updates':NI_LR_PLANNED_UPDATES},
                 CFG.run_dir/'ni_optimizer_plan.json')
# Global AlFe EMA state remains untouched throughout Ni.
configure_ema_phase('Ni',0,reset=False)
RETENTION_STREAM_LOADER=(make_loader(pretrain_train_frame,ni_normalization,True,balance_elements=True)
                         if CFG.retention_stream_enabled else None)
with isolated_rng():
    _,ni_initial_extras=ni_retention_validation_score(model,{},None,0,1)
ni_candidates=[{'stage':'before_Ni','checkpoint':str(teacher_path),**ni_initial_extras}]
for ACTIVE_NI_STAGE,epochs in ni_schedule:
    history=train_stage(model,ACTIVE_NI_STAGE,ni_stage_loaders[ACTIVE_NI_STAGE],ni_validation_loader,epochs,
        CFG.ni_learning_rate,ni_normalization,validation_score_callback=ni_retention_validation_score)
    display(history.tail())
    archive=CFG.run_dir/'checkpoints'/ACTIVE_NI_STAGE/'pareto_archive.json'
    if archive.exists():
        ni_candidates.extend([{**c,**v29_candidate_field_audit(c['checkpoint'])} for c in json.loads(archive.read_text())])
    with isolated_rng(): _,extras=ni_retention_validation_score(model,{},None,0,1)
    ni_candidates.append({'stage':ACTIVE_NI_STAGE,'checkpoint':history.attrs['best_checkpoint'],**extras, **v29_candidate_field_audit(history.attrs['best_checkpoint'])})
    V29_STAGE_VALIDATION_FRAMES.extend(v29_evaluate_validation_stage(model,ACTIVE_NI_STAGE))

# Repair a genuinely adapted model, not the accidentally restored pre-Ni baseline.
adapted=[c for c in ni_candidates if c['stage']!='before_Ni']
repair_start=min(adapted,key=lambda c:c['validation_unbounded_ni_score']
    +CFG.ni_exploration_retention_weight*c['validation_retention_excess'])
v29_load_model_candidate(repair_start)
V29_SELECTION_PHASE='repair'
V29_REPAIR_NI_BUDGET=(repair_start['validation_unbounded_ni_score']*(1+CFG.ni_repair_budget_relative)
    +CFG.ni_repair_budget_absolute)
if repair_start['validation_retention_excess']>1e-12:
    RETENTION_STREAM_LOADER=make_loader(pretrain_train_frame,ni_normalization,True,balance_elements=True)
    atomic_json_dump(dict(enabled=True,reason='conditional_retention_repair',source_ids=pretrain_train_frame.calc_id.tolist(),
        separate_from_ni_loader=True),CFG.run_dir/'retention_stream_repair_manifest.json')
    ACTIVE_NI_STAGE='G_retention_repair'
    repair_epochs=1 if CFG.run_mode in ('SMOKE','QUICK') else CFG.ni_repair_epochs
    repair_history=train_stage(model,ACTIVE_NI_STAGE,joint_finetune_loader,ni_validation_loader,repair_epochs,
        CFG.ni_repair_learning_rate,ni_normalization,validation_score_callback=ni_retention_validation_score)
    archive=CFG.run_dir/'checkpoints'/ACTIVE_NI_STAGE/'pareto_archive.json'
    if archive.exists():
        ni_candidates.extend([{**c,**v29_candidate_field_audit(c['checkpoint'])} for c in json.loads(archive.read_text())])
    with isolated_rng(): _,extras=ni_retention_validation_score(model,{},None,0,1)
    ni_candidates.append({'stage':ACTIVE_NI_STAGE,'checkpoint':repair_history.attrs['best_checkpoint'],**extras, **v29_candidate_field_audit(repair_history.attrs['best_checkpoint'])})
    V29_STAGE_VALIDATION_FRAMES.extend(v29_evaluate_validation_stage(model,ACTIVE_NI_STAGE))

# Never use holdout results for selection, and never disguise a fallback as fine-tuning.
feasible=[c for c in ni_candidates if c['validation_retention_excess']<=1e-12
    and c.get('validation_field_constraints_passed',True)
    and (c['stage']=='before_Ni' or (c.get('validation_energy_gates_passed',False) and c.get('candidate_trained',False)))]
if not feasible: raise RuntimeError('No feasible candidate, including baseline: inspect retention audit')
NI_SELECTED_CANDIDATE=min(feasible,key=lambda c:c['validation_unbounded_ni_score'])
v29_load_model_candidate(NI_SELECTED_CANDIDATE)
NI_SELECTION_AUDIT={'candidates':ni_candidates,'selected':NI_SELECTED_CANDIDATE,
    'selection_pool':'Ni validation + Al/Fe validation ONLY; no holdout selection',
    'strict_final_retention':True,'repair_start':repair_start,'baseline_is_eligible':True,
    'adaptation_accepted':NI_SELECTED_CANDIDATE['stage']!='before_Ni'}
atomic_json_dump(NI_SELECTION_AUDIT,CFG.run_dir/'ni_candidate_selection.json')
pd.DataFrame(ni_candidates).to_csv(CFG.run_dir/'checkpoint_selection_metrics.csv',index=False)
atomic_torch_save({'model_state':model.state_dict(),'training_contract':training_contract(),
    'architecture_version':ARCHITECTURE_VERSION,
    'training_protocol_version':TRAINING_PROTOCOL_VERSION,
    'curriculum_version':CURRICULUM_VERSION,
    'checkpoint_schema_version':CHECKPOINT_SCHEMA_VERSION,
    'selection':NI_SELECTED_CANDIDATE,'checkpoint_role':'inference_selection_only',
    'units':UNIT_CONTRACT,'normalization':asdict(ni_normalization),
    'normalization_hash':stable_contract_hash(asdict(ni_normalization)),
    'ni_manifest_hash':NI_SELECTION_MANIFEST['hash'],'split_manifest_hash':stable_contract_hash(split_manifest),
    'ema_phase_state':dict(EMA_PHASE_STATE),'ema_frozen':True},
    CFG.run_dir/'checkpoints'/'ni_global_best.pt')
FINAL_NI_LATENT_AUDIT=audit_stage_quality(model,ni_validation_loader,'D_latent',enforce=False,suffix='_Ni_post_G')
FINAL_ALFE_LATENT_AUDIT=audit_stage_quality(model,pretrain_validation_loader,'D_latent',enforce=False,suffix='_AlFe_post_G')
V29_STAGE_VALIDATION_FRAMES.extend(v29_evaluate_validation_stage(model,'selected_final'))
validation_goals=[]
for row in FINAL_NI_LATENT_AUDIT['rows']:
    if row.get('field')=='potential':
        validation_goals += [dict(target='potential',criterion='latent_cosine',value=row['cosine_similarity'],
            threshold=.95,passed=row['cosine_similarity']>=.95),
            dict(target='potential',criterion='high_band_latent_cosine',value=row['cosine_by_band']['high'],
            threshold=.90,passed=row['cosine_by_band']['high']>=.90)]
ni_selected_summary=evaluate_loader(model,ni_validation_loader,ni_normalization,'Ni_validation_selected')[1]
for target,key,limit in [('rho','rho_relative_l2_mean',CFG.final_rho_l2_goal),
    ('potential','potential_relative_l2_mean',CFG.final_potential_l2_goal),
    ('cohesive_energy','cohesive_bias_ev_per_atom',CFG.final_energy_bias_goal_ev)]:
    value=float(ni_selected_summary[key])
    validation_goals.append(dict(target=target,criterion=key,value=value,threshold=limit,passed=abs(value)<=limit))
pd.DataFrame(validation_goals).to_csv(CFG.run_dir/'Ni_validation_quality_goals.csv',index=False)
if not all(r['passed'] for r in validation_goals):
    warnings.warn('Selected model misses one or more Ni quality goals; retain diagnostics and do not claim successful transfer.')
print('Selected:',NI_SELECTED_CANDIDATE['stage'],'; adaptation accepted:',NI_SELECTION_AUDIT['adaptation_accepted'])


In [ ]:
record_evaluation(model,al_holdout_loader,'Al_ID_after_Ni')
record_evaluation(model,fe_holdout_loader,'Fe_ID_after_Ni')
for element,loader in ood_loaders.items(): record_evaluation(model,loader,f'{element}_OOD_after_Ni')
record_evaluation(model,ni_id_holdout_loader,'Ni_ID_selected')
record_evaluation(model,ni_ood_holdout_loader,'Ni_OOD_selected')
all_metrics=pd.concat(evaluation_frames,ignore_index=True)
# Apply train-only Ni thresholds retrospectively to saved zero-shot predictions.
for element,threshold in COHESIVE_WEAK_THRESHOLDS.items():
    mask=all_metrics.element.eq(element)
    normalized=(all_metrics.loc[mask,'cohesive_target_ev_per_atom']-ni_normalization.cohesive_center_ev_per_atom)/ni_normalization.cohesive_scale_ev_per_atom
    all_metrics.loc[mask,'cohesive_weak_binding']=normalized<=threshold
    all_metrics.loc[mask,'cohesive_extreme_weak_binding']=normalized<=COHESIVE_EXTREME_WEAK_THRESHOLDS[element]
summary_metrics=pd.DataFrame([{'category':c,**summarize_configuration_metrics(g)} for c,g in all_metrics.groupby('category')])
family_metrics=pd.DataFrame([{'category':c,'element':e,'family':f,**summarize_configuration_metrics(g)}
                            for (c,e,f),g in all_metrics.groupby(['category','element','family'])])
all_metrics.to_csv(CFG.run_dir/'per_configuration_metrics.csv',index=False)
summary_metrics.to_csv(CFG.run_dir/'summary_metrics.csv',index=False)
family_metrics.to_csv(CFG.run_dir/'metrics_by_family.csv',index=False)
display(summary_metrics)


## Bootstrap and paired comparisons
Resample whole configurations, never voxels. For before/after comparisons, IDs are
paired and the same bootstrap indices are used for both models. The 95% intervals
describe sampling of configurations in this split, not uncertainty across training seeds.

In [ ]:
def bootstrap_mean_interval(values,repetitions=None,seed=None):
    values=np.asarray(values,dtype=float)
    if values.size==0: return (float('nan'),)*3
    if not np.isfinite(values).all(): raise ValueError('Bootstrap received nonfinite values')
    repetitions=CFG.bootstrap_repetitions if repetitions is None else int(repetitions)
    if repetitions<1: raise ValueError('Bootstrap requires at least one repetition')
    rng=np.random.default_rng(CFG.seed+811 if seed is None else seed)
    draws=np.empty(repetitions)
    for start in range(0,repetitions,64):
        stop=min(start+64,repetitions)
        indices=rng.integers(0,len(values),size=(stop-start,len(values)))
        draws[start:stop]=values[indices].mean(axis=1)
    low,high=np.quantile(draws,[.025,.975])
    return float(values.mean()),float(low),float(high)

bootstrap_columns=('rho_relative_l2','potential_relative_l2','potential_rmse_ry',
                   'cohesive_abs_error_ev_per_atom','cohesive_abs_error_normalized',
                   'cohesive_signed_error_ev_per_atom')
bootstrap_rows=[]
for (category,element),group in all_metrics.groupby(['category','element']):
    for metric in bootstrap_columns:
        mean,low,high=bootstrap_mean_interval(group[metric])
        bootstrap_rows.append(dict(category=category,element=element,metric=metric,
                                   mean=mean,ci95_low=low,ci95_high=high,n=len(group)))
bootstrap_intervals=pd.DataFrame(bootstrap_rows)
bootstrap_intervals.to_csv(CFG.run_dir/'bootstrap_intervals.csv',index=False)
comparison_pairs=[('Al_ID_before_Ni','Al_ID_after_Ni'),('Fe_ID_before_Ni','Fe_ID_after_Ni'),
                  ('Ni_ID_zero_shot','Ni_ID_selected'),('Ni_OOD_zero_shot','Ni_OOD_selected')]
comparison_pairs += [(f'{e}_OOD_before_Ni',f'{e}_OOD_after_Ni') for e in ood_loaders]
paired_rows=[]
for before,after in comparison_pairs:
    left=all_metrics.loc[all_metrics.category.eq(before)].set_index('calc_id')
    right=all_metrics.loc[all_metrics.category.eq(after)].set_index('calc_id')
    if set(left.index)!=set(right.index): raise AssertionError(f'Before/after IDs do not match: {before}')
    right=right.loc[left.index]
    for metric in bootstrap_columns:
        delta=right[metric].to_numpy()-left[metric].to_numpy()
        mean,low,high=bootstrap_mean_interval(delta)
        paired_rows.append(dict(before=before,after=after,metric=metric,
                                mean_delta_after_minus_before=mean,ci95_low=low,ci95_high=high,n=len(delta)))
paired_transfer=pd.DataFrame(paired_rows)
paired_transfer.to_csv(CFG.run_dir/'paired_transfer_and_retention.csv',index=False)
display(paired_transfer)


## Supplementary classification: micro and macro by configuration
Fields remain regression outputs: high density and potential sign are
voxel segmentation diagnostics, not physical material classes. We report
micro counts and the mean of per-configuration metrics (macro), without treating voxels
as independent observations in the bootstrap. Weak cohesion is classification by structure;
for it we also report macro F1 across the two classes. Zero division yields zero.

In [ ]:
def fit_density_classification_thresholds(frame, normalization):
    thresholds, fitted_ids = {}, []
    for offset, (element, group) in enumerate(frame.groupby("element", sort=True)):
        sample = deterministic_sample(
            group, min(CFG.classification_threshold_records_per_element, len(group)),
            CFG.seed + 710 + offset)
        values = []
        for _, row in sample.iterrows():
            record = cached_raw_record(row.to_dict())
            values.append(np.asarray(record["rho_scaled"], dtype=np.float32).reshape(-1))
            fitted_ids.append(str(row.calc_id))
        thresholds[str(element)] = float(np.quantile(
            np.concatenate(values), CFG.density_classification_quantile))
    return thresholds, fitted_ids

def binary_confusion_counts(target_positive, prediction_positive):
    target_positive = target_positive.bool().reshape(-1)
    prediction_positive = prediction_positive.bool().reshape(-1)
    return np.asarray([
        int((~target_positive & ~prediction_positive).sum()),
        int((~target_positive & prediction_positive).sum()),
        int((target_positive & ~prediction_positive).sum()),
        int((target_positive & prediction_positive).sum()),
    ], dtype=np.int64)  # TN, FP, FN, TP

def metrics_from_binary_counts(counts):
    tn, fp, fn, tp = map(float, counts)
    # Same zero-division convention used by common classification libraries.
    safe = lambda numerator, denominator: numerator / denominator if denominator else 0.0
    recall = safe(tp, tp+fn)
    specificity = safe(tn, tn+fp)
    precision = safe(tp, tp+fp)
    return {
        "precision": precision,
        "recall": recall,
        "f1": safe(2*tp, 2*tp+fp+fn),
        "specificity": specificity,
        "balanced_accuracy": 0.5*(recall+specificity),
        "accuracy": safe(tp+tn, tp+tn+fp+fn),
        "support_positive": int(tp+fn),
        "support_negative": int(tn+fp),
    }
@torch.no_grad()
def classify_fields(model,loader,category):
    model.eval(); rows=[]
    for data in loader:
        data=move_batch(data); out=model.forward_operator(operator_batch(data))
        for i,element in enumerate(map(str,data.element)):
            threshold=DENSITY_CLASSIFICATION_THRESHOLDS[element]
            tasks={'density_high':(data.rho_target[i]>threshold,out['rho_scaled'][i]>threshold),
                   'potential_positive':(gauge_center(data.potential_target[i:i+1])>0,
                                         gauge_center(out['potential_scaled'][i:i+1])>0)}
            for task,(truth,prediction) in tasks.items():
                counts=binary_confusion_counts(truth,prediction)
                rows.append({'category':category,'calc_id':str(data.calc_id[i]),'element':element,'task':task,
                             **dict(zip(('tn','fp','fn','tp'),map(int,counts))),**metrics_from_binary_counts(counts)})
    return rows

classification_sets={'Al_ID_after_Ni':al_holdout_loader,'Fe_ID_after_Ni':fe_holdout_loader,
                     'Ni_ID_selected':ni_id_holdout_loader,'Ni_OOD_selected':ni_ood_holdout_loader,
                     **{f'{e}_OOD_after_Ni':l for e,l in ood_loaders.items()}}
field_classification_rows=[]
for category,loader in classification_sets.items(): field_classification_rows+=classify_fields(model,loader,category)
field_classification=pd.DataFrame(field_classification_rows)
field_classification.to_csv(CFG.run_dir/'classification_per_configuration.csv',index=False)
classification_rows=[]
for (category,task),group in field_classification.groupby(['category','task']):
    counts=group[['tn','fp','fn','tp']].sum().to_numpy()
    micro=metrics_from_binary_counts(counts)
    row={'category':category,'task':task,'configurations':len(group),
         **dict(zip(('tn','fp','fn','tp'),map(int,counts))),**micro}
    for metric in ('f1','recall','precision','balanced_accuracy'):
        mean,low,high=bootstrap_mean_interval(group[metric])
        row[f'macro_configuration_{metric}']=mean
        row[f'macro_configuration_{metric}_ci95_low']=low
        row[f'macro_configuration_{metric}_ci95_high']=high
    classification_rows.append(row)
for category,group in all_metrics.groupby('category'):
    thresholds=np.asarray([pretrain_normalization.decode_cohesive_energy(COHESIVE_WEAK_THRESHOLDS[e]) for e in group.element])
    truth=group.cohesive_target_ev_per_atom.to_numpy()<=thresholds
    predicted=group.cohesive_pred_ev_per_atom.to_numpy()<=thresholds
    counts=np.asarray([np.sum(~truth & ~predicted),np.sum(~truth & predicted),np.sum(truth & ~predicted),np.sum(truth & predicted)])
    metrics=metrics_from_binary_counts(counts)
    tn,fp,fn,tp=counts
    negative_f1=2*tn/max(2*tn+fn+fp,1)
    classification_rows.append({'category':category,'task':'weak_cohesion','configurations':len(group),
        **dict(zip(('tn','fp','fn','tp'),map(int,counts))),**metrics,
        'macro_class_f1':.5*(metrics['f1']+negative_f1)})
classification_metrics=pd.DataFrame(classification_rows)
classification_metrics.to_csv(CFG.run_dir/'classification_metrics.csv',index=False)
classification_metrics[['category','task','tn','fp','fn','tp']].to_csv(CFG.run_dir/'confusion_matrix_counts.csv',index=False)
display(classification_metrics)


## V29 — Complete metrics, confusion matrices and tail diagnostics
No target in this section is used for model selection.

In [ ]:
# Regressions remain primary. Classification is threshold-based diagnostics only.
def v29_export_all_metrics():
    fields=pd.DataFrame(V29_DETAIL_ROWS).drop_duplicates(['category','calc_id','target'],keep='last')
    spectra=pd.DataFrame(V29_SPECTRAL_ROWS).drop_duplicates(['category','calc_id','target','band'],keep='last')
    validation_frames=globals().get('V29_STAGE_VALIDATION_FRAMES',[])
    energy=pd.concat([all_metrics,*validation_frames],ignore_index=True).drop_duplicates(['category','calc_id'],keep='last')
    # Full energy predictions, no global field means masquerading as field predictions.
    energy.to_csv(CFG.run_dir/'per_configuration_predictions.csv',index=False)
    fields.to_csv(CFG.run_dir/'field_metrics_per_configuration.csv',index=False)
    for target in ('rho','potential'):
        spectra.loc[spectra.target.eq(target)].to_csv(CFG.run_dir/f'{target}_spectral_metrics.csv',index=False)
    spectra.loc[spectra.target.eq('rho')].to_csv(CFG.run_dir/'density_spectral_metrics.csv',index=False)
    regression=[]; tails=[]; cls=list(V29_CLASS_ROWS); ci=[]
    metric_keys=['mae','rmse','median_absolute_error','bias','error_std','relative_l2','r2',
        'pearson','spearman','slope','intercept','error_p90','error_p95','error_p99','rms_ratio','cosine']
    for (category,element,target),group in fields.groupby(['category','element','target']):
        regression.append(dict(category=category,element=element,target=target,configurations=len(group),
            aggregation='macro_mean_of_per_configuration_voxel_metrics',unit=group.unit.iloc[0],
            **{k:float(group[k].mean()) for k in metric_keys}))
        for metric in metric_keys:
            values=group[metric].to_numpy(); valid=values[np.isfinite(values)]
            if len(valid):
                mean,lo,hi=bootstrap_mean_interval(valid)
                ci.append(dict(category=category,element=element,target=target,metric=metric,
                    estimate=mean,ci95_low=lo,ci95_high=hi,n_valid=len(valid),n_undefined=len(values)-len(valid),
                    method='configuration_bootstrap_macro_mean'))
    for (category,element),group in energy.groupby(['category','element']):
        t=group.cohesive_target_ev_per_atom.to_numpy(); p=group.cohesive_pred_ev_per_atom.to_numpy()
        meta=dict(category=category,element=element,target='cohesive_energy',unit='eV/atom')
        regression.append(dict(**meta,configurations=len(t),aggregation='configuration_regression',**v29_regression_metrics(t,p)))
        # Descriptive deciles use test targets only for reporting, NEVER training or model selection.
        edges=np.quantile(t,np.linspace(0,1,11)); deciles=np.searchsorted(edges[1:-1],t,side='right')
        masks=[(f'descriptive_decile_{i+1}',deciles==i,np.nan,'descriptive_only') for i in range(10)]
        masks += [(f'upper_{int(round((1-q)*100))}pct_train_threshold',t>=V29_UPPER_ENERGY_THRESHOLDS[element][str(q)],
                   V29_UPPER_ENERGY_THRESHOLDS[element][str(q)],'train_only') for q in (.90,.95,.99)]
        weak=pretrain_normalization.decode_cohesive_energy(COHESIVE_WEAK_THRESHOLDS[element])
        masks += [('weak_cohesion_train_threshold',t<=weak,weak,'train_only')]
        for name,mask,threshold,scope in masks:
            row=dict(**meta,region=name,threshold=threshold,threshold_scope=scope,records=int(mask.sum()))
            if mask.any(): row.update(v29_regression_metrics(t[mask],p[mask]))
            tails.append(row)
        for task,threshold,is_upper in [('weak_cohesion',weak,False),
            ('energy_high_tail',V29_UPPER_ENERGY_THRESHOLDS[element]['0.9'],True)]:
            for i,cid in enumerate(group.calc_id):
                cls.append(v29_binary_row(t[i]>=threshold if is_upper else t[i]<=threshold,
                    p[i]>=threshold if is_upper else p[i]<=threshold,
                    category=category,element=element,calc_id=str(cid),task=task))
        # Bootstrap whole energy configurations, including slope/R2 rather than voxel resampling.
        rng=np.random.default_rng(CFG.seed+1701)
        energy_keys=['mae','rmse','bias','r2','slope']
        draws={k:[] for k in energy_keys}
        for _ in range(CFG.bootstrap_repetitions):
            ids=rng.integers(len(t),size=len(t)); tt,pp=t[ids],p[ids]; error=pp-tt
            tc=tt-tt.mean(); var=np.dot(tc,tc)
            values=dict(mae=np.abs(error).mean(),rmse=np.sqrt(np.mean(error**2)),bias=error.mean(),
                r2=1-np.dot(error,error)/var if var>1e-20 else np.nan,
                slope=np.dot(tc,pp-pp.mean())/var if var>1e-20 else np.nan)
            for k in energy_keys: draws[k].append(values[k])
        estimates=v29_regression_metrics(t,p)
        for metric,values in draws.items():
            valid=np.asarray(values); valid=valid[np.isfinite(valid)]
            ci.append(dict(**meta,metric=metric,estimate=estimates[metric],
                ci95_low=float(np.quantile(valid,.025)) if len(valid) else np.nan,
                ci95_high=float(np.quantile(valid,.975)) if len(valid) else np.nan,
                n_valid=len(t),n_undefined=0,method='configuration_bootstrap_regression'))
    regression=pd.DataFrame(regression)
    regression.to_csv(CFG.run_dir/'metrics_regression_summary.csv',index=False)
    regression.to_csv(CFG.run_dir/'metrics_by_target_element_domain.csv',index=False)
    pd.DataFrame(tails).to_csv(CFG.run_dir/'energy_tail_metrics.csv',index=False)
    pd.DataFrame(ci).to_csv(CFG.run_dir/'bootstrap_confidence_intervals.csv',index=False)
    per=pd.DataFrame(cls).drop_duplicates(['category','calc_id','task'],keep='last')
    per.to_csv(CFG.run_dir/'all_targets_classification_per_configuration.csv',index=False)
    summaries=[]; matrices=[]
    for (category,element,task),group in per.groupby(['category','element','task']):
        tn,fp,fn,tp=map(int,group[['tn','fp','fn','tp']].sum())
        meta=dict(category=category,element=element,task=task)
        summaries.append(dict(**meta,configurations=len(group),tn=tn,fp=fp,fn=fn,tp=tp,
            **v29_binary_metrics(tn,fp,fn,tp)))
        cm=np.array([[tn,fp],[fn,tp]],dtype=float)
        for i in range(2):
            for j in range(2):
                matrices.append(dict(**meta,true_class=i,predicted_class=j,count=int(cm[i,j]),
                    normalized_true=cm[i,j]/cm[i].sum() if cm[i].sum() else np.nan,
                    normalized_predicted=cm[i,j]/cm[:,j].sum() if cm[:,j].sum() else np.nan))
    comprehensive=pd.DataFrame(summaries)
    # Keep legacy macro/bootstrap columns; enrich same canonical CSV with every target/stage.
    legacy=classification_metrics.copy()
    missing=[c for c in legacy if c not in comprehensive and c not in ('category','task')]
    comprehensive=comprehensive.merge(legacy[['category','task',*missing]],on=['category','task'],how='left')
    comprehensive.to_csv(CFG.run_dir/'classification_metrics.csv',index=False)
    matrix=pd.DataFrame(matrices)
    matrix.to_csv(CFG.run_dir/'confusion_matrices_normalized.csv',index=False)
    matrix.drop(columns=['normalized_true','normalized_predicted']).to_csv(CFG.run_dir/'confusion_matrices_counts.csv',index=False)
    # Final physical diagnostics are reported, not retrospectively used to select on test data.
    gates=[]
    for row in regression.to_dict('records'):
        if '_selected' not in row['category'] and '_after_Ni' not in row['category']: continue
        if row['target']=='cohesive_energy':
            passed=abs(row['bias'])<=CFG.final_energy_bias_goal_ev
        else:
            goal=CFG.final_rho_l2_goal if row['target']=='rho' else CFG.final_potential_l2_goal
            passed=row['relative_l2']<=goal and .8<=row['rms_ratio']<=1.2
            if row['target']=='potential': passed=passed and row['cosine']>=CFG.final_potential_cosine_goal
        gates.append(dict(category=row['category'],target=row['target'],passed=passed,
            scope='post-selection descriptive audit, not a holdout selection rule'))
    pd.DataFrame(gates).to_csv(CFG.run_dir/'final_physical_quality_audit.csv',index=False)
    # Explicitly label the selected checkpoint in every final per-configuration row.
    selection_map=[]
    for category in energy.category.unique():
        role=('selected_final' if '_selected' in category or '_after_Ni' in category else
              'validation_candidate' if '_validation_' in category else 'pretrain_baseline')
        selection_map.append(dict(category=category,role=role,
            selected_checkpoint_stage=NI_SELECTED_CANDIDATE['stage'] if role=='selected_final' else '',
            selected_checkpoint_path=NI_SELECTED_CANDIDATE['checkpoint'] if role=='selected_final' else ''))
    pd.DataFrame(selection_map).to_csv(CFG.run_dir/'evaluation_stage_manifest.csv',index=False)
    display(regression[['category','target','mae','relative_l2','r2','bias']])
    return regression,comprehensive

V29_REGRESSION_METRICS,V29_CLASSIFICATION_METRICS=v29_export_all_metrics()

# Export additional confusion figures for the energy upper tail (existing figures preserved).
rows=V29_CLASSIFICATION_METRICS.loc[V29_CLASSIFICATION_METRICS.task.eq('energy_high_tail') &
    V29_CLASSIFICATION_METRICS.category.str.contains('selected|after_Ni')]
if len(rows):
    fig,axes=plt.subplots(1,len(rows),figsize=(3.2*len(rows),3.3),squeeze=False)
    for ax,(_,r) in zip(axes.ravel(),rows.iterrows()):
        cm=np.asarray([[r.tn,r.fp],[r.fn,r.tp]],dtype=float)
        norm=cm/np.maximum(cm.sum(1,keepdims=True),1)
        ax.imshow(norm,vmin=0,vmax=1,cmap='Blues')
        for i in range(2):
            for j in range(2): ax.text(j,i,f'{norm[i,j]:.3f}\n(n={int(cm[i,j])})',ha='center',va='center',color='white' if norm[i,j]>.5 else 'black')
        ax.set(title=r.category,xlabel=f'Predicted\nF1={r.f1:.3f}; recall={r.recall:.3f}',ylabel='Reference',xticks=[0,1],yticks=[0,1])
    fig.tight_layout(); fig.savefig(CFG.run_dir/'energy_upper_tail_confusion.png',dpi=300,bbox_inches='tight')
    fig.savefig(CFG.run_dir/'energy_upper_tail_confusion.pdf',bbox_inches='tight'); plt.show()


## V29 — Publication-ready ID/OOD results
The principal baseline is the Al/Fe-pretrained model evaluated on Ni zero-shot. Error reductions are exported separately for density, potential and cohesive energy; an optional composite is explicitly defined and must not replace the target-specific results.

In [ ]:
def v29_publication_metrics(regression):
    """Export the exact quantities needed for the article's main results sentence.

    The principal transfer baseline is the Al/Fe-pretrained model evaluated on Ni
    before Ni adaptation (zero-shot). Improvements are error reductions, reported
    separately for ID and structural OOD data and separately for every target.
    """
    domain_categories={
        'Ni_ID':('Ni_ID_zero_shot','Ni_ID_selected'),
        'Ni_structural_OOD':('Ni_OOD_zero_shot','Ni_OOD_selected'),
    }
    target_metrics={
        'density':('rho','relative_l2','dimensionless'),
        'potential':('potential','relative_l2','dimensionless'),
        'cohesive_energy':('cohesive_energy','mae','eV/atom'),
    }
    comparison=[]; publication=[]; composite=[]
    for domain,(baseline_category,selected_category) in domain_categories.items():
        ratios=[]
        for display_target,(target,metric,unit) in target_metrics.items():
            def unique_value(category):
                rows=regression.loc[(regression.category==category)&(regression.target==target)]
                if len(rows)!=1:
                    raise RuntimeError(
                        f'Expected exactly one {target} row for {category}; found {len(rows)}. '
                        'Publication metrics require unambiguous aggregate categories.')
                value=float(rows.iloc[0][metric])
                if not np.isfinite(value):
                    raise RuntimeError(f'Non-finite {metric} for {category}/{target}.')
                return value
            baseline=unique_value(baseline_category); selected=unique_value(selected_category)
            reduction=100.0*(baseline-selected)/baseline if baseline>0 else np.nan
            ratios.append(selected/baseline if baseline>0 else np.nan)
            comparison.append(dict(
                domain=domain,target=display_target,metric=metric,unit=unit,
                principal_baseline='Al/Fe-pretrained model evaluated on Ni zero-shot',
                baseline_category=baseline_category,selected_category=selected_category,
                baseline_value=baseline,selected_value=selected,
                improvement_percent=reduction,
                interpretation='positive = error reduction; negative = degradation'))
            publication.append(dict(
                domain=domain,target=display_target,
                article_metric=('cohesive_energy_mae' if display_target=='cohesive_energy' else 'relative_l2_error'),
                value=(1000.0*selected if display_target=='cohesive_energy' else selected),
                unit=('meV/atom' if display_target=='cohesive_energy' else 'dimensionless'),
                source_category=selected_category))
        valid=np.asarray(ratios,dtype=float); valid=valid[np.isfinite(valid)]
        composite.append(dict(
            domain=domain,
            composite_improvement_percent=100.0*(1.0-valid.mean()) if len(valid) else np.nan,
            formula='100 * (1 - mean(selected_error / baseline_error))',
            components='density relative L2; potential relative L2; cohesive-energy MAE',
            weighting='equal weight across the three target-specific normalized error ratios'))
    comparison=pd.DataFrame(comparison); publication=pd.DataFrame(publication); composite=pd.DataFrame(composite)
    publication.to_csv(CFG.run_dir/'publication_metrics.csv',index=False)
    comparison.to_csv(CFG.run_dir/'baseline_improvement_metrics.csv',index=False)
    composite.to_csv(CFG.run_dir/'publication_composite_improvement.csv',index=False)

    sentences=[]
    for domain in domain_categories:
        values=publication.loc[publication.domain.eq(domain)].set_index('target').value
        gains=comparison.loc[comparison.domain.eq(domain)].set_index('target').improvement_percent
        sentences.append(
            f"Periodic Field JEPA achieves a density relative-L2 error of {values['density']:.6g}, "
            f"a potential relative-L2 error of {values['potential']:.6g}, and a cohesive-energy "
            f"MAE of {values['cohesive_energy']:.6g} meV atom^-1 on {domain}; relative to the "
            f"Al/Fe-pretrained zero-shot baseline, the corresponding error reductions are "
            f"{gains['density']:.2f}%, {gains['potential']:.2f}%, and {gains['cohesive_energy']:.2f}%."
        )
    (CFG.run_dir/'publication_results_sentences.txt').write_text('\n'.join(sentences),encoding='utf-8')
    display(publication); display(comparison); display(composite)
    return publication,comparison,composite

V29_PUBLICATION_METRICS,V29_BASELINE_IMPROVEMENTS,V29_COMPOSITE_IMPROVEMENT = \
    v29_publication_metrics(V29_REGRESSION_METRICS)


## Channel usage and symmetry audit
Interventions use only training/validation configurations, without optimizing on
holdouts. Numerical sensitivity does not establish physical causality, but detects a
constant head or one that ignores a channel. The report compares independent shuffling
of each channel and gradients; if the gate fails, do not consider sharing validated.

In [ ]:
def shared_latent_intervention_audit(model,loader):
    model.eval(); latents=[]
    with torch.no_grad():
        for data in loader:
            out=model.forward_operator(operator_batch(move_batch(data)))
            latents.append(out['z_pred'].detach())
            if sum(len(z) for z in latents)>=max(8,CFG.preset.batch_size*2): break
    z=torch.cat(latents)
    if len(z)<2: raise ValueError('Intervention requires at least two configurations')
    with torch.no_grad(): baseline=model.predict_cohesive_energy(z)
    rows=[]
    for channel,name in enumerate(FIELD_NAMES):
        altered=z.clone(); altered[:,channel]=z[:,channel].roll(1,0)
        with torch.no_grad(): prediction=model.predict_cohesive_energy(altered)
        delta=float((prediction-baseline).abs().mean())
        relative=delta/max(float(baseline.std(unbiased=False)),1e-4)
        differentiable=z.clone().requires_grad_(True)
        gradient=torch.autograd.grad(model.predict_cohesive_energy(differentiable).sum(),differentiable)[0]
        gradient_norm=float(gradient[:,channel,2:].abs().mean())
        passed=(delta>=CFG.shared_latent_intervention_min_delta
                and relative>=CFG.shared_latent_intervention_min_relative_change and gradient_norm>0)
        rows.append(dict(channel=name,mean_prediction_change_normalized=delta,
                         relative_change=relative,gradient_mean_abs=gradient_norm,passed=passed))
    report={'rows':rows,'passed':all(r['passed'] for r in rows),'source':'pretrain_validation',
            'scope':'functional sensitivity, not physical causality'}
    atomic_json_dump(report,CFG.run_dir/'shared_latent_intervention_audit.json')
    pd.DataFrame(rows).to_csv(CFG.run_dir/'shared_latent_intervention_audit.csv',index=False)
    return report

@torch.no_grad()
def translation_audit(model,loader):
    model.eval(); rows=[]
    for data in loader:
        data=move_batch(data); op=operator_batch(data)
        shifts=tuple(max(1,n//5) for n in CFG.canonical_grid)
        transformed=translate_periodic_batch(op,shifts)
        first=model.forward_operator(op); second=model.forward_operator(transformed)
        ec=(second['cohesive_energy_normalized']-first['cohesive_energy_normalized']).abs()
        re=relative_field_error(second['rho_scaled'],torch.roll(first['rho_scaled'],shifts,(-3,-2,-1)))
        ve=relative_field_error(second['potential_scaled'],torch.roll(first['potential_scaled'],shifts,(-3,-2,-1)))
        for i,calc_id in enumerate(data.calc_id):
            rows.append(dict(calc_id=calc_id,cohesive_translation_error_normalized=float(ec[i]),
                             rho_covariance_relative_l2=float(re[i]),potential_covariance_relative_l2=float(ve[i])))
        if len(rows)>=24: break
    frame=pd.DataFrame(rows)
    frame.to_csv(CFG.run_dir/'translation_covariance_audit.csv',index=False)
    return frame

INTERVENTION_AUDIT=shared_latent_intervention_audit(model,pretrain_validation_loader)
TRANSLATION_AUDIT=translation_audit(model,pretrain_validation_loader)
display(INTERVENTION_AUDIT); display(TRANSLATION_AUDIT)


## Inference from structure only
The output grid size is a user choice; it is not read from the target NetCDF.
The example uses the canonical grid but explicitly accepts `output_grid=(nz,ny,nx)`.

In [ ]:
def canonical_prediction_to_native(
    rho_scaled: np.ndarray,
    potential_scaled: np.ndarray,
    native_shape: tuple[int, int, int],
    electron_count: float,
    volume_bohr3: float,
    normalization: Normalization,
) -> tuple[np.ndarray, np.ndarray]:
    rho_native_scaled = periodic_fourier_resample(rho_scaled, native_shape)
    rho_native_scaled = np.maximum(rho_native_scaled, 0.0)
    rho_native_scaled /= max(float(rho_native_scaled.mean()), 1e-12)
    rho_native = rho_native_scaled * electron_count / volume_bohr3
    potential_native = periodic_fourier_resample(potential_scaled, native_shape)
    potential_native -= float(potential_native.mean())
    potential_native *= normalization.potential_scale_ry
    return rho_native.astype(np.float32), potential_native.astype(np.float32)
@torch.no_grad()
def infer_structure(model,row,normalization,output_grid=None):
    record=load_structural_record(row)
    data=graph_from_record(record,normalization,False)
    batch=Batch.from_data_list([data]).to(DEVICE)
    model.eval(); out=model.forward_operator(batch)
    shape=tuple(output_grid or CFG.canonical_grid)
    if len(shape)!=3 or min(shape)<2: raise ValueError('Invalid grid')
    rho,potential=canonical_prediction_to_native(out['rho_scaled'][0,0].cpu().numpy(),
        out['potential_scaled'][0,0].cpu().numpy(),shape,record['electron_count'],record['volume_bohr3'],normalization)
    cohesive=normalization.decode_cohesive_energy(float(out['cohesive_energy_normalized'][0]))
    return {'rho_e_per_bohr3':rho,'potential_ry':potential,'cohesive_ev_per_atom':cohesive,
            'output_grid':shape,'input_keys':sorted(data.keys())}

row=splits['ni_id_holdout_test'].iloc[0]
structural_input={k:row[k] for k in ('calc_id','element','struct_out_path','ion_xml_path')}
prediction=infer_structure(model,structural_input,ni_normalization)
print({'cohesive_ev_per_atom':prediction['cohesive_ev_per_atom'],'output_grid':prediction['output_grid'],
       'input_keys':prediction['input_keys']})

@torch.no_grad()
def benchmark_inference(model,row,normalization):
    t0=time.perf_counter(); record=load_structural_record(row)
    data=graph_from_record(record,normalization,False)
    graph_seconds=time.perf_counter()-t0
    batch=Batch.from_data_list([data]).to(DEVICE); model.eval()
    for _ in range(CFG.inference_benchmark_warmup): model.forward_operator(batch)
    if DEVICE.type=='cuda': torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    times=[]
    for _ in range(CFG.inference_benchmark_repetitions):
        t0=time.perf_counter(); model.forward_operator(batch)
        if DEVICE.type=='cuda': torch.cuda.synchronize()
        times.append(time.perf_counter()-t0)
    return {'graph_and_structural_io_seconds':graph_seconds,'forward_median_seconds':float(np.median(times)),
            'forward_configurations_per_second':float(1/np.mean(times)),
            'peak_allocated_bytes':torch.cuda.max_memory_allocated() if DEVICE.type=='cuda' else None,
            'edges':data.edge_index.shape[1],'device':str(DEVICE),'batch_size':1}
atomic_json_dump(benchmark_inference(model,structural_input,ni_normalization),CFG.run_dir/'inference_benchmark.json')


## Figures: physical cohesion, fields, and retention
All energy units are eV/atom; figures do not affect model selection.

In [ ]:
plt.rcParams.update({'figure.dpi':120,'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
final_categories=[c for c in all_metrics.category.unique() if 'after_Ni' in c or 'selected' in c]
figure,axes=plt.subplots(2,3,figsize=(14,8),squeeze=False)
for axis,category in zip(axes.flat,final_categories):
    group=all_metrics.loc[all_metrics.category.eq(category)]
    axis.scatter(group.cohesive_target_ev_per_atom,group.cohesive_pred_ev_per_atom,s=10,alpha=.5)
    lower=min(group.cohesive_target_ev_per_atom.min(),group.cohesive_pred_ev_per_atom.min())
    upper=max(group.cohesive_target_ev_per_atom.max(),group.cohesive_pred_ev_per_atom.max())
    axis.plot([lower,upper],[lower,upper],'k--',lw=1)
    axis.set(title=category,xlabel='DFT cohesive energy (eV/atom)',ylabel='Predicted cohesive energy (eV/atom)')
for axis in list(axes.flat)[len(final_categories):]: axis.set_visible(False)
figure.tight_layout(); figure.savefig(CFG.run_dir/'cohesive_scatter.png',dpi=200); plt.show()

figure,axes=plt.subplots(1,2,figsize=(12,4))
for element,group in all_metrics.loc[all_metrics.category.isin(final_categories)].groupby('element'):
    axes[0].scatter(group.cohesive_target_ev_per_atom,group.cohesive_signed_error_ev_per_atom,s=7,alpha=.35,label=element)
    ordered=group.sort_values('cohesive_target_ev_per_atom').copy()
    ordered['decile']=np.minimum(np.arange(len(ordered))*10//max(len(ordered),1),9)+1
    means=ordered.groupby('decile').cohesive_abs_error_ev_per_atom.mean()
    axes[1].plot(means.index,means.values,'o-',label=element)
axes[0].axhline(0,color='black',lw=.8)
axes[0].set(xlabel='DFT cohesive energy (eV/atom)',ylabel='Predicted − DFT (eV/atom)')
axes[1].set(xlabel='Cohesive energy decile (1 = weak binding)',ylabel='MAE (eV/atom)')
for axis in axes: axis.legend()
figure.tight_layout(); figure.savefig(CFG.run_dir/'cohesive_tail_errors.png',dpi=200); plt.show()

figure,axes=plt.subplots(1,3,figsize=(15,4))
for axis,metric in zip(axes,('rho_relative_l2_mean','potential_relative_l2_mean','cohesive_mae_ev_per_atom')):
    x=np.arange(len(summary_metrics)); axis.bar(x,summary_metrics[metric])
    axis.set_xticks(x,summary_metrics.category,rotation=75,ha='right'); axis.set_title(metric)
figure.tight_layout(); figure.savefig(CFG.run_dir/'performance_and_retention.png',dpi=200); plt.show()

histories=[pd.read_csv(p) for p in sorted(CFG.run_dir.glob('history_*.csv'))]
if histories:
    order=['autoencoder','D_latent','E_density','E_density_potential','E_full','F_periodic_ramp',
           'P_cohesive_refinement','G_cohesive_calibration','G_adapter_warmup','G_joint_finetune','G_retention_repair']
    training_history=pd.concat(histories,ignore_index=True)
    training_history['stage_order']=training_history.stage.map({s:i for i,s in enumerate(order)})
    training_history=training_history.sort_values(['stage_order','epoch']).reset_index(drop=True)
    training_history.to_csv(CFG.run_dir/'training_history_all_stages.csv',index=False)
    active_stages=[s for s in order if (training_history.stage==s).any()]
    figure,axes=plt.subplots(len(active_stages),3,figsize=(15,2.6*len(active_stages)),squeeze=False)
    for row,stage in enumerate(active_stages):
        group=training_history.loc[training_history.stage.eq(stage)]
        metrics=('jepa','latent_cosine','total') if stage=='D_latent' else ('rho','potential','cohesive')
        for axis,metric in zip(axes[row],metrics):
            plotted=False
            for prefix,label in [('train','Training'),('validation','Validation')]:
                key=f'{prefix}_{metric}'
                if key in group and group[key].notna().any():
                    axis.plot(group.epoch+1,group[key],label=label,linewidth=1.3); plotted=True
            if not plotted: axis.text(.5,.5,'Not computed',ha='center',va='center',transform=axis.transAxes)
            else: axis.legend(fontsize=7)
            axis.set(title=f'{stage} | {metric}',xlabel='Stage epoch',ylabel='Loss (own scale)')
    figure.suptitle('Raw curves by stage; training/validation may use different weights',fontsize=12)
    figure.tight_layout(rect=(0,0,1,.99)); figure.savefig(CFG.run_dir/'training_curves.png',dpi=170); plt.show()
    figure,axes=plt.subplots(len(active_stages),2,figsize=(13,2.4*len(active_stages)),squeeze=False)
    for row,stage in enumerate(active_stages):
        group=training_history.loc[training_history.stage.eq(stage)]
        axes[row,0].plot(group.epoch+1,group.validation_score,label='selection score')
        if 'validation_monitor_total' in group and group.validation_monitor_total.notna().any():
            axes[row,0].plot(group.epoch+1,group.validation_monitor_total,label='fixed monitor',alpha=.8)
        axes[row,0].set(title=stage+' | validation',xlabel='Epoch',ylabel='Score / monitor')
        axes[row,0].legend(fontsize=7)
        for key in group.columns:
            if key.startswith('train_lr_') and key.endswith('_mean') and group[key].notna().any():
                axes[row,1].plot(group.epoch+1,group[key],label=key[9:-5])
        axes[row,1].set(title=stage+' | mean applied LR',xlabel='Epoch',ylabel='Learning rate')
        axes[row,1].set_yscale('log'); axes[row,1].legend(fontsize=6)
    figure.tight_layout(); figure.savefig(CFG.run_dir/'training_selection_and_lr.png',dpi=170); plt.show()

# Density/potential maps use fixed color scales for target/prediction within each pair.
for element,loader in [('Al',al_holdout_loader),('Fe',fe_holdout_loader),('Ni',ni_id_holdout_loader)]:
    data=move_batch(next(iter(loader))); model.eval()
    with torch.no_grad(): out=model.forward_operator(operator_batch(data))
    figure,axes=plt.subplots(2,3,figsize=(11,6))
    for f,(truth,pred,title) in enumerate([(data.rho_target,out['rho_scaled'],'Normalized density'),
                                          (data.potential_target,out['potential_scaled'],'Potential (Ry)')]):
        scale=pretrain_normalization.potential_scale_ry if f else 1.
        a=truth[0,0,truth.shape[-3]//2].detach().cpu().numpy()*scale
        b=pred[0,0,pred.shape[-3]//2].detach().cpu().numpy()*scale
        vmin=min(a.min(),b.min()); vmax=max(a.max(),b.max())
        for col,(values,label) in enumerate([(a,'DFT'),(b,'Predicted'),(b-a,'Error')]):
            kwargs={'vmin':vmin,'vmax':vmax} if col<2 else {'vmin':-max(abs(b-a).max(),1e-8),'vmax':max(abs(b-a).max(),1e-8)}
            im=axes[f,col].imshow(values,origin='lower',cmap='viridis' if col<2 else 'coolwarm',**kwargs)
            axes[f,col].set_title(title+' — '+label); figure.colorbar(im,ax=axes[f,col],shrink=.7)
    figure.suptitle(element+' — slice on the fractional grid'); figure.tight_layout()
    figure.savefig(CFG.run_dir/f'field_maps_{element}.png',dpi=200); plt.show()

weak_class=classification_metrics.loc[classification_metrics.task.eq('weak_cohesion')]
figure,axes=plt.subplots(math.ceil(len(weak_class)/3),3,figsize=(12,3*math.ceil(len(weak_class)/3)),squeeze=False)
for axis,(_,row) in zip(axes.flat,weak_class.iterrows()):
    counts=np.asarray([[row.tn,row.fp],[row.fn,row.tp]],dtype=float)
    normalized=counts/np.maximum(counts.sum(1,keepdims=True),1)
    axis.imshow(normalized,vmin=0,vmax=1,cmap='Blues')
    for a in range(2):
        for b in range(2): axis.text(b,a,f'{int(counts[a,b])}',ha='center',va='center')
    axis.set(title=str(row.category),xlabel='Predicted',ylabel='DFT')
    axis.set_xticks([0,1],['normal','weak']); axis.set_yticks([0,1],['normal','weak'])
for axis in list(axes.flat)[len(weak_class):]: axis.set_visible(False)
figure.tight_layout(); figure.savefig(CFG.run_dir/'weak_cohesion_confusion.png',dpi=200); plt.show()


## Additional publication analyses — history by optimizer update and confusion matrices

This cell is exclusively post-training. It does not participate in forward, backward, checkpoint selection, or weight updates. The curriculum curve uses `optimizer_history_<stage>.csv` files recorded during training. The confusion matrices reuse the auxiliary classifications defined in V24 for high-density regions, potential sign, and weak cohesion.

In [ ]:
# =============================================================================
# Curriculum curve by optimizer update + auxiliary confusion matrices
# This cell does NOT change weights, architecture, checkpoints, or model selection.
# =============================================================================

CURRICULUM_STAGE_ORDER = [
    'D_latent',
    'E_density',
    'E_density_potential',
    'E_full',
    'F_periodic_ramp',
    'P_cohesive_refinement',
    'G_cohesive_calibration',
    'G_adapter_warmup',
    'G_joint_finetune',
]

CURRICULUM_STAGE_LABELS = {
    'D_latent': 'D latent JEPA',
    'E_density': 'E density',
    'E_density_potential': 'E density + potential',
    'E_full': 'E full',
    'F_periodic_ramp': 'F periodic ramp',
    'P_cohesive_refinement': 'P cohesive refinement',
    'G_cohesive_calibration': 'G cohesive calibration',
    'G_adapter_warmup': 'G adapter warmup',
    'G_joint_finetune': 'G joint finetune',
}


def build_curriculum_optimizer_history() -> tuple[pd.DataFrame, dict[str, int], dict[str, int]]:
    parts = []
    stage_offsets: dict[str, int] = {}
    stage_ends: dict[str, int] = {}
    offset = 0
    for stage in CURRICULUM_STAGE_ORDER:
        path = optimizer_update_history_path(stage)
        if not path.exists():
            continue
        group = pd.read_csv(path).sort_values('stage_optimizer_update').copy()
        if group.empty:
            continue
        stage_offsets[stage] = offset
        group['curriculum_update'] = offset + group['stage_optimizer_update'].astype(int)
        offset += int(group['stage_optimizer_update'].max())
        stage_ends[stage] = offset
        parts.append(group)
    if not parts:
        return pd.DataFrame(), stage_offsets, stage_ends
    frame = pd.concat(parts, ignore_index=True)
    frame.to_csv(CFG.run_dir/'curriculum_optimizer_history.csv', index=False)
    return frame, stage_offsets, stage_ends


curriculum_updates, stage_offsets, stage_ends = build_curriculum_optimizer_history()

if len(curriculum_updates):
    figure, axis = plt.subplots(figsize=(11, 7))
    for stage in CURRICULUM_STAGE_ORDER:
        group = curriculum_updates.loc[curriculum_updates.stage.eq(stage)].sort_values('curriculum_update').copy()
        if group.empty:
            continue
        x = group.curriculum_update.to_numpy(dtype=int)
        y = group.training_objective.to_numpy(dtype=float)
        valid = np.isfinite(y) & (y > 0)
        x, y = x[valid], y[valid]
        if not len(y):
            continue

        # Thin raw line; the thick line is only a visualization EMA.
        raw_line, = axis.plot(x, y, linewidth=.65, alpha=.18)
        span = max(3, min(75, len(y)))
        smooth = pd.Series(y).ewm(span=span, adjust=False).mean().to_numpy()
        axis.plot(
            x, smooth, linewidth=2.0, color=raw_line.get_color(),
            label=CURRICULUM_STAGE_LABELS.get(stage, stage),
        )
        marker_stride = max(1, len(x)//80)
        axis.scatter(x[::marker_stride], smooth[::marker_stride], s=12,
                     color=raw_line.get_color(), zorder=3)

    present_stages = [stage for stage in CURRICULUM_STAGE_ORDER if stage in stage_ends]
    for stage in present_stages[:-1]:
        axis.axvline(stage_ends[stage], linestyle=':', linewidth=.8, alpha=.45)

    # Mark the selected Ni checkpoint when information is available.
    candidate = globals().get('NI_SELECTED_CANDIDATE', None)
    if isinstance(candidate, Mapping) and candidate.get('checkpoint'):
        try:
            selected_path = Path(candidate['checkpoint'])
            if selected_path.is_file():
                selected_payload = torch.load(selected_path, map_location='cpu', weights_only=False)
                selected_stage = str(selected_payload.get('stage', ''))
                selected_step = int(selected_payload.get('global_step', 0))
                if selected_stage in stage_offsets and selected_step > 0:
                    selected_x = stage_offsets[selected_stage] + selected_step
                    axis.axvline(selected_x, linestyle='--', linewidth=1.4,
                                 label='Selected checkpoint')
                    axis.text(
                        selected_x, .98,
                        f' selected: {selected_stage}, update {selected_step}',
                        rotation=90, va='top', ha='right',
                        transform=axis.get_xaxis_transform(), fontsize=8,
                    )
        except Exception as exc:
            print('Selected checkpoint could not be marked:', exc)

    axis.set_yscale('log')
    axis.set_xlabel('Optimizer update')
    axis.set_ylabel('Training objective')
    axis.set_title('Stage-specific optimization objectives (not directly comparable across stages)')
    axis.grid(alpha=.2)
    axis.legend(fontsize=8, ncol=2)
    figure.tight_layout()
    figure.savefig(CFG.run_dir/'curriculum_training_objective.png', dpi=240, bbox_inches='tight')
    plt.show()
else:
    print('No optimizer_history_<stage>.csv found. Run training for this revision first.')


# Fixed-monitor and stage-normalized curriculum diagnostics.
if len(curriculum_updates):
    if 'loss_monitor_total' in curriculum_updates.columns:
        fig,ax=plt.subplots(figsize=(11,5.5))
        for macro_name,stages in [('Al/Fe',('D_latent','E_density','E_density_potential','E_full','F_periodic_ramp')),
                                  ('Ni',('G_cohesive_calibration','G_adapter_warmup','G_joint_finetune'))]:
            g=curriculum_updates.loc[curriculum_updates.stage.isin(stages)].sort_values('curriculum_update')
            if g.empty: continue
            y=g.loss_monitor_total.to_numpy(float); x=g.curriculum_update.to_numpy(int)
            valid=np.isfinite(y)&(y>0); x,y=x[valid],y[valid]
            if not len(y): continue
            smooth=pd.Series(y).ewm(span=max(5,min(75,len(y))),adjust=False).mean().to_numpy()
            ax.plot(x,smooth,linewidth=2,label=f'{macro_name} fixed monitor')
        ax.set(xlabel='Optimizer update',ylabel='Fixed monitor objective',
               title='Fixed monitor objective (same weights inside each macro-phase)')
        ax.set_yscale('log'); ax.grid(alpha=.2); ax.legend(); fig.tight_layout()
        fig.savefig(CFG.run_dir/'curriculum_fixed_monitor_objective.png',dpi=240,bbox_inches='tight'); plt.show()

    fig,ax=plt.subplots(figsize=(11,5.5))
    for stage in CURRICULUM_STAGE_ORDER:
        g=curriculum_updates.loc[curriculum_updates.stage.eq(stage)].sort_values('curriculum_update')
        if g.empty: continue
        y=g.training_objective.to_numpy(float); x=g.curriculum_update.to_numpy(int)
        valid=np.isfinite(y)&(y>0); x,y=x[valid],y[valid]
        if not len(y): continue
        baseline=float(pd.Series(y).ewm(span=max(3,min(25,len(y))),adjust=False).mean().iloc[0])
        normalized=y/max(baseline,1e-12)
        smooth=pd.Series(normalized).ewm(span=max(3,min(75,len(y))),adjust=False).mean().to_numpy()
        ax.plot(x,smooth,linewidth=1.8,label=CURRICULUM_STAGE_LABELS.get(stage,stage))
    ax.axhline(1.0,linestyle=':',linewidth=1.0)
    ax.set(xlabel='Optimizer update',ylabel='Stage-normalized objective',
           title='Stage-normalized training objective (each stage starts near 1)')
    ax.grid(alpha=.2); ax.legend(fontsize=8,ncol=2); fig.tight_layout()
    fig.savefig(CFG.run_dir/'curriculum_stage_normalized_objective.png',dpi=240,bbox_inches='tight'); plt.show()


# V31 optimizer diagnostics: adaptive task weights and stage readiness.
if len(curriculum_updates):
    balance_columns=[column for column in curriculum_updates.columns if column.startswith('balance_')]
    if balance_columns:
        fig,ax=plt.subplots(figsize=(11,5.5))
        for column in balance_columns:
            values=pd.to_numeric(curriculum_updates[column],errors='coerce')
            mask=values.notna()
            if mask.any():
                ax.plot(curriculum_updates.loc[mask,'curriculum_update'],values.loc[mask],
                        linewidth=1.4,label=column.replace('balance_',''))
        ax.axhline(1.0,linestyle=':',linewidth=1.0)
        ax.set(xlabel='Optimizer update',ylabel='Adaptive task factor',
               title='V31 conflict-aware gradient-balance factors')
        ax.grid(alpha=.2); ax.legend(fontsize=8,ncol=3); fig.tight_layout()
        fig.savefig(CFG.run_dir/'gradient_balance_factors.png',dpi=240,bbox_inches='tight'); plt.show()

readiness_rows=[]
for stage in CURRICULUM_STAGE_ORDER:
    path=CFG.run_dir/f'history_{stage}.csv'
    if not path.exists(): continue
    frame=pd.read_csv(path)
    if frame.empty: continue
    last=frame.iloc[-1]
    readiness_rows.append({
        'stage':stage,
        'stage_readiness_passed':bool(last.get('stage_readiness_passed',False)),
        'checkpoint_eligible':bool(last.get('checkpoint_eligible',False)),
        'checkpoint_reason':str(last.get('checkpoint_reason','')),
        'global_step':int(last.get('global_step',0)),
    })
if readiness_rows:
    pd.DataFrame(readiness_rows).to_csv(CFG.run_dir/'stage_readiness_summary.csv',index=False)

UNIT_AUDIT_SUMMARY=finalize_netcdf_unit_audit()
print('NetCDF unit audit:',json.dumps(UNIT_AUDIT_SUMMARY,indent=2))

# -----------------------------------------------------------------------------
# Auxiliary confusion matrices for discretized continuous outputs
# -----------------------------------------------------------------------------
def plot_binary_confusion_task(
    metrics_frame: pd.DataFrame,
    task: str,
    class_labels: tuple[str, str],
    filename: str,
    figure_title: str,
) -> None:
    subset = metrics_frame.loc[metrics_frame.task.eq(task)].copy()
    if subset.empty:
        print(f'No rows found for task={task}')
        return
    ncols = 3
    nrows = math.ceil(len(subset)/ncols)
    figure, axes = plt.subplots(nrows, ncols, figsize=(4.1*ncols, 3.6*nrows), squeeze=False)
    for axis, (_, row) in zip(axes.flat, subset.iterrows()):
        counts = np.asarray([[row.tn, row.fp], [row.fn, row.tp]], dtype=float)
        normalized = counts/np.maximum(counts.sum(axis=1, keepdims=True), 1.0)
        axis.imshow(normalized, vmin=0, vmax=1, cmap='Blues')
        for i in range(2):
            for j in range(2):
                axis.text(j, i, f'{100.0*normalized[i,j]:.1f}%\n(n={int(counts[i,j])})',
                          ha='center', va='center', fontsize=9)
        axis.set_title(str(row.category))
        axis.set_xlabel('Predicted')
        axis.set_ylabel('Reference')
        axis.set_xticks([0, 1], class_labels)
        axis.set_yticks([0, 1], class_labels)
    for axis in list(axes.flat)[len(subset):]:
        axis.set_visible(False)
    figure.suptitle(figure_title, fontsize=12)
    figure.tight_layout(rect=(0, 0, 1, .97))
    figure.savefig(CFG.run_dir/filename, dpi=220, bbox_inches='tight')
    plt.show()


plot_binary_confusion_task(
    classification_metrics,
    task='density_high',
    class_labels=('not high ρ', 'high ρ'),
    filename='density_high_confusion.png',
    figure_title='Electron-density high-region confusion matrices',
)

plot_binary_confusion_task(
    classification_metrics,
    task='potential_positive',
    class_labels=('V ≤ 0', 'V > 0'),
    filename='potential_sign_confusion.png',
    figure_title='Gauge-centered electrostatic-potential sign confusion matrices',
)

# Also keep the normalized version of the weak cohesion matrix.
plot_binary_confusion_task(
    classification_metrics,
    task='weak_cohesion',
    class_labels=('normal', 'weak'),
    filename='weak_cohesion_confusion_normalized.png',
    figure_title='Weak-cohesion confusion matrices',
)


## Final checks and scientific run card

In [ ]:
POST_TESTS={}
assert not all_metrics.duplicated(['category','calc_id']).any()
POST_TESTS['one_row_per_configuration']=True
for metric in bootstrap_columns:
    assert np.isfinite(all_metrics[metric]).all(),metric
assert (bootstrap_intervals.ci95_low<=bootstrap_intervals.ci95_high).all()
POST_TESTS['finite_metrics_and_bootstrap_including_normalized_cohesion']=True
assert np.max(np.abs(all_metrics.cohesive_abs_error_normalized*pretrain_normalization.cohesive_scale_ev_per_atom
                     -all_metrics.cohesive_abs_error_ev_per_atom))<1e-8
POST_TESTS['energy_units_consistent']=True
assert NI_FINETUNE_DATA_AUDIT['used_fraction']<=CFG.ni_finetune_fraction_cap
POST_TESTS['ni_ten_percent_cap']=True
assert all(p.grad is None and not p.requires_grad for p in RETENTION_TEACHER.parameters())
POST_TESTS['teacher_frozen']=True
assert all_metrics.charge_constraint_error_e.max()<1e-3
POST_TESTS['charge_constraint_only_not_accuracy']=True
POST_TESTS['shared_channel_intervention']=INTERVENTION_AUDIT['passed']
atomic_json_dump(POST_TESTS,CFG.run_dir/'post_training_tests.json')
FINAL_CARD={'architecture':architecture_manifest,'config':json_canonical(asdict(CFG)),
            'normalization':asdict(pretrain_normalization),'data_audit':DATA_AUDIT_MANIFEST,
            'pretraining_tests':CONTRACT_TESTS,'post_tests':POST_TESTS,
            'ni_training':NI_FINETUNE_DATA_AUDIT,'shared_intervention':INTERVENTION_AUDIT,
            'source_revision':SOURCE_REVISION,'pretrain_sampling':PRETRAIN_SAMPLING_AUDIT,
            'pretrain_selection':PRETRAIN_GLOBAL_BEST,'energy_refinement':ENERGY_REFINEMENT_AUDIT,
            'ni_selection':NI_SELECTION_AUDIT,
            'metrics':summary_metrics.to_dict('records'),
            'limitations':['experimentally anchored 298.15 K target, not isolated-atom DFT cohesion at 0 K',
                           'Ni zero-shot weights use a target zero calibrated from ni_adapt_train',
                           'truncated normalized spectral fields',
                           'single training seed; bootstrap is configuration uncertainty',
                           'cell-basis dependent graph; not invariant to supercell replication',
                           'classification thresholds are diagnostic, not material phase labels',
                           'field decoder covariance is empirical, unlike invariant scalar head']}
atomic_json_dump(FINAL_CARD,CFG.run_dir/'scientific_run_card.json')
if CFG.require_shared_latent_intervention and not INTERVENTION_AUDIT['passed']:
    raise RuntimeError('Head did not pass the two-channel usage gate; reports/checkpoints preserved. Do not claim validated sharing.')
print('V29 completed. See scientific_run_card.json, candidate selections, and ID/OOD metrics.')

FINAL_CARD['v29_validation_goals_are_not_guarantees']=True
FINAL_CARD['energy_input_contract']='shared latent + explicit known elemental anchor; no graph-context bypass'
atomic_json_dump(FINAL_CARD,CFG.run_dir/'scientific_run_card.json')


## Validation scope

All cells are analyzed by AST. tests/test_v29_protocol.py tests real functions with
synthetic inputs, without replacing scientific validation on the data. QUICK uses the same
contract and may block JEPA if physical gates are not reached within the short budget.
STANDARD, DEVELOPMENT, and FULL are not run automatically.


## Additional CSV outputs
- `metrics_regression_summary.csv`, `metrics_by_target_element_domain.csv`
- `per_configuration_predictions.csv` (energy truth/prediction and existing field errors)
- `field_metrics_per_configuration.csv` (rho scaled by mean density; potential in Ry)
- `rho_spectral_metrics.csv`, `potential_spectral_metrics.csv`
- `classification_metrics.csv`, `all_targets_classification_per_configuration.csv`
- `confusion_matrices_counts.csv`, `confusion_matrices_normalized.csv`
- `energy_tail_metrics.csv`, `bootstrap_confidence_intervals.csv`
- `retention_metrics_by_component.csv`, `checkpoint_selection_metrics.csv`
- `physical_validation_<stage>.csv`, `evaluation_stage_manifest.csv`
- `final_physical_quality_audit.csv`
- `publication_metrics.csv`, `baseline_improvement_metrics.csv`
- `publication_composite_improvement.csv`, `publication_results_sentences.txt`

Original CSVs, maps for Al/Fe/Ni, paired comparisons and diagnostics remain. Intermediate adapted/repaired candidates are assessed on validation only. Final ID/OOD results correspond to the explicitly selected checkpoint. Raw voxel arrays are not duplicated into gigantic CSV files.

- `data_curriculum_manifest_pretrain_AlFe.csv`, `data_curriculum_manifest_Ni_joint_with_AlFe_replay.csv`
- `data_curriculum_history.csv`, `sample_exposure_counts.csv`

V29 uses fixed full validation at every epoch. Test and OOD holdouts never enter manifests, scheduling, early stopping or checkpoint selection.


## V20 — automatic cohesive energy calibration

The notebook does not request atomic energies or reference IDs. The
thermochemical anchors are embedded NIST-JANAF values converted using
`1 eV/atom = 96.48533212331002 kJ/mol`. The bulk reference is chosen only
from training sets using a robust, deterministic, auditable criterion.
The `cohesive_calibration_manifest.json` file records all candidate families,
lower-tail IDs, selected record, energy per atom, and anchor used.

For each configuration `i`:

`E_coh(i) = H_atomization_NIST + epsilon_bulk_ref(training) - E_total(i)/N_i`.

The experimental offset establishes the absolute scale; differences between structures
remain fully determined by the dataset’s SIESTA energies.


